# FLEX-445 — AGARD 445.6 Benchmark

Configuration: 2.5-foot wall-mounted weakened model 3.

Source:
E. Carson Yates Jr., NASA TM-100492, August 1987.
Table 2: structural grid coordinates.
Table 4: calculated mode shapes for weakened model 3.
Appendix Table I: measured frequencies and panel mass.
Appendix Table II: flutter measurements in air.

Current task:
Reconstruct and inspect the published structural grid.

Coordinates:
x — streamwise, measured from the root leading edge
y — spanwise
z — perpendicular to the chord plane

Internal units: SI.

## Publication / reproducibility note

This GitHub copy preserves the analysis code and narrative while clearing code-cell outputs and execution counts for reliable rendering. Re-run the notebook to regenerate numerical tables and figures. Public AGARD 445.6 benchmark data are attributed to NASA TM-100492 in the source cells and in the repository `REFERENCES.md`. PanelAero is an external BSD-3-Clause dependency; see `THIRD_PARTY_NOTICES.md`.

This is Notebook 02 of the FLEX-445 project. Notebook 01 establishes the typical-section aeroelastic foundations; this notebook carries the work through the AGARD 445.6 benchmark, DLM/p-k analysis, passive and active architecture studies, DOE/ML screening, and physics re-verification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INCH_TO_M = 0.0254

# NASA TM-100492, Table 2; coordinates in inches
span_stations_in = np.arange(0.0, 30.0 + 3.0, 3.0)

leading_edge_x_in = np.array([
    0.0000, 3.1866, 6.3732, 9.5598, 12.7464,
    15.9330, 19.1196, 22.3062, 25.4928, 28.6794,
    31.8660
])

trailing_edge_x_in = np.array([
    21.9600, 24.4002, 26.8404, 29.2806, 31.7208,
    34.1610, 36.6012, 39.0414, 41.4816, 43.9218,
    46.3620
])

sixth_node_x_in = np.array([
    10.75, 13.65, 16.60, 19.50, 22.30,
    25.20, 28.10, 30.90, 33.70, 36.70, 39.50
])

# Uniform chordwise spacing, with the published sixth-node offsets
chord_fraction = np.linspace(0.0, 1.0, 11)
local_chord_in = trailing_edge_x_in - leading_edge_x_in

x_grid_in = (
    leading_edge_x_in[:, None]
    + local_chord_in[:, None] * chord_fraction[None, :]
)
x_grid_in[:, 5] = sixth_node_x_in

y_grid_in = np.repeat(span_stations_in[:, None], 11, axis=1)

x_grid = x_grid_in * INCH_TO_M
y_grid = y_grid_in * INCH_TO_M

nodes = pd.DataFrame({
    "node_id": np.arange(1, 122),
    "x_m": x_grid.ravel(),
    "y_m": y_grid.ravel(),
    "z_m": np.zeros(121)
})

print(nodes.head(11).to_string(index=False))

In [ ]:
panel_span = np.ptp(y_grid[:, 0])
root_chord = x_grid[0, -1] - x_grid[0, 0]
tip_chord = x_grid[-1, -1] - x_grid[-1, 0]

panel_area = 0.5 * (root_chord + tip_chord) * panel_span
panel_taper = tip_chord / root_chord

quarter_chord_x = (
    x_grid[:, 0] + 0.25 * (x_grid[:, -1] - x_grid[:, 0])
)
quarter_chord_sweep = np.rad2deg(np.arctan2(
    quarter_chord_x[-1] - quarter_chord_x[0],
    panel_span
))

assert len(nodes) == 121
assert nodes["node_id"].is_unique
assert np.all(np.diff(x_grid, axis=1) > 0.0)
assert np.all(np.diff(y_grid[:, 0]) > 0.0)

# Selected published coordinates: node ID -> [x, y] in inches
reference_nodes = {
    1: [0.0, 0.0],
    6: [10.75, 0.0],
    11: [21.96, 0.0],
    61: [25.20, 15.0],
    111: [31.866, 30.0],
    116: [39.50, 30.0],
    121: [46.362, 30.0]
}

for node_id, reference_xy in reference_nodes.items():
    actual_xy = nodes.loc[
        nodes["node_id"] == node_id, ["x_m", "y_m"]
    ].to_numpy()[0]

    assert np.allclose(
        actual_xy,
        np.array(reference_xy) * INCH_TO_M,
        rtol=0.0,
        atol=1e-10
    ), f"Coordinate mismatch at node {node_id}"

print(f"Nodes: {len(nodes)}")
print(f"Panel span: {panel_span:.6f} m")
print(f"Root chord: {root_chord:.6f} m")
print(f"Tip chord: {tip_chord:.6f} m")
print(f"Panel area: {panel_area:.6f} m^2")
print(f"Panel taper ratio: {panel_taper:.6f}")
print(f"Quarter-chord sweep: {quarter_chord_sweep:.3f} deg")
print("Grid checks passed.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for station in range(11):
    ax.plot(
        y_grid[station, :], x_grid[station, :],
        color="0.65", linewidth=0.8
    )

for chord_index in range(11):
    ax.plot(
        y_grid[:, chord_index], x_grid[:, chord_index],
        color="0.65", linewidth=0.8
    )

ax.scatter(
    nodes["y_m"], nodes["x_m"],
    s=12, color="tab:blue", zorder=3
)
ax.plot(
    y_grid[:, 0], quarter_chord_x,
    "--", color="tab:orange", label="Quarter-chord line"
)

for node_id in reference_nodes:
    node = nodes.iloc[node_id - 1]
    ax.annotate(
        str(node_id),
        (node["y_m"], node["x_m"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

ax.set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title="AGARD 445.6 — Table 2 structural grid"
)
ax.set_aspect("equal", adjustable="box")
ax.invert_yaxis()
ax.grid(True, alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
element_rows = []

for span_index in range(10):
    for chord_index in range(10):
        n1 = span_index * 11 + chord_index + 1
        n2 = n1 + 1
        n3 = n2 + 11
        n4 = n1 + 11

        element_rows.append([
            len(element_rows) + 1, n1, n2, n3, n4
        ])

elements = pd.DataFrame(
    element_rows,
    columns=["element_id", "n1", "n2", "n3", "n4"]
)

# Check element areas in the physical x-y coordinate system
xy = nodes[["x_m", "y_m"]].to_numpy()
element_areas = []

for row in elements.itertuples(index=False):
    node_indices = np.array([row.n1, row.n2, row.n3, row.n4]) - 1
    corners = xy[node_indices]

    x = corners[:, 0]
    y = corners[:, 1]

    signed_area = 0.5 * (
        np.dot(x, np.roll(y, -1))
        - np.dot(y, np.roll(x, -1))
    )
    element_areas.append(signed_area)

elements["area_m2"] = element_areas

assert len(elements) == 100
assert np.all(elements["area_m2"] > 0.0)
assert np.isclose(
    elements["area_m2"].sum(),
    panel_area,
    rtol=1e-12
)

print(elements.head().to_string(index=False))
print(f"\nElements: {len(elements)}")
print(f"Summed element area: {elements['area_m2'].sum():.6f} m^2")
print("Mesh connectivity checks passed.")

In [ ]:
# NASA TM-100492:
# Table 4 — calculated modes for weakened model 3
# Appendix Table I — measured frequencies for the same model
modal_reference = pd.DataFrame({
    "mode_id": [1, 2, 3, 4],
    "description": [
        "First bending",
        "First torsion",
        "Second bending",
        "Second torsion"
    ],
    "calculated_frequency_hz": [
        9.5992, 38.1650, 48.3482, 91.5448
    ],
    "measured_frequency_hz": [
        9.60, 38.10, 50.70, 98.50
    ]
})

modal_reference["frequency_difference_pct"] = 100 * (
    modal_reference["calculated_frequency_hz"]
    / modal_reference["measured_frequency_hz"]
    - 1.0
)

# Follow the report's recommendation for the first four modes
modal_reference["analysis_frequency_hz"] = (
    modal_reference["measured_frequency_hz"]
)

omega_modes = (
    2 * np.pi
    * modal_reference["analysis_frequency_hz"].to_numpy()
)

print(modal_reference.to_string(index=False, float_format="%.4f"))

In [ ]:
# NASA TM-100492, Table 4(a), printed pages 19–20.
# Transverse modal coefficients in the original tabulated normalization.
# Rows follow spanwise stations; columns follow chordwise node order.

mode1_z_table = np.array([
    [-0.0405, -0.0153, 0, 0, 0, 0, 0, 0, 0, -0.0524, -0.107],
    [0.00638, 0.0352, 0.0690, 0.113, 0.166, 0.225,
     0.306, 0.402, 0.538, 0.697, 0.914],
    [0.195, 0.317, 0.462, 0.628, 0.816, 1.03,
     1.27, 1.56, 1.88, 2.25, 2.68],
    [0.815, 1.08, 1.38, 1.70, 2.05, 2.45,
     2.86, 3.32, 3.84, 4.41, 5.03],
    [2.01, 2.42, 2.87, 3.35, 3.86, 4.43,
     5.00, 5.63, 6.30, 7.03, 7.80],
    [3.80, 4.36, 4.95, 5.57, 6.22, 6.97,
     7.63, 8.39, 9.19, 10.0, 10.9],
    [6.16, 6.85, 7.56, 8.29, 9.06, 9.96,
     10.7, 11.5, 12.4, 13.3, 14.3],
    [9.05, 9.82, 10.6, 11.4, 12.3, 13.3,
     14.0, 14.9, 15.9, 16.9, 17.9],
    [12.4, 13.2, 14.0, 14.9, 15.8, 16.8,
     17.6, 18.5, 19.5, 20.5, 21.5],
    [16.0, 16.8, 17.7, 18.6, 19.5, 20.6,
     21.3, 22.2, 23.2, 24.2, 25.1],
    [19.8, 20.6, 21.5, 22.4, 23.2, 24.4,
     25.0, 26.0, 26.9, 27.8, 28.8],
], dtype=float)

assert mode1_z_table.shape == (11, 11)
assert np.isfinite(mode1_z_table).all()
assert np.array_equal(nodes["node_id"].to_numpy(), np.arange(1, 122))

mode1 = nodes[["node_id", "x_m", "y_m"]].copy()
mode1["z_coefficient_table"] = mode1_z_table.ravel()

# Scale a separate copy for visualization only.
plot_scale = np.max(np.abs(mode1_z_table))
mode1_z_plot = mode1_z_table / plot_scale

reference_values = {
    1: -0.0405,
    6: 0.0,
    61: 6.97,
    111: 19.8,
    116: 24.4,
    121: 28.8,
}

for node_id, expected in reference_values.items():
    actual = mode1.loc[
        mode1["node_id"] == node_id, "z_coefficient_table"
    ].item()
    assert np.isclose(actual, expected, rtol=0, atol=1e-12)

print(f"Imported transverse coefficients: {mode1_z_table.size}")
print(f"Visualization scale factor: {plot_scale:.4f}")
print("Mode 1 data checks passed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

contour = axes[0].contourf(
    y_grid,
    x_grid,
    mode1_z_plot,
    levels=np.linspace(-1, 1, 21),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)

axes[0].scatter(
    y_grid.ravel(),
    x_grid.ravel(),
    s=7,
    color="black",
    alpha=0.35,
)

axes[0].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title="First bending mode — normalized transverse shape",
)
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
fig.colorbar(contour, ax=axes[0], label="Normalized modal coefficient")

for column, label in [
    (0, "Leading edge"),
    (5, "Sixth node in each station"),
    (10, "Trailing edge"),
]:
    axes[1].plot(
        y_grid[:, column],
        mode1_z_plot[:, column],
        "o-",
        markersize=4,
        label=label,
    )

axes[1].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Normalized modal coefficient",
    title="Spanwise variation",
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# NASA TM-100492, Table 4(b), printed pages 21–22.
# Transverse modal coefficients in the original tabulated normalization.

mode2_z_table = np.array([
    [-0.315, -0.128, 0, 0, 0, 0, 0, 0, 0, -0.686, -2.28],
    [0.137, 0.335, 0.514, 0.668, 0.767, 0.778,
     0.636, 0.238, -0.719, -2.35, -4.79],
    [1.62, 2.16, 2.59, 2.83, 2.83, 2.50,
     1.74, 0.444, -1.50, -4.11, -7.53],
    [5.22, 5.84, 6.13, 6.03, 5.48, 4.35,
     2.76, 0.476, -2.47, -6.10, -10.6],
    [10.5, 10.7, 10.4, 9.51, 8.05, 5.90,
     3.28, -0.0744, -4.09, -8.80, -14.4],
    [16.5, 15.8, 14.4, 12.5, 9.91, 6.41,
     2.86, -1.61, -6.72, -12.5, -19.2],
    [22.0, 20.0, 17.4, 14.3, 10.5, 5.47,
     1.16, -4.39, -10.5, -17.3, -24.9],
    [25.9, 22.6, 18.7, 14.3, 9.40, 3.17,
     -2.01, -8.48, -15.5, -23.0, -31.3],
    [27.4, 22.9, 17.9, 12.4, 6.52, -0.653,
     -6.50, -13.6, -21.2, -29.2, -37.9],
    [26.3, 20.7, 14.8, 8.64, 2.11, -6.59,
     -11.9, -19.4, -27.3, -35.6, -44.5],
    [22.6, 16.5, 10.2, 3.58, -3.28, -12.4,
     -17.8, -25.6, -33.7, -42.3, -52.6],
], dtype=float)

assert mode2_z_table.shape == (11, 11)
assert np.isfinite(mode2_z_table).all()

reference_values = {
    1: -0.315,
    11: -2.28,
    52: -0.0744,
    61: 6.41,
    94: -0.653,
    111: 22.6,
    121: -52.6,
}

for node_id, expected in reference_values.items():
    actual = mode2_z_table.ravel()[node_id - 1]
    assert np.isclose(actual, expected, rtol=0, atol=1e-12)

# Preserve the source coefficients separately from plotting values.
mode2_plot_scale = np.max(np.abs(mode2_z_table))
mode2_z_plot = mode2_z_table / mode2_plot_scale

print(f"Imported transverse coefficients: {mode2_z_table.size}")
print(f"Visualization scale factor: {mode2_plot_scale:.4f}")
print("Mode 2 data checks passed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

contour = axes[0].contourf(
    y_grid,
    x_grid,
    mode2_z_plot,
    levels=np.linspace(-1, 1, 21),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)

axes[0].contour(
    y_grid,
    x_grid,
    mode2_z_plot,
    levels=[0],
    colors="black",
    linewidths=1.2,
)

axes[0].scatter(
    y_grid.ravel(),
    x_grid.ravel(),
    s=7,
    color="black",
    alpha=0.35,
)

axes[0].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title="First torsion mode — normalized transverse shape",
)
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
fig.colorbar(contour, ax=axes[0], label="Normalized modal coefficient")

for column, label in [
    (0, "Leading edge"),
    (5, "Sixth node in each station"),
    (10, "Trailing edge"),
]:
    axes[1].plot(
        y_grid[:, column],
        mode2_z_plot[:, column],
        "o-",
        markersize=4,
        label=label,
    )

axes[1].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Normalized modal coefficient",
    title="Spanwise variation",
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# NASA TM-100492, Table 4(c), printed pages 23–24.
# Mode 3: calculated second-bending mode, f3 = 48.3482 Hz.
#
# These are the published transverse z modal coefficients in their
# ORIGINAL tabulated normalization. Do not rescale this source array.
# A separate normalized copy is created only for later visualization.
#
# Rows: spanwise stations from root to tip.
# Columns: chordwise node order from leading edge to trailing edge.

mode3_z_table = np.array([
    [-0.0829, -0.0280, 0, 0, 0, 0, 0, 0, 0, 0.566, 2.30],

    [-0.00421, 0.0344, 0.0916, 0.196, 0.371, 0.631,
     1.12, 1.95, 3.60, 6.19, 10.3],

    [0.162, 0.366, 0.694, 1.20, 1.95, 3.06,
     4.68, 6.99, 10.2, 14.3, 20.0],

    [0.714, 1.25, 2.02, 3.13, 4.64, 6.76,
     9.32, 12.7, 16.8, 21.9, 28.4],

    [1.45, 2.36, 3.62, 5.29, 7.44, 10.2,
     13.4, 17.2, 21.7, 26.9, 33.2],

    [1.70, 2.93, 4.55, 6.59, 9.06, 12.2,
     15.3, 19.1, 23.3, 27.9, 33.4],

    [0.549, 1.96, 3.72, 5.83, 8.27, 11.4,
     14.1, 17.4, 20.8, 24.5, 28.7],

    [-2.87, -1.46, 0.219, 2.15, 4.31, 6.98,
     9.13, 11.7, 14.3, 16.8, 19.6],

    [-9.08, -7.77, -6.27, -4.61, -2.83, -0.748,
     0.857, 2.67, 4.39, 5.96, 7.42],

    [-17.9, -16.6, -15.3, -13.9, -12.4, -10.7,
     -9.73, -8.52, -7.48, -6.67, -6.20],

    [-28.2, -26.9, -25.7, -24.5, -23.4, -22.1,
     -21.4, -20.7, -20.3, -20.2, -21.3],
], dtype=float)


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert mode3_z_table.shape == (11, 11)
assert mode3_z_table.size == 121
assert np.isfinite(mode3_z_table).all()

# Confirm that the existing structural-grid node ordering remains
# consistent with the 11 x 11 modal table.
assert np.array_equal(
    nodes["node_id"].to_numpy(),
    np.arange(1, 122)
)

# Check that the notebook's stored calculated frequency agrees
# with the Table 4(c) heading.
mode3_calculated_frequency = modal_reference.loc[
    modal_reference["mode_id"] == 3,
    "calculated_frequency_hz"
].item()

assert np.isclose(
    mode3_calculated_frequency,
    48.3482,
    rtol=0.0,
    atol=1e-10
)


# ------------------------------------------------------------
# Spot checks against the scanned Table 4(c)
# ------------------------------------------------------------

reference_values = {
    1:   -0.0829,
    11:   2.30,
    22:  10.3,
    44:  28.4,
    61:  12.2,
    66:  33.4,
    78:  -2.87,
    89:  -9.08,
    100: -17.9,
    111: -28.2,
    121: -21.3,
}

for node_id, expected in reference_values.items():
    actual = mode3_z_table.ravel()[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 3 coefficient mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# Preserve source coefficients; normalize a separate plot copy
# ------------------------------------------------------------

mode3_plot_scale = np.max(np.abs(mode3_z_table))
mode3_z_plot = mode3_z_table / mode3_plot_scale

print(f"Imported transverse coefficients: {mode3_z_table.size}")
print(f"Calculated Mode 3 frequency: {mode3_calculated_frequency:.4f} Hz")
print(f"Analysis Mode 3 frequency: "
      f"{modal_reference.loc[modal_reference['mode_id'] == 3, 'analysis_frequency_hz'].item():.4f} Hz")
print(f"Visualization scale factor: {mode3_plot_scale:.4f}")
print("Mode 3 data checks passed.")

In [ ]:
# Cell 11 — Mode 3 transverse-shape visualization
# Second bending mode, NASA TM-100492 Table 4(c).
#
# Only mode3_z_plot is used here.
# mode3_z_table remains untouched in its original published normalization.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ------------------------------------------------------------
# 1. Wing-planform contour
# ------------------------------------------------------------

contour = axes[0].contourf(
    y_grid,
    x_grid,
    mode3_z_plot,
    levels=np.linspace(-1, 1, 21),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)

# Zero-displacement contour(s)
axes[0].contour(
    y_grid,
    x_grid,
    mode3_z_plot,
    levels=[0],
    colors="black",
    linewidths=1.2,
)

# Published structural-grid locations
axes[0].scatter(
    y_grid.ravel(),
    x_grid.ravel(),
    s=7,
    color="black",
    alpha=0.35,
)

axes[0].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title="Second bending mode — normalized transverse shape",
)

axes[0].invert_yaxis()
axes[0].set_aspect("equal")

fig.colorbar(
    contour,
    ax=axes[0],
    label="Normalized modal coefficient",
)


# ------------------------------------------------------------
# 2. Spanwise traces at representative chordwise locations
# ------------------------------------------------------------

for column, label in [
    (0, "Leading edge"),
    (5, "Sixth node in each station"),
    (10, "Trailing edge"),
]:
    axes[1].plot(
        y_grid[:, column],
        mode3_z_plot[:, column],
        "o-",
        markersize=4,
        label=label,
    )

axes[1].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Normalized modal coefficient",
    title="Spanwise variation",
)

axes[1].axhline(
    0,
    color="black",
    linewidth=0.8,
)

axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Import Mode 4 transverse modal coefficients
#
# NASA TM-100492, Table 4(d), printed pages 25–26.
# 2.5-ft weakened model 3.
#
# Mode 4: second torsion
# Calculated frequency = 91.5448 Hz
#
# IMPORTANT:
# mode4_z_table contains the ORIGINAL published z coefficients.
# Do not overwrite or normalize this source array.
# Any scaling is performed separately for plotting only.

mode4_z_table = np.array([
    # Nodes 1–11: root station
    [-1.08, -0.416, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, -1.42, -5.22],

    # Nodes 12–22
    [0.482, 1.01, 1.43, 1.73, 1.85, 1.77,
     1.34, 0.436, -1.56, -4.92, -10.7],

    # Nodes 23–33
    [4.61, 5.67, 6.33, 6.46, 6.01, 4.90,
     3.04, 0.289, -3.49, -8.37, -15.5],

    # Nodes 34–44
    [12.8, 13.2, 12.9, 11.7, 9.63, 6.71,
     3.29, -0.953, -5.84, -11.4, -18.8],

    # Nodes 45–55
    [21.7, 20.1, 17.6, 14.4, 10.5, 5.98,
     1.43, -3.46, -8.44, -13.4, -19.6],

    # Nodes 56–66
    [26.5, 22.3, 17.6, 12.6, 7.55, 2.16,
     -2.14, -6.40, -10.1, -13.0, -16.0],

    # Nodes 67–77
    [23.7, 17.6, 11.8, 6.36, 1.49, -3.13,
     -5.83, -7.94, -8.79, -8.18, -6.13],

    # Nodes 78–88
    [13.0, 6.90, 1.72, -2.42, -5.38, -7.16,
     -7.28, -5.93, -2.81, 2.36, 10.5],

    # Nodes 89–99
    [-2.49, -6.48, -9.10, -10.3, -9.97, -7.71,
     -4.51, 0.890, 8.34, 18.2, 32.5],

    # Nodes 100–110
    [-17.3, -17.6, -16.5, -14.1, -10.1, -2.75,
      2.83, 12.1, 23.7, 38.2, 58.3],

    # Nodes 111–121: tip station
    [-26.2, -22.9, -18.6, -13.0, -5.87, 5.57,
      13.6, 26.7, 42.8, 63.6, 104.0],
], dtype=float)


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert mode4_z_table.shape == (11, 11)
assert mode4_z_table.size == 121
assert np.isfinite(mode4_z_table).all()

assert np.array_equal(
    nodes["node_id"].to_numpy(),
    np.arange(1, 122)
)


# ------------------------------------------------------------
# Frequency check
# ------------------------------------------------------------

mode4_calculated_frequency = modal_reference.loc[
    modal_reference["mode_id"] == 4,
    "calculated_frequency_hz"
].item()

mode4_analysis_frequency = modal_reference.loc[
    modal_reference["mode_id"] == 4,
    "analysis_frequency_hz"
].item()

assert np.isclose(
    mode4_calculated_frequency,
    91.5448,
    rtol=0.0,
    atol=1e-10
)


# ------------------------------------------------------------
# Spot checks against scanned Table 4(d)
#
# These deliberately sample several span stations and both
# positive and negative portions of the torsional mode.
# ------------------------------------------------------------

reference_values = {
    1:   -1.08,
    11:  -5.22,
    22: -10.7,
    33: -15.5,
    44: -18.8,
    55: -19.6,
    66: -16.0,
    77:  -6.13,
    88:  10.5,
    99:  32.5,
    110: 58.3,
    121: 104.0,
}

mode4_flat = mode4_z_table.ravel()

for node_id, expected in reference_values.items():
    actual = mode4_flat[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 4 coefficient mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# Separate visualization normalization
# ------------------------------------------------------------

mode4_plot_scale = np.max(np.abs(mode4_z_table))

mode4_z_plot = (
    mode4_z_table.copy()
    / mode4_plot_scale
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"Imported transverse coefficients: {mode4_z_table.size}")
print(
    f"Calculated Mode 4 frequency: "
    f"{mode4_calculated_frequency:.4f} Hz"
)
print(
    f"Analysis Mode 4 frequency: "
    f"{mode4_analysis_frequency:.4f} Hz"
)
print(
    f"Visualization scale factor: "
    f"{mode4_plot_scale:.4f}"
)
print("Mode 4 data checks passed.")

In [ ]:
# Cell 13 — Mode 4 transverse-shape visualization
# Second torsion mode, NASA TM-100492 Table 4(d).
#
# mode4_z_table remains in its ORIGINAL published normalization.
# Only the separate mode4_z_plot array is used for visualization.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))


# ------------------------------------------------------------
# 1. Wing-planform contour
# ------------------------------------------------------------

contour = axes[0].contourf(
    y_grid,
    x_grid,
    mode4_z_plot,
    levels=np.linspace(-1, 1, 21),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)

# Modal zero-displacement contours
axes[0].contour(
    y_grid,
    x_grid,
    mode4_z_plot,
    levels=[0],
    colors="black",
    linewidths=1.2,
)

# Structural-grid nodes
axes[0].scatter(
    y_grid.ravel(),
    x_grid.ravel(),
    s=7,
    color="black",
    alpha=0.35,
)

axes[0].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title="Second torsion mode — normalized transverse shape",
)

axes[0].invert_yaxis()
axes[0].set_aspect("equal")

fig.colorbar(
    contour,
    ax=axes[0],
    label="Normalized modal coefficient",
)


# ------------------------------------------------------------
# 2. Spanwise traces at representative chordwise locations
# ------------------------------------------------------------

for column, label in [
    (0, "Leading edge"),
    (5, "Sixth node in each station"),
    (10, "Trailing edge"),
]:
    axes[1].plot(
        y_grid[:, column],
        mode4_z_plot[:, column],
        "o-",
        markersize=4,
        label=label,
    )

axes[1].axhline(
    0.0,
    color="black",
    linewidth=0.8,
)

axes[1].set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Normalized modal coefficient",
    title="Spanwise variation",
)

axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Four-mode SI structural modal model
#
# Builds the generalized modal mass and stiffness matrices using:
#   - the ORIGINAL published mode-shape normalization
#   - measured frequencies for Modes 1–4
#
# No visualization-normalized modal arrays are used here.

# ------------------------------------------------------------
# 1. Assemble original published transverse mode shapes
# ------------------------------------------------------------

Phi_z_4 = np.column_stack([
    mode1_z_table.ravel(),
    mode2_z_table.ravel(),
    mode3_z_table.ravel(),
    mode4_z_table.ravel(),
])

assert Phi_z_4.shape == (121, 4)
assert np.isfinite(Phi_z_4).all()


# ------------------------------------------------------------
# 2. Generalized-mass conversion
#
# Published normalization:
#     M_gen = I [lbf*s^2/in]
#
# 1 lbf*s^2/in is dimensionally a mass.
# Convert that unit directly to kg.
# ------------------------------------------------------------

LBF_TO_N = 4.4482216152605
IN_TO_M = 0.0254

generalized_mass_unit_kg = LBF_TO_N / IN_TO_M

print(
    "1 lbf*s^2/in = "
    f"{generalized_mass_unit_kg:.6f} kg"
)


# Each published mode has generalized mass = 1 in the
# report's normalization.
M_modal_SI = (
    generalized_mass_unit_kg
    * np.eye(4)
)


# ------------------------------------------------------------
# 3. Use measured frequencies for the structural analysis
# ------------------------------------------------------------

analysis_frequencies_hz = (
    modal_reference
    .sort_values("mode_id")
    ["analysis_frequency_hz"]
    .to_numpy(dtype=float)
)

assert analysis_frequencies_hz.shape == (4,)

omega_analysis = (
    2.0 * np.pi * analysis_frequencies_hz
)


# ------------------------------------------------------------
# 4. Generalized stiffness matrix
#
# For an uncoupled mass-normalized modal basis:
#
#     K_i = M_i * omega_i^2
#
# With modal coordinates expressed in metres,
# K_modal is in N/m.
# ------------------------------------------------------------

K_modal_SI = np.diag(
    generalized_mass_unit_kg
    * omega_analysis**2
)


# ------------------------------------------------------------
# 5. Numerical verification
#
# Solve:
#
#     K q = lambda M q
#
# where lambda = omega^2
# ------------------------------------------------------------

eigvals = np.linalg.eigvals(
    np.linalg.solve(M_modal_SI, K_modal_SI)
)

eigvals = np.sort(np.real(eigvals))

recovered_omega = np.sqrt(eigvals)

recovered_frequency_hz = (
    recovered_omega
    / (2.0 * np.pi)
)


# ------------------------------------------------------------
# 6. Verification table
# ------------------------------------------------------------

structural_modal_check = pd.DataFrame({
    "mode_id": np.arange(1, 5),
    "analysis_frequency_hz": analysis_frequencies_hz,
    "recovered_frequency_hz": recovered_frequency_hz,
    "frequency_error_hz": (
        recovered_frequency_hz
        - analysis_frequencies_hz
    ),
    "generalized_mass_kg": np.diag(M_modal_SI),
    "generalized_stiffness_N_per_m": np.diag(K_modal_SI),
})


# ------------------------------------------------------------
# 7. Assertions
# ------------------------------------------------------------

assert np.allclose(
    recovered_frequency_hz,
    analysis_frequencies_hz,
    rtol=1e-12,
    atol=1e-12,
)

assert np.allclose(
    M_modal_SI,
    M_modal_SI.T,
)

assert np.allclose(
    K_modal_SI,
    K_modal_SI.T,
)

assert np.all(
    np.linalg.eigvalsh(M_modal_SI) > 0
)

assert np.all(
    np.linalg.eigvalsh(K_modal_SI) > 0
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("\nFour-mode transverse basis shape:")
print(Phi_z_4.shape)

print("\nGeneralized mass matrix M [kg]:")
print(M_modal_SI)

print("\nGeneralized stiffness matrix K [N/m]:")
print(K_modal_SI)

print("\nStructural modal verification:")
display(structural_modal_check)

print("\nFour-mode SI structural matrices verified.")

In [ ]:
# Cell 15 — Equivalent-viscous modal damping model
#
# NASA TM-100492 recommends a structural damping coefficient
# of approximately g = 0.02 for all modes.
#
# For the time-domain model used later for active control,
# use the standard aeroelastic relation:
#
#       g = 2*zeta
#
# giving:
#
#       zeta = 0.01
#
# This is an equivalent-viscous representation of the reported
# structural damping coefficient, not a statement about the
# physical damping mechanism of the test article.

# ------------------------------------------------------------
# 1. Structural damping specification
# ------------------------------------------------------------

g_structural = 0.02

zeta_modal = np.full(
    4,
    g_structural / 2.0,
    dtype=float
)

assert np.allclose(zeta_modal, 0.01)


# ------------------------------------------------------------
# 2. Equivalent modal viscous damping matrix
#
# For each uncoupled modal equation:
#
#   M_i qddot_i + C_i qdot_i + K_i q_i = Q_i
#
# with:
#
#   C_i = 2*zeta_i*M_i*omega_i
# ------------------------------------------------------------

C_modal_SI = np.diag(
    2.0
    * zeta_modal
    * np.diag(M_modal_SI)
    * omega_analysis
)


# ------------------------------------------------------------
# 3. Recover damping ratios from M, C and K
#
# zeta_i = C_i / (2*M_i*omega_i)
# ------------------------------------------------------------

recovered_zeta = (
    np.diag(C_modal_SI)
    /
    (
        2.0
        * np.diag(M_modal_SI)
        * omega_analysis
    )
)

recovered_g = 2.0 * recovered_zeta


# ------------------------------------------------------------
# 4. Build the purely structural first-order state-space matrix
#
# State vector:
#
#       x_s = [q1 q2 q3 q4 qdot1 qdot2 qdot3 qdot4]^T
#
# With no aerodynamic or control forces:
#
#       xdot_s = A_struct x_s
# ------------------------------------------------------------

n_modes = 4

Z4 = np.zeros((n_modes, n_modes))
I4 = np.eye(n_modes)

M_inv = np.linalg.inv(M_modal_SI)

A_struct = np.block([
    [Z4,                          I4],
    [-M_inv @ K_modal_SI, -M_inv @ C_modal_SI]
])

assert A_struct.shape == (8, 8)


# ------------------------------------------------------------
# 5. Structural eigenvalue verification
# ------------------------------------------------------------

structural_eigenvalues = np.linalg.eigvals(A_struct)

# Keep one eigenvalue from each complex-conjugate pair:
positive_imag = structural_eigenvalues[
    np.imag(structural_eigenvalues) > 0
]

positive_imag = positive_imag[
    np.argsort(np.imag(positive_imag))
]

assert positive_imag.size == 4


recovered_damped_frequency_hz = (
    np.abs(np.imag(positive_imag))
    / (2.0 * np.pi)
)

recovered_state_zeta = (
    -np.real(positive_imag)
    / np.abs(positive_imag)
)


# ------------------------------------------------------------
# 6. Verification table
# ------------------------------------------------------------

damping_check = pd.DataFrame({
    "mode_id": np.arange(1, 5),
    "undamped_frequency_hz": analysis_frequencies_hz,
    "specified_g": np.full(4, g_structural),
    "specified_zeta": zeta_modal,
    "C_modal_Ns_per_m": np.diag(C_modal_SI),
    "recovered_g": recovered_g,
    "state_space_zeta": recovered_state_zeta,
    "damped_frequency_hz": recovered_damped_frequency_hz,
})


# ------------------------------------------------------------
# 7. Assertions
# ------------------------------------------------------------

assert np.allclose(
    recovered_zeta,
    zeta_modal,
    rtol=1e-12,
    atol=1e-12
)

assert np.allclose(
    recovered_g,
    g_structural,
    rtol=1e-12,
    atol=1e-12
)

assert np.all(
    np.real(structural_eigenvalues) < 0.0
)

assert np.allclose(
    recovered_state_zeta,
    zeta_modal,
    rtol=1e-10,
    atol=1e-10
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print(f"Specified structural damping coefficient g: {g_structural:.4f}")
print(f"Equivalent viscous damping ratio zeta: {zeta_modal[0]:.4f}")

print("\nEquivalent modal damping matrix C [N*s/m]:")
print(C_modal_SI)

print("\nStructural damping verification:")
display(damping_check)

print("\nAll uncoupled structural poles are stable.")
print("Equivalent-viscous four-mode structural model verified.")

In [ ]:
# Cell 16 — Reproducible DLM environment setup
#
# Public open-source aerodynamic solver:
# DLR Institute of Aeroelasticity — PanelAero
#
# We pin the package version so that the notebook remains reproducible.

!pip -q install "PanelAero==2025.8"

from importlib.metadata import version
import inspect

import panelaero
from panelaero import DLM, VLM


# ------------------------------------------------------------
# 1. Confirm installed package
# ------------------------------------------------------------

panelaero_version = version("PanelAero")

print(f"PanelAero version: {panelaero_version}")
print(f"PanelAero module: {panelaero.__file__}")


# ------------------------------------------------------------
# 2. Inspect public DLM interface
#
# We deliberately inspect the installed package rather than
# guessing the API from an older example or tutorial.
# ------------------------------------------------------------

dlm_public_members = [
    name
    for name, obj in inspect.getmembers(DLM)
    if not name.startswith("_")
]

print("\nPublic members available in panelaero.DLM:")

for name in dlm_public_members:
    print(f"  {name}")


# ------------------------------------------------------------
# 3. Basic availability checks
# ------------------------------------------------------------

assert DLM is not None
assert VLM is not None

print("\nDLM/VLM imports successful.")
print("Aerodynamic environment setup passed.")

In [ ]:
# Cell 17 — AGARD 445.6 aerodynamic lattice for PanelAero DLM
#
# Separate aerodynamic discretization from the structural 10x10 plate mesh.
#
# Baseline aerodynamic lattice:
#     10 spanwise panels
#      8 chordwise panels
#     80 total panels
#
# Geometry:
#     2.5-ft weakened model 3 semispan wing
#     span              = 0.762 m
#     root chord        = 0.557784 m
#     tip chord         = 0.3681984 m
#     quarter-chord sweep = 45 deg
#
# Coordinate convention retained from the AGARD structural model:
#     x = streamwise
#     y = spanwise
#     z = transverse


def build_agard_aerogrid(n_span=10, n_chord=8):
    """
    Build a flat trapezoidal aerodynamic lattice compatible with
    PanelAero VLM/DLM.

    Panel numbering proceeds:
        span strip by span strip,
        leading edge -> trailing edge within each strip.

    Each aerodynamic panel is defined:
          4 -------- 3
          |          |
       -> |          |
          |          |
          1 -------- 2

    giving a positive +z panel normal.
    """

    # --------------------------------------------------------
    # 1. Reference geometry
    # --------------------------------------------------------

    span = 0.762
    c_root = 0.557784
    c_tip = 0.3681984

    sweep_c4 = np.deg2rad(45.0)

    # Span stations
    y_stations = np.linspace(
        0.0,
        span,
        n_span + 1
    )

    # Chord fractions:
    # xi = 0 -> LE
    # xi = 1 -> TE
    xi_stations = np.linspace(
        0.0,
        1.0,
        n_chord + 1
    )


    # --------------------------------------------------------
    # 2. Planform geometry
    #
    # Chord varies linearly with span.
    #
    # Quarter-chord line:
    #
    # x_c/4(y) = 0.25*c_root + y*tan(Lambda_c/4)
    #
    # Therefore:
    #
    # x_LE(y) = x_c/4(y) - 0.25*c(y)
    # --------------------------------------------------------

    chord_at_y = (
        c_root
        + (c_tip - c_root)
        * (y_stations / span)
    )

    x_quarter_at_y = (
        0.25 * c_root
        + y_stations * np.tan(sweep_c4)
    )

    x_le_at_y = (
        x_quarter_at_y
        - 0.25 * chord_at_y
    )


    # --------------------------------------------------------
    # 3. Aerodynamic corner-point grid
    #
    # Shape:
    #   [chord station, span station, xyz]
    # --------------------------------------------------------

    aero_xyz = np.zeros(
        (n_chord + 1, n_span + 1, 3),
        dtype=float
    )

    for i_span in range(n_span + 1):

        y = y_stations[i_span]
        chord = chord_at_y[i_span]
        x_le = x_le_at_y[i_span]

        for i_chord in range(n_chord + 1):

            xi = xi_stations[i_chord]

            aero_xyz[i_chord, i_span] = [
                x_le + xi * chord,
                y,
                0.0,
            ]


    # --------------------------------------------------------
    # 4. Assign unique corner-point IDs
    # --------------------------------------------------------

    grid_ids = np.arange(
        1,
        aero_xyz.shape[0] * aero_xyz.shape[1] + 1,
        dtype=int
    ).reshape(
        n_chord + 1,
        n_span + 1
    )


    # --------------------------------------------------------
    # 5. Construct PanelAero aerodynamic-panel quantities
    # --------------------------------------------------------

    panel_ID = []

    panel_corner_ids = []

    panel_length = []
    panel_area = []
    panel_normal = []

    offset_l = []   # 25% chord / mid-span
    offset_k = []   # 50% chord / mid-span
    offset_j = []   # 75% chord / mid-span control point

    offset_P1 = []  # 25% chord, inboard edge
    offset_P3 = []  # 25% chord, outboard edge

    span_vector_r = []

    panel_counter = 1

    for i_span in range(n_span):

        for i_chord in range(n_chord):

            # Panel corners
            P1 = aero_xyz[i_chord,     i_span]
            P2 = aero_xyz[i_chord + 1, i_span]
            P3 = aero_xyz[i_chord + 1, i_span + 1]
            P4 = aero_xyz[i_chord,     i_span + 1]

            # Corresponding IDs
            id1 = grid_ids[i_chord,     i_span]
            id2 = grid_ids[i_chord + 1, i_span]
            id3 = grid_ids[i_chord + 1, i_span + 1]
            id4 = grid_ids[i_chord,     i_span + 1]

            # Chordwise edge vectors
            l1 = P2 - P1
            l2 = P3 - P4

            # Spanwise edge vectors
            b1 = P4 - P1
            b2 = P3 - P2

            # Mean chordwise/spanwise panel vectors
            l_mean = 0.5 * (l1 + l2)
            b_mean = 0.5 * (b1 + b2)

            # Panel area
            area = np.linalg.norm(
                np.cross(l_mean, b_mean)
            )

            # Positive-normal convention
            normal = np.cross(l1, b1)
            normal /= np.linalg.norm(normal)

            # PanelAero reference points
            p_l = (
                P1
                + 0.25 * l_mean
                + 0.50 * b1
            )

            p_k = (
                P1
                + 0.50 * l_mean
                + 0.50 * b1
            )

            p_j = (
                P1
                + 0.75 * l_mean
                + 0.50 * b1
            )

            p_P1 = P1 + 0.25 * l1
            p_P3 = P4 + 0.25 * l2

            r = p_P3 - p_P1

            panel_ID.append(panel_counter)

            panel_corner_ids.append([
                id1,
                id2,
                id3,
                id4,
            ])

            # PanelAero uses the streamwise chord length here
            panel_length.append(l_mean[0])

            panel_area.append(area)
            panel_normal.append(normal)

            offset_l.append(p_l)
            offset_k.append(p_k)
            offset_j.append(p_j)

            offset_P1.append(p_P1)
            offset_P3.append(p_P3)

            span_vector_r.append(r)

            panel_counter += 1


    # --------------------------------------------------------
    # 6. Convert to arrays
    # --------------------------------------------------------

    panel_ID = np.asarray(panel_ID, dtype=int)

    panel_corner_ids = np.asarray(
        panel_corner_ids,
        dtype=int
    )

    panel_length = np.asarray(panel_length)
    panel_area = np.asarray(panel_area)
    panel_normal = np.asarray(panel_normal)

    offset_l = np.asarray(offset_l)
    offset_k = np.asarray(offset_k)
    offset_j = np.asarray(offset_j)

    offset_P1 = np.asarray(offset_P1)
    offset_P3 = np.asarray(offset_P3)

    span_vector_r = np.asarray(span_vector_r)

    n_panels = len(panel_ID)


    # --------------------------------------------------------
    # 7. PanelAero aerogrid dictionary
    # --------------------------------------------------------

    set_l = np.arange(
        n_panels * 6
    ).reshape(n_panels, 6)

    set_k = np.arange(
        n_panels * 6
    ).reshape(n_panels, 6)

    set_j = np.arange(
        n_panels * 6
    ).reshape(n_panels, 6)


    cornerpoint_grids = np.column_stack([
        grid_ids.ravel(),
        aero_xyz.reshape(-1, 3)
    ])


    aerogrid = {
        "ID": panel_ID,

        "l": panel_length,
        "A": panel_area,
        "N": panel_normal,

        "offset_l": offset_l,
        "offset_k": offset_k,
        "offset_j": offset_j,

        "offset_P1": offset_P1,
        "offset_P3": offset_P3,

        "r": span_vector_r,

        "set_l": set_l,
        "set_k": set_k,
        "set_j": set_j,

        "CD": np.zeros(n_panels, dtype=int),
        "CP": np.zeros(n_panels, dtype=int),

        "n": n_panels,

        "coord_desc": "bodyfixed",

        "cornerpoint_panels": panel_corner_ids,
        "cornerpoint_grids": cornerpoint_grids,
    }


    return (
        aerogrid,
        aero_xyz,
        y_stations,
        chord_at_y,
        x_le_at_y,
    )


# ============================================================
# Build baseline aerodynamic lattice
# ============================================================

aero_n_span = 10
aero_n_chord = 8

(
    aerogrid,
    aero_xyz,
    aero_y_stations,
    aero_chords,
    aero_x_le,
) = build_agard_aerogrid(
    n_span=aero_n_span,
    n_chord=aero_n_chord,
)


# ------------------------------------------------------------
# 8. Geometry verification
# ------------------------------------------------------------

expected_area = 0.3527992944

assert aerogrid["n"] == (
    aero_n_span * aero_n_chord
)

assert np.isclose(
    np.sum(aerogrid["A"]),
    expected_area,
    rtol=0.0,
    atol=1e-12,
)

# Every panel must point in +z
assert np.all(
    aerogrid["N"][:, 2] > 0.0
)

# Flat aerodynamic surface
assert np.allclose(
    aero_xyz[:, :, 2],
    0.0
)

# Verify span
calculated_span = (
    aero_y_stations[-1]
    - aero_y_stations[0]
)

assert np.isclose(
    calculated_span,
    0.762
)

# Verify root/tip chords
root_chord_check = (
    aero_xyz[-1, 0, 0]
    - aero_xyz[0, 0, 0]
)

tip_chord_check = (
    aero_xyz[-1, -1, 0]
    - aero_xyz[0, -1, 0]
)

assert np.isclose(
    root_chord_check,
    0.557784
)

assert np.isclose(
    tip_chord_check,
    0.3681984
)


# Quarter-chord sweep reconstructed from geometry
x_qc_root = (
    aero_xyz[0, 0, 0]
    + 0.25 * root_chord_check
)

x_qc_tip = (
    aero_xyz[0, -1, 0]
    + 0.25 * tip_chord_check
)

quarter_chord_sweep_check_deg = np.rad2deg(
    np.arctan2(
        x_qc_tip - x_qc_root,
        calculated_span
    )
)

assert np.isclose(
    quarter_chord_sweep_check_deg,
    45.0,
    atol=1e-12
)


# ------------------------------------------------------------
# 9. Plot aerodynamic lattice
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 7))

# Chordwise grid lines
for i_chord in range(aero_n_chord + 1):

    ax.plot(
        aero_xyz[i_chord, :, 1],
        aero_xyz[i_chord, :, 0],
        color="black",
        linewidth=0.8,
    )

# Spanwise grid lines
for i_span in range(aero_n_span + 1):

    ax.plot(
        aero_xyz[:, i_span, 1],
        aero_xyz[:, i_span, 0],
        color="black",
        linewidth=0.8,
    )

# DLM receiving/control points: 75% chord
ax.scatter(
    aerogrid["offset_j"][:, 1],
    aerogrid["offset_j"][:, 0],
    s=12,
    label="DLM receiving points (75% chord)",
)

ax.set(
    xlabel="Spanwise coordinate y (m)",
    ylabel="Streamwise coordinate x (m)",
    title=(
        f"AGARD 445.6 aerodynamic lattice — "
        f"{aero_n_span} × {aero_n_chord}"
    ),
)

ax.invert_yaxis()
ax.set_aspect("equal")
ax.grid(alpha=0.15)
ax.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

print(
    f"Aerodynamic lattice: "
    f"{aero_n_span} span x "
    f"{aero_n_chord} chord"
)

print(
    f"Total aerodynamic panels: "
    f"{aerogrid['n']}"
)

print(
    f"Corner points: "
    f"{(aero_n_span + 1) * (aero_n_chord + 1)}"
)

print(
    f"Summed aerodynamic area: "
    f"{np.sum(aerogrid['A']):.10f} m^2"
)

print(
    f"Root chord check: "
    f"{root_chord_check:.7f} m"
)

print(
    f"Tip chord check: "
    f"{tip_chord_check:.7f} m"
)

print(
    f"Quarter-chord sweep check: "
    f"{quarter_chord_sweep_check_deg:.6f} deg"
)

print(
    f"Minimum panel-normal z component: "
    f"{np.min(aerogrid['N'][:, 2]):.6f}"
)

print("\nAGARD aerodynamic lattice checks passed.")

In [ ]:
# Cell 18 — PanelAero DLM smoke test
#
# Corrected for PanelAero 2025.8:
#     VLM.calc_Qjj(...) -> (Qjj, Bjj)
#     DLM.calc_Qjj(...) -> Qjj
#
# This is only a numerical implementation check.
# It is NOT yet an AGARD flutter result.

# ------------------------------------------------------------
# 1. Test conditions
# ------------------------------------------------------------

Ma_test = 0.30

# PanelAero DLM convention:
#
#       k_PA = omega / U
#
# Units are 1/m when dimensional geometry is used.
k_test = 2.0


# ------------------------------------------------------------
# 2. Steady aerodynamic matrices
# ------------------------------------------------------------

Qjj_dlm_steady = DLM.calc_Qjj(
    aerogrid,
    Ma=Ma_test,
    k=0.0
)

# PanelAero VLM returns:
#     Qjj, Bjj
Qjj_vlm, Bjj_vlm = VLM.calc_Qjj(
    aerogrid,
    Ma=Ma_test
)


# ------------------------------------------------------------
# 3. Unsteady DLM matrix
# ------------------------------------------------------------

Qjj_dlm_unsteady = DLM.calc_Qjj(
    aerogrid,
    Ma=Ma_test,
    k=k_test
)


# ------------------------------------------------------------
# 4. Basic dimensions and finiteness
# ------------------------------------------------------------

n_aero = aerogrid["n"]

assert Qjj_dlm_steady.shape == (n_aero, n_aero)
assert Qjj_vlm.shape == (n_aero, n_aero)
assert Qjj_dlm_unsteady.shape == (n_aero, n_aero)

assert Bjj_vlm.shape == (n_aero, n_aero)

assert np.isfinite(Qjj_dlm_steady).all()
assert np.isfinite(Qjj_vlm).all()
assert np.isfinite(Qjj_dlm_unsteady).all()
assert np.isfinite(Bjj_vlm).all()


# ------------------------------------------------------------
# 5. Verify DLM k = 0 limit against VLM
# ------------------------------------------------------------

steady_difference = (
    Qjj_dlm_steady
    - Qjj_vlm
)

steady_relative_error = (
    np.linalg.norm(steady_difference)
    /
    np.linalg.norm(Qjj_vlm)
)

assert np.allclose(
    Qjj_dlm_steady,
    Qjj_vlm,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 6. Confirm genuinely unsteady response
# ------------------------------------------------------------

real_norm = np.linalg.norm(
    np.real(Qjj_dlm_unsteady)
)

imag_norm = np.linalg.norm(
    np.imag(Qjj_dlm_unsteady)
)

assert imag_norm > 0.0


# ------------------------------------------------------------
# 7. Numerical conditioning
# ------------------------------------------------------------

condition_steady = np.linalg.cond(
    Qjj_dlm_steady
)

condition_unsteady = np.linalg.cond(
    Qjj_dlm_unsteady
)


# ------------------------------------------------------------
# 8. Difference between steady and unsteady AICs
# ------------------------------------------------------------

unsteady_change_norm = (
    np.linalg.norm(
        Qjj_dlm_unsteady
        - Qjj_dlm_steady
    )
    /
    np.linalg.norm(
        Qjj_dlm_steady
    )
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(f"Mach number: {Ma_test:.3f}")
print(
    f"PanelAero unsteady k = omega/U: "
    f"{k_test:.4f} 1/m"
)

print(
    f"\nAIC matrix dimensions: "
    f"{Qjj_dlm_unsteady.shape}"
)

print(
    f"VLM Bjj matrix dimensions: "
    f"{Bjj_vlm.shape}"
)

print(
    f"\nSteady DLM-VLM relative difference: "
    f"{steady_relative_error:.3e}"
)

print(
    f"\nUnsteady real-part norm: "
    f"{real_norm:.6e}"
)

print(
    f"Unsteady imaginary-part norm: "
    f"{imag_norm:.6e}"
)

print(
    f"Relative change from steady to unsteady: "
    f"{unsteady_change_norm:.6e}"
)

print(
    f"\nCondition number, steady: "
    f"{condition_steady:.6e}"
)

print(
    f"Condition number, unsteady: "
    f"{condition_unsteady:.6e}"
)

print(
    "\nDLM aerodynamic influence matrix "
    "smoke test passed."
)

In [ ]:
# Cell 19 — Structural-to-aerodynamic interpolation operators
#
# Purpose:
#   Transfer the published AGARD structural modal fields from the
#   121 structural nodes to the independent DLM aerodynamic lattice.
#
# We create TWO reusable operators:
#
#   H_j : structural nodes -> DLM receiving points (75% chord)
#   H_k : structural nodes -> aerodynamic force points (50% chord)
#
# These operators will later also transfer the published dz/dx fields.
#
# No modal source data are modified or renormalized.

from scipy.spatial import Delaunay


# ------------------------------------------------------------
# 1. Structural interpolation coordinates
# ------------------------------------------------------------

structural_xy = np.column_stack([
    x_grid.ravel(),
    y_grid.ravel(),
])

assert structural_xy.shape == (121, 2)

# Published four-mode transverse basis already assembled in Cell 14
assert Phi_z_4.shape == (121, 4)


# ------------------------------------------------------------
# 2. Aerodynamic query locations
# ------------------------------------------------------------

aero_j_xy = np.column_stack([
    aerogrid["offset_j"][:, 0],
    aerogrid["offset_j"][:, 1],
])

aero_k_xy = np.column_stack([
    aerogrid["offset_k"][:, 0],
    aerogrid["offset_k"][:, 1],
])

assert aero_j_xy.shape == (80, 2)
assert aero_k_xy.shape == (80, 2)


# ------------------------------------------------------------
# 3. Build a linear barycentric interpolation matrix
#
# For each aerodynamic query point:
#
#     f_aero = H @ f_structural
#
# Each row contains the three barycentric weights belonging
# to one triangle of the structural-node triangulation.
# ------------------------------------------------------------

structural_triangulation = Delaunay(structural_xy)


def build_barycentric_transfer(triangulation, query_xy, n_source):
    """
    Construct a dense linear interpolation matrix from a Delaunay
    triangulation.

    Parameters
    ----------
    triangulation : scipy.spatial.Delaunay
        Triangulation of the source points.

    query_xy : ndarray, shape (n_query, 2)
        Target coordinates.

    n_source : int
        Number of source nodes.

    Returns
    -------
    H : ndarray, shape (n_query, n_source)
        Linear interpolation operator.
    """

    simplex = triangulation.find_simplex(query_xy)

    # Every DLM point should lie inside the structural wing planform.
    if np.any(simplex < 0):
        bad = np.where(simplex < 0)[0]

        raise ValueError(
            "Some aerodynamic interpolation points lie outside "
            f"the structural-node convex hull: {bad.tolist()}"
        )

    H = np.zeros(
        (len(query_xy), n_source),
        dtype=float
    )

    for i, simplex_id in enumerate(simplex):

        transform = triangulation.transform[simplex_id]

        # First two barycentric coordinates
        bary_first = (
            transform[:2]
            @ (query_xy[i] - transform[2])
        )

        # Third coordinate closes the partition of unity
        bary = np.array([
            bary_first[0],
            bary_first[1],
            1.0 - bary_first.sum(),
        ])

        vertices = triangulation.simplices[simplex_id]

        H[i, vertices] = bary

    return H


H_j = build_barycentric_transfer(
    structural_triangulation,
    aero_j_xy,
    n_source=121,
)

H_k = build_barycentric_transfer(
    structural_triangulation,
    aero_k_xy,
    n_source=121,
)


# ------------------------------------------------------------
# 4. Operator checks
# ------------------------------------------------------------

assert H_j.shape == (80, 121)
assert H_k.shape == (80, 121)

assert np.isfinite(H_j).all()
assert np.isfinite(H_k).all()

# Linear interpolation must preserve a constant field.
assert np.allclose(
    H_j.sum(axis=1),
    1.0,
    rtol=0.0,
    atol=1e-12,
)

assert np.allclose(
    H_k.sum(axis=1),
    1.0,
    rtol=0.0,
    atol=1e-12,
)

# Since all target points are inside the triangulation,
# barycentric weights should be non-negative apart from tiny
# round-off errors.
assert H_j.min() > -1e-12
assert H_k.min() > -1e-12


# ------------------------------------------------------------
# 5. Transfer the four original modal z-fields
# ------------------------------------------------------------

Phi_z_j = H_j @ Phi_z_4
Phi_z_k = H_k @ Phi_z_4

assert Phi_z_j.shape == (80, 4)
assert Phi_z_k.shape == (80, 4)

assert np.isfinite(Phi_z_j).all()
assert np.isfinite(Phi_z_k).all()


# ------------------------------------------------------------
# 6. Verify the interpolation machinery at the ORIGINAL nodes
#
# Constructing a transfer operator evaluated at the source
# coordinates should reproduce all four modal values.
# ------------------------------------------------------------

H_nodes = build_barycentric_transfer(
    structural_triangulation,
    structural_xy,
    n_source=121,
)

Phi_z_reconstructed = H_nodes @ Phi_z_4

node_reconstruction_error = np.max(
    np.abs(
        Phi_z_reconstructed
        - Phi_z_4
    )
)

assert np.allclose(
    Phi_z_reconstructed,
    Phi_z_4,
    rtol=1e-12,
    atol=1e-12,
)


# ------------------------------------------------------------
# 7. Preserve source-array check
# ------------------------------------------------------------

assert np.allclose(
    Phi_z_4[:, 0],
    mode1_z_table.ravel(),
)

assert np.allclose(
    Phi_z_4[:, 1],
    mode2_z_table.ravel(),
)

assert np.allclose(
    Phi_z_4[:, 2],
    mode3_z_table.ravel(),
)

assert np.allclose(
    Phi_z_4[:, 3],
    mode4_z_table.ravel(),
)


# ------------------------------------------------------------
# 8. Compact transfer summary
# ------------------------------------------------------------

transfer_summary = pd.DataFrame({
    "mode_id": np.arange(1, 5),

    "structural_min": np.min(
        Phi_z_4,
        axis=0
    ),

    "structural_max": np.max(
        Phi_z_4,
        axis=0
    ),

    "receiving_point_min": np.min(
        Phi_z_j,
        axis=0
    ),

    "receiving_point_max": np.max(
        Phi_z_j,
        axis=0
    ),

    "force_point_min": np.min(
        Phi_z_k,
        axis=0
    ),

    "force_point_max": np.max(
        Phi_z_k,
        axis=0
    ),
})


print("Structural-to-aerodynamic transfer matrices:")
print(f"H_j shape: {H_j.shape}")
print(f"H_k shape: {H_k.shape}")

print(
    "\nMaximum source-node reconstruction error: "
    f"{node_reconstruction_error:.3e}"
)

print(
    "Minimum H_j interpolation weight: "
    f"{H_j.min():.3e}"
)

print(
    "Minimum H_k interpolation weight: "
    f"{H_k.min():.3e}"
)

print("\nTransferred modal-field summary:")
display(transfer_summary)

print(
    "\nFour published z-mode fields transferred "
    "to the aerodynamic lattice successfully."
)

In [ ]:
# Cell 20 — Mode 1 published streamwise modal slope dz/dx
#
# Source:
# NASA TM-100492, Table 4(a), PDF pages 21–22
# 2.5-ft weakened model 3
#
# IMPORTANT:
# These are the ORIGINAL published dz/dx modal coefficients.
# They use the same modal normalization as mode1_z_table.
#
# Do not derive these slopes numerically from z:
# the report provides them directly.

mode1_dzdx_table = np.array([
    # Nodes 1–11
    [-0.00101, -0.000468, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.163, 0.237],

    # Nodes 12–22
    [0.00836, 0.0209, 0.0415, 0.0679, 0.0992,
     0.133, 0.177, 0.224, 0.261, 0.328, 0.386],

    # Nodes 23–33
    [0.0629, 0.0960, 0.134, 0.174, 0.217,
     0.262, 0.309, 0.358, 0.407, 0.461, 0.511],

    # Nodes 34–44
    [0.160, 0.201, 0.242, 0.285, 0.329,
     0.375, 0.417, 0.461, 0.504, 0.546, 0.584],

    # Nodes 45–55
    [0.270, 0.309, 0.349, 0.389, 0.429,
     0.469, 0.506, 0.543, 0.578, 0.612, 0.640],

    # Nodes 56–66
    [0.375, 0.412, 0.447, 0.482, 0.516,
     0.551, 0.579, 0.608, 0.635, 0.661, 0.679],

    # Nodes 67–77
    [0.471, 0.502, 0.531, 0.559, 0.586,
     0.614, 0.634, 0.655, 0.674, 0.691, 0.700],

    # Nodes 78–88
    [0.552, 0.576, 0.598, 0.618, 0.637,
     0.656, 0.669, 0.683, 0.694, 0.703, 0.702],

    # Nodes 89–99
    [0.614, 0.630, 0.644, 0.657, 0.668,
     0.679, 0.686, 0.692, 0.697, 0.699, 0.692],

    # Nodes 100–110
    [0.654, 0.662, 0.669, 0.675, 0.679,
     0.684, 0.685, 0.687, 0.688, 0.688, 0.681],

    # Nodes 111–121
    [0.672, 0.672, 0.673, 0.674, 0.675,
     0.677, 0.677, 0.677, 0.678, 0.679, 0.688],
], dtype=float)


# ------------------------------------------------------------
# 1. Basic integrity checks
# ------------------------------------------------------------

assert mode1_dzdx_table.shape == (11, 11)
assert mode1_dzdx_table.size == 121
assert np.isfinite(mode1_dzdx_table).all()


# ------------------------------------------------------------
# 2. Spot checks against scanned Table 4(a)
# ------------------------------------------------------------

mode1_dzdx_reference = {
    1:   -0.00101,
    2:   -0.000468,
    10:   0.163,
    11:   0.237,
    22:   0.386,
    44:   0.584,
    66:   0.679,
    77:   0.700,
    88:   0.702,
    99:   0.692,
    110:  0.681,
    121:  0.688,
}

mode1_dzdx_flat = mode1_dzdx_table.ravel()

for node_id, expected in mode1_dzdx_reference.items():

    actual = mode1_dzdx_flat[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 1 dz/dx mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# 3. Transfer published slope to aerodynamic locations
#
# Use exactly the same interpolation operators validated
# in Cell 19.
# ------------------------------------------------------------

mode1_dzdx_j = (
    H_j @ mode1_dzdx_flat
)

mode1_dzdx_k = (
    H_k @ mode1_dzdx_flat
)

assert mode1_dzdx_j.shape == (80,)
assert mode1_dzdx_k.shape == (80,)

assert np.isfinite(mode1_dzdx_j).all()
assert np.isfinite(mode1_dzdx_k).all()


# ------------------------------------------------------------
# 4. Verify interpolation at original structural nodes
# ------------------------------------------------------------

mode1_dzdx_reconstructed = (
    H_nodes @ mode1_dzdx_flat
)

mode1_dzdx_reconstruction_error = np.max(
    np.abs(
        mode1_dzdx_reconstructed
        - mode1_dzdx_flat
    )
)

assert np.allclose(
    mode1_dzdx_reconstructed,
    mode1_dzdx_flat,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print(
    f"Imported Mode 1 dz/dx coefficients: "
    f"{mode1_dzdx_table.size}"
)

print(
    f"Structural dz/dx range: "
    f"{mode1_dzdx_flat.min():.6f} to "
    f"{mode1_dzdx_flat.max():.6f}"
)

print(
    f"DLM receiving-point dz/dx range: "
    f"{mode1_dzdx_j.min():.6f} to "
    f"{mode1_dzdx_j.max():.6f}"
)

print(
    f"Aerodynamic force-point dz/dx range: "
    f"{mode1_dzdx_k.min():.6f} to "
    f"{mode1_dzdx_k.max():.6f}"
)

print(
    "Maximum source-node slope reconstruction error: "
    f"{mode1_dzdx_reconstruction_error:.3e}"
)

print("\nMode 1 published dz/dx field checks passed.")

In [ ]:
# Cell 21 — Mode 2 published streamwise modal slope dz/dx
#
# Source:
# NASA TM-100492, Table 4(b), PDF pages 23–24
# 2.5-ft weakened model 3
#
# Mode 2 = first torsion
#
# These are the ORIGINAL published dz/dx coefficients.
# They retain the same generalized-mass normalization as
# mode2_z_table.

mode2_dzdx_table = np.array([
    # Nodes 1–11
    [0.0213, 0.0250, 0.0, 0.0, 0.0, 0.0,
     0.0, 0.0, 0.0, -0.0972, -0.0149],

    # Nodes 12–22
    [0.120, 0.210, 0.318, 0.416, 0.480,
     0.494, 0.439, 0.316, 0.289, 0.213, 0.218],

    # Nodes 23–33
    [0.538, 0.680, 0.797, 0.869, 0.894,
     0.874, 0.822, 0.757, 0.693, 0.620, 0.612],

    # Nodes 34–44
    [1.13, 1.23, 1.29, 1.31, 1.30,
     1.25, 1.19, 1.11, 1.03, 0.936, 0.928],

    # Nodes 45–55
    [1.72, 1.75, 1.75, 1.72, 1.67,
     1.59, 1.50, 1.40, 1.29, 1.18, 1.19],

    # Nodes 56–66
    [2.24, 2.19, 2.13, 2.06, 1.96,
     1.85, 1.74, 1.62, 1.51, 1.40, 1.45],

    # Nodes 67–77
    [2.62, 2.51, 2.41, 2.30, 2.18,
     2.04, 1.93, 1.82, 1.71, 1.63, 1.75],

    # Nodes 78–88
    [2.85, 2.70, 2.58, 2.45, 2.32,
     2.19, 2.10, 2.01, 1.93, 1.90, 2.10],

    # Nodes 89–99
    [2.95, 2.79, 2.66, 2.54, 2.43,
     2.33, 2.27, 2.22, 2.20, 2.21, 2.50],

    # Nodes 100–110
    [2.95, 2.79, 2.69, 2.61, 2.56,
     2.51, 2.49, 2.48, 2.48, 2.50, 2.74],

    # Nodes 111–121
    [2.90, 2.80, 2.77, 2.75, 2.74,
     2.73, 2.73, 2.73, 2.74, 2.73, 2.49],
], dtype=float)


# ------------------------------------------------------------
# 1. Basic integrity checks
# ------------------------------------------------------------

assert mode2_dzdx_table.shape == (11, 11)
assert mode2_dzdx_table.size == 121
assert np.isfinite(mode2_dzdx_table).all()


# ------------------------------------------------------------
# 2. Spot checks against the rendered Table 4(b) scan
# ------------------------------------------------------------

mode2_dzdx_reference = {
    1:    0.0213,
    2:    0.0250,
    10:  -0.0972,
    11:  -0.0149,
    22:   0.218,
    33:   0.612,
    44:   0.928,
    55:   1.19,
    61:   1.85,
    66:   1.45,
    77:   1.75,
    88:   2.10,
    99:   2.50,
    110:  2.74,
    121:  2.49,
}

mode2_dzdx_flat = mode2_dzdx_table.ravel()

for node_id, expected in mode2_dzdx_reference.items():

    actual = mode2_dzdx_flat[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 2 dz/dx mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# 3. Transfer to aerodynamic locations
# ------------------------------------------------------------

mode2_dzdx_j = (
    H_j @ mode2_dzdx_flat
)

mode2_dzdx_k = (
    H_k @ mode2_dzdx_flat
)

assert mode2_dzdx_j.shape == (80,)
assert mode2_dzdx_k.shape == (80,)

assert np.isfinite(mode2_dzdx_j).all()
assert np.isfinite(mode2_dzdx_k).all()


# ------------------------------------------------------------
# 4. Verify interpolation at original structural nodes
# ------------------------------------------------------------

mode2_dzdx_reconstructed = (
    H_nodes @ mode2_dzdx_flat
)

mode2_dzdx_reconstruction_error = np.max(
    np.abs(
        mode2_dzdx_reconstructed
        - mode2_dzdx_flat
    )
)

assert np.allclose(
    mode2_dzdx_reconstructed,
    mode2_dzdx_flat,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print(
    f"Imported Mode 2 dz/dx coefficients: "
    f"{mode2_dzdx_table.size}"
)

print(
    f"Structural dz/dx range: "
    f"{mode2_dzdx_flat.min():.6f} to "
    f"{mode2_dzdx_flat.max():.6f}"
)

print(
    f"DLM receiving-point dz/dx range: "
    f"{mode2_dzdx_j.min():.6f} to "
    f"{mode2_dzdx_j.max():.6f}"
)

print(
    f"Aerodynamic force-point dz/dx range: "
    f"{mode2_dzdx_k.min():.6f} to "
    f"{mode2_dzdx_k.max():.6f}"
)

print(
    "Maximum source-node slope reconstruction error: "
    f"{mode2_dzdx_reconstruction_error:.3e}"
)

print("\nMode 2 published dz/dx field checks passed.")

In [ ]:
# Cell 22 — Mode 3 published streamwise modal slope dz/dx
#
# Source:
# NASA TM-100492, Table 4(c), PDF pages 25–26
# 2.5-ft weakened model 3
#
# Mode 3 = second bending
#
# These are the ORIGINAL published dz/dx coefficients.
# They retain the same generalized-mass normalization as
# mode3_z_table.

mode3_dzdx_table = np.array([
    # Nodes 1–11
    [-0.00246, -0.00492, 0.0, 0.0, 0.0, 0.0,
     0.0, 0.0, 0.0, 0.948, 1.14],

    # Nodes 12–22
    [-0.000732, 0.0104, 0.0455, 0.104, 0.198,
     0.337, 0.572, 0.870, 0.974, 1.25, 1.25],

    # Nodes 23–33
    [0.0282, 0.0682, 0.129, 0.214, 0.325,
     0.460, 0.595, 0.710, 0.773, 0.805, 0.480],

    # Nodes 34–44
    [0.0143, 0.0346, 0.0605, 0.0951, 0.131,
     0.158, 0.154, 0.114, 0.0132, -0.166, -0.791],

    # Nodes 45–55
    [-0.204, -0.243, -0.287, -0.336, -0.402,
     -0.496, -0.622, -0.795, -1.03, -1.35, -2.16],

    # Nodes 56–66
    [-0.701, -0.808, -0.928, -1.06, -1.22,
     -1.42, -1.62, -1.88, -2.19, -2.58, -3.43],

    # Nodes 67–77
    [-1.46, -1.62, -1.80, -1.99, -2.20,
     -2.46, -2.68, -2.96, -3.28, -3.66, -4.39],

    # Nodes 78–88
    [-2.38, -2.56, -2.76, -2.96, -3.16,
     -3.41, -3.61, -3.86, -4.12, -4.42, -4.92],

    # Nodes 89–99
    [-3.33, -3.47, -3.63, -3.79, -3.94,
     -4.12, -4.26, -4.42, -4.58, -4.75, -4.97],

    # Nodes 100–110
    [-4.10, -4.14, -4.23, -4.31, -4.38,
     -4.48, -4.53, -4.60, -4.67, -4.73, -4.77],

    # Nodes 111–121
    [-4.48, -4.43, -4.44, -4.45, -4.47,
     -4.49, -4.50, -4.52, -4.55, -4.58, -4.74],
], dtype=float)


# ------------------------------------------------------------
# 1. Basic integrity checks
# ------------------------------------------------------------

assert mode3_dzdx_table.shape == (11, 11)
assert mode3_dzdx_table.size == 121
assert np.isfinite(mode3_dzdx_table).all()


# ------------------------------------------------------------
# 2. Spot checks against rendered Table 4(c)
# ------------------------------------------------------------

mode3_dzdx_reference = {
    1:   -0.00246,
    2:   -0.00492,
    10:   0.948,
    11:   1.14,
    22:   1.25,
    33:   0.480,
    44:  -0.791,
    55:  -2.16,
    61:  -1.42,
    62:  -1.62,
    66:  -3.43,
    77:  -4.39,
    88:  -4.92,
    99:  -4.97,
    110: -4.77,
    121: -4.74,
}

mode3_dzdx_flat = mode3_dzdx_table.ravel()

for node_id, expected in mode3_dzdx_reference.items():

    actual = mode3_dzdx_flat[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 3 dz/dx mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# 3. Transfer to aerodynamic locations
# ------------------------------------------------------------

mode3_dzdx_j = (
    H_j @ mode3_dzdx_flat
)

mode3_dzdx_k = (
    H_k @ mode3_dzdx_flat
)

assert mode3_dzdx_j.shape == (80,)
assert mode3_dzdx_k.shape == (80,)

assert np.isfinite(mode3_dzdx_j).all()
assert np.isfinite(mode3_dzdx_k).all()


# ------------------------------------------------------------
# 4. Verify interpolation at original structural nodes
# ------------------------------------------------------------

mode3_dzdx_reconstructed = (
    H_nodes @ mode3_dzdx_flat
)

mode3_dzdx_reconstruction_error = np.max(
    np.abs(
        mode3_dzdx_reconstructed
        - mode3_dzdx_flat
    )
)

assert np.allclose(
    mode3_dzdx_reconstructed,
    mode3_dzdx_flat,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print(
    f"Imported Mode 3 dz/dx coefficients: "
    f"{mode3_dzdx_table.size}"
)

print(
    f"Structural dz/dx range: "
    f"{mode3_dzdx_flat.min():.6f} to "
    f"{mode3_dzdx_flat.max():.6f}"
)

print(
    f"DLM receiving-point dz/dx range: "
    f"{mode3_dzdx_j.min():.6f} to "
    f"{mode3_dzdx_j.max():.6f}"
)

print(
    f"Aerodynamic force-point dz/dx range: "
    f"{mode3_dzdx_k.min():.6f} to "
    f"{mode3_dzdx_k.max():.6f}"
)

print(
    "Maximum source-node slope reconstruction error: "
    f"{mode3_dzdx_reconstruction_error:.3e}"
)

print("\nMode 3 published dz/dx field checks passed.")

In [ ]:
# Cell 23 — Mode 4 published streamwise modal slope dz/dx
#
# Source:
# NASA TM-100492, Table 4(d), PDF pages 27–28
# 2.5-ft weakened model 3
#
# Mode 4 = second torsion
#
# These are the ORIGINAL published dz/dx coefficients.
# They retain the same generalized-mass normalization as
# mode4_z_table.

mode4_dzdx_table = np.array([
    # Nodes 1–11
    [0.0950, 0.0965, 0.0, 0.0, 0.0, 0.0,
     0.0, 0.0, 0.0, -0.203, 0.385],

    # Nodes 12–22
    [0.393, 0.619, 0.876, 1.07, 1.15,
     1.11, 0.916, 0.604, 0.531, 0.441, 1.12],

    # Nodes 23–33
    [1.49, 1.73, 1.88, 1.90, 1.80,
     1.62, 1.40, 1.21, 1.12, 1.16, 2.14],

    # Nodes 34–44
    [2.66, 2.60, 2.48, 2.28, 2.03,
     1.76, 1.55, 1.42, 1.40, 1.58, 2.82],

    # Nodes 45–55
    [3.20, 2.80, 2.45, 2.10, 1.77,
     1.51, 1.36, 1.33, 1.45, 1.81, 3.25],

    # Nodes 56–66
    [2.80, 2.18, 1.73, 1.36, 1.09,
     0.934, 0.927, 1.07, 1.38, 1.92, 3.31],

    # Nodes 67–77
    [1.47, 0.871, 0.500, 0.268, 0.171,
     0.234, 0.400, 0.725, 1.19, 1.81, 2.74],

    # Nodes 78–88
    [-0.395, -0.739, -0.881, -0.873, -0.741,
     -0.453, -0.159, 0.261, 0.742, 1.24, 1.25],

    # Nodes 89–99
    [-2.22, -2.18, -2.04, -1.81, -1.51,
     -1.13, -0.834, -0.488, -0.174, 0.0213, -1.32],

    # Nodes 100–110
    [-3.50, -3.08, -2.78, -2.49, -2.25,
     -1.99, -1.86, -1.72, -1.59, -1.50, -2.75],

    # Nodes 111–121
    [-3.88, -3.47, -3.40, -3.34, -3.30,
     -3.30, -3.35, -3.42, -3.53, -3.59, -1.06],
], dtype=float)


# ------------------------------------------------------------
# 1. Basic integrity checks
# ------------------------------------------------------------

assert mode4_dzdx_table.shape == (11, 11)
assert mode4_dzdx_table.size == 121
assert np.isfinite(mode4_dzdx_table).all()


# ------------------------------------------------------------
# 2. Spot checks against rendered Table 4(d)
# ------------------------------------------------------------

mode4_dzdx_reference = {
    1:    0.0950,
    2:    0.0965,
    10:  -0.203,
    11:   0.385,
    22:   1.12,
    33:   2.14,
    44:   2.82,
    55:   3.25,
    61:   0.934,
    62:   0.927,
    66:   3.31,
    77:   2.74,
    78:  -0.395,
    88:   1.25,
    89:  -2.22,
    98:   0.0213,
    99:  -1.32,
    110: -2.75,
    111: -3.88,
    121: -1.06,
}

mode4_dzdx_flat = mode4_dzdx_table.ravel()

for node_id, expected in mode4_dzdx_reference.items():

    actual = mode4_dzdx_flat[node_id - 1]

    assert np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=1e-12
    ), (
        f"Mode 4 dz/dx mismatch at node {node_id}: "
        f"expected {expected}, obtained {actual}"
    )


# ------------------------------------------------------------
# 3. Transfer to aerodynamic locations
# ------------------------------------------------------------

mode4_dzdx_j = (
    H_j @ mode4_dzdx_flat
)

mode4_dzdx_k = (
    H_k @ mode4_dzdx_flat
)

assert mode4_dzdx_j.shape == (80,)
assert mode4_dzdx_k.shape == (80,)

assert np.isfinite(mode4_dzdx_j).all()
assert np.isfinite(mode4_dzdx_k).all()


# ------------------------------------------------------------
# 4. Verify interpolation at original structural nodes
# ------------------------------------------------------------

mode4_dzdx_reconstructed = (
    H_nodes @ mode4_dzdx_flat
)

mode4_dzdx_reconstruction_error = np.max(
    np.abs(
        mode4_dzdx_reconstructed
        - mode4_dzdx_flat
    )
)

assert np.allclose(
    mode4_dzdx_reconstructed,
    mode4_dzdx_flat,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print(
    f"Imported Mode 4 dz/dx coefficients: "
    f"{mode4_dzdx_table.size}"
)

print(
    f"Structural dz/dx range: "
    f"{mode4_dzdx_flat.min():.6f} to "
    f"{mode4_dzdx_flat.max():.6f}"
)

print(
    f"DLM receiving-point dz/dx range: "
    f"{mode4_dzdx_j.min():.6f} to "
    f"{mode4_dzdx_j.max():.6f}"
)

print(
    f"Aerodynamic force-point dz/dx range: "
    f"{mode4_dzdx_k.min():.6f} to "
    f"{mode4_dzdx_k.max():.6f}"
)

print(
    "Maximum source-node slope reconstruction error: "
    f"{mode4_dzdx_reconstruction_error:.3e}"
)

print("\nMode 4 published dz/dx field checks passed.")

In [ ]:
# Cell 24 — Assemble the four published dz/dx modes and
# diagnose the plate-element slope sign convention.
#
# IMPORTANT:
# We are NOT modifying the published slope data here.
# This cell only compares them with a numerical chordwise derivative
# of the published z displacement field.
#
# Table 2 structural coordinates are published in inches.
# Our x_grid is SI (m), so convert x_grid back to inches for
# a dimensionally consistent comparison.

# ------------------------------------------------------------
# 1. Assemble published four-mode slope basis
# ------------------------------------------------------------

Phi_dzdx_pub_4 = np.column_stack([
    mode1_dzdx_table.ravel(),
    mode2_dzdx_table.ravel(),
    mode3_dzdx_table.ravel(),
    mode4_dzdx_table.ravel(),
])

assert Phi_dzdx_pub_4.shape == (121, 4)
assert np.isfinite(Phi_dzdx_pub_4).all()


# ------------------------------------------------------------
# 2. Transfer complete slope basis to DLM locations
# ------------------------------------------------------------

Phi_dzdx_pub_j = H_j @ Phi_dzdx_pub_4
Phi_dzdx_pub_k = H_k @ Phi_dzdx_pub_4

assert Phi_dzdx_pub_j.shape == (80, 4)
assert Phi_dzdx_pub_k.shape == (80, 4)

assert np.isfinite(Phi_dzdx_pub_j).all()
assert np.isfinite(Phi_dzdx_pub_k).all()


# ------------------------------------------------------------
# 3. Convert structural x coordinates back to inches
#
# Table 2 coordinates are given in inches.
# ------------------------------------------------------------

x_grid_in = x_grid / IN_TO_M


# ------------------------------------------------------------
# 4. Numerical chordwise derivative of z
#
# This is ONLY a diagnostic.
#
# The tabulated plate-element slope DOFs are the source data
# we intend to use. Numerical differentiation is not being used
# to replace them.
#
# We exclude:
#   - root row, because of the constrained-root treatment
#   - first/last chord nodes, to avoid one-sided derivative effects
# ------------------------------------------------------------

slope_diagnostic_rows = []

for mode_index in range(4):

    z_mode = (
        Phi_z_4[:, mode_index]
        .reshape(11, 11)
    )

    slope_pub = (
        Phi_dzdx_pub_4[:, mode_index]
        .reshape(11, 11)
    )

    dzdx_numeric = np.zeros_like(z_mode)

    for i_span in range(11):

        dzdx_numeric[i_span, :] = np.gradient(
            z_mode[i_span, :],
            x_grid_in[i_span, :],
            edge_order=2,
        )

    # Interior comparison only
    numeric_compare = (
        dzdx_numeric[1:, 1:-1]
        .ravel()
    )

    published_compare = (
        slope_pub[1:, 1:-1]
        .ravel()
    )

    # Correlation with published sign
    corr_same = np.corrcoef(
        numeric_compare,
        published_compare
    )[0, 1]

    # Correlation after reversing published sign
    corr_reversed = np.corrcoef(
        numeric_compare,
        -published_compare
    )[0, 1]

    # Least-squares scalar:
    #
    # numeric derivative ~= alpha * published slope
    #
    alpha = (
        np.dot(
            published_compare,
            numeric_compare
        )
        /
        np.dot(
            published_compare,
            published_compare
        )
    )

    # Normalized RMS discrepancy for both sign possibilities
    rms_reference = np.sqrt(
        np.mean(numeric_compare**2)
    )

    nrmse_same = (
        np.sqrt(
            np.mean(
                (
                    numeric_compare
                    - published_compare
                )**2
            )
        )
        / rms_reference
    )

    nrmse_reversed = (
        np.sqrt(
            np.mean(
                (
                    numeric_compare
                    + published_compare
                )**2
            )
        )
        / rms_reference
    )

    slope_diagnostic_rows.append({
        "mode_id": mode_index + 1,
        "corr_numeric_vs_published": corr_same,
        "corr_numeric_vs_minus_published": corr_reversed,
        "least_squares_alpha": alpha,
        "NRMSE_same_sign": nrmse_same,
        "NRMSE_reversed_sign": nrmse_reversed,
    })


slope_convention_check = pd.DataFrame(
    slope_diagnostic_rows
)


# ------------------------------------------------------------
# 5. Basic source-data reconstruction checks
# ------------------------------------------------------------

Phi_dzdx_nodes_reconstructed = (
    H_nodes @ Phi_dzdx_pub_4
)

slope_basis_reconstruction_error = np.max(
    np.abs(
        Phi_dzdx_nodes_reconstructed
        - Phi_dzdx_pub_4
    )
)

assert np.allclose(
    Phi_dzdx_nodes_reconstructed,
    Phi_dzdx_pub_4,
    rtol=1e-12,
    atol=1e-12,
)


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print("Published four-mode slope basis:")
print(f"Phi_dzdx_pub_4 shape: {Phi_dzdx_pub_4.shape}")

print("\nTransferred slope bases:")
print(f"Phi_dzdx_pub_j shape: {Phi_dzdx_pub_j.shape}")
print(f"Phi_dzdx_pub_k shape: {Phi_dzdx_pub_k.shape}")

print(
    "\nMaximum source-node slope reconstruction error: "
    f"{slope_basis_reconstruction_error:.3e}"
)

print(
    "\nDiagnostic comparison:"
)
print(
    "numerical dz/dx from z versus published plate-element dz/dx"
)

display(slope_convention_check)

print(
    "\nNo aerodynamic sign convention has been applied yet."
)
print(
    "Inspect the diagnostic before constructing modal normalwash."
)

In [ ]:
# Cell 25 — Four-mode DLM normalwash kinematics
#
# Purpose:
#   Convert the published AGARD slope basis to SI units and define
#   reusable structural -> aerodynamic normalwash operators.
#
# Modal coordinate convention:
#
#       z(x,y,t) = Phi_z(x,y) @ q(t)
#
# with:
#       Phi_z dimensionless
#       q       in metres
#
# Published dz/dx values are associated with Table 2 coordinates
# given in inches, so:
#
#       [dz/dx]_SI = [dz/dx]_published / 0.0254
#
# PanelAero convention:
# positive downwash -> positive pressure -> positive/upward Fz.
#
# With structural z positive upward, the normalized aerodynamic
# normalwash generated by structural motion is taken as
#
#       w_j / U
#         = - Phi_x q
#           - Phi_z qdot / U
#
# where Phi_x is the physical modal streamwise slope [1/m].
#
# For harmonic motion q = q_hat exp(i*omega*t):
#
#       w_hat_j
#         = -(Phi_x + i*(omega/U)*Phi_z) q_hat
#
# PanelAero uses k_PA = omega/U [1/m].

# ------------------------------------------------------------
# 1. Convert published slope basis from 1/in to 1/m
# ------------------------------------------------------------

Phi_dzdx_j_SI = (
    Phi_dzdx_pub_j
    / IN_TO_M
)

Phi_dzdx_k_SI = (
    Phi_dzdx_pub_k
    / IN_TO_M
)

assert Phi_dzdx_j_SI.shape == (80, 4)
assert Phi_dzdx_k_SI.shape == (80, 4)

assert np.isfinite(Phi_dzdx_j_SI).all()
assert np.isfinite(Phi_dzdx_k_SI).all()


# ------------------------------------------------------------
# 2. Define reusable normalwash operators
#
# For arbitrary time-domain q and qdot:
#
#   w_norm =
#       Wq_j @ q
#       + Wqdot_j @ qdot / U
#
# where w_norm is dimensionless.
# ------------------------------------------------------------

Wq_j = -Phi_dzdx_j_SI
Wqdot_j = -Phi_z_j.copy()

assert Wq_j.shape == (80, 4)
assert Wqdot_j.shape == (80, 4)


# ------------------------------------------------------------
# 3. Helper: time-domain normalized modal normalwash
# ------------------------------------------------------------

def modal_normalwash_time(q, qdot, U):
    """
    Return dimensionless DLM/VLM normalwash at the 80 receiving points.

    Parameters
    ----------
    q : array-like, shape (4,)
        Modal coordinates [m]

    qdot : array-like, shape (4,)
        Modal velocities [m/s]

    U : float
        Freestream velocity [m/s]
    """

    q = np.asarray(q, dtype=float)
    qdot = np.asarray(qdot, dtype=float)

    assert q.shape == (4,)
    assert qdot.shape == (4,)
    assert U > 0.0

    return (
        Wq_j @ q
        + (Wqdot_j @ qdot) / U
    )


# ------------------------------------------------------------
# 4. Helper: harmonic normalwash matrix
#
# Maps modal displacement amplitude q_hat [m] directly to
# dimensionless complex normalwash.
# ------------------------------------------------------------

def modal_normalwash_harmonic(omega, U):
    """
    Return complex modal normalwash matrix, shape (80,4).

    omega : rad/s
    U     : m/s
    """

    assert omega >= 0.0
    assert U > 0.0

    k_PA = omega / U   # PanelAero convention [1/m]

    return -(
        Phi_dzdx_j_SI
        + 1j * k_PA * Phi_z_j
    )


# ------------------------------------------------------------
# 5. Internal consistency check:
# harmonic formulation versus time-domain formulation
# ------------------------------------------------------------

U_check = 50.0
omega_check = 2.0 * np.pi * 38.1

q_hat_check = np.array([
    0.0,
    1.0e-3,
    0.0,
    0.0
], dtype=complex)

W_harmonic_check = modal_normalwash_harmonic(
    omega_check,
    U_check
)

wj_from_harmonic = (
    W_harmonic_check @ q_hat_check
)

# For exp(i*omega*t):
# qdot_hat = i*omega*q_hat
wj_from_components = (
    Wq_j @ q_hat_check
    + (
        Wqdot_j
        @ (1j * omega_check * q_hat_check)
    ) / U_check
)

harmonic_consistency_error = np.max(
    np.abs(
        wj_from_harmonic
        - wj_from_components
    )
)

assert np.allclose(
    wj_from_harmonic,
    wj_from_components,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 6. PanelAero sign sanity check using static incidence
#
# The official PanelAero tutorial uses positive normalized
# downwash for positive angle of attack.
# ------------------------------------------------------------

alpha_check_deg = 5.0
alpha_check_rad = np.deg2rad(alpha_check_deg)

wj_alpha = np.full(
    aerogrid["n"],
    alpha_check_rad
)

cp_alpha = (
    Qjj_vlm @ wj_alpha
)

# Unit dynamic pressure is sufficient for a sign check.
Fz_alpha_unit_q = np.sum(
    aerogrid["A"]
    * aerogrid["N"][:, 2]
    * cp_alpha
)

assert Fz_alpha_unit_q > 0.0


# ------------------------------------------------------------
# 7. Summary table
# ------------------------------------------------------------

kinematic_summary = pd.DataFrame({
    "mode_id": np.arange(1, 5),

    "Phi_z_j_abs_max": np.max(
        np.abs(Phi_z_j),
        axis=0
    ),

    "published_dzdx_j_abs_max_1_per_in": np.max(
        np.abs(Phi_dzdx_pub_j),
        axis=0
    ),

    "SI_dzdx_j_abs_max_1_per_m": np.max(
        np.abs(Phi_dzdx_j_SI),
        axis=0
    ),
})


print("Four-mode aerodynamic kinematic operators:")
print(f"Wq_j shape:     {Wq_j.shape}")
print(f"Wqdot_j shape:  {Wqdot_j.shape}")

print(
    "\nSlope-unit conversion:"
)
print(
    f"1/in = {1.0 / IN_TO_M:.6f} 1/m"
)

print(
    "\nHarmonic/time-domain consistency error: "
    f"{harmonic_consistency_error:.3e}"
)

print(
    f"\nPanelAero +5 deg incidence sign check, "
    f"unit-q total Fz: {Fz_alpha_unit_q:.6e}"
)

print("\nModal kinematic summary:")
display(kinematic_summary)

print(
    "\nFour-mode SI normalwash operators verified."
)

In [ ]:
# Cell 26 — Four-mode generalized aerodynamic matrix
#
# Purpose:
#   Combine
#
#       modal normalwash
#           ->
#       DLM pressure coefficients
#           ->
#       panel aerodynamic forces
#           ->
#       generalized modal forces
#
# at one controlled test condition.
#
# No physical density or flutter speed is introduced yet.
# The result is expressed PER UNIT DYNAMIC PRESSURE.
#
# Structural convention:
#
#       z = Phi_z q
#
# where q is in metres.
#
# Generalized force follows from virtual work:
#
#       Q_h = Phi_z^T F_z
#
# using midpoint quadrature over each aerodynamic panel.


# ------------------------------------------------------------
# 1. Test aerodynamic condition
# ------------------------------------------------------------

Ma_qhh_test = 0.30

# PanelAero convention:
#     k_PA = omega/U   [1/m]
k_qhh_test = 2.0


# ------------------------------------------------------------
# 2. DLM pressure AIC at the test condition
# ------------------------------------------------------------

Qjj_qhh_test = DLM.calc_Qjj(
    aerogrid,
    Ma=Ma_qhh_test,
    k=k_qhh_test
)

assert Qjj_qhh_test.shape == (80, 80)
assert np.isfinite(Qjj_qhh_test).all()


# ------------------------------------------------------------
# 3. Harmonic structural -> aerodynamic normalwash operator
#
# For q_hat:
#
#   w_hat_j = W_harmonic q_hat
#
# where w_hat_j is dimensionless.
# ------------------------------------------------------------

W_harmonic_test = -(
    Phi_dzdx_j_SI
    + 1j * k_qhh_test * Phi_z_j
)

assert W_harmonic_test.shape == (80, 4)


# ------------------------------------------------------------
# 4. Panel-force -> generalized-force projection
#
# For unit dynamic pressure:
#
#   Fz_panel / q_dyn
#       = A_panel * N_z * Delta_cp
#
# Midpoint mode shapes Phi_z_k are used as the panel
# virtual-displacement values.
# ------------------------------------------------------------

panel_force_weight = (
    aerogrid["A"]
    * aerogrid["N"][:, 2]
)

assert panel_force_weight.shape == (80,)
assert np.all(panel_force_weight > 0.0)


G_panel_to_modal = (
    Phi_z_k.T
    @ np.diag(panel_force_weight)
)

assert G_panel_to_modal.shape == (4, 80)


# ------------------------------------------------------------
# 5. Generalized aerodynamic matrix per unit dynamic pressure
#
#   Q_h / q_dyn
#
#       = G_panel_to_modal
#         @ Qjj
#         @ W_harmonic
#         @ q_hat
#
# Define:
#
#       Qhh_per_qdyn
#         = G_panel_to_modal @ Qjj @ W_harmonic
#
# Units:
#       metres
#
# because:
#       q_dyn [N/m^2]
#       Qhh_per_qdyn [m]
#       q [m]
#
# gives generalized force [N].
# ------------------------------------------------------------

Qhh_per_qdyn_test = (
    G_panel_to_modal
    @ Qjj_qhh_test
    @ W_harmonic_test
)

assert Qhh_per_qdyn_test.shape == (4, 4)
assert np.isfinite(Qhh_per_qdyn_test).all()


# ------------------------------------------------------------
# 6. Direct-work verification
#
# Use one deterministic complex modal displacement vector.
#
# Route A:
#   q -> normalwash -> cp -> panel forces -> generalized forces
#
# Route B:
#   q -> Qhh directly
#
# The two routes must agree.
# ------------------------------------------------------------

q_hat_check = np.array([
    1.00e-3 + 0.00e-3j,
   -0.40e-3 + 0.20e-3j,
    0.25e-3 - 0.10e-3j,
   -0.15e-3 + 0.05e-3j,
], dtype=complex)


# Route A
wj_check = (
    W_harmonic_test
    @ q_hat_check
)

cp_check = (
    Qjj_qhh_test
    @ wj_check
)

Fz_per_qdyn_check = (
    panel_force_weight
    * cp_check
)

Qh_direct_per_qdyn = (
    Phi_z_k.T
    @ Fz_per_qdyn_check
)


# Route B
Qh_matrix_per_qdyn = (
    Qhh_per_qdyn_test
    @ q_hat_check
)


generalized_force_consistency_error = np.max(
    np.abs(
        Qh_direct_per_qdyn
        - Qh_matrix_per_qdyn
    )
)

assert np.allclose(
    Qh_direct_per_qdyn,
    Qh_matrix_per_qdyn,
    rtol=1e-12,
    atol=1e-12
)


# ------------------------------------------------------------
# 7. Separate real and imaginary parts for inspection
# ------------------------------------------------------------

Qhh_real_df = pd.DataFrame(
    np.real(Qhh_per_qdyn_test),
    index=[
        "Q1", "Q2", "Q3", "Q4"
    ],
    columns=[
        "q1", "q2", "q3", "q4"
    ]
)

Qhh_imag_df = pd.DataFrame(
    np.imag(Qhh_per_qdyn_test),
    index=[
        "Q1", "Q2", "Q3", "Q4"
    ],
    columns=[
        "q1", "q2", "q3", "q4"
    ]
)


# ------------------------------------------------------------
# 8. Diagnostic matrix measures
#
# DLM generalized matrices are NOT expected to be symmetric
# or Hermitian, so we only report useful norms.
# ------------------------------------------------------------

Qhh_real_norm = np.linalg.norm(
    np.real(Qhh_per_qdyn_test)
)

Qhh_imag_norm = np.linalg.norm(
    np.imag(Qhh_per_qdyn_test)
)

Qhh_total_norm = np.linalg.norm(
    Qhh_per_qdyn_test
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(
    f"Generalized aerodynamic test condition:"
)
print(
    f"Mach = {Ma_qhh_test:.3f}"
)
print(
    f"k_PA = omega/U = "
    f"{k_qhh_test:.4f} 1/m"
)

print(
    f"\nPanel-to-modal projection shape: "
    f"{G_panel_to_modal.shape}"
)

print(
    f"Generalized aerodynamic matrix shape: "
    f"{Qhh_per_qdyn_test.shape}"
)

print(
    "\nReal part of Qhh / q_dyn [m]:"
)
display(Qhh_real_df)

print(
    "\nImaginary part of Qhh / q_dyn [m]:"
)
display(Qhh_imag_df)

print(
    f"\nReal-part norm: "
    f"{Qhh_real_norm:.6e} m"
)

print(
    f"Imaginary-part norm: "
    f"{Qhh_imag_norm:.6e} m"
)

print(
    f"Total complex norm: "
    f"{Qhh_total_norm:.6e} m"
)

print(
    "\nDirect panel-force vs matrix "
    "generalized-force error: "
    f"{generalized_force_consistency_error:.3e}"
)

print(
    "\nFour-mode generalized aerodynamic "
    "matrix construction verified."
)

In [ ]:
# Cell 27 — DLM aerodynamic lattice convergence
#
# Compare generalized aerodynamic matrices Qhh/q_dyn
# at one fixed aerodynamic condition using progressively
# refined aerodynamic lattices.
#
# This is numerical convergence only.
# It is NOT experimental validation.

# ------------------------------------------------------------
# 1. Fixed convergence-test condition
# ------------------------------------------------------------

Ma_mesh_test = 0.30
k_mesh_test = 2.0   # omega/U [1/m]


# ------------------------------------------------------------
# 2. Meshes to compare
#
# Kept deliberately modest so this remains lightweight.
# ------------------------------------------------------------

mesh_cases = [
    (6, 4),
    (8, 6),
    (10, 8),
    (12, 10),
    (16, 12),
]


# ------------------------------------------------------------
# 3. Helper: build Qhh/q_dyn for any aerodynamic lattice
# ------------------------------------------------------------

def calculate_Qhh_for_mesh(
    n_span,
    n_chord,
    Ma,
    k_PA,
):
    """
    Construct the generalized aerodynamic matrix Qhh/q_dyn
    for a specified aerodynamic lattice.

    Returns
    -------
    result : dict
        Contains grid, transfer matrices and 4x4 Qhh matrix.
    """

    # Build aerodynamic lattice
    (
        aerogrid_local,
        aero_xyz_local,
        aero_y_local,
        aero_chords_local,
        aero_xle_local,
    ) = build_agard_aerogrid(
        n_span=n_span,
        n_chord=n_chord,
    )

    # --------------------------------------------------------
    # Structural -> aerodynamic coordinates
    # --------------------------------------------------------

    j_xy_local = np.column_stack([
        aerogrid_local["offset_j"][:, 0],
        aerogrid_local["offset_j"][:, 1],
    ])

    k_xy_local = np.column_stack([
        aerogrid_local["offset_k"][:, 0],
        aerogrid_local["offset_k"][:, 1],
    ])

    # Transfer operators
    H_j_local = build_barycentric_transfer(
        structural_triangulation,
        j_xy_local,
        n_source=121,
    )

    H_k_local = build_barycentric_transfer(
        structural_triangulation,
        k_xy_local,
        n_source=121,
    )

    # --------------------------------------------------------
    # Transfer published modal fields
    # --------------------------------------------------------

    Phi_z_j_local = (
        H_j_local @ Phi_z_4
    )

    Phi_z_k_local = (
        H_k_local @ Phi_z_4
    )

    Phi_dzdx_j_local = (
        H_j_local
        @ Phi_dzdx_pub_4
        / IN_TO_M
    )

    # --------------------------------------------------------
    # Harmonic modal normalwash matrix
    # --------------------------------------------------------

    W_local = -(
        Phi_dzdx_j_local
        + 1j * k_PA * Phi_z_j_local
    )

    # --------------------------------------------------------
    # DLM pressure AIC
    # --------------------------------------------------------

    Qjj_local = DLM.calc_Qjj(
        aerogrid_local,
        Ma=Ma,
        k=k_PA,
    )

    # --------------------------------------------------------
    # Panel force -> modal force projection
    # --------------------------------------------------------

    force_weight_local = (
        aerogrid_local["A"]
        * aerogrid_local["N"][:, 2]
    )

    G_local = (
        Phi_z_k_local.T
        @ np.diag(force_weight_local)
    )

    # --------------------------------------------------------
    # Generalized aerodynamic matrix per q_dyn
    # --------------------------------------------------------

    Qhh_local = (
        G_local
        @ Qjj_local
        @ W_local
    )

    assert Qhh_local.shape == (4, 4)
    assert np.isfinite(Qhh_local).all()

    return {
        "n_span": n_span,
        "n_chord": n_chord,
        "n_panels": aerogrid_local["n"],
        "aerogrid": aerogrid_local,
        "H_j": H_j_local,
        "H_k": H_k_local,
        "Phi_z_j": Phi_z_j_local,
        "Phi_z_k": Phi_z_k_local,
        "Phi_dzdx_j_SI": Phi_dzdx_j_local,
        "Qjj": Qjj_local,
        "Qhh_per_qdyn": Qhh_local,
    }


# ------------------------------------------------------------
# 4. Calculate all mesh cases
# ------------------------------------------------------------

mesh_convergence_results = {}

for n_span, n_chord in mesh_cases:

    print(
        f"Calculating "
        f"{n_span} x {n_chord} "
        f"({n_span * n_chord} panels)..."
    )

    result = calculate_Qhh_for_mesh(
        n_span=n_span,
        n_chord=n_chord,
        Ma=Ma_mesh_test,
        k_PA=k_mesh_test,
    )

    mesh_convergence_results[
        (n_span, n_chord)
    ] = result


# ------------------------------------------------------------
# 5. Finest mesh = numerical reference
# ------------------------------------------------------------

reference_key = mesh_cases[-1]

Qhh_mesh_reference = (
    mesh_convergence_results[
        reference_key
    ]["Qhh_per_qdyn"]
)

reference_norm = np.linalg.norm(
    Qhh_mesh_reference
)

reference_real_norm = np.linalg.norm(
    np.real(Qhh_mesh_reference)
)

reference_imag_norm = np.linalg.norm(
    np.imag(Qhh_mesh_reference)
)


# ------------------------------------------------------------
# 6. Convergence metrics
# ------------------------------------------------------------

mesh_rows = []

for key in mesh_cases:

    result = mesh_convergence_results[key]

    Q = result["Qhh_per_qdyn"]

    total_error = (
        np.linalg.norm(
            Q - Qhh_mesh_reference
        )
        / reference_norm
    )

    real_error = (
        np.linalg.norm(
            np.real(Q)
            - np.real(Qhh_mesh_reference)
        )
        / reference_real_norm
    )

    imag_error = (
        np.linalg.norm(
            np.imag(Q)
            - np.imag(Qhh_mesh_reference)
        )
        / reference_imag_norm
    )

    mesh_rows.append({
        "n_span": result["n_span"],
        "n_chord": result["n_chord"],
        "n_panels": result["n_panels"],
        "Qhh_norm_m": np.linalg.norm(Q),
        "relative_total_error_pct":
            100.0 * total_error,
        "relative_real_error_pct":
            100.0 * real_error,
        "relative_imag_error_pct":
            100.0 * imag_error,
    })


mesh_convergence_table = pd.DataFrame(
    mesh_rows
)


# ------------------------------------------------------------
# 7. Difference between current 10x8 mesh and finest mesh
# ------------------------------------------------------------

Qhh_10x8 = (
    mesh_convergence_results[
        (10, 8)
    ]["Qhh_per_qdyn"]
)

error_10x8 = (
    np.linalg.norm(
        Qhh_10x8
        - Qhh_mesh_reference
    )
    / reference_norm
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print(
    "\nDLM mesh-convergence condition:"
)
print(
    f"Mach = {Ma_mesh_test:.3f}"
)
print(
    f"k_PA = {k_mesh_test:.4f} 1/m"
)

print(
    "\nReference mesh:"
    f" {reference_key[0]} x {reference_key[1]}"
    f" = {reference_key[0] * reference_key[1]} panels"
)

print(
    "\nGeneralized aerodynamic matrix convergence:"
)

display(mesh_convergence_table)

print(
    f"\n10 x 8 relative Qhh error "
    f"versus finest mesh: "
    f"{100.0 * error_10x8:.4f}%"
)

print(
    "\nAerodynamic mesh-convergence study completed."
)

In [ ]:
# Cell 28 — Extended DLM aerodynamic-mesh convergence
#
# Add two finer grids:
#
#     20 x 16 = 320 panels
#     24 x 18 = 432 panels
#
# Existing Cell 27 results are reused.
#
# We inspect BOTH:
#   1. error relative to the finest available mesh
#   2. change from one refinement level to the next
#
# This remains a numerical-convergence study only.


# ------------------------------------------------------------
# 1. Add finer meshes
# ------------------------------------------------------------

additional_mesh_cases = [
    (20, 16),
    (24, 18),
]

for n_span, n_chord in additional_mesh_cases:

    key = (n_span, n_chord)

    if key not in mesh_convergence_results:

        print(
            f"Calculating "
            f"{n_span} x {n_chord} "
            f"({n_span * n_chord} panels)..."
        )

        mesh_convergence_results[key] = (
            calculate_Qhh_for_mesh(
                n_span=n_span,
                n_chord=n_chord,
                Ma=Ma_mesh_test,
                k_PA=k_mesh_test,
            )
        )


# ------------------------------------------------------------
# 2. Full refinement sequence
# ------------------------------------------------------------

extended_mesh_cases = [
    (6, 4),
    (8, 6),
    (10, 8),
    (12, 10),
    (16, 12),
    (20, 16),
    (24, 18),
]


# ------------------------------------------------------------
# 3. Finest available solution
# ------------------------------------------------------------

finest_key = extended_mesh_cases[-1]

Qhh_finest = (
    mesh_convergence_results[
        finest_key
    ]["Qhh_per_qdyn"]
)

finest_total_norm = np.linalg.norm(
    Qhh_finest
)

finest_real_norm = np.linalg.norm(
    np.real(Qhh_finest)
)

finest_imag_norm = np.linalg.norm(
    np.imag(Qhh_finest)
)


# ------------------------------------------------------------
# 4. Convergence metrics
# ------------------------------------------------------------

extended_rows = []

previous_Q = None
previous_key = None

for key in extended_mesh_cases:

    result = mesh_convergence_results[key]

    Q = result["Qhh_per_qdyn"]

    # ----------------------------------------
    # Difference from finest available mesh
    # ----------------------------------------

    relative_total_error = (
        np.linalg.norm(
            Q - Qhh_finest
        )
        / finest_total_norm
    )

    relative_real_error = (
        np.linalg.norm(
            np.real(Q)
            - np.real(Qhh_finest)
        )
        / finest_real_norm
    )

    relative_imag_error = (
        np.linalg.norm(
            np.imag(Q)
            - np.imag(Qhh_finest)
        )
        / finest_imag_norm
    )


    # ----------------------------------------
    # Successive refinement change
    # ----------------------------------------

    if previous_Q is None:

        successive_change = np.nan

    else:

        successive_change = (
            np.linalg.norm(
                Q - previous_Q
            )
            / np.linalg.norm(Q)
        )

    extended_rows.append({
        "n_span": key[0],
        "n_chord": key[1],
        "n_panels": result["n_panels"],

        "Qhh_norm_m":
            np.linalg.norm(Q),

        "error_vs_finest_pct":
            100.0 * relative_total_error,

        "real_error_vs_finest_pct":
            100.0 * relative_real_error,

        "imag_error_vs_finest_pct":
            100.0 * relative_imag_error,

        "successive_change_pct":
            (
                np.nan
                if np.isnan(successive_change)
                else 100.0 * successive_change
            ),
    })

    previous_Q = Q
    previous_key = key


extended_mesh_convergence = pd.DataFrame(
    extended_rows
)


# ------------------------------------------------------------
# 5. Specific comparisons of candidate production meshes
# ------------------------------------------------------------

candidate_keys = [
    (10, 8),
    (12, 10),
    (16, 12),
    (20, 16),
]

candidate_rows = []

for key in candidate_keys:

    Q_candidate = (
        mesh_convergence_results[
            key
        ]["Qhh_per_qdyn"]
    )

    candidate_error = (
        np.linalg.norm(
            Q_candidate
            - Qhh_finest
        )
        / finest_total_norm
    )

    candidate_rows.append({
        "mesh": f"{key[0]}x{key[1]}",
        "panels": key[0] * key[1],
        "error_vs_24x18_pct":
            100.0 * candidate_error,
    })


candidate_mesh_summary = pd.DataFrame(
    candidate_rows
)


# ------------------------------------------------------------
# 6. Condition numbers of DLM panel matrices
#
# This checks whether refinement is introducing a serious
# conditioning problem.
# ------------------------------------------------------------

conditioning_rows = []

for key in [
    (12, 10),
    (16, 12),
    (20, 16),
    (24, 18),
]:

    Qjj_local = (
        mesh_convergence_results[
            key
        ]["Qjj"]
    )

    conditioning_rows.append({
        "mesh":
            f"{key[0]}x{key[1]}",

        "panels":
            key[0] * key[1],

        "Qjj_condition_number":
            np.linalg.cond(Qjj_local),
    })


mesh_conditioning = pd.DataFrame(
    conditioning_rows
)


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print(
    "\nExtended DLM mesh-convergence condition:"
)
print(
    f"Mach = {Ma_mesh_test:.3f}"
)
print(
    f"k_PA = {k_mesh_test:.4f} 1/m"
)

print(
    f"\nFinest available mesh: "
    f"{finest_key[0]} x {finest_key[1]} "
    f"= {finest_key[0] * finest_key[1]} panels"
)

print(
    "\nExtended convergence table:"
)

display(
    extended_mesh_convergence
)

print(
    "\nCandidate production-mesh comparison:"
)

display(
    candidate_mesh_summary
)

print(
    "\nDLM matrix conditioning:"
)

display(
    mesh_conditioning
)

print(
    "\nExtended aerodynamic mesh-convergence "
    "study completed."
)

In [ ]:
# Cell 29 — Production DLM model and reusable generalized-aerodynamic function
#
# Production aerodynamic lattice:
#       20 span x 16 chord = 320 panels
#
# Verification lattice:
#       24 span x 18 chord = 432 panels
#
# The production choice is based on Cell 28:
#       ~0.84% difference from the 24x18 Qhh matrix
#       at Ma = 0.30, k_PA = 2.0 1/m.
#
# IMPORTANT:
# This is a numerical discretization choice, not an experimental result.


# ------------------------------------------------------------
# 1. Select production and verification meshes
# ------------------------------------------------------------

production_mesh_key = (20, 16)
verification_mesh_key = (24, 18)

production_result = (
    mesh_convergence_results[
        production_mesh_key
    ]
)

verification_result = (
    mesh_convergence_results[
        verification_mesh_key
    ]
)


# ------------------------------------------------------------
# 2. Extract production aerodynamic data
# ------------------------------------------------------------

aerogrid_prod = (
    production_result["aerogrid"]
)

H_j_prod = (
    production_result["H_j"]
)

H_k_prod = (
    production_result["H_k"]
)

Phi_z_j_prod = (
    production_result["Phi_z_j"]
)

Phi_z_k_prod = (
    production_result["Phi_z_k"]
)

Phi_dzdx_j_SI_prod = (
    production_result["Phi_dzdx_j_SI"]
)


n_aero_prod = aerogrid_prod["n"]

assert n_aero_prod == 320

assert Phi_z_j_prod.shape == (320, 4)
assert Phi_z_k_prod.shape == (320, 4)
assert Phi_dzdx_j_SI_prod.shape == (320, 4)


# ------------------------------------------------------------
# 3. Production panel-force weights
# ------------------------------------------------------------

panel_force_weight_prod = (
    aerogrid_prod["A"]
    * aerogrid_prod["N"][:, 2]
)

assert panel_force_weight_prod.shape == (320,)
assert np.all(panel_force_weight_prod > 0.0)


# ------------------------------------------------------------
# 4. Generalized-aerodynamic function
#
# Returns:
#
#       Qhh / q_dyn
#
# such that
#
#       Q_a_hat
#           = q_dyn
#             * Qhh_per_qdyn(Ma, k)
#             * q_hat
#
# PanelAero:
#
#       k_PA = omega / U    [1/m]
#
# n_modes may be:
#       2, 3, or 4
#
# which will later allow modal-convergence studies.
# ------------------------------------------------------------

def calculate_Qhh_production(
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Calculate the complex generalized aerodynamic matrix
    using the 20x16 production DLM lattice.

    Parameters
    ----------
    Ma : float
        Mach number.

    k_PA : float
        PanelAero reduced-frequency parameter:
            k_PA = omega / U   [1/m]

    n_modes : int
        Number of structural modes retained.
        Allowed values: 2, 3, 4.

    Returns
    -------
    Qhh : complex ndarray, shape (n_modes, n_modes)
        Generalized aerodynamic matrix per unit dynamic pressure.
        Units: metres.
    """

    if n_modes not in (2, 3, 4):
        raise ValueError(
            "n_modes must be 2, 3, or 4."
        )

    if Ma < 0.0:
        raise ValueError(
            "Mach number must be non-negative."
        )

    if k_PA < 0.0:
        raise ValueError(
            "k_PA must be non-negative."
        )


    # --------------------------------------------------------
    # DLM pressure AIC
    # --------------------------------------------------------

    Qjj = DLM.calc_Qjj(
        aerogrid_prod,
        Ma=Ma,
        k=k_PA,
    )


    # --------------------------------------------------------
    # Retained modal fields
    # --------------------------------------------------------

    Phi_z_j_n = (
        Phi_z_j_prod[:, :n_modes]
    )

    Phi_z_k_n = (
        Phi_z_k_prod[:, :n_modes]
    )

    Phi_dzdx_j_n = (
        Phi_dzdx_j_SI_prod[:, :n_modes]
    )


    # --------------------------------------------------------
    # Harmonic modal normalwash
    #
    #   w_j =
    #       -(Phi_x + i*k_PA*Phi_z) q
    # --------------------------------------------------------

    W_n = -(
        Phi_dzdx_j_n
        + 1j * k_PA * Phi_z_j_n
    )


    # --------------------------------------------------------
    # Panel force -> modal generalized force
    # --------------------------------------------------------

    G_n = (
        Phi_z_k_n.T
        @ np.diag(
            panel_force_weight_prod
        )
    )


    # --------------------------------------------------------
    # Generalized aerodynamic matrix
    # --------------------------------------------------------

    Qhh = (
        G_n
        @ Qjj
        @ W_n
    )

    assert Qhh.shape == (
        n_modes,
        n_modes
    )

    assert np.isfinite(Qhh).all()

    return Qhh


# ------------------------------------------------------------
# 5. Structural-matrix helper for modal truncation
# ------------------------------------------------------------

def get_structural_modal_matrices(
    n_modes=4
):
    """
    Return truncated structural modal matrices M, C, K.
    """

    if n_modes not in (2, 3, 4):
        raise ValueError(
            "n_modes must be 2, 3, or 4."
        )

    return (
        M_modal_SI[:n_modes, :n_modes].copy(),
        C_modal_SI[:n_modes, :n_modes].copy(),
        K_modal_SI[:n_modes, :n_modes].copy(),
    )


# ------------------------------------------------------------
# 6. Reproduce Cell 28 production-mesh result
# ------------------------------------------------------------

Qhh_prod_check = (
    calculate_Qhh_production(
        Ma=0.30,
        k_PA=2.0,
        n_modes=4,
    )
)

Qhh_prod_stored = (
    production_result[
        "Qhh_per_qdyn"
    ]
)

production_reproduction_error = (
    np.linalg.norm(
        Qhh_prod_check
        - Qhh_prod_stored
    )
    /
    np.linalg.norm(
        Qhh_prod_stored
    )
)

assert np.allclose(
    Qhh_prod_check,
    Qhh_prod_stored,
    rtol=1e-12,
    atol=1e-12,
)


# ------------------------------------------------------------
# 7. Verify modal truncation dimensions
# ------------------------------------------------------------

modal_dimension_rows = []

for n_modes in (2, 3, 4):

    Qhh_n = (
        calculate_Qhh_production(
            Ma=0.30,
            k_PA=2.0,
            n_modes=n_modes,
        )
    )

    M_n, C_n, K_n = (
        get_structural_modal_matrices(
            n_modes
        )
    )

    assert Qhh_n.shape == (
        n_modes,
        n_modes
    )

    assert M_n.shape == (
        n_modes,
        n_modes
    )

    assert C_n.shape == (
        n_modes,
        n_modes
    )

    assert K_n.shape == (
        n_modes,
        n_modes
    )

    modal_dimension_rows.append({
        "retained_modes":
            n_modes,

        "Qhh_shape":
            str(Qhh_n.shape),

        "structural_matrix_shape":
            str(M_n.shape),

        "Qhh_norm_m":
            np.linalg.norm(Qhh_n),
    })


modal_dimension_check = pd.DataFrame(
    modal_dimension_rows
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print(
    "Production aerodynamic lattice:"
)
print(
    f"{production_mesh_key[0]} x "
    f"{production_mesh_key[1]} "
    f"= {n_aero_prod} panels"
)

print(
    "\nVerification aerodynamic lattice:"
)
print(
    f"{verification_mesh_key[0]} x "
    f"{verification_mesh_key[1]} "
    f"= "
    f"{verification_mesh_key[0] * verification_mesh_key[1]} "
    f"panels"
)

print(
    "\nProduction-function reproduction error: "
    f"{production_reproduction_error:.3e}"
)

print(
    "\nModal truncation interface check:"
)

display(
    modal_dimension_check
)

print(
    "\nProduction DLM generalized-aerodynamic "
    "model ready."
)

In [ ]:
# Cell 30 — p-k iteration smoke test
#
# Purpose:
#   Verify that the aeroelastic nonlinear frequency-consistency
#   iteration works before performing any flutter-speed sweep.
#
# Governing frozen-k system:
#
#   M qddot + C qdot + (K - q_dyn*Qhh) q = 0
#
# where:
#
#   Qhh = Qhh(Mach, k_PA)
#   k_PA = omega / U
#
# This cell uses an artificial numerical condition only.
# It is NOT an AGARD validation point.


# ------------------------------------------------------------
# 1. Frozen-k aeroelastic poles
# ------------------------------------------------------------

def frozen_k_aeroelastic_poles(
    U,
    q_dyn,
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Solve the complex aeroelastic eigenproblem with the
    aerodynamic matrix frozen at a specified k_PA.

    Parameters
    ----------
    U : float
        Freestream velocity [m/s]

    q_dyn : float
        Dynamic pressure [Pa]

    Ma : float
        Mach number

    k_PA : float
        PanelAero reduced-frequency parameter
        omega/U [1/m]

    n_modes : int
        Number of retained structural modes: 2, 3, or 4

    Returns
    -------
    poles : complex ndarray
        Aeroelastic poles p = sigma + i*omega [1/s]

    eigvecs : complex ndarray
        Corresponding first-order state eigenvectors.
    """

    if U <= 0.0:
        raise ValueError("U must be positive.")

    if q_dyn < 0.0:
        raise ValueError("q_dyn must be non-negative.")

    M, C, K = get_structural_modal_matrices(
        n_modes=n_modes
    )

    Qhh = calculate_Qhh_production(
        Ma=Ma,
        k_PA=k_PA,
        n_modes=n_modes,
    )

    # Effective complex stiffness
    K_eff = (
        K
        - q_dyn * Qhh
    )

    M_inv = np.linalg.inv(M)

    Z = np.zeros(
        (n_modes, n_modes),
        dtype=complex
    )

    I = np.eye(
        n_modes,
        dtype=complex
    )

    # First-order state:
    #
    #   x = [q, qdot]^T
    #
    #   xdot = A x
    #
    A = np.block([
        [Z, I],
        [
            -M_inv @ K_eff,
            -M_inv @ C
        ],
    ])

    poles, eigvecs = np.linalg.eig(A)

    return poles, eigvecs


# ------------------------------------------------------------
# 2. Single-mode p-k consistency iteration
# ------------------------------------------------------------

def pk_iterate_mode(
    mode_index,
    U,
    q_dyn,
    Ma,
    n_modes=4,
    tol=1e-8,
    max_iter=60,
    relaxation=0.6,
):
    """
    Iterate one aeroelastic mode until:

        k_PA = imag(p) / U

    Parameters
    ----------
    mode_index : int
        Zero-based structural mode index.

    Returns
    -------
    result : dict
        Converged pole and iteration information.
    """

    if not 0 <= mode_index < n_modes:
        raise ValueError(
            "mode_index must be smaller than n_modes."
        )

    # Initial guess from dry structural frequency
    omega_target = (
        omega_analysis[mode_index]
    )

    k_current = (
        omega_target / U
    )

    pole_previous = complex(
        -zeta_modal[mode_index] * omega_target,
        omega_target
    )

    history = []

    converged = False

    for iteration in range(
        1,
        max_iter + 1
    ):

        poles, eigvecs = (
            frozen_k_aeroelastic_poles(
                U=U,
                q_dyn=q_dyn,
                Ma=Ma,
                k_PA=k_current,
                n_modes=n_modes,
            )
        )

        # Use only positive-frequency branches.
        candidates = poles[
            np.imag(poles) > 0.0
        ]

        if candidates.size == 0:
            raise RuntimeError(
                "No positive-frequency aeroelastic "
                "poles were found."
            )

        # ----------------------------------------------------
        # Mode tracking for this smoke test:
        #
        # Select the pole closest to the previous pole in the
        # complex plane.
        #
        # Later, for the full flutter sweep, we will improve
        # this with eigenvector/MAC-based branch tracking.
        # ----------------------------------------------------

        selected_pole = candidates[
            np.argmin(
                np.abs(
                    candidates
                    - pole_previous
                )
            )
        ]

        omega_new = np.imag(
            selected_pole
        )

        k_raw = (
            omega_new / U
        )

        # Relax the reduced-frequency update
        k_next = (
            relaxation * k_raw
            + (1.0 - relaxation) * k_current
        )

        relative_k_change = (
            abs(k_next - k_current)
            /
            max(
                abs(k_current),
                1e-14
            )
        )

        history.append({
            "iteration": iteration,
            "k_current_1_per_m":
                k_current,
            "sigma_1_per_s":
                np.real(selected_pole),
            "omega_rad_per_s":
                omega_new,
            "frequency_hz":
                omega_new / (2.0 * np.pi),
            "k_raw_1_per_m":
                k_raw,
            "relative_k_change":
                relative_k_change,
        })

        pole_previous = selected_pole
        k_current = k_next

        if relative_k_change < tol:
            converged = True
            break


    # --------------------------------------------------------
    # Final damping metric
    #
    # For pole:
    #
    #   p = sigma + i omega
    #
    # modal damping ratio:
    #
    #   zeta = -sigma / |p|
    #
    # Positive zeta -> stable
    # Zero          -> flutter boundary
    # Negative      -> unstable
    # --------------------------------------------------------

    sigma_final = np.real(
        selected_pole
    )

    omega_final = np.imag(
        selected_pole
    )

    zeta_final = (
        -sigma_final
        / abs(selected_pole)
    )

    return {
        "mode_index": mode_index,
        "converged": converged,
        "iterations": iteration,
        "pole": selected_pole,
        "sigma_1_per_s": sigma_final,
        "omega_rad_per_s": omega_final,
        "frequency_hz":
            omega_final / (2.0 * np.pi),
        "k_PA_1_per_m":
            omega_final / U,
        "zeta":
            zeta_final,
        "history":
            pd.DataFrame(history),
    }


# ------------------------------------------------------------
# 3. Controlled numerical test condition
#
# IMPORTANT:
# These are NOT experimental values.
# ------------------------------------------------------------

U_pk_test = 100.0       # m/s
q_dyn_pk_test = 250.0   # Pa
Ma_pk_test = 0.30

n_modes_pk_test = 4


# ------------------------------------------------------------
# 4. Iterate all four aeroelastic branches
# ------------------------------------------------------------

pk_smoke_results = []

for mode_index in range(
    n_modes_pk_test
):

    result = pk_iterate_mode(
        mode_index=mode_index,
        U=U_pk_test,
        q_dyn=q_dyn_pk_test,
        Ma=Ma_pk_test,
        n_modes=n_modes_pk_test,
    )

    assert result["converged"]

    pk_smoke_results.append(
        result
    )


# ------------------------------------------------------------
# 5. Summary table
# ------------------------------------------------------------

pk_smoke_table = pd.DataFrame({
    "mode_id": [
        r["mode_index"] + 1
        for r in pk_smoke_results
    ],

    "dry_frequency_hz":
        analysis_frequencies_hz,

    "aeroelastic_frequency_hz": [
        r["frequency_hz"]
        for r in pk_smoke_results
    ],

    "sigma_1_per_s": [
        r["sigma_1_per_s"]
        for r in pk_smoke_results
    ],

    "damping_ratio": [
        r["zeta"]
        for r in pk_smoke_results
    ],

    "k_PA_1_per_m": [
        r["k_PA_1_per_m"]
        for r in pk_smoke_results
    ],

    "iterations": [
        r["iterations"]
        for r in pk_smoke_results
    ],
})


# ------------------------------------------------------------
# 6. Final consistency checks
# ------------------------------------------------------------

for result in pk_smoke_results:

    k_consistency_error = abs(
        result["k_PA_1_per_m"]
        - result["omega_rad_per_s"]
          / U_pk_test
    )

    assert k_consistency_error < 1e-12

    assert np.isfinite(
        result["sigma_1_per_s"]
    )

    assert np.isfinite(
        result["frequency_hz"]
    )


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print(
    "p-k numerical smoke-test condition:"
)

print(
    f"Mach number:       {Ma_pk_test:.3f}"
)

print(
    f"Freestream speed:  {U_pk_test:.3f} m/s"
)

print(
    f"Dynamic pressure:  {q_dyn_pk_test:.3f} Pa"
)

print(
    "\nNOTE: this is an artificial numerical "
    "condition, not an AGARD test point."
)

print(
    "\nConverged aeroelastic branches:"
)

display(
    pk_smoke_table
)

print(
    "\nIteration history for Mode 1:"
)

display(
    pk_smoke_results[0]["history"]
)

print(
    "\np-k reduced-frequency iteration "
    "smoke test passed."
)

In [ ]:
# Cell 31 — MAC-based p-k aeroelastic branch tracking
#
# Purpose:
#   Replace nearest-pole tracking with complex mode-shape tracking.
#
# For two complex modal vectors a and b:
#
#                    |a^H M b|^2
#   MAC(a,b) = ----------------------------
#              (a^H M a)(b^H M b)
#
# MAC -> 1 : highly correlated modal content
# MAC -> 0 : weak/orthogonal modal content
#
# This is important near bending-torsion interaction, where
# frequency ordering alone can swap branches.


# ------------------------------------------------------------
# 1. Mass-normalize a complex modal displacement vector
# ------------------------------------------------------------

def mass_normalize_modal_vector(v, M):
    """
    Mass-normalize a complex modal displacement vector.
    """

    v = np.asarray(v, dtype=complex)

    norm_sq = np.real(
        np.vdot(v, M @ v)
    )

    if norm_sq <= 0.0:
        raise ValueError(
            "Modal vector has non-positive mass norm."
        )

    return v / np.sqrt(norm_sq)


# ------------------------------------------------------------
# 2. Complex mass-weighted MAC
# ------------------------------------------------------------

def complex_modal_mac(
    reference,
    candidate,
    M,
):
    """
    Complex mass-weighted Modal Assurance Criterion.
    """

    reference = np.asarray(
        reference,
        dtype=complex
    )

    candidate = np.asarray(
        candidate,
        dtype=complex
    )

    numerator = abs(
        np.vdot(
            reference,
            M @ candidate
        )
    )**2

    denominator = (
        np.real(
            np.vdot(
                reference,
                M @ reference
            )
        )
        *
        np.real(
            np.vdot(
                candidate,
                M @ candidate
            )
        )
    )

    if denominator <= 0.0:
        return 0.0

    return float(
        np.real(numerator / denominator)
    )


# ------------------------------------------------------------
# 3. Phase-align candidate vector to reference
#
# Eigenvector phase is arbitrary.
# Phase alignment makes continuation easier to inspect.
# ------------------------------------------------------------

def phase_align_modal_vector(
    candidate,
    reference,
    M,
):
    """
    Rotate candidate complex phase so that its mass-weighted
    correlation with reference is real and positive.
    """

    correlation = np.vdot(
        reference,
        M @ candidate
    )

    if abs(correlation) == 0.0:
        return candidate

    phase = np.angle(
        correlation
    )

    return (
        candidate
        * np.exp(-1j * phase)
    )


# ------------------------------------------------------------
# 4. MAC-based p-k iteration
# ------------------------------------------------------------

def pk_iterate_mode_mac(
    mode_index,
    U,
    q_dyn,
    Ma,
    n_modes=4,
    tol=1e-8,
    max_iter=60,
    relaxation=0.6,
    initial_k=None,
    reference_q=None,
):
    """
    Iterate one aeroelastic branch using modal MAC for
    branch selection.

    Optional inputs initial_k and reference_q allow continuation
    from a previously converged flight condition.
    """

    if not 0 <= mode_index < n_modes:
        raise ValueError(
            "mode_index must be smaller than n_modes."
        )

    M, C, K = (
        get_structural_modal_matrices(
            n_modes=n_modes
        )
    )


    # --------------------------------------------------------
    # Initial reduced-frequency guess
    # --------------------------------------------------------

    if initial_k is None:

        k_current = (
            omega_analysis[mode_index]
            / U
        )

    else:

        k_current = float(initial_k)


    # --------------------------------------------------------
    # Initial modal reference
    #
    # If no previous aeroelastic eigenvector is available,
    # begin from the corresponding dry structural modal basis
    # vector e_i.
    # --------------------------------------------------------

    if reference_q is None:

        reference_q_current = np.zeros(
            n_modes,
            dtype=complex
        )

        reference_q_current[
            mode_index
        ] = 1.0

    else:

        reference_q_current = np.asarray(
            reference_q,
            dtype=complex
        ).copy()

        if reference_q_current.shape != (
            n_modes,
        ):
            raise ValueError(
                "reference_q has incorrect shape."
            )


    reference_q_current = (
        mass_normalize_modal_vector(
            reference_q_current,
            M
        )
    )


    history = []

    converged = False

    selected_pole = None
    selected_q = None
    selected_mac = None


    # --------------------------------------------------------
    # p-k iteration
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):

        poles, eigvecs = (
            frozen_k_aeroelastic_poles(
                U=U,
                q_dyn=q_dyn,
                Ma=Ma,
                k_PA=k_current,
                n_modes=n_modes,
            )
        )


        # Positive-frequency branches only
        candidate_indices = np.where(
            np.imag(poles) > 0.0
        )[0]

        if candidate_indices.size == 0:
            raise RuntimeError(
                "No positive-frequency aeroelastic "
                "poles were found."
            )


        # ----------------------------------------------------
        # Evaluate modal MAC for every candidate
        #
        # For state vector:
        #
        #       x = [q, qdot]
        #
        # only the displacement part q is used for MAC.
        # ----------------------------------------------------

        candidate_macs = []

        candidate_q_vectors = []

        for idx in candidate_indices:

            q_candidate = (
                eigvecs[
                    :n_modes,
                    idx
                ]
            )

            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M
                )
            )

            mac_value = complex_modal_mac(
                reference_q_current,
                q_candidate,
                M
            )

            candidate_q_vectors.append(
                q_candidate
            )

            candidate_macs.append(
                mac_value
            )


        candidate_macs = np.asarray(
            candidate_macs,
            dtype=float
        )


        # Highest-MAC candidate defines the tracked branch
        local_best = int(
            np.argmax(candidate_macs)
        )

        selected_index = (
            candidate_indices[
                local_best
            ]
        )

        selected_pole = (
            poles[selected_index]
        )

        selected_mac = (
            candidate_macs[
                local_best
            ]
        )

        selected_q = (
            candidate_q_vectors[
                local_best
            ]
        )


        # ----------------------------------------------------
        # Align arbitrary complex eigenvector phase
        # ----------------------------------------------------

        selected_q = (
            phase_align_modal_vector(
                selected_q,
                reference_q_current,
                M
            )
        )

        selected_q = (
            mass_normalize_modal_vector(
                selected_q,
                M
            )
        )


        # ----------------------------------------------------
        # Reduced-frequency consistency update
        # ----------------------------------------------------

        omega_new = np.imag(
            selected_pole
        )

        k_raw = (
            omega_new / U
        )

        k_next = (
            relaxation * k_raw
            + (1.0 - relaxation)
            * k_current
        )

        relative_k_change = (
            abs(k_next - k_current)
            /
            max(
                abs(k_current),
                1e-14
            )
        )


        history.append({
            "iteration":
                iteration,

            "k_current_1_per_m":
                k_current,

            "sigma_1_per_s":
                np.real(
                    selected_pole
                ),

            "frequency_hz":
                omega_new
                / (2.0 * np.pi),

            "selected_MAC":
                selected_mac,

            "k_raw_1_per_m":
                k_raw,

            "relative_k_change":
                relative_k_change,
        })


        # Update modal reference continuously
        reference_q_current = (
            selected_q.copy()
        )

        k_current = k_next


        if relative_k_change < tol:

            converged = True
            break


    # --------------------------------------------------------
    # Final damping
    # --------------------------------------------------------

    sigma_final = np.real(
        selected_pole
    )

    omega_final = np.imag(
        selected_pole
    )

    zeta_final = (
        -sigma_final
        / abs(selected_pole)
    )


    return {
        "mode_index":
            mode_index,

        "converged":
            converged,

        "iterations":
            iteration,

        "pole":
            selected_pole,

        "sigma_1_per_s":
            sigma_final,

        "omega_rad_per_s":
            omega_final,

        "frequency_hz":
            omega_final
            / (2.0 * np.pi),

        "k_PA_1_per_m":
            omega_final / U,

        "zeta":
            zeta_final,

        "selected_MAC":
            selected_mac,

        "modal_vector":
            selected_q.copy(),

        "history":
            pd.DataFrame(
                history
            ),
    }


# ------------------------------------------------------------
# 5. Repeat the Cell 30 artificial condition
# ------------------------------------------------------------

pk_mac_results = []

for mode_index in range(4):

    result = pk_iterate_mode_mac(
        mode_index=mode_index,
        U=U_pk_test,
        q_dyn=q_dyn_pk_test,
        Ma=Ma_pk_test,
        n_modes=4,
    )

    assert result["converged"]

    pk_mac_results.append(
        result
    )


# ------------------------------------------------------------
# 6. Compare against previous nearest-pole result
# ------------------------------------------------------------

comparison_rows = []

for old, new in zip(
    pk_smoke_results,
    pk_mac_results,
):

    comparison_rows.append({
        "mode_id":
            new["mode_index"] + 1,

        "nearest_pole_frequency_hz":
            old["frequency_hz"],

        "MAC_frequency_hz":
            new["frequency_hz"],

        "frequency_difference_hz":
            new["frequency_hz"]
            - old["frequency_hz"],

        "nearest_pole_sigma":
            old["sigma_1_per_s"],

        "MAC_sigma":
            new["sigma_1_per_s"],

        "sigma_difference":
            new["sigma_1_per_s"]
            - old["sigma_1_per_s"],

        "final_MAC":
            new["selected_MAC"],

        "iterations":
            new["iterations"],
    })


pk_tracking_comparison = pd.DataFrame(
    comparison_rows
)


# ------------------------------------------------------------
# 7. Sanity checks
# ------------------------------------------------------------

for result in pk_mac_results:

    assert (
        0.0
        <= result["selected_MAC"]
        <= 1.0 + 1e-12
    )

    assert np.isfinite(
        result["frequency_hz"]
    )

    assert np.isfinite(
        result["sigma_1_per_s"]
    )


# At this benign condition both tracking methods should recover
# effectively the same physical branches.
assert np.allclose(
    [
        r["frequency_hz"]
        for r in pk_mac_results
    ],
    [
        r["frequency_hz"]
        for r in pk_smoke_results
    ],
    rtol=1e-7,
    atol=1e-7,
)

assert np.allclose(
    [
        r["sigma_1_per_s"]
        for r in pk_mac_results
    ],
    [
        r["sigma_1_per_s"]
        for r in pk_smoke_results
    ],
    rtol=1e-7,
    atol=1e-7,
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print(
    "MAC-based branch-tracking verification:"
)

display(
    pk_tracking_comparison
)

print(
    "\nMode 1 MAC-based p-k iteration history:"
)

display(
    pk_mac_results[0][
        "history"
    ]
)

print(
    "\nMAC-based p-k branch tracker verified."
)

In [ ]:
# Cell 32 — MAC-based dynamic-pressure continuation sweep
#
# Purpose:
#   Verify robust continuation of all four aeroelastic branches
#   as aerodynamic loading increases.
#
# Fixed numerical conditions:
#
#       Mach = 0.30
#       U    = 100 m/s
#
# Dynamic pressure is swept independently.
#
# IMPORTANT:
# This is a numerical continuation test.
# It is NOT an AGARD experimental flutter boundary.


# ------------------------------------------------------------
# 1. Controlled sweep conditions
# ------------------------------------------------------------

Ma_continuation = 0.30
U_continuation = 100.0   # m/s

q_dyn_values = np.array([
       0.0,
     250.0,
     500.0,
     750.0,
    1000.0,
    1250.0,
    1500.0,
], dtype=float)

n_modes_continuation = 4


# ------------------------------------------------------------
# 2. Storage for branch continuation
# ------------------------------------------------------------

continuation_results = {
    mode_id: []
    for mode_id in range(1, 5)
}

# Previous converged modal vector and reduced frequency
# for each tracked branch.
previous_modal_vectors = {
    mode_id: None
    for mode_id in range(1, 5)
}

previous_k_values = {
    mode_id: None
    for mode_id in range(1, 5)
}


# ------------------------------------------------------------
# 3. Sweep dynamic pressure
# ------------------------------------------------------------

for q_dyn in q_dyn_values:

    print(
        f"Solving q_dyn = "
        f"{q_dyn:.1f} Pa ..."
    )

    for mode_index in range(
        n_modes_continuation
    ):

        mode_id = mode_index + 1

        previous_q = (
            previous_modal_vectors[
                mode_id
            ]
        )

        previous_k = (
            previous_k_values[
                mode_id
            ]
        )


        # ----------------------------------------------------
        # Solve branch using continuation from previous point
        # ----------------------------------------------------

        result = pk_iterate_mode_mac(
            mode_index=mode_index,
            U=U_continuation,
            q_dyn=q_dyn,
            Ma=Ma_continuation,
            n_modes=n_modes_continuation,
            initial_k=previous_k,
            reference_q=previous_q,
        )

        if not result["converged"]:
            raise RuntimeError(
                f"Mode {mode_id} failed to converge "
                f"at q_dyn = {q_dyn:.1f} Pa."
            )


        # ----------------------------------------------------
        # Continuation MAC:
        #
        # Compare the newly converged eigenvector with the
        # converged eigenvector from the previous loading point.
        #
        # At the first point, compare against the dry
        # structural basis vector.
        # ----------------------------------------------------

        M4, _, _ = (
            get_structural_modal_matrices(
                n_modes=4
            )
        )

        if previous_q is None:

            reference_vector = np.zeros(
                4,
                dtype=complex
            )

            reference_vector[
                mode_index
            ] = 1.0

        else:

            reference_vector = previous_q


        continuation_mac = (
            complex_modal_mac(
                reference_vector,
                result["modal_vector"],
                M4,
            )
        )


        # ----------------------------------------------------
        # Save result
        # ----------------------------------------------------

        continuation_results[
            mode_id
        ].append({
            "q_dyn_Pa":
                q_dyn,

            "sigma_1_per_s":
                result[
                    "sigma_1_per_s"
                ],

            "frequency_hz":
                result[
                    "frequency_hz"
                ],

            "damping_ratio":
                result[
                    "zeta"
                ],

            "k_PA_1_per_m":
                result[
                    "k_PA_1_per_m"
                ],

            "continuation_MAC":
                continuation_mac,

            "pk_iterations":
                result[
                    "iterations"
                ],
        })


        # ----------------------------------------------------
        # Update continuation state
        # ----------------------------------------------------

        previous_modal_vectors[
            mode_id
        ] = (
            result[
                "modal_vector"
            ].copy()
        )

        previous_k_values[
            mode_id
        ] = (
            result[
                "k_PA_1_per_m"
            ]
        )


# ------------------------------------------------------------
# 4. Combine into one engineering table
# ------------------------------------------------------------

continuation_rows = []

for mode_id in range(1, 5):

    for row in continuation_results[
        mode_id
    ]:

        continuation_rows.append({
            "mode_id":
                mode_id,

            **row,
        })


continuation_table = pd.DataFrame(
    continuation_rows
)

continuation_table = (
    continuation_table
    .sort_values(
        [
            "q_dyn_Pa",
            "mode_id",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Dry-structure verification at q_dyn = 0
# ------------------------------------------------------------

dry_rows = (
    continuation_table[
        continuation_table[
            "q_dyn_Pa"
        ] == 0.0
    ]
    .sort_values(
        "mode_id"
    )
)

assert np.allclose(
    dry_rows[
        "frequency_hz"
    ].to_numpy(),
    analysis_frequencies_hz,
    rtol=1e-4,
    atol=1e-4,
)

assert np.allclose(
    dry_rows[
        "damping_ratio"
    ].to_numpy(),
    zeta_modal,
    rtol=1e-5,
    atol=1e-5,
)


# ------------------------------------------------------------
# 6. Check continuation quality
# ------------------------------------------------------------

minimum_continuation_mac = (
    continuation_table[
        "continuation_MAC"
    ].min()
)

maximum_iterations = (
    continuation_table[
        "pk_iterations"
    ].max()
)


# ------------------------------------------------------------
# 7. Detect sign changes in sigma
#
# A change from sigma < 0 to sigma > 0 would indicate
# that the sweep has bracketed an instability boundary.
#
# We only REPORT this here.
# No interpolation to a flutter point yet.
# ------------------------------------------------------------

crossing_rows = []

for mode_id in range(1, 5):

    mode_data = (
        continuation_table[
            continuation_table[
                "mode_id"
            ] == mode_id
        ]
        .sort_values(
            "q_dyn_Pa"
        )
        .reset_index(drop=True)
    )

    sigma = (
        mode_data[
            "sigma_1_per_s"
        ].to_numpy()
    )

    qvals = (
        mode_data[
            "q_dyn_Pa"
        ].to_numpy()
    )

    for i in range(
        len(mode_data) - 1
    ):

        if (
            sigma[i] < 0.0
            and sigma[i + 1] >= 0.0
        ):

            crossing_rows.append({
                "mode_id":
                    mode_id,

                "q_lower_Pa":
                    qvals[i],

                "q_upper_Pa":
                    qvals[i + 1],

                "sigma_lower":
                    sigma[i],

                "sigma_upper":
                    sigma[i + 1],
            })


instability_brackets = pd.DataFrame(
    crossing_rows
)


# ------------------------------------------------------------
# 8. Plot real part of aeroelastic poles
#
# sigma < 0 : stable
# sigma = 0 : neutral stability
# sigma > 0 : unstable
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

for mode_id in range(1, 5):

    mode_data = (
        continuation_table[
            continuation_table[
                "mode_id"
            ] == mode_id
        ]
    )

    plt.plot(
        mode_data[
            "q_dyn_Pa"
        ],
        mode_data[
            "sigma_1_per_s"
        ],
        "o-",
        label=f"Mode {mode_id}",
    )


plt.axhline(
    0.0,
    color="black",
    linewidth=1.0,
)

plt.xlabel(
    "Dynamic pressure q∞ (Pa)"
)

plt.ylabel(
    "Pole real part σ (1/s)"
)

plt.title(
    "Aeroelastic branch continuation — artificial test"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. Compact reporting table
# ------------------------------------------------------------

print(
    "\nDynamic-pressure continuation test:"
)

print(
    f"Mach = {Ma_continuation:.3f}"
)

print(
    f"U = {U_continuation:.2f} m/s"
)

print(
    "\nNOTE: this is an artificial numerical sweep, "
    "not an AGARD experimental boundary."
)

print(
    "\nContinuation results:"
)

display(
    continuation_table[
        [
            "q_dyn_Pa",
            "mode_id",
            "sigma_1_per_s",
            "frequency_hz",
            "damping_ratio",
            "k_PA_1_per_m",
            "continuation_MAC",
            "pk_iterations",
        ]
    ]
)

print(
    f"\nMinimum continuation MAC: "
    f"{minimum_continuation_mac:.6f}"
)

print(
    f"Maximum p-k iterations: "
    f"{maximum_iterations}"
)


if len(
    instability_brackets
) == 0:

    print(
        "\nNo stable-to-unstable sigma crossing "
        "was bracketed in this pressure range."
    )

else:

    print(
        "\nPotential instability bracket(s):"
    )

    display(
        instability_brackets
    )


print(
    "\nMAC-based dynamic-pressure "
    "continuation sweep completed."
)

In [ ]:
# Cell 33 — Published AGARD 445.6 weakened Model 3 flutter data
#
# Source:
# NASA TM-100492, Appendix Table II
# "FLUTTER DATA MEASURED IN AIR"
#
# Configuration:
#   Panel span: 2.500 ft
#   Mounting:   Wall
#   Structure:  Weakened
#   Model:      3
#
# IMPORTANT:
# These are published experimental test points.
# They are NOT generated or fitted values.

# ------------------------------------------------------------
# 1. Raw published Table II values
# ------------------------------------------------------------

agard_model3_air = pd.DataFrame({
    "Mach": [
        0.901,
        0.678,
        0.499,
        0.954,
        0.960,
        0.957,
        1.072,
        1.141,
    ],

    "rho_slug_ft3": [
        0.000193,
        0.000404,
        0.000830,
        0.000123,
        0.000123,
        0.000123,
        0.000107,
        0.000152,
    ],

    "mass_ratio_mu": [
        143.920,
        68.753,
        33.465,
        225.820,
        225.820,
        225.820,
        259.590,
        182.740,
    ],

    # Published torsional reference natural frequency
    "omega_alpha_rad_s": [
        239.3,
        239.3,
        239.3,
        239.3,
        239.3,
        239.3,
        239.3,
        239.3,
    ],

    # Published flutter angular frequency
    "omega_flutter_rad_s": [
        101.1,
        113.0,
        128.1,
        91.1,
        87.3,
        87.9,
        86.7,
        109.9,
    ],

    "V_ft_s": [
        973.4,
        759.1,
        565.8,
        1008.4,
        1013.8,
        1020.2,
        1131.0,
        1195.3,
    ],

    "q_lb_ft2": [
        89.3,
        115.7,
        133.1,
        60.6,
        61.3,
        61.7,
        66.1,
        105.3,
    ],

    # Published nondimensional flutter-speed index
    "flutter_speed_index": [
        0.3700,
        0.4174,
        0.4459,
        0.3059,
        0.3076,
        0.3095,
        0.3201,
        0.4031,
    ],
})


# ------------------------------------------------------------
# 2. Unit conversions
# ------------------------------------------------------------

FT_TO_M = 0.3048

SLUG_TO_KG = 14.5939029372

SLUG_FT3_TO_KG_M3 = (
    SLUG_TO_KG
    / FT_TO_M**3
)

PSF_TO_PA = (
    LBF_TO_N
    / FT_TO_M**2
)


agard_model3_air["rho_kg_m3"] = (
    agard_model3_air["rho_slug_ft3"]
    * SLUG_FT3_TO_KG_M3
)

agard_model3_air["V_m_s"] = (
    agard_model3_air["V_ft_s"]
    * FT_TO_M
)

agard_model3_air["q_Pa"] = (
    agard_model3_air["q_lb_ft2"]
    * PSF_TO_PA
)

agard_model3_air["flutter_frequency_Hz"] = (
    agard_model3_air["omega_flutter_rad_s"]
    / (2.0 * np.pi)
)

agard_model3_air["alpha_frequency_Hz"] = (
    agard_model3_air["omega_alpha_rad_s"]
    / (2.0 * np.pi)
)


# ------------------------------------------------------------
# 3. Independent q = 1/2 rho V^2 consistency diagnostic
#
# This is ONLY a check.
#
# For the actual benchmark calculations we will use the
# PUBLISHED q value directly rather than replacing it with
# a recomputed value.
# ------------------------------------------------------------

agard_model3_air["q_from_rhoV2_Pa"] = (
    0.5
    * agard_model3_air["rho_kg_m3"]
    * agard_model3_air["V_m_s"]**2
)

agard_model3_air["q_consistency_difference_pct"] = (
    100.0
    * (
        agard_model3_air["q_from_rhoV2_Pa"]
        - agard_model3_air["q_Pa"]
    )
    / agard_model3_air["q_Pa"]
)


# ------------------------------------------------------------
# 4. Flag points compatible with current subsonic DLM model
# ------------------------------------------------------------

agard_model3_air["subsonic_DLM_candidate"] = (
    agard_model3_air["Mach"] < 1.0
)


# ------------------------------------------------------------
# 5. Sort by Mach for later benchmark plots
# ------------------------------------------------------------

agard_model3_air = (
    agard_model3_air
    .sort_values("Mach")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Source-data checks
# ------------------------------------------------------------

assert len(agard_model3_air) == 8

assert (
    agard_model3_air[
        "subsonic_DLM_candidate"
    ].sum()
    == 6
)

assert np.allclose(
    agard_model3_air[
        "omega_alpha_rad_s"
    ],
    239.3,
)

# Published torsional reference frequency should be
# consistent with the measured ~38.1 Hz value used earlier.
alpha_frequency_mean = (
    agard_model3_air[
        "alpha_frequency_Hz"
    ].mean()
)

alpha_frequency_error = (
    alpha_frequency_mean
    - 38.1
)

assert abs(
    alpha_frequency_error
) < 0.05


# The tabulated rho, V and q values are rounded independently,
# so exact equality is not expected.
max_q_consistency_difference = (
    agard_model3_air[
        "q_consistency_difference_pct"
    ]
    .abs()
    .max()
)

assert max_q_consistency_difference < 4.0


# ------------------------------------------------------------
# 7. Compact benchmark table
# ------------------------------------------------------------

benchmark_columns = [
    "Mach",
    "mass_ratio_mu",
    "rho_kg_m3",
    "V_m_s",
    "q_Pa",
    "flutter_frequency_Hz",
    "flutter_speed_index",
    "q_consistency_difference_pct",
    "subsonic_DLM_candidate",
]


print(
    "Published AGARD 445.6 weakened Model 3 "
    "flutter points imported."
)

print(
    f"\nTotal published Model 3 air points: "
    f"{len(agard_model3_air)}"
)

print(
    "Subsonic points available for current DLM: "
    f"{agard_model3_air['subsonic_DLM_candidate'].sum()}"
)

print(
    "\nPublished torsional reference frequency:"
)

print(
    f"omega_alpha = 239.3 rad/s "
    f"= {alpha_frequency_mean:.4f} Hz"
)

print(
    "\nMaximum q consistency difference "
    "from 0.5*rho*V^2:"
)

print(
    f"{max_q_consistency_difference:.3f}%"
)

print(
    "\nExperimental benchmark dataset:"
)

display(
    agard_model3_air[
        benchmark_columns
    ]
)

print(
    "\nCell 33 source-data import checks passed."
)

In [ ]:
# Cell 34 — Four-mode DLM benchmark at published AGARD flutter points
#
# Purpose:
#   Evaluate the aeroelastic model directly at each published
#   subsonic experimental flutter condition for weakened Model 3.
#
# At a perfect prediction:
#
#       sigma_critical ≈ 0
#
# at the experimental flutter condition.
#
# We also compare the predicted critical-branch frequency with the
# published experimental flutter frequency.
#
# IMPORTANT:
# This is the first experimental benchmark comparison.
# It is NOT yet a flutter-speed prediction sweep.


# ------------------------------------------------------------
# 1. Extract six published subsonic experimental points
# ------------------------------------------------------------

agard_subsonic_points = (
    agard_model3_air[
        agard_model3_air[
            "subsonic_DLM_candidate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(agard_subsonic_points) == 6


# ------------------------------------------------------------
# 2. Structural mass matrix for MAC diagnostics
# ------------------------------------------------------------

M4, _, _ = (
    get_structural_modal_matrices(
        n_modes=4
    )
)


# ------------------------------------------------------------
# 3. Storage
# ------------------------------------------------------------

benchmark_branch_rows = []

benchmark_point_rows = []


# ------------------------------------------------------------
# 4. Solve each published experimental condition
#
# Each point is treated independently.
#
# We initialize each tracked branch from its corresponding
# dry structural mode.
# ------------------------------------------------------------

for test_id, row in (
    agard_subsonic_points.iterrows()
):

    Ma_test = float(
        row["Mach"]
    )

    U_test = float(
        row["V_m_s"]
    )

    q_test = float(
        row["q_Pa"]
    )

    f_exp = float(
        row["flutter_frequency_Hz"]
    )

    print(
        f"\nSolving AGARD test point {test_id + 1}/6:"
    )

    print(
        f"Mach = {Ma_test:.3f}, "
        f"U = {U_test:.3f} m/s, "
        f"q = {q_test:.1f} Pa"
    )


    point_results = []


    # --------------------------------------------------------
    # Solve all four aeroelastic branches
    # --------------------------------------------------------

    for mode_index in range(4):

        result = pk_iterate_mode_mac(
            mode_index=mode_index,
            U=U_test,
            q_dyn=q_test,
            Ma=Ma_test,
            n_modes=4,
        )

        if not result["converged"]:

            raise RuntimeError(
                f"Mode {mode_index + 1} failed "
                f"to converge at Mach {Ma_test:.3f}."
            )


        # ----------------------------------------------------
        # MAC of final aeroelastic vector against its
        # original dry structural basis vector.
        #
        # This is different from selected_MAC inside the
        # p-k iteration, which compares successive vectors.
        # ----------------------------------------------------

        dry_reference = np.zeros(
            4,
            dtype=complex
        )

        dry_reference[
            mode_index
        ] = 1.0


        dry_basis_mac = (
            complex_modal_mac(
                dry_reference,
                result["modal_vector"],
                M4,
            )
        )


        branch_row = {
            "test_id":
                test_id + 1,

            "Mach":
                Ma_test,

            "mode_id":
                mode_index + 1,

            "sigma_1_per_s":
                result[
                    "sigma_1_per_s"
                ],

            "frequency_Hz":
                result[
                    "frequency_hz"
                ],

            "damping_ratio":
                result[
                    "zeta"
                ],

            "k_PA_1_per_m":
                result[
                    "k_PA_1_per_m"
                ],

            "dry_basis_MAC":
                dry_basis_mac,

            "pk_iterations":
                result[
                    "iterations"
                ],
        }


        point_results.append(
            branch_row
        )

        benchmark_branch_rows.append(
            branch_row
        )


    # --------------------------------------------------------
    # Identify least-stable branch
    #
    # Largest sigma:
    #
    #   sigma < 0 : stable
    #   sigma = 0 : neutral/flutter
    #   sigma > 0 : unstable
    # --------------------------------------------------------

    critical = max(
        point_results,
        key=lambda x:
            x["sigma_1_per_s"]
    )


    f_pred = (
        critical[
            "frequency_Hz"
        ]
    )

    frequency_error_pct = (
        100.0
        * (
            f_pred
            - f_exp
        )
        / f_exp
    )


    benchmark_point_rows.append({
        "test_id":
            test_id + 1,

        "Mach":
            Ma_test,

        "mass_ratio_mu":
            float(
                row["mass_ratio_mu"]
            ),

        "U_m_s":
            U_test,

        "q_Pa":
            q_test,

        "experimental_flutter_Hz":
            f_exp,

        "critical_mode_id":
            critical[
                "mode_id"
            ],

        "critical_sigma_1_per_s":
            critical[
                "sigma_1_per_s"
            ],

        "critical_damping_ratio":
            critical[
                "damping_ratio"
            ],

        "predicted_critical_frequency_Hz":
            f_pred,

        "frequency_error_pct":
            frequency_error_pct,

        "critical_dry_basis_MAC":
            critical[
                "dry_basis_MAC"
            ],

        "critical_pk_iterations":
            critical[
                "pk_iterations"
            ],
    })


# ------------------------------------------------------------
# 5. Assemble tables
# ------------------------------------------------------------

agard_branch_benchmark = (
    pd.DataFrame(
        benchmark_branch_rows
    )
)

agard_point_benchmark = (
    pd.DataFrame(
        benchmark_point_rows
    )
)


# ------------------------------------------------------------
# 6. Basic numerical checks
# ------------------------------------------------------------

assert len(
    agard_branch_benchmark
) == 24

assert len(
    agard_point_benchmark
) == 6

assert np.isfinite(
    agard_branch_benchmark[
        [
            "sigma_1_per_s",
            "frequency_Hz",
            "damping_ratio",
            "k_PA_1_per_m",
        ]
    ].to_numpy()
).all()


# ------------------------------------------------------------
# 7. Classify model state at the experimental flutter point
#
# This is descriptive only.
# No arbitrary "validation pass/fail" threshold is imposed.
# ------------------------------------------------------------

def stability_label(sigma):

    if sigma > 0.0:
        return "model unstable"

    elif sigma < 0.0:
        return "model stable"

    else:
        return "neutral"


agard_point_benchmark[
    "model_state_at_experiment"
] = (
    agard_point_benchmark[
        "critical_sigma_1_per_s"
    ]
    .apply(
        stability_label
    )
)


# ------------------------------------------------------------
# 8. Display complete branch table
# ------------------------------------------------------------

print(
    "\nAll four aeroelastic branches "
    "at each published flutter point:"
)

display(
    agard_branch_benchmark[
        [
            "test_id",
            "Mach",
            "mode_id",
            "sigma_1_per_s",
            "frequency_Hz",
            "damping_ratio",
            "k_PA_1_per_m",
            "dry_basis_MAC",
            "pk_iterations",
        ]
    ]
)


# ------------------------------------------------------------
# 9. Display critical branch comparison
# ------------------------------------------------------------

print(
    "\nCritical-branch comparison "
    "at experimental flutter conditions:"
)

display(
    agard_point_benchmark[
        [
            "test_id",
            "Mach",
            "mass_ratio_mu",
            "U_m_s",
            "q_Pa",
            "experimental_flutter_Hz",
            "critical_mode_id",
            "critical_sigma_1_per_s",
            "critical_damping_ratio",
            "predicted_critical_frequency_Hz",
            "frequency_error_pct",
            "critical_dry_basis_MAC",
            "model_state_at_experiment",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Plot stability residual at experimental boundary
#
# Ideal prediction:
#
#       sigma = 0
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    agard_point_benchmark[
        "Mach"
    ],
    agard_point_benchmark[
        "critical_sigma_1_per_s"
    ],
    "o-",
)

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Mach number"
)

plt.ylabel(
    "Critical pole real part σ (1/s)"
)

plt.title(
    "DLM stability residual at published AGARD flutter points"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. Plot frequency comparison
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    agard_point_benchmark[
        "Mach"
    ],
    agard_point_benchmark[
        "experimental_flutter_Hz"
    ],
    "o-",
    label="Experiment",
)

plt.plot(
    agard_point_benchmark[
        "Mach"
    ],
    agard_point_benchmark[
        "predicted_critical_frequency_Hz"
    ],
    "s--",
    label="4-mode DLM",
)

plt.xlabel(
    "Mach number"
)

plt.ylabel(
    "Flutter / critical frequency (Hz)"
)

plt.title(
    "AGARD 445.6 Model 3 — frequency comparison"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 12. Summary diagnostics
# ------------------------------------------------------------

mean_abs_frequency_error = (
    agard_point_benchmark[
        "frequency_error_pct"
    ]
    .abs()
    .mean()
)

max_abs_frequency_error = (
    agard_point_benchmark[
        "frequency_error_pct"
    ]
    .abs()
    .max()
)


print(
    "\nBenchmark diagnostics:"
)

print(
    f"Mean absolute critical-frequency error: "
    f"{mean_abs_frequency_error:.3f}%"
)

print(
    f"Maximum absolute critical-frequency error: "
    f"{max_abs_frequency_error:.3f}%"
)

print(
    "\nIdeal stability residual at an experimental "
    "flutter point is sigma = 0."
)

print(
    "\nCell 34 published-condition DLM benchmark completed."
)

In [ ]:
# Cell 35 — Kinematic consistency audit
#
# Purpose:
#   Compare two streamwise modal slope representations:
#
#   A) Current model:
#      published dz/dx values interpolated independently
#
#   B) Kinematically consistent:
#      d(Phi_z)/dx computed directly from the SAME
#      displacement interpolation used for Phi_z
#
# This is a diagnostic only.
# We are NOT replacing the production model yet.


# ------------------------------------------------------------
# 1. Production DLM receiving-point coordinates
# ------------------------------------------------------------

prod_j_xy = np.column_stack([
    aerogrid_prod["offset_j"][:, 0],
    aerogrid_prod["offset_j"][:, 1],
])

assert prod_j_xy.shape == (320, 2)


# ------------------------------------------------------------
# 2. Exact gradient of the piecewise-linear Delaunay field
#
# Within each structural triangle:
#
#   Phi_z(x,y) = a*x + b*y + c
#
# Therefore:
#
#   dPhi_z/dx = a
#   dPhi_z/dy = b
#
# Units:
#   Phi_z          dimensionless
#   x,y            metres
#   gradient       1/m
# ------------------------------------------------------------

def piecewise_linear_modal_gradient(
    triangulation,
    source_modal_values,
    query_xy,
):
    """
    Compute exact spatial gradients of a piecewise-linear
    triangular interpolation.

    Parameters
    ----------
    triangulation : scipy.spatial.Delaunay
        Structural triangulation.

    source_modal_values : ndarray, shape (n_nodes, n_modes)
        Modal z values at structural nodes.

    query_xy : ndarray, shape (n_query, 2)
        Query coordinates [m].

    Returns
    -------
    grad_x : ndarray, shape (n_query, n_modes)
        dPhi/dx [1/m]

    grad_y : ndarray, shape (n_query, n_modes)
        dPhi/dy [1/m]
    """

    source_modal_values = np.asarray(
        source_modal_values,
        dtype=float
    )

    query_xy = np.asarray(
        query_xy,
        dtype=float
    )

    simplex_ids = triangulation.find_simplex(
        query_xy
    )

    if np.any(simplex_ids < 0):
        raise ValueError(
            "At least one query point lies outside "
            "the structural interpolation domain."
        )

    n_query = query_xy.shape[0]
    n_modes = source_modal_values.shape[1]

    grad_x = np.zeros(
        (n_query, n_modes)
    )

    grad_y = np.zeros(
        (n_query, n_modes)
    )

    for i in range(n_query):

        simplex_id = simplex_ids[i]

        vertex_ids = (
            triangulation.simplices[
                simplex_id
            ]
        )

        xy_triangle = (
            triangulation.points[
                vertex_ids
            ]
        )

        z_triangle = (
            source_modal_values[
                vertex_ids,
                :
            ]
        )

        # Solve:
        #
        #   [x y 1] [a]   [Phi]
        #           [b] =
        #           [c]
        #
        # simultaneously for all modal columns.
        A_triangle = np.column_stack([
            xy_triangle[:, 0],
            xy_triangle[:, 1],
            np.ones(3),
        ])

        coefficients = np.linalg.solve(
            A_triangle,
            z_triangle,
        )

        grad_x[i, :] = (
            coefficients[0, :]
        )

        grad_y[i, :] = (
            coefficients[1, :]
        )

    return grad_x, grad_y


# ------------------------------------------------------------
# 3. Compute displacement-derived slopes
# ------------------------------------------------------------

(
    Phi_dzdx_from_z_j_SI,
    Phi_dzdy_from_z_j_SI,
) = piecewise_linear_modal_gradient(
    structural_triangulation,
    Phi_z_4,
    prod_j_xy,
)

assert Phi_dzdx_from_z_j_SI.shape == (320, 4)
assert Phi_dzdy_from_z_j_SI.shape == (320, 4)

assert np.isfinite(
    Phi_dzdx_from_z_j_SI
).all()


# ------------------------------------------------------------
# 4. Compare against current published-slope model
# ------------------------------------------------------------

kinematic_audit_rows = []

for mode_index in range(4):

    slope_published = (
        Phi_dzdx_j_SI_prod[
            :,
            mode_index
        ]
    )

    slope_from_z = (
        Phi_dzdx_from_z_j_SI[
            :,
            mode_index
        ]
    )

    corr = np.corrcoef(
        slope_from_z,
        slope_published
    )[0, 1]

    corr_reversed = np.corrcoef(
        slope_from_z,
        -slope_published
    )[0, 1]

    alpha = (
        np.dot(
            slope_published,
            slope_from_z
        )
        /
        np.dot(
            slope_published,
            slope_published
        )
    )

    relative_difference = (
        np.linalg.norm(
            slope_from_z
            - slope_published
        )
        /
        np.linalg.norm(
            slope_from_z
        )
    )

    relative_difference_reversed = (
        np.linalg.norm(
            slope_from_z
            + slope_published
        )
        /
        np.linalg.norm(
            slope_from_z
        )
    )

    kinematic_audit_rows.append({
        "mode_id":
            mode_index + 1,

        "published_slope_abs_max_1_per_m":
            np.max(
                np.abs(
                    slope_published
                )
            ),

        "z_derived_slope_abs_max_1_per_m":
            np.max(
                np.abs(
                    slope_from_z
                )
            ),

        "corr_zderived_vs_published":
            corr,

        "corr_zderived_vs_minus_published":
            corr_reversed,

        "least_squares_alpha":
            alpha,

        "relative_difference":
            relative_difference,

        "relative_difference_if_sign_reversed":
            relative_difference_reversed,
    })


kinematic_audit_table = pd.DataFrame(
    kinematic_audit_rows
)


# ------------------------------------------------------------
# 5. Build an alternate Qhh only for sensitivity inspection
#
# This is NOT yet the production model.
# ------------------------------------------------------------

def calculate_Qhh_zgradient_diagnostic(
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Diagnostic generalized aerodynamic matrix using
    dPhi_z/dx derived directly from the displacement field.
    """

    Qjj = DLM.calc_Qjj(
        aerogrid_prod,
        Ma=Ma,
        k=k_PA,
    )

    Phi_z_j_n = (
        Phi_z_j_prod[
            :,
            :n_modes
        ]
    )

    Phi_z_k_n = (
        Phi_z_k_prod[
            :,
            :n_modes
        ]
    )

    Phi_x_n = (
        Phi_dzdx_from_z_j_SI[
            :,
            :n_modes
        ]
    )

    W_n = -(
        Phi_x_n
        + 1j
        * k_PA
        * Phi_z_j_n
    )

    G_n = (
        Phi_z_k_n.T
        @ np.diag(
            panel_force_weight_prod
        )
    )

    return (
        G_n
        @ Qjj
        @ W_n
    )


# ------------------------------------------------------------
# 6. Compare generalized aerodynamic matrices
# at one representative low-subsonic condition
#
# Use experimental point near Mach 0.5.
# ------------------------------------------------------------

audit_point = (
    agard_subsonic_points
    .iloc[0]
)

Ma_audit = float(
    audit_point["Mach"]
)

U_audit = float(
    audit_point["V_m_s"]
)

omega_exp_audit = (
    2.0
    * np.pi
    * float(
        audit_point[
            "flutter_frequency_Hz"
        ]
    )
)

k_audit = (
    omega_exp_audit
    / U_audit
)


Qhh_published_slope = (
    calculate_Qhh_production(
        Ma=Ma_audit,
        k_PA=k_audit,
        n_modes=4,
    )
)

Qhh_zderived_slope = (
    calculate_Qhh_zgradient_diagnostic(
        Ma=Ma_audit,
        k_PA=k_audit,
        n_modes=4,
    )
)


Qhh_slope_model_difference = (
    np.linalg.norm(
        Qhh_zderived_slope
        - Qhh_published_slope
    )
    /
    np.linalg.norm(
        Qhh_published_slope
    )
)


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print(
    "Modal kinematic consistency audit:"
)

display(
    kinematic_audit_table
)

print(
    "\nRepresentative aerodynamic condition:"
)

print(
    f"Mach = {Ma_audit:.3f}"
)

print(
    f"k_PA = {k_audit:.6f} 1/m"
)

print(
    "\nRelative Qhh difference between:"
)

print(
    "  independently interpolated published dz/dx"
)

print(
    "and"
)

print(
    "  displacement-derived dPhi_z/dx"
)

print(
    f"\nDifference = "
    f"{100.0 * Qhh_slope_model_difference:.3f}%"
)

print(
    "\nCell 35 kinematic audit completed."
)

In [ ]:
# Cell 36 — Wall-mounted semispan symmetry audit
#
# The experimental AGARD Model 3 configuration is wall-mounted.
#
# Current model:
#     isolated aerodynamic semispan
#
# Audit model:
#     semispan + xz mirror symmetry using PanelAero
#
# We retain the SAME published z and dz/dx modal data.
#
# No structural interpolation changes are made here.


# ------------------------------------------------------------
# 1. Choose the published Mach 0.960 test point
# ------------------------------------------------------------

symmetry_test_row = (
    agard_subsonic_points[
        np.isclose(
            agard_subsonic_points["Mach"],
            0.960,
        )
    ]
    .iloc[0]
)

Ma_sym = float(
    symmetry_test_row["Mach"]
)

U_sym = float(
    symmetry_test_row["V_m_s"]
)

q_sym = float(
    symmetry_test_row["q_Pa"]
)

f_exp_sym = float(
    symmetry_test_row[
        "flutter_frequency_Hz"
    ]
)

omega_exp_sym = (
    2.0
    * np.pi
    * f_exp_sym
)

k_exp_sym = (
    omega_exp_sym
    / U_sym
)


# ------------------------------------------------------------
# 2. Current unsymmetric DLM AIC
# ------------------------------------------------------------

Qjj_unsym = DLM.calc_Qjj(
    aerogrid_prod,
    Ma=Ma_sym,
    k=k_exp_sym,
)

assert Qjj_unsym.shape == (
    320,
    320
)


# ------------------------------------------------------------
# 3. Wall-symmetric DLM AIC
#
# PanelAero calc_Qjjs supports xz_symmetry.
#
# The function returns:
#
#   shape = (n_Mach, n_k, n_panel, n_panel)
# ------------------------------------------------------------

Qjj_sym_all = DLM.calc_Qjjs(
    aerogrid_prod,
    Ma=np.array([
        Ma_sym
    ]),
    k=np.array([
        k_exp_sym
    ]),
    xz_symmetry=True,
)

Qjj_sym = (
    Qjj_sym_all[
        0,
        0,
        :,
        :
    ]
)

assert Qjj_sym.shape == (
    320,
    320
)

assert np.isfinite(
    Qjj_sym
).all()


# ------------------------------------------------------------
# 4. Use exactly the SAME structural kinematics
#
# Published slope model retained.
# ------------------------------------------------------------

W_sym_audit = -(
    Phi_dzdx_j_SI_prod
    + 1j
    * k_exp_sym
    * Phi_z_j_prod
)

G_sym_audit = (
    Phi_z_k_prod.T
    @ np.diag(
        panel_force_weight_prod
    )
)


# ------------------------------------------------------------
# 5. Build generalized aerodynamic matrices
# ------------------------------------------------------------

Qhh_unsym = (
    G_sym_audit
    @ Qjj_unsym
    @ W_sym_audit
)

Qhh_sym = (
    G_sym_audit
    @ Qjj_sym
    @ W_sym_audit
)

assert Qhh_unsym.shape == (4, 4)
assert Qhh_sym.shape == (4, 4)


# ------------------------------------------------------------
# 6. Quantify aerodynamic change due to wall symmetry
# ------------------------------------------------------------

Qjj_symmetry_difference = (
    np.linalg.norm(
        Qjj_sym
        - Qjj_unsym
    )
    /
    np.linalg.norm(
        Qjj_unsym
    )
)

Qhh_symmetry_difference = (
    np.linalg.norm(
        Qhh_sym
        - Qhh_unsym
    )
    /
    np.linalg.norm(
        Qhh_unsym
    )
)

Qhh_norm_ratio = (
    np.linalg.norm(
        Qhh_sym
    )
    /
    np.linalg.norm(
        Qhh_unsym
    )
)


# ------------------------------------------------------------
# 7. Evaluate dynamic stiffness at the EXPERIMENTAL
#    flutter frequency
#
# At an exact flutter solution:
#
# det(D) = 0
#
# where:
#
# D =
#   -omega^2 M
#   + i omega C
#   + K
#   - q_dyn Qhh
#
# Instead of determinant, use singular values.
#
# A smaller:
#
#       s_min / s_max
#
# indicates a matrix closer to singularity.
# ------------------------------------------------------------

D_unsym = (
    -omega_exp_sym**2
    * M_modal_SI
    + 1j
    * omega_exp_sym
    * C_modal_SI
    + K_modal_SI
    - q_sym
    * Qhh_unsym
)

D_sym = (
    -omega_exp_sym**2
    * M_modal_SI
    + 1j
    * omega_exp_sym
    * C_modal_SI
    + K_modal_SI
    - q_sym
    * Qhh_sym
)


svals_unsym = np.linalg.svd(
    D_unsym,
    compute_uv=False,
)

svals_sym = np.linalg.svd(
    D_sym,
    compute_uv=False,
)


singularity_metric_unsym = (
    np.min(
        svals_unsym
    )
    /
    np.max(
        svals_unsym
    )
)

singularity_metric_sym = (
    np.min(
        svals_sym
    )
    /
    np.max(
        svals_sym
    )
)


condition_unsym = (
    np.max(
        svals_unsym
    )
    /
    np.min(
        svals_unsym
    )
)

condition_sym = (
    np.max(
        svals_sym
    )
    /
    np.min(
        svals_sym
    )
)


# ------------------------------------------------------------
# 8. Compare modal generalized-aero column norms
#
# Each column corresponds to aerodynamic response
# produced by one structural modal coordinate.
# ------------------------------------------------------------

modal_aero_rows = []

for mode_index in range(4):

    norm_unsym = np.linalg.norm(
        Qhh_unsym[
            :,
            mode_index
        ]
    )

    norm_sym = np.linalg.norm(
        Qhh_sym[
            :,
            mode_index
        ]
    )

    modal_aero_rows.append({
        "mode_id":
            mode_index + 1,

        "unsymmetric_column_norm_m":
            norm_unsym,

        "symmetric_column_norm_m":
            norm_sym,

        "symmetry_change_pct":
            100.0
            * (
                norm_sym
                - norm_unsym
            )
            / norm_unsym,
    })


wall_symmetry_modal_table = (
    pd.DataFrame(
        modal_aero_rows
    )
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(
    "Wall-symmetry audit at published test condition:"
)

print(
    f"Mach = {Ma_sym:.3f}"
)

print(
    f"U = {U_sym:.3f} m/s"
)

print(
    f"q = {q_sym:.3f} Pa"
)

print(
    f"Experimental flutter frequency = "
    f"{f_exp_sym:.6f} Hz"
)

print(
    f"k_PA at experimental frequency = "
    f"{k_exp_sym:.6f} 1/m"
)


print(
    "\nRelative aerodynamic-matrix differences:"
)

print(
    f"Qjj difference due to symmetry: "
    f"{100.0 * Qjj_symmetry_difference:.3f}%"
)

print(
    f"Qhh difference due to symmetry: "
    f"{100.0 * Qhh_symmetry_difference:.3f}%"
)

print(
    f"Qhh norm ratio "
    f"(symmetric / unsymmetric): "
    f"{Qhh_norm_ratio:.6f}"
)


print(
    "\nPer-mode generalized-aerodynamic change:"
)

display(
    wall_symmetry_modal_table
)


print(
    "\nDynamic-stiffness proximity to singularity "
    "at the measured flutter frequency:"
)

print(
    f"Unsymmetric s_min/s_max = "
    f"{singularity_metric_unsym:.6e}"
)

print(
    f"Symmetric   s_min/s_max = "
    f"{singularity_metric_sym:.6e}"
)

print(
    f"\nUnsymmetric condition number = "
    f"{condition_unsym:.6e}"
)

print(
    f"Symmetric   condition number = "
    f"{condition_sym:.6e}"
)


if (
    singularity_metric_sym
    <
    singularity_metric_unsym
):

    print(
        "\nWall symmetry moves the model "
        "closer to the experimental neutral-stability condition."
    )

else:

    print(
        "\nWall symmetry does not move the model "
        "closer to singularity at this test point."
    )


print(
    "\nCell 36 wall-symmetry audit completed."
)

In [ ]:
# Cell 37 — Explicit wall-image DLM model
#
# Purpose:
#   Replace PanelAero's internal xz_symmetry helper with an
#   explicitly constructed mirrored aerodynamic wing.
#
# Physical interpretation:
#
#   The AGARD model is a wall-mounted semispan.
#   Inviscid wall symmetry is represented by an image wing
#   reflected across y = 0.
#
# IMPORTANT:
#
#   - Structural model remains the physical semispan only.
#   - Aerodynamics are solved on right wing + image wing.
#   - Generalized forces are integrated ONLY on the physical
#     right semispan.
#
#   This avoids doubling structural mass or aerodynamic force.


# ------------------------------------------------------------
# 1. Helper: reflect points across the x-z plane
# ------------------------------------------------------------

def reflect_points_about_xz(points):
    """
    Mirror Cartesian points across y = 0.
    """

    mirrored = np.asarray(
        points,
        dtype=float
    ).copy()

    mirrored[:, 1] *= -1.0

    return mirrored


# ------------------------------------------------------------
# 2. Physical right-wing aerodynamic grid
# ------------------------------------------------------------

right_grid = aerogrid_prod

n_right = right_grid["n"]

assert n_right == 320


# ------------------------------------------------------------
# 3. Build left image wing
#
# For panel orientation:
#
# Right wing:
#     P1 = inboard
#     P3 = outboard
#
# After reflection, simply mirroring P1/P3 would reverse the
# spanwise orientation.
#
# Therefore:
#
#     left P1 = mirror(right P3)
#     left P3 = mirror(right P1)
#
# This keeps panel orientation increasing in global +y and
# preserves upward normals.
# ------------------------------------------------------------

left_offset_j = (
    reflect_points_about_xz(
        right_grid["offset_j"]
    )
)

left_offset_k = (
    reflect_points_about_xz(
        right_grid["offset_k"]
    )
)

left_offset_l = (
    reflect_points_about_xz(
        right_grid["offset_l"]
    )
)


left_offset_P1 = (
    reflect_points_about_xz(
        right_grid["offset_P3"]
    )
)

left_offset_P3 = (
    reflect_points_about_xz(
        right_grid["offset_P1"]
    )
)


# Reflection of a normal vector across y = 0:
#
#     [Nx, Ny, Nz] -> [Nx, -Ny, Nz]
#
# For this flat wing Ny = 0, Nz remains positive.
left_N = (
    right_grid["N"].copy()
)

left_N[:, 1] *= -1.0


left_A = (
    right_grid["A"].copy()
)

left_l = (
    right_grid["l"].copy()
)


# ------------------------------------------------------------
# 4. Assemble explicit full aerodynamic image system
#
# Ordering:
#
#     first  320 panels = physical right semispan
#     second 320 panels = image left semispan
# ------------------------------------------------------------

aerogrid_wall_explicit = {
    "offset_j":
        np.vstack([
            right_grid["offset_j"],
            left_offset_j,
        ]),

    "offset_k":
        np.vstack([
            right_grid["offset_k"],
            left_offset_k,
        ]),

    "offset_l":
        np.vstack([
            right_grid["offset_l"],
            left_offset_l,
        ]),

    "offset_P1":
        np.vstack([
            right_grid["offset_P1"],
            left_offset_P1,
        ]),

    "offset_P3":
        np.vstack([
            right_grid["offset_P3"],
            left_offset_P3,
        ]),

    "N":
        np.vstack([
            right_grid["N"],
            left_N,
        ]),

    "A":
        np.hstack([
            right_grid["A"],
            left_A,
        ]),

    "l":
        np.hstack([
            right_grid["l"],
            left_l,
        ]),

    "n":
        2 * n_right,
}


assert (
    aerogrid_wall_explicit["n"]
    == 640
)


# ------------------------------------------------------------
# 5. Geometry/orientation verification
# ------------------------------------------------------------

right_span_direction = (
    right_grid["offset_P3"][:, 1]
    - right_grid["offset_P1"][:, 1]
)

left_span_direction = (
    left_offset_P3[:, 1]
    - left_offset_P1[:, 1]
)


minimum_Nz = np.min(
    aerogrid_wall_explicit[
        "N"
    ][:, 2]
)


assert np.all(
    right_span_direction > 0.0
)

assert np.all(
    left_span_direction > 0.0
)

assert minimum_Nz > 0.0


# ------------------------------------------------------------
# 6. Reconstruct the published Mach 0.960 point
# ------------------------------------------------------------

wall_test_row = (
    agard_subsonic_points[
        np.isclose(
            agard_subsonic_points["Mach"],
            0.960,
        )
    ]
    .iloc[0]
)


Ma_wall = float(
    wall_test_row["Mach"]
)

U_wall = float(
    wall_test_row["V_m_s"]
)

q_wall = float(
    wall_test_row["q_Pa"]
)

f_wall_exp = float(
    wall_test_row[
        "flutter_frequency_Hz"
    ]
)


omega_wall_exp = (
    2.0
    * np.pi
    * f_wall_exp
)

k_wall_exp = (
    omega_wall_exp
    / U_wall
)


# ------------------------------------------------------------
# 7. Compute explicit 640-panel DLM AIC
#
# IMPORTANT:
# xz_symmetry=False.
#
# The geometry itself now contains the image wing.
# ------------------------------------------------------------

print(
    "Calculating explicit 640-panel wall-image DLM..."
)

Qjj_wall_full = (
    DLM.calc_Qjj(
        aerogrid_wall_explicit,
        Ma=Ma_wall,
        k=k_wall_exp,
    )
)


assert Qjj_wall_full.shape == (
    640,
    640
)

assert np.isfinite(
    Qjj_wall_full
).all()


# ------------------------------------------------------------
# 8. Partition full AIC
#
#             input downwash
#
#              R        L
#          +----------------
# output R |  Q_RR     Q_RL
# output L |  Q_LR     Q_LL
#
# For symmetric wall motion:
#
#       w_L = w_R
#
# Therefore physical right-wing pressure is:
#
#       cp_R
#         = (Q_RR + Q_RL) w_R
#
# Define effective wall AIC:
#
#       Q_wall = Q_RR + Q_RL
# ------------------------------------------------------------

Q_RR = (
    Qjj_wall_full[
        :n_right,
        :n_right,
    ]
)

Q_RL = (
    Qjj_wall_full[
        :n_right,
        n_right:,
    ]
)

Q_LR = (
    Qjj_wall_full[
        n_right:,
        :n_right,
    ]
)

Q_LL = (
    Qjj_wall_full[
        n_right:,
        n_right:,
    ]
)


Qjj_wall_effective = (
    Q_RR
    + Q_RL
)


assert Qjj_wall_effective.shape == (
    320,
    320
)


# ------------------------------------------------------------
# 9. Symmetric modal normalwash
#
# Same modal deformation occurs on the image wing.
# ------------------------------------------------------------

W_right_wall = -(
    Phi_dzdx_j_SI_prod
    + 1j
    * k_wall_exp
    * Phi_z_j_prod
)


W_full_wall = np.vstack([
    W_right_wall,
    W_right_wall,
])


# ------------------------------------------------------------
# 10. Verify full-system pressure symmetry
# ------------------------------------------------------------

cp_full_modal = (
    Qjj_wall_full
    @ W_full_wall
)


cp_right_modal = (
    cp_full_modal[
        :n_right,
        :
    ]
)

cp_left_modal = (
    cp_full_modal[
        n_right:,
        :
    ]
)


pressure_symmetry_error = (
    np.linalg.norm(
        cp_right_modal
        - cp_left_modal
    )
    /
    np.linalg.norm(
        cp_right_modal
    )
)


# Effective AIC route
cp_right_effective = (
    Qjj_wall_effective
    @ W_right_wall
)


effective_AIC_error = (
    np.linalg.norm(
        cp_right_effective
        - cp_right_modal
    )
    /
    np.linalg.norm(
        cp_right_modal
    )
)


# ------------------------------------------------------------
# 11. Physical semispan generalized-force projection
#
# DO NOT integrate the left image wing.
#
# The left wing is aerodynamic image geometry only.
# ------------------------------------------------------------

G_right_wall = (
    Phi_z_k_prod.T
    @ np.diag(
        panel_force_weight_prod
    )
)


Qhh_wall_explicit = (
    G_right_wall
    @ Qjj_wall_effective
    @ W_right_wall
)


# ------------------------------------------------------------
# 12. Compare against isolated semispan
# ------------------------------------------------------------

Qjj_isolated_wall_test = (
    DLM.calc_Qjj(
        aerogrid_prod,
        Ma=Ma_wall,
        k=k_wall_exp,
    )
)


Qhh_isolated_wall_test = (
    G_right_wall
    @ Qjj_isolated_wall_test
    @ W_right_wall
)


explicit_wall_Qhh_change = (
    np.linalg.norm(
        Qhh_wall_explicit
        - Qhh_isolated_wall_test
    )
    /
    np.linalg.norm(
        Qhh_isolated_wall_test
    )
)


# ------------------------------------------------------------
# 13. Dynamic stiffness at measured flutter frequency
# ------------------------------------------------------------

D_wall_explicit = (
    -omega_wall_exp**2
    * M_modal_SI

    + 1j
    * omega_wall_exp
    * C_modal_SI

    + K_modal_SI

    - q_wall
    * Qhh_wall_explicit
)


wall_singular_values = (
    np.linalg.svd(
        D_wall_explicit,
        compute_uv=False,
    )
)


wall_singularity_metric = (
    np.min(
        wall_singular_values
    )
    /
    np.max(
        wall_singular_values
    )
)


wall_condition_number = (
    np.max(
        wall_singular_values
    )
    /
    np.min(
        wall_singular_values
    )
)


# ------------------------------------------------------------
# 14. Modal aerodynamic changes
# ------------------------------------------------------------

wall_modal_rows = []

for mode_index in range(4):

    isolated_norm = np.linalg.norm(
        Qhh_isolated_wall_test[
            :,
            mode_index
        ]
    )

    wall_norm = np.linalg.norm(
        Qhh_wall_explicit[
            :,
            mode_index
        ]
    )

    wall_modal_rows.append({
        "mode_id":
            mode_index + 1,

        "isolated_norm_m":
            isolated_norm,

        "explicit_wall_norm_m":
            wall_norm,

        "wall_change_pct":
            100.0
            * (
                wall_norm
                - isolated_norm
            )
            / isolated_norm,
    })


explicit_wall_modal_table = (
    pd.DataFrame(
        wall_modal_rows
    )
)


# ------------------------------------------------------------
# 15. Report
# ------------------------------------------------------------

print(
    "\nExplicit wall-image aerodynamic model:"
)

print(
    f"Physical panels: {n_right}"
)

print(
    f"Image panels:    {n_right}"
)

print(
    f"Total panels:    "
    f"{aerogrid_wall_explicit['n']}"
)

print(
    f"\nMinimum panel Nz: "
    f"{minimum_Nz:.6f}"
)

print(
    f"Minimum right span direction: "
    f"{np.min(right_span_direction):.6e} m"
)

print(
    f"Minimum left span direction:  "
    f"{np.min(left_span_direction):.6e} m"
)


print(
    "\nFull-wing symmetric-pressure check:"
)

print(
    f"Right-vs-left pressure error: "
    f"{pressure_symmetry_error:.3e}"
)

print(
    f"Effective-AIC reconstruction error: "
    f"{effective_AIC_error:.3e}"
)


print(
    "\nWall effect on generalized aerodynamic matrix:"
)

print(
    f"Relative Qhh change from isolated semispan: "
    f"{100.0 * explicit_wall_Qhh_change:.3f}%"
)


print(
    "\nPer-mode aerodynamic effect:"
)

display(
    explicit_wall_modal_table
)


print(
    "\nDynamic stiffness at published "
    "Mach 0.960 flutter condition:"
)

print(
    f"s_min/s_max = "
    f"{wall_singularity_metric:.6e}"
)

print(
    f"Condition number = "
    f"{wall_condition_number:.6e}"
)


print(
    "\nCell 37 explicit wall-image model completed."
)

In [ ]:
# Cell 38 — Explicit-wall p-k audit at Mach 0.960
#
# Purpose:
#   Determine how much the corrected wall-mounted aerodynamics
#   actually move the four converged aeroelastic poles.
#
# Only ONE published experimental condition is evaluated:
#
#       Mach = 0.960
#
# This avoids an expensive six-point sweep before we know
# whether wall correction is sufficient.
#
# Production status:
#   Structural coupling = still under audit
#   Aerodynamic wall model = explicit 640-panel image system


# ------------------------------------------------------------
# 1. Reusable explicit-wall generalized aerodynamic matrix
# ------------------------------------------------------------

G_right_wall_4 = (
    Phi_z_k_prod.T
    @ np.diag(
        panel_force_weight_prod
    )
)


def calculate_Qhh_wall_explicit(
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Generalized aerodynamic matrix for the wall-mounted
    semispan using an explicit mirrored image wing.

    Only forces on the physical right semispan are projected
    into the structural modal coordinates.

    Parameters
    ----------
    Ma : float
        Mach number.

    k_PA : float
        PanelAero frequency parameter:
            omega/U [1/m]

    n_modes : int
        2, 3 or 4 retained structural modes.

    Returns
    -------
    Qhh : complex ndarray
        Generalized aerodynamic matrix per unit dynamic pressure.
    """

    if n_modes not in (2, 3, 4):
        raise ValueError(
            "n_modes must be 2, 3, or 4."
        )

    if not (0.0 <= Ma < 1.0):
        raise ValueError(
            "Current PanelAero benchmark is restricted "
            "to subsonic Mach numbers."
        )

    if k_PA < 0.0:
        raise ValueError(
            "k_PA must be non-negative."
        )


    # --------------------------------------------------------
    # Full physical + image aerodynamic system
    # --------------------------------------------------------

    Qjj_full = DLM.calc_Qjj(
        aerogrid_wall_explicit,
        Ma=Ma,
        k=k_PA,
    )


    # --------------------------------------------------------
    # Partition pressure AIC
    #
    # Right-wing pressure under symmetric motion:
    #
    #   cp_R = (Q_RR + Q_RL) w_R
    # --------------------------------------------------------

    Q_RR_local = (
        Qjj_full[
            :n_right,
            :n_right,
        ]
    )

    Q_RL_local = (
        Qjj_full[
            :n_right,
            n_right:,
        ]
    )

    Q_wall_effective = (
        Q_RR_local
        + Q_RL_local
    )


    # --------------------------------------------------------
    # Retained structural modal fields
    # --------------------------------------------------------

    Phi_z_j_n = (
        Phi_z_j_prod[
            :,
            :n_modes
        ]
    )

    Phi_z_k_n = (
        Phi_z_k_prod[
            :,
            :n_modes
        ]
    )

    Phi_x_j_n = (
        Phi_dzdx_j_SI_prod[
            :,
            :n_modes
        ]
    )


    # --------------------------------------------------------
    # Harmonic normalwash
    # --------------------------------------------------------

    W_n = -(
        Phi_x_j_n
        + 1j
        * k_PA
        * Phi_z_j_n
    )


    # --------------------------------------------------------
    # Physical semispan generalized-force projection
    # --------------------------------------------------------

    G_n = (
        Phi_z_k_n.T
        @ np.diag(
            panel_force_weight_prod
        )
    )


    # --------------------------------------------------------
    # Generalized aerodynamic matrix
    # --------------------------------------------------------

    Qhh = (
        G_n
        @ Q_wall_effective
        @ W_n
    )

    assert Qhh.shape == (
        n_modes,
        n_modes
    )

    assert np.isfinite(
        Qhh
    ).all()

    return Qhh


# ------------------------------------------------------------
# 2. Frozen-k wall aeroelastic poles
# ------------------------------------------------------------

def frozen_k_wall_poles(
    U,
    q_dyn,
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Solve frozen-k aeroelastic eigenproblem using
    explicit-wall DLM aerodynamics.
    """

    M, C, K = (
        get_structural_modal_matrices(
            n_modes=n_modes
        )
    )

    Qhh = (
        calculate_Qhh_wall_explicit(
            Ma=Ma,
            k_PA=k_PA,
            n_modes=n_modes,
        )
    )

    K_eff = (
        K
        - q_dyn * Qhh
    )

    M_inv = np.linalg.inv(
        M
    )

    Z = np.zeros(
        (n_modes, n_modes),
        dtype=complex,
    )

    I = np.eye(
        n_modes,
        dtype=complex,
    )

    A = np.block([
        [
            Z,
            I,
        ],
        [
            -M_inv @ K_eff,
            -M_inv @ C,
        ],
    ])

    poles, eigvecs = (
        np.linalg.eig(A)
    )

    return poles, eigvecs


# ------------------------------------------------------------
# 3. MAC-tracked p-k iteration using explicit wall
# ------------------------------------------------------------

def pk_iterate_mode_wall(
    mode_index,
    U,
    q_dyn,
    Ma,
    n_modes=4,
    initial_k=None,
    tol=1e-6,
    max_iter=40,
    relaxation=0.65,
):
    """
    MAC-based p-k iteration using explicit-wall DLM.

    A slightly looser tolerance than Cell 31 is used because
    each iteration now solves a 640-panel DLM system.
    """

    if not 0 <= mode_index < n_modes:
        raise ValueError(
            "mode_index must be smaller than n_modes."
        )


    M, _, _ = (
        get_structural_modal_matrices(
            n_modes=n_modes
        )
    )


    # --------------------------------------------------------
    # Initial k
    # --------------------------------------------------------

    if initial_k is None:

        k_current = (
            omega_analysis[
                mode_index
            ]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    # Dry modal reference
    reference_q = np.zeros(
        n_modes,
        dtype=complex,
    )

    reference_q[
        mode_index
    ] = 1.0

    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    history = []

    converged = False

    selected_pole = None
    selected_q = None
    selected_mac = None


    # --------------------------------------------------------
    # Iteration
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):

        poles, eigvecs = (
            frozen_k_wall_poles(
                U=U,
                q_dyn=q_dyn,
                Ma=Ma,
                k_PA=k_current,
                n_modes=n_modes,
            )
        )


        candidate_indices = np.where(
            np.imag(poles) > 0.0
        )[0]


        if candidate_indices.size == 0:
            raise RuntimeError(
                "No positive-frequency poles found."
            )


        candidate_macs = []
        candidate_vectors = []


        for idx in candidate_indices:

            q_candidate = (
                eigvecs[
                    :n_modes,
                    idx,
                ]
            )

            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )

            mac = complex_modal_mac(
                reference_q,
                q_candidate,
                M,
            )

            candidate_macs.append(
                mac
            )

            candidate_vectors.append(
                q_candidate
            )


        candidate_macs = np.asarray(
            candidate_macs
        )


        local_best = int(
            np.argmax(
                candidate_macs
            )
        )

        selected_index = (
            candidate_indices[
                local_best
            ]
        )

        selected_pole = (
            poles[
                selected_index
            ]
        )

        selected_q = (
            candidate_vectors[
                local_best
            ]
        )

        selected_mac = float(
            candidate_macs[
                local_best
            ]
        )


        # Phase alignment
        selected_q = (
            phase_align_modal_vector(
                selected_q,
                reference_q,
                M,
            )
        )

        selected_q = (
            mass_normalize_modal_vector(
                selected_q,
                M,
            )
        )


        omega_new = np.imag(
            selected_pole
        )

        k_raw = (
            omega_new / U
        )

        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_k_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(k_current),
                1e-14,
            )
        )


        history.append({
            "iteration":
                iteration,

            "k_current":
                k_current,

            "sigma":
                np.real(
                    selected_pole
                ),

            "frequency_Hz":
                omega_new
                / (
                    2.0
                    * np.pi
                ),

            "MAC":
                selected_mac,

            "relative_k_change":
                relative_k_change,
        })


        reference_q = (
            selected_q.copy()
        )

        k_current = (
            k_next
        )


        if (
            relative_k_change
            < tol
        ):

            converged = True
            break


    if not converged:

        raise RuntimeError(
            f"Wall p-k iteration did not converge "
            f"for mode {mode_index + 1}."
        )


    sigma_final = (
        np.real(
            selected_pole
        )
    )

    omega_final = (
        np.imag(
            selected_pole
        )
    )

    zeta_final = (
        -sigma_final
        / abs(
            selected_pole
        )
    )


    return {
        "mode_index":
            mode_index,

        "pole":
            selected_pole,

        "sigma":
            sigma_final,

        "frequency_Hz":
            omega_final
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            zeta_final,

        "k_PA":
            omega_final / U,

        "MAC":
            selected_mac,

        "iterations":
            iteration,

        "modal_vector":
            selected_q,

        "history":
            pd.DataFrame(
                history
            ),
    }


# ------------------------------------------------------------
# 4. Published Mach 0.960 point
# ------------------------------------------------------------

test960 = (
    agard_subsonic_points[
        np.isclose(
            agard_subsonic_points[
                "Mach"
            ],
            0.960,
        )
    ]
    .iloc[0]
)


Ma_960 = float(
    test960["Mach"]
)

U_960 = float(
    test960["V_m_s"]
)

q_960 = float(
    test960["q_Pa"]
)

f_exp_960 = float(
    test960[
        "flutter_frequency_Hz"
    ]
)


print(
    "Explicit-wall p-k audit:"
)

print(
    f"Mach = {Ma_960:.3f}"
)

print(
    f"U = {U_960:.3f} m/s"
)

print(
    f"q = {q_960:.3f} Pa"
)

print(
    f"Experimental flutter frequency = "
    f"{f_exp_960:.6f} Hz"
)


# ------------------------------------------------------------
# 5. Get isolated-semispan results from Cell 34
# ------------------------------------------------------------

isolated_960 = (
    agard_branch_benchmark[
        np.isclose(
            agard_branch_benchmark[
                "Mach"
            ],
            0.960,
        )
    ]
    .sort_values(
        "mode_id"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 6. Solve all four branches with explicit wall
#
# Use the already-converged isolated k values as initial
# guesses to reduce computational cost.
# ------------------------------------------------------------

wall_pk_results_960 = []

for mode_index in range(4):

    initial_k = float(
        isolated_960
        .iloc[
            mode_index
        ][
            "k_PA_1_per_m"
        ]
    )

    print(
        f"\nSolving wall Mode "
        f"{mode_index + 1} ..."
    )

    result = (
        pk_iterate_mode_wall(
            mode_index=
                mode_index,

            U=
                U_960,

            q_dyn=
                q_960,

            Ma=
                Ma_960,

            n_modes=
                4,

            initial_k=
                initial_k,
        )
    )

    wall_pk_results_960.append(
        result
    )


# ------------------------------------------------------------
# 7. Compare isolated versus explicit wall
# ------------------------------------------------------------

wall_comparison_rows = []

for mode_index in range(4):

    isolated_row = (
        isolated_960
        .iloc[
            mode_index
        ]
    )

    wall_result = (
        wall_pk_results_960[
            mode_index
        ]
    )


    wall_comparison_rows.append({
        "mode_id":
            mode_index + 1,

        "isolated_sigma":
            float(
                isolated_row[
                    "sigma_1_per_s"
                ]
            ),

        "wall_sigma":
            wall_result[
                "sigma"
            ],

        "sigma_change":
            wall_result[
                "sigma"
            ]
            - float(
                isolated_row[
                    "sigma_1_per_s"
                ]
            ),

        "isolated_frequency_Hz":
            float(
                isolated_row[
                    "frequency_Hz"
                ]
            ),

        "wall_frequency_Hz":
            wall_result[
                "frequency_Hz"
            ],

        "wall_MAC":
            wall_result[
                "MAC"
            ],

        "wall_iterations":
            wall_result[
                "iterations"
            ],
    })


wall_pk_comparison_960 = (
    pd.DataFrame(
        wall_comparison_rows
    )
)


# ------------------------------------------------------------
# 8. Identify least-stable explicit-wall branch
# ------------------------------------------------------------

critical_wall_result = max(
    wall_pk_results_960,
    key=lambda r:
        r["sigma"],
)


critical_wall_mode = (
    critical_wall_result[
        "mode_index"
    ]
    + 1
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(
    "\nIsolated-semispan vs explicit-wall p-k:"
)

display(
    wall_pk_comparison_960
)


print(
    "\nLeast-stable explicit-wall branch:"
)

print(
    f"Mode = "
    f"{critical_wall_mode}"
)

print(
    f"sigma = "
    f"{critical_wall_result['sigma']:.6f} 1/s"
)

print(
    f"damping ratio = "
    f"{critical_wall_result['damping_ratio']:.6f}"
)

print(
    f"frequency = "
    f"{critical_wall_result['frequency_Hz']:.6f} Hz"
)

print(
    f"experimental flutter frequency = "
    f"{f_exp_960:.6f} Hz"
)


if (
    critical_wall_result[
        "sigma"
    ] < 0.0
):

    print(
        "\nThe corrected wall model still predicts "
        "the published test condition as stable."
    )

elif (
    critical_wall_result[
        "sigma"
    ] > 0.0
):

    print(
        "\nThe corrected wall model predicts the "
        "published test condition as unstable."
    )

else:

    print(
        "\nThe corrected wall model is neutral "
        "at the published test condition."
    )


print(
    "\nCell 38 explicit-wall p-k audit completed."
)

In [ ]:
# Cell 39 — Infinite-plate-spline-type modal transfer
#
# Purpose:
#   Construct ONE smooth deformation surface from the published
#   structural z modal values.
#
# From that same surface obtain:
#
#       Phi_z at DLM receiving points
#       Phi_z at aerodynamic force points
#       dPhi_z/dx at receiving points
#
# This removes the current inconsistency where displacement and
# slope are interpolated independently.
#
# The implementation uses the classical 2-D thin-plate spline:
#
#   f(x,y) =
#       sum_i w_i * r_i^2 log(r_i)
#       + a0 + a1*x + a2*y
#
# This is an IPS-type diagnostic implementation.
# It is NOT claimed to reproduce MSC/Nastran SPLINE1 bit-for-bit.


# ------------------------------------------------------------
# 1. Structural interpolation coordinates
# ------------------------------------------------------------

ips_source_xy = (
    structural_xy.copy()
)

n_struct_ips = (
    ips_source_xy.shape[0]
)

assert n_struct_ips == 121


# Use one common dimensional scale for x and y.
# This improves numerical conditioning without changing the
# physical aspect ratio of the point cloud.
ips_length_ref = (
    np.ptp(
        ips_source_xy[:, 1]
    )
)

assert ips_length_ref > 0.0


ips_source_xy_nd = (
    ips_source_xy
    / ips_length_ref
)


# ------------------------------------------------------------
# 2. Thin-plate radial basis
# ------------------------------------------------------------

def tps_kernel_from_r2(r2):
    """
    phi(r) = r^2 log(r)

    with phi(0) = 0.
    """

    r2 = np.asarray(
        r2,
        dtype=float,
    )

    r = np.sqrt(r2)

    phi = np.zeros_like(
        r
    )

    mask = (
        r > 1e-14
    )

    phi[mask] = (
        r2[mask]
        * np.log(
            r[mask]
        )
    )

    return phi


# ------------------------------------------------------------
# 3. Assemble IPS interpolation system
# ------------------------------------------------------------

dx_ss = (
    ips_source_xy_nd[:, 0][:, None]
    - ips_source_xy_nd[:, 0][None, :]
)

dy_ss = (
    ips_source_xy_nd[:, 1][:, None]
    - ips_source_xy_nd[:, 1][None, :]
)

r2_ss = (
    dx_ss**2
    + dy_ss**2
)

K_ips = (
    tps_kernel_from_r2(
        r2_ss
    )
)


P_ips = np.column_stack([
    np.ones(
        n_struct_ips
    ),
    ips_source_xy_nd[:, 0],
    ips_source_xy_nd[:, 1],
])


L_ips = np.block([
    [
        K_ips,
        P_ips,
    ],
    [
        P_ips.T,
        np.zeros(
            (3, 3)
        ),
    ],
])


# ------------------------------------------------------------
# 4. Build a transfer operator rather than fitting one mode
#
# Once C_ips is known:
#
#       H @ source_values
#
# interpolates ANY structural scalar field.
# ------------------------------------------------------------

rhs_ips = np.vstack([
    np.eye(
        n_struct_ips
    ),
    np.zeros(
        (
            3,
            n_struct_ips,
        )
    ),
])


C_ips = np.linalg.solve(
    L_ips,
    rhs_ips,
)


ips_system_condition = (
    np.linalg.cond(
        L_ips
    )
)


# ------------------------------------------------------------
# 5. Generic IPS transfer + spatial derivatives
# ------------------------------------------------------------

def build_ips_transfer(
    query_xy,
):
    """
    Return IPS interpolation matrices:

        H
        H_dx
        H_dy

    such that:

        value(query) = H @ value(source)

        dvalue/dx = H_dx @ value(source)

        dvalue/dy = H_dy @ value(source)

    H_dx and H_dy are with respect to physical SI
    coordinates and therefore have units 1/m.
    """

    query_xy = np.asarray(
        query_xy,
        dtype=float,
    )

    query_nd = (
        query_xy
        / ips_length_ref
    )


    dx = (
        query_nd[:, 0][:, None]
        - ips_source_xy_nd[:, 0][None, :]
    )

    dy = (
        query_nd[:, 1][:, None]
        - ips_source_xy_nd[:, 1][None, :]
    )

    r2 = (
        dx**2
        + dy**2
    )

    r = np.sqrt(
        r2
    )


    # --------------------------------------------------------
    # TPS kernel
    # --------------------------------------------------------

    Kq = (
        tps_kernel_from_r2(
            r2
        )
    )


    # --------------------------------------------------------
    # Derivatives of:
    #
    #       r^2 log(r)
    #
    # wrt nondimensional x/y:
    #
    #       dx * (2 log(r) + 1)
    # --------------------------------------------------------

    dKdx_nd = np.zeros_like(
        r
    )

    dKdy_nd = np.zeros_like(
        r
    )

    mask = (
        r > 1e-14
    )

    common = (
        2.0
        * np.log(
            r[mask]
        )
        + 1.0
    )

    dKdx_nd[mask] = (
        dx[mask]
        * common
    )

    dKdy_nd[mask] = (
        dy[mask]
        * common
    )


    # --------------------------------------------------------
    # Polynomial terms
    # --------------------------------------------------------

    Pq = np.column_stack([
        np.ones(
            query_xy.shape[0]
        ),
        query_nd[:, 0],
        query_nd[:, 1],
    ])


    dPdx_nd = np.tile(
        np.array([
            0.0,
            1.0,
            0.0,
        ]),
        (
            query_xy.shape[0],
            1,
        ),
    )


    dPdy_nd = np.tile(
        np.array([
            0.0,
            0.0,
            1.0,
        ]),
        (
            query_xy.shape[0],
            1,
        ),
    )


    # --------------------------------------------------------
    # Full interpolation operators
    # --------------------------------------------------------

    B = np.hstack([
        Kq,
        Pq,
    ])

    B_dx_nd = np.hstack([
        dKdx_nd,
        dPdx_nd,
    ])

    B_dy_nd = np.hstack([
        dKdy_nd,
        dPdy_nd,
    ])


    H = (
        B
        @ C_ips
    )

    H_dx = (
        B_dx_nd
        @ C_ips
        / ips_length_ref
    )

    H_dy = (
        B_dy_nd
        @ C_ips
        / ips_length_ref
    )

    return (
        H,
        H_dx,
        H_dy,
    )


# ------------------------------------------------------------
# 6. Exact reconstruction at structural nodes
# ------------------------------------------------------------

(
    H_ips_nodes,
    H_ips_dx_nodes,
    H_ips_dy_nodes,
) = build_ips_transfer(
    ips_source_xy
)


ips_identity_error = (
    np.max(
        np.abs(
            H_ips_nodes
            - np.eye(
                n_struct_ips
            )
        )
    )
)


assert np.allclose(
    H_ips_nodes,
    np.eye(
        n_struct_ips
    ),
    rtol=1e-8,
    atol=1e-8,
)


# ------------------------------------------------------------
# 7. Production DLM receiving and force-point coordinates
# ------------------------------------------------------------

ips_j_xy = np.column_stack([
    aerogrid_prod[
        "offset_j"
    ][:, 0],

    aerogrid_prod[
        "offset_j"
    ][:, 1],
])


ips_k_xy = np.column_stack([
    aerogrid_prod[
        "offset_k"
    ][:, 0],

    aerogrid_prod[
        "offset_k"
    ][:, 1],
])


# ------------------------------------------------------------
# 8. Build IPS operators
# ------------------------------------------------------------

(
    H_j_ips,
    H_dx_j_ips,
    H_dy_j_ips,
) = build_ips_transfer(
    ips_j_xy
)


(
    H_k_ips,
    H_dx_k_ips,
    H_dy_k_ips,
) = build_ips_transfer(
    ips_k_xy
)


assert H_j_ips.shape == (
    320,
    121,
)

assert H_k_ips.shape == (
    320,
    121,
)


# ------------------------------------------------------------
# 9. Transfer the four NASA modal displacement fields
# ------------------------------------------------------------

Phi_z_j_ips = (
    H_j_ips
    @ Phi_z_4
)

Phi_z_k_ips = (
    H_k_ips
    @ Phi_z_4
)


# Streamwise slope derived from exactly the SAME deformation
# surface.
Phi_dzdx_j_ips = (
    H_dx_j_ips
    @ Phi_z_4
)


Phi_dzdy_j_ips = (
    H_dy_j_ips
    @ Phi_z_4
)


assert Phi_z_j_ips.shape == (
    320,
    4,
)

assert Phi_z_k_ips.shape == (
    320,
    4,
)

assert Phi_dzdx_j_ips.shape == (
    320,
    4,
)


# ------------------------------------------------------------
# 10. Compare old versus IPS modal transfer
# ------------------------------------------------------------

ips_comparison_rows = []

for mode_index in range(4):

    old_z = (
        Phi_z_j_prod[
            :,
            mode_index
        ]
    )

    ips_z = (
        Phi_z_j_ips[
            :,
            mode_index
        ]
    )

    old_slope = (
        Phi_dzdx_j_SI_prod[
            :,
            mode_index
        ]
    )

    ips_slope = (
        Phi_dzdx_j_ips[
            :,
            mode_index
        ]
    )


    displacement_difference = (
        np.linalg.norm(
            ips_z
            - old_z
        )
        /
        np.linalg.norm(
            old_z
        )
    )


    slope_difference = (
        np.linalg.norm(
            ips_slope
            - old_slope
        )
        /
        np.linalg.norm(
            old_slope
        )
    )


    slope_corr = (
        np.corrcoef(
            ips_slope,
            old_slope,
        )[0, 1]
    )


    ips_comparison_rows.append({
        "mode_id":
            mode_index + 1,

        "old_z_abs_max":
            np.max(
                np.abs(
                    old_z
                )
            ),

        "IPS_z_abs_max":
            np.max(
                np.abs(
                    ips_z
                )
            ),

        "z_difference_pct":
            100.0
            * displacement_difference,

        "old_slope_abs_max_1_per_m":
            np.max(
                np.abs(
                    old_slope
                )
            ),

        "IPS_slope_abs_max_1_per_m":
            np.max(
                np.abs(
                    ips_slope
                )
            ),

        "slope_difference_pct":
            100.0
            * slope_difference,

        "slope_correlation":
            slope_corr,
    })


ips_transfer_comparison = (
    pd.DataFrame(
        ips_comparison_rows
    )
)


# ------------------------------------------------------------
# 11. Build IPS-wall Qhh at the SAME Mach 0.960
#     experimental-frequency condition used in Cell 37
#
# Reuse the already verified explicit-wall effective AIC:
#
#     Qjj_wall_effective
#
# This avoids another expensive 640x640 DLM solve.
# ------------------------------------------------------------

W_ips_960 = -(
    Phi_dzdx_j_ips
    + 1j
    * k_wall_exp
    * Phi_z_j_ips
)


G_ips_960 = (
    Phi_z_k_ips.T
    @ np.diag(
        panel_force_weight_prod
    )
)


Qhh_wall_ips_960 = (
    G_ips_960
    @ Qjj_wall_effective
    @ W_ips_960
)


# ------------------------------------------------------------
# 12. Compare current coupling and IPS coupling
# ------------------------------------------------------------

ips_Qhh_difference = (
    np.linalg.norm(
        Qhh_wall_ips_960
        - Qhh_wall_explicit
    )
    /
    np.linalg.norm(
        Qhh_wall_explicit
    )
)


# ------------------------------------------------------------
# 13. Dynamic stiffness at the experimental flutter frequency
# ------------------------------------------------------------

D_wall_ips_960 = (
    -omega_wall_exp**2
    * M_modal_SI

    + 1j
    * omega_wall_exp
    * C_modal_SI

    + K_modal_SI

    - q_wall
    * Qhh_wall_ips_960
)


svals_ips_960 = (
    np.linalg.svd(
        D_wall_ips_960,
        compute_uv=False,
    )
)


ips_singularity_metric = (
    np.min(
        svals_ips_960
    )
    /
    np.max(
        svals_ips_960
    )
)


ips_condition_number = (
    np.max(
        svals_ips_960
    )
    /
    np.min(
        svals_ips_960
    )
)


# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print(
    "IPS structural-to-aerodynamic transfer audit:"
)

print(
    f"\nIPS system condition number: "
    f"{ips_system_condition:.6e}"
)

print(
    f"Maximum structural-node reconstruction error: "
    f"{ips_identity_error:.3e}"
)


print(
    "\nBarycentric/published-slope coupling "
    "versus smooth IPS coupling:"
)

display(
    ips_transfer_comparison
)


print(
    "\nAt Mach 0.960 and the published "
    "experimental flutter frequency:"
)

print(
    f"Relative Qhh change from coupling method: "
    f"{100.0 * ips_Qhh_difference:.3f}%"
)


print(
    "\nDynamic-stiffness singularity metric:"
)

print(
    f"Current wall model: "
    f"{wall_singularity_metric:.6e}"
)

print(
    f"IPS wall model:     "
    f"{ips_singularity_metric:.6e}"
)


print(
    "\nDynamic-stiffness condition number:"
)

print(
    f"Current wall model: "
    f"{wall_condition_number:.6e}"
)

print(
    f"IPS wall model:     "
    f"{ips_condition_number:.6e}"
)


if (
    ips_singularity_metric
    <
    wall_singularity_metric
):

    print(
        "\nThe IPS coupling moves the model "
        "closer to neutral stability at the "
        "experimental condition."
    )

else:

    print(
        "\nThe IPS coupling does not move the model "
        "closer to neutral stability at this condition."
    )


print(
    "\nCell 39 IPS coupling audit completed."
)

In [ ]:
# Cell 40 — Explicit-wall + IPS p-k audit at Mach 0.960
#
# Purpose:
#   Combine the two important corrections identified so far:
#
#       1. explicit aerodynamic wall-image wing
#       2. smooth IPS structural-to-aerodynamic transfer
#
# and solve the actual nonlinear p-k problem at the published
# Mach 0.960 flutter condition.
#
# This is still a ONE-POINT audit.
# Do not yet run all six experimental points.


# ------------------------------------------------------------
# 1. Generalized aerodynamic matrix:
#    explicit wall + IPS coupling
# ------------------------------------------------------------

import copy

# Cache only the Mach-dependent steady VLM AIC and the tiny 4x4 Qhh
# matrices. This avoids storing hundreds of large aerodynamic matrices.
_wall_vlm_A_cache = {}
_wall_qhh_cache = {}


def _wall_effective_Qjj_symmetry_reduced(Ma, k_PA):
    """
    Exact symmetry-reduced wall AIC.

    Instead of:
        1) building a 640-panel full-wing AIC,
        2) inverting the full 640x640 matrix,
        3) extracting Q_RR + Q_RL,

    exploit symmetric motion directly:

        Q_wall = -(A_RR + A_RL)^(-1)

    This reduces the dense inversion from 640x640 to 320x320.
    """

    ma_key = round(float(Ma), 10)

    # Steady VLM part depends on Mach only.
    if ma_key not in _wall_vlm_A_cache:
        A_vlm_full, _ = VLM.calc_Ajj(
            aerogrid=copy.deepcopy(aerogrid_wall_explicit),
            Ma=float(Ma),
        )
        _wall_vlm_A_cache[ma_key] = A_vlm_full
    else:
        A_vlm_full = _wall_vlm_A_cache[ma_key]

    # Unsteady DLM contribution depends on Mach and k.
    if abs(float(k_PA)) < 1.0e-14:
        A_dlm_full = np.zeros_like(A_vlm_full, dtype=complex)
    else:
        A_dlm_full = DLM.calc_Ajj(
            aerogrid=copy.deepcopy(aerogrid_wall_explicit),
            Ma=float(Ma),
            k=float(k_PA),
            method="parabolic",
        )

    A_full = A_vlm_full + A_dlm_full

    A_RR = A_full[:n_right, :n_right]
    A_RL = A_full[:n_right, n_right:]

    Q_wall = -np.linalg.inv(A_RR + A_RL)

    return Q_wall


def calculate_Qhh_wall_ips(
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Generalized aerodynamic matrix using:

        explicit wall-image aerodynamics
        +
        IPS displacement/slope transfer
        +
        exact symmetry-reduced aerodynamic solve

    Only forces on the physical right semispan are projected into
    structural generalized coordinates.
    """

    if n_modes not in (2, 3, 4):
        raise ValueError("n_modes must be 2, 3, or 4.")

    if not (0.0 <= Ma < 1.0):
        raise ValueError(
            "Current DLM benchmark is restricted to subsonic Mach numbers."
        )

    if k_PA < 0.0:
        raise ValueError("k_PA must be non-negative.")

    # A small rounded cache is safe here because the retained object is only
    # a 2x2/3x3/4x4 generalized matrix, not the full aerodynamic AIC.
    cache_key = (
        round(float(Ma), 8),
        round(float(k_PA), 8),
        int(n_modes),
    )

    if cache_key in _wall_qhh_cache:
        return _wall_qhh_cache[cache_key].copy()

    Q_wall = _wall_effective_Qjj_symmetry_reduced(
        Ma=Ma,
        k_PA=k_PA,
    )

    Phi_z_j_n = Phi_z_j_ips[:, :n_modes]
    Phi_z_k_n = Phi_z_k_ips[:, :n_modes]
    Phi_x_j_n = Phi_dzdx_j_ips[:, :n_modes]

    # Harmonic normalwash.
    W_n = -(
        Phi_x_j_n
        + 1j * k_PA * Phi_z_j_n
    )

    # Physical right-semispan generalized-force projection.
    G_n = (
        Phi_z_k_n.T
        @ np.diag(panel_force_weight_prod)
    )

    Qhh = (
        G_n
        @ Q_wall
        @ W_n
    )

    assert Qhh.shape == (n_modes, n_modes)
    assert np.isfinite(Qhh).all()

    _wall_qhh_cache[cache_key] = Qhh.copy()

    return Qhh


# ------------------------------------------------------------
# Optional equivalence check against Cell 37's explicit full inverse
# ------------------------------------------------------------

if "Qjj_wall_effective" in globals() and "k_wall_exp" in globals():
    Q_wall_fast_check = _wall_effective_Qjj_symmetry_reduced(
        Ma=Ma_wall,
        k_PA=k_wall_exp,
    )

    wall_reduction_error = (
        np.linalg.norm(Q_wall_fast_check - Qjj_wall_effective)
        / np.linalg.norm(Qjj_wall_effective)
    )

    print(
        "Symmetry-reduced wall-AIC equivalence error: "
        f"{wall_reduction_error:.3e}"
    )

    assert wall_reduction_error < 1.0e-10

# ------------------------------------------------------------
# 2. Frozen-k poles with wall + IPS
# ------------------------------------------------------------

def frozen_k_wall_ips_poles(
    U,
    q_dyn,
    Ma,
    k_PA,
    n_modes=4,
):
    """
    Frozen-k aeroelastic eigenproblem using
    explicit-wall DLM + IPS coupling.
    """

    M, C, K = (
        get_structural_modal_matrices(
            n_modes=n_modes
        )
    )

    Qhh = (
        calculate_Qhh_wall_ips(
            Ma=Ma,
            k_PA=k_PA,
            n_modes=n_modes,
        )
    )

    K_eff = (
        K
        - q_dyn
        * Qhh
    )

    M_inv = np.linalg.inv(
        M
    )

    Z = np.zeros(
        (
            n_modes,
            n_modes,
        ),
        dtype=complex,
    )

    I = np.eye(
        n_modes,
        dtype=complex,
    )

    A = np.block([
        [
            Z,
            I,
        ],
        [
            -M_inv @ K_eff,
            -M_inv @ C,
        ],
    ])

    poles, eigvecs = (
        np.linalg.eig(A)
    )

    return poles, eigvecs


# ------------------------------------------------------------
# 3. MAC-tracked p-k iteration
# ------------------------------------------------------------

def pk_iterate_mode_wall_ips(
    mode_index,
    U,
    q_dyn,
    Ma,
    n_modes=4,
    initial_k=None,
    initial_reference=None,
    tol=1e-6,
    max_iter=50,
    relaxation=0.65,
):
    """
    MAC-tracked p-k iteration using:

        explicit wall aerodynamics
        +
        IPS structural/aerodynamic transfer
    """

    if not 0 <= mode_index < n_modes:
        raise ValueError(
            "mode_index must be smaller than n_modes."
        )


    M, _, _ = (
        get_structural_modal_matrices(
            n_modes=n_modes
        )
    )


    # --------------------------------------------------------
    # Initial reduced-frequency guess
    # --------------------------------------------------------

    if initial_k is None:

        k_current = (
            omega_analysis[
                mode_index
            ]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    # --------------------------------------------------------
    # Initial mode-shape reference
    # --------------------------------------------------------

    if initial_reference is None:

        reference_q = np.zeros(
            n_modes,
            dtype=complex,
        )

        reference_q[
            mode_index
        ] = 1.0

    else:

        reference_q = np.asarray(
            initial_reference,
            dtype=complex,
        ).copy()


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    history = []

    converged = False

    selected_pole = None
    selected_q = None
    selected_mac = None


    # --------------------------------------------------------
    # p-k iteration
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):

        poles, eigvecs = (
            frozen_k_wall_ips_poles(
                U=U,
                q_dyn=q_dyn,
                Ma=Ma,
                k_PA=k_current,
                n_modes=n_modes,
            )
        )


        candidate_indices = np.where(
            np.imag(poles) > 0.0
        )[0]


        if candidate_indices.size == 0:

            raise RuntimeError(
                "No positive-frequency poles found."
            )


        candidate_macs = []

        candidate_vectors = []


        for idx in candidate_indices:

            q_candidate = (
                eigvecs[
                    :n_modes,
                    idx,
                ]
            )

            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )

            mac = complex_modal_mac(
                reference_q,
                q_candidate,
                M,
            )

            candidate_macs.append(
                mac
            )

            candidate_vectors.append(
                q_candidate
            )


        candidate_macs = np.asarray(
            candidate_macs
        )


        best_local = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_index = (
            candidate_indices[
                best_local
            ]
        )


        selected_pole = (
            poles[
                selected_index
            ]
        )


        selected_q = (
            candidate_vectors[
                best_local
            ]
        )


        selected_mac = float(
            candidate_macs[
                best_local
            ]
        )


        # ----------------------------------------------------
        # Phase align selected modal vector
        # ----------------------------------------------------

        selected_q = (
            phase_align_modal_vector(
                selected_q,
                reference_q,
                M,
            )
        )


        selected_q = (
            mass_normalize_modal_vector(
                selected_q,
                M,
            )
        )


        # ----------------------------------------------------
        # p-k consistency
        # ----------------------------------------------------

        omega_new = np.imag(
            selected_pole
        )

        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw
            +
            (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_k_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(k_current),
                1e-14,
            )
        )


        history.append({
            "iteration":
                iteration,

            "k_current":
                k_current,

            "sigma_1_per_s":
                np.real(
                    selected_pole
                ),

            "frequency_Hz":
                omega_new
                / (
                    2.0
                    * np.pi
                ),

            "MAC":
                selected_mac,

            "relative_k_change":
                relative_k_change,
        })


        reference_q = (
            selected_q.copy()
        )

        k_current = (
            k_next
        )


        if (
            relative_k_change
            < tol
        ):

            converged = True
            break


    if not converged:

        raise RuntimeError(
            f"Wall+IPS p-k iteration did not converge "
            f"for mode {mode_index + 1}."
        )


    sigma_final = np.real(
        selected_pole
    )

    omega_final = np.imag(
        selected_pole
    )


    damping_final = (
        -sigma_final
        / abs(
            selected_pole
        )
    )


    return {
        "mode_index":
            mode_index,

        "sigma":
            sigma_final,

        "frequency_Hz":
            omega_final
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            damping_final,

        "k_PA":
            omega_final
            / U,

        "MAC":
            selected_mac,

        "iterations":
            iteration,

        "modal_vector":
            selected_q.copy(),

        "history":
            pd.DataFrame(
                history
            ),
    }


# ------------------------------------------------------------
# 4. Mach 0.960 experimental condition
# ------------------------------------------------------------

print(
    "Wall + IPS p-k benchmark audit:"
)

print(
    f"Mach = {Ma_960:.3f}"
)

print(
    f"U = {U_960:.3f} m/s"
)

print(
    f"q = {q_960:.3f} Pa"
)

print(
    f"Experimental flutter frequency = "
    f"{f_exp_960:.6f} Hz"
)


# ------------------------------------------------------------
# 5. Solve all four branches
#
# Use Cell 38 wall-only solution as initial k/reference.
# This reduces computational cost and strengthens continuity.
# ------------------------------------------------------------

wall_ips_results_960 = []


for mode_index in range(4):

    previous_wall = (
        wall_pk_results_960[
            mode_index
        ]
    )


    print(
        f"\nSolving wall+IPS Mode "
        f"{mode_index + 1} ..."
    )


    result = (
        pk_iterate_mode_wall_ips(
            mode_index=
                mode_index,

            U=
                U_960,

            q_dyn=
                q_960,

            Ma=
                Ma_960,

            n_modes=
                4,

            initial_k=
                previous_wall[
                    "k_PA"
                ],

            initial_reference=
                previous_wall[
                    "modal_vector"
                ],
        )
    )


    wall_ips_results_960.append(
        result
    )


# ------------------------------------------------------------
# 6. Comparison:
#
# isolated
# wall-only
# wall + IPS
# ------------------------------------------------------------

wall_ips_comparison_rows = []


for mode_index in range(4):

    isolated_row = (
        isolated_960
        .iloc[
            mode_index
        ]
    )


    wall_result = (
        wall_pk_results_960[
            mode_index
        ]
    )


    ips_result = (
        wall_ips_results_960[
            mode_index
        ]
    )


    wall_ips_comparison_rows.append({
        "mode_id":
            mode_index + 1,

        "isolated_sigma":
            float(
                isolated_row[
                    "sigma_1_per_s"
                ]
            ),

        "wall_sigma":
            wall_result[
                "sigma"
            ],

        "wall_IPS_sigma":
            ips_result[
                "sigma"
            ],

        "isolated_frequency_Hz":
            float(
                isolated_row[
                    "frequency_Hz"
                ]
            ),

        "wall_frequency_Hz":
            wall_result[
                "frequency_Hz"
            ],

        "wall_IPS_frequency_Hz":
            ips_result[
                "frequency_Hz"
            ],

        "wall_IPS_damping":
            ips_result[
                "damping_ratio"
            ],

        "wall_IPS_MAC":
            ips_result[
                "MAC"
            ],

        "iterations":
            ips_result[
                "iterations"
            ],
    })


wall_ips_comparison_960 = (
    pd.DataFrame(
        wall_ips_comparison_rows
    )
)


# ------------------------------------------------------------
# 7. Identify least-stable IPS branch
# ------------------------------------------------------------

critical_ips = max(
    wall_ips_results_960,
    key=lambda r:
        r["sigma"],
)


critical_ips_mode = (
    critical_ips[
        "mode_index"
    ]
    + 1
)


critical_frequency_error = (
    100.0
    * (
        critical_ips[
            "frequency_Hz"
        ]
        - f_exp_960
    )
    / f_exp_960
)


# ------------------------------------------------------------
# 8. Also find branch closest in frequency to experiment
#
# This is descriptive only.
# It is NOT automatically called the flutter mode.
# ------------------------------------------------------------

frequency_closest_ips = min(
    wall_ips_results_960,
    key=lambda r:
        abs(
            r["frequency_Hz"]
            - f_exp_960
        ),
)


frequency_closest_mode = (
    frequency_closest_ips[
        "mode_index"
    ]
    + 1
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(
    "\nThree-stage coupling comparison:"
)

display(
    wall_ips_comparison_960
)


print(
    "\nLeast-stable wall+IPS branch:"
)

print(
    f"Mode = "
    f"{critical_ips_mode}"
)

print(
    f"sigma = "
    f"{critical_ips['sigma']:.6f} 1/s"
)

print(
    f"damping ratio = "
    f"{critical_ips['damping_ratio']:.6f}"
)

print(
    f"frequency = "
    f"{critical_ips['frequency_Hz']:.6f} Hz"
)

print(
    f"frequency difference from experiment = "
    f"{critical_frequency_error:.3f}%"
)


print(
    "\nBranch closest to experimental frequency:"
)

print(
    f"Mode = "
    f"{frequency_closest_mode}"
)

print(
    f"sigma = "
    f"{frequency_closest_ips['sigma']:.6f} 1/s"
)

print(
    f"frequency = "
    f"{frequency_closest_ips['frequency_Hz']:.6f} Hz"
)

print(
    f"experimental frequency = "
    f"{f_exp_960:.6f} Hz"
)


if (
    critical_ips[
        "sigma"
    ] < 0.0
):

    print(
        "\nThe wall+IPS model remains stable "
        "at the measured flutter condition."
    )

elif (
    critical_ips[
        "sigma"
    ] > 0.0
):

    print(
        "\nThe wall+IPS model is already unstable "
        "at the measured flutter condition."
    )

else:

    print(
        "\nThe wall+IPS model is neutral "
        "at the measured flutter condition."
    )


print(
    "\nCell 40 wall+IPS p-k audit completed."
)

In [ ]:
# Cell 41 — Coarse flutter-boundary continuation
# Mach 0.960, explicit wall + IPS coupling
#
# Goal:
#   Track Mode 1 through the experimental condition and find
#   a bracket where sigma changes sign.
#
# Flutter condition:
#
#       sigma = 0
#
# Scale definition:
#
#       U = lambda * U_exp
#       q = lambda^2 * q_exp
#
# Thus lambda = 1.0 is exactly the published experimental point.


# ------------------------------------------------------------
# 1. Experimental reference condition
# ------------------------------------------------------------

Ma_search = Ma_960
U_exp_search = U_960
q_exp_search = q_960
f_exp_search = f_exp_960


# ------------------------------------------------------------
# 2. Coarse continuation points
#
# Concentrated around lambda = 1 because Cell 40 already
# showed that Mode 1 is relatively close to neutral.
# ------------------------------------------------------------

lambda_values = np.array([
    0.90,
    0.95,
    1.00,
    1.05,
    1.10,
    1.15,
    1.20,
])


# ------------------------------------------------------------
# 3. Initial continuation state
#
# Begin from the converged Mode-1 solution from Cell 40.
# ------------------------------------------------------------

mode1_cell40 = (
    wall_ips_results_960[0]
)

previous_k = (
    mode1_cell40["k_PA"]
)

previous_reference = (
    mode1_cell40[
        "modal_vector"
    ].copy()
)


# ------------------------------------------------------------
# 4. Storage
# ------------------------------------------------------------

mode1_boundary_rows = []


# ------------------------------------------------------------
# 5. Continuation sweep
#
# We move through lambda in ascending order.
#
# For each point:
#
#       U(lambda)
#       q(lambda)
#
# and track the same aeroelastic branch using MAC.
# ------------------------------------------------------------

for lam in lambda_values:

    U_test = (
        lam
        * U_exp_search
    )

    q_test = (
        lam**2
        * q_exp_search
    )


    print(
        f"\nSolving lambda = {lam:.3f}"
    )

    print(
        f"U = {U_test:.3f} m/s, "
        f"q = {q_test:.1f} Pa"
    )


    result = (
        pk_iterate_mode_wall_ips(
            mode_index=0,

            U=U_test,

            q_dyn=q_test,

            Ma=Ma_search,

            n_modes=4,

            initial_k=previous_k,

            initial_reference=
                previous_reference,

            tol=1e-6,

            max_iter=50,

            relaxation=0.65,
        )
    )


    mode1_boundary_rows.append({
        "lambda":
            lam,

        "U_m_s":
            U_test,

        "q_Pa":
            q_test,

        "sigma_1_per_s":
            result[
                "sigma"
            ],

        "damping_ratio":
            result[
                "damping_ratio"
            ],

        "frequency_Hz":
            result[
                "frequency_Hz"
            ],

        "frequency_error_pct":
            100.0
            * (
                result[
                    "frequency_Hz"
                ]
                - f_exp_search
            )
            / f_exp_search,

        "k_PA_1_per_m":
            result[
                "k_PA"
            ],

        "MAC":
            result[
                "MAC"
            ],

        "iterations":
            result[
                "iterations"
            ],
    })


    # Continue using the converged solution
    previous_k = (
        result[
            "k_PA"
        ]
    )

    previous_reference = (
        result[
            "modal_vector"
        ].copy()
    )


# ------------------------------------------------------------
# 6. Assemble table
# ------------------------------------------------------------

mode1_boundary_scan = (
    pd.DataFrame(
        mode1_boundary_rows
    )
)


# ------------------------------------------------------------
# 7. Detect stable-to-unstable sign changes
#
# sigma < 0 : stable
# sigma > 0 : unstable
# ------------------------------------------------------------

flutter_brackets = []

for i in range(
    len(
        mode1_boundary_scan
    ) - 1
):

    sigma_a = float(
        mode1_boundary_scan
        .iloc[i][
            "sigma_1_per_s"
        ]
    )

    sigma_b = float(
        mode1_boundary_scan
        .iloc[i + 1][
            "sigma_1_per_s"
        ]
    )


    if (
        sigma_a <= 0.0
        and sigma_b >= 0.0
    ):

        flutter_brackets.append({
            "lambda_low":
                float(
                    mode1_boundary_scan
                    .iloc[i][
                        "lambda"
                    ]
                ),

            "lambda_high":
                float(
                    mode1_boundary_scan
                    .iloc[i + 1][
                        "lambda"
                    ]
                ),

            "U_low_m_s":
                float(
                    mode1_boundary_scan
                    .iloc[i][
                        "U_m_s"
                    ]
                ),

            "U_high_m_s":
                float(
                    mode1_boundary_scan
                    .iloc[i + 1][
                        "U_m_s"
                    ]
                ),

            "sigma_low":
                sigma_a,

            "sigma_high":
                sigma_b,
        })


flutter_bracket_table = (
    pd.DataFrame(
        flutter_brackets
    )
)


# ------------------------------------------------------------
# 8. Display continuation results
# ------------------------------------------------------------

print(
    "\nMode-1 wall+IPS continuation:"
)

display(
    mode1_boundary_scan
)


# ------------------------------------------------------------
# 9. Plot sigma
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    mode1_boundary_scan[
        "U_m_s"
    ],
    mode1_boundary_scan[
        "sigma_1_per_s"
    ],
    "o-",
)

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.axvline(
    U_exp_search,
    linestyle="--",
    linewidth=1.0,
    label="Experiment",
)

plt.xlabel(
    "Velocity U (m/s)"
)

plt.ylabel(
    "Mode-1 pole real part σ (1/s)"
)

plt.title(
    "AGARD 445.6 Mode-1 flutter-boundary search — Mach 0.960"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. Plot frequency evolution
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    mode1_boundary_scan[
        "U_m_s"
    ],
    mode1_boundary_scan[
        "frequency_Hz"
    ],
    "o-",
    label="Predicted Mode 1",
)

plt.axhline(
    f_exp_search,
    linestyle="--",
    linewidth=1.0,
    label="Experimental flutter frequency",
)

plt.axvline(
    U_exp_search,
    linestyle=":",
    linewidth=1.0,
    label="Experimental flutter velocity",
)

plt.xlabel(
    "Velocity U (m/s)"
)

plt.ylabel(
    "Frequency (Hz)"
)

plt.title(
    "Mode-1 frequency evolution — Mach 0.960"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. Report bracket
# ------------------------------------------------------------

if len(
    flutter_bracket_table
) > 0:

    print(
        "\nStable-to-unstable flutter bracket found:"
    )

    display(
        flutter_bracket_table
    )

else:

    print(
        "\nNo sigma = 0 crossing found "
        "inside the current lambda range."
    )


print(
    "\nExperimental reference:"
)

print(
    f"U_exp = "
    f"{U_exp_search:.3f} m/s"
)

print(
    f"q_exp = "
    f"{q_exp_search:.3f} Pa"
)

print(
    f"f_exp = "
    f"{f_exp_search:.6f} Hz"
)


print(
    "\nCell 41 coarse Mode-1 boundary search completed."
)

In [ ]:
# Cell 42 — Refined flutter-boundary solution at Mach 0.960
#
# Corrected baseline:
#   - 4 structural modes
#   - measured modal frequencies
#   - explicit wall-image DLM
#   - IPS structural/aerodynamic coupling
#   - MAC-tracked nonlinear p-k iteration
#
# Flutter criterion:
#
#       sigma(lambda_f) = 0
#
# Starting bracket from Cell 41:
#
#       lambda = 1.10 ... 1.15


# ------------------------------------------------------------
# 1. Initial flutter bracket from Cell 41
# ------------------------------------------------------------

lambda_low = 1.10
lambda_high = 1.15

sigma_tolerance = 1.0e-3       # 1/s
lambda_tolerance = 1.0e-5
max_root_iterations = 15


# ------------------------------------------------------------
# 2. Helper for one Mode-1 solution at a given lambda
# ------------------------------------------------------------

def solve_mode1_at_lambda(
    lam,
    initial_k,
    initial_reference,
):
    """
    Solve the corrected wall+IPS Mode-1 p-k problem
    at a velocity/dynamic-pressure scale lambda.
    """

    U_test = (
        lam
        * U_exp_search
    )

    q_test = (
        lam**2
        * q_exp_search
    )

    result = (
        pk_iterate_mode_wall_ips(
            mode_index=0,

            U=U_test,

            q_dyn=q_test,

            Ma=Ma_search,

            n_modes=4,

            initial_k=initial_k,

            initial_reference=
                initial_reference,

            tol=1e-7,

            max_iter=60,

            relaxation=0.65,
        )
    )

    return (
        U_test,
        q_test,
        result,
    )


# ------------------------------------------------------------
# 3. Solve lower bracket endpoint
# ------------------------------------------------------------

print(
    "Re-solving lower flutter bracket..."
)

U_low, q_low, result_low = (
    solve_mode1_at_lambda(
        lam=lambda_low,

        initial_k=
            mode1_cell40[
                "k_PA"
            ],

        initial_reference=
            mode1_cell40[
                "modal_vector"
            ],
    )
)

sigma_low = (
    result_low[
        "sigma"
    ]
)


# ------------------------------------------------------------
# 4. Solve upper bracket endpoint
# ------------------------------------------------------------

print(
    "Re-solving upper flutter bracket..."
)

U_high, q_high, result_high = (
    solve_mode1_at_lambda(
        lam=lambda_high,

        initial_k=
            result_low[
                "k_PA"
            ],

        initial_reference=
            result_low[
                "modal_vector"
            ],
    )
)

sigma_high = (
    result_high[
        "sigma"
    ]
)


assert sigma_low < 0.0
assert sigma_high > 0.0


print(
    "\nInitial verified bracket:"
)

print(
    f"lambda_low  = {lambda_low:.6f}, "
    f"sigma_low  = {sigma_low:.6f} 1/s"
)

print(
    f"lambda_high = {lambda_high:.6f}, "
    f"sigma_high = {sigma_high:.6f} 1/s"
)


# ------------------------------------------------------------
# 5. Safeguarded secant / false-position root search
#
# Because sigma(lambda) is smooth in Cell 41, a secant estimate
# is efficient.
#
# If the estimate becomes too close to a bracket edge,
# use the midpoint instead.
# ------------------------------------------------------------

root_history = []

flutter_result = None
lambda_flutter = None
U_flutter = None
q_flutter = None


for root_iteration in range(
    1,
    max_root_iterations + 1
):

    # --------------------------------------------------------
    # False-position estimate
    # --------------------------------------------------------

    lambda_trial = (
        lambda_low
        - sigma_low
        * (
            lambda_high
            - lambda_low
        )
        / (
            sigma_high
            - sigma_low
        )
    )


    # --------------------------------------------------------
    # Safeguard against pathological secant placement
    # --------------------------------------------------------

    bracket_width = (
        lambda_high
        - lambda_low
    )

    edge_margin = (
        0.10
        * bracket_width
    )

    if (
        lambda_trial
        <= lambda_low
        + edge_margin
        or
        lambda_trial
        >= lambda_high
        - edge_margin
    ):

        lambda_trial = (
            0.5
            * (
                lambda_low
                + lambda_high
            )
        )


    # --------------------------------------------------------
    # Use closer endpoint as continuation reference
    # --------------------------------------------------------

    if (
        abs(
            lambda_trial
            - lambda_low
        )
        <=
        abs(
            lambda_high
            - lambda_trial
        )
    ):

        initial_result = (
            result_low
        )

    else:

        initial_result = (
            result_high
        )


    # --------------------------------------------------------
    # Solve p-k at trial point
    # --------------------------------------------------------

    U_trial, q_trial, result_trial = (
        solve_mode1_at_lambda(
            lam=lambda_trial,

            initial_k=
                initial_result[
                    "k_PA"
                ],

            initial_reference=
                initial_result[
                    "modal_vector"
                ],
        )
    )


    sigma_trial = (
        result_trial[
            "sigma"
        ]
    )


    root_history.append({
        "iteration":
            root_iteration,

        "lambda":
            lambda_trial,

        "U_m_s":
            U_trial,

        "q_Pa":
            q_trial,

        "sigma_1_per_s":
            sigma_trial,

        "frequency_Hz":
            result_trial[
                "frequency_Hz"
            ],

        "damping_ratio":
            result_trial[
                "damping_ratio"
            ],

        "k_PA_1_per_m":
            result_trial[
                "k_PA"
            ],

        "MAC":
            result_trial[
                "MAC"
            ],

        "pk_iterations":
            result_trial[
                "iterations"
            ],

        "bracket_width":
            bracket_width,
    })


    print(
        f"\nRoot iteration {root_iteration}:"
    )

    print(
        f"lambda = {lambda_trial:.8f}"
    )

    print(
        f"U = {U_trial:.4f} m/s"
    )

    print(
        f"q = {q_trial:.3f} Pa"
    )

    print(
        f"sigma = {sigma_trial:.6f} 1/s"
    )

    print(
        f"f = "
        f"{result_trial['frequency_Hz']:.6f} Hz"
    )


    # --------------------------------------------------------
    # Convergence
    # --------------------------------------------------------

    if (
        abs(
            sigma_trial
        )
        < sigma_tolerance
        or
        bracket_width
        < lambda_tolerance
    ):

        flutter_result = (
            result_trial
        )

        lambda_flutter = (
            lambda_trial
        )

        U_flutter = (
            U_trial
        )

        q_flutter = (
            q_trial
        )

        break


    # --------------------------------------------------------
    # Update sign-changing bracket
    # --------------------------------------------------------

    if sigma_trial < 0.0:

        lambda_low = (
            lambda_trial
        )

        sigma_low = (
            sigma_trial
        )

        result_low = (
            result_trial
        )

        U_low = (
            U_trial
        )

        q_low = (
            q_trial
        )

    else:

        lambda_high = (
            lambda_trial
        )

        sigma_high = (
            sigma_trial
        )

        result_high = (
            result_trial
        )

        U_high = (
            U_trial
        )

        q_high = (
            q_trial
        )


else:

    raise RuntimeError(
        "Flutter root search did not converge."
    )


# ------------------------------------------------------------
# 6. Root-search history
# ------------------------------------------------------------

flutter_root_history = (
    pd.DataFrame(
        root_history
    )
)


print(
    "\nFlutter-root convergence history:"
)

display(
    flutter_root_history
)


# ------------------------------------------------------------
# 7. Experimental comparison
# ------------------------------------------------------------

f_flutter = (
    flutter_result[
        "frequency_Hz"
    ]
)


velocity_error_pct = (
    100.0
    * (
        U_flutter
        - U_exp_search
    )
    / U_exp_search
)


q_error_pct = (
    100.0
    * (
        q_flutter
        - q_exp_search
    )
    / q_exp_search
)


frequency_error_pct = (
    100.0
    * (
        f_flutter
        - f_exp_search
    )
    / f_exp_search
)


# ------------------------------------------------------------
# 8. Final comparison table
# ------------------------------------------------------------

flutter_comparison_096 = (
    pd.DataFrame({
        "quantity": [
            "Velocity U (m/s)",
            "Dynamic pressure q (Pa)",
            "Flutter frequency (Hz)",
        ],

        "experiment": [
            U_exp_search,
            q_exp_search,
            f_exp_search,
        ],

        "wall_IPS_prediction": [
            U_flutter,
            q_flutter,
            f_flutter,
        ],

        "error_pct": [
            velocity_error_pct,
            q_error_pct,
            frequency_error_pct,
        ],
    })
)


print(
    "\nMach 0.960 refined flutter comparison:"
)

display(
    flutter_comparison_096
)


# ------------------------------------------------------------
# 9. Final flutter result
# ------------------------------------------------------------

print(
    "\nPredicted flutter solution:"
)

print(
    f"Mach                  = "
    f"{Ma_search:.3f}"
)

print(
    f"lambda_flutter        = "
    f"{lambda_flutter:.8f}"
)

print(
    f"U_flutter             = "
    f"{U_flutter:.4f} m/s"
)

print(
    f"q_flutter             = "
    f"{q_flutter:.3f} Pa"
)

print(
    f"f_flutter             = "
    f"{f_flutter:.6f} Hz"
)

print(
    f"sigma_flutter         = "
    f"{flutter_result['sigma']:.6e} 1/s"
)

print(
    f"damping ratio         = "
    f"{flutter_result['damping_ratio']:.6e}"
)

print(
    f"MAC                    = "
    f"{flutter_result['MAC']:.6f}"
)


print(
    "\nExperimental errors:"
)

print(
    f"Velocity error         = "
    f"{velocity_error_pct:+.3f}%"
)

print(
    f"Dynamic-pressure error = "
    f"{q_error_pct:+.3f}%"
)

print(
    f"Frequency error        = "
    f"{frequency_error_pct:+.3f}%"
)


print(
    "\nCell 42 refined flutter solution completed."
)

In [ ]:
# Cell 43 — Automated representative AGARD validation (runtime-safe)
#
# Representative points:
#     Mach 0.499
#     Mach 0.678
#     Mach 0.901
# plus the already completed Mach 0.960 result from Cell 42.
#
# This version avoids the previous brute-force 0.70...1.50 sweep.
# It adaptively expands from the experimental condition until a
# stable-to-unstable Mode-1 bracket is found.
#
# IMPORTANT:
#   Failure to find a bracket is reported in the output table rather
#   than terminating the whole notebook with RuntimeError.


# ------------------------------------------------------------
# 1. Representative experimental points
# ------------------------------------------------------------

validation_machs = np.array([
    0.499,
    0.678,
    0.901,
])

validation_mask = np.zeros(
    len(agard_subsonic_points),
    dtype=bool,
)

for ma_target in validation_machs:
    validation_mask |= np.isclose(
        agard_subsonic_points["Mach"].to_numpy(dtype=float),
        ma_target,
        atol=1e-9,
        rtol=0.0,
    )

validation_points = (
    agard_subsonic_points[
        validation_mask
    ]
    .sort_values("Mach")
    .reset_index(drop=True)
)

assert len(validation_points) == 3


# ------------------------------------------------------------
# 2. One Mode-1 solve at a lambda-scaled test condition
# ------------------------------------------------------------

def _solve_validation_lambda(
    Ma,
    U_exp,
    q_exp,
    lam,
    previous_result=None,
):
    U_test = float(lam) * float(U_exp)
    q_test = float(lam) ** 2 * float(q_exp)

    if previous_result is None:
        initial_k = omega_analysis[0] / U_test
        initial_reference = None
    else:
        initial_k = previous_result["k_PA"]
        initial_reference = previous_result["modal_vector"]

    result = pk_iterate_mode_wall_ips(
        mode_index=0,
        U=U_test,
        q_dyn=q_test,
        Ma=float(Ma),
        n_modes=4,
        initial_k=initial_k,
        initial_reference=initial_reference,
        tol=2e-6,
        max_iter=45,
        relaxation=0.65,
    )

    return {
        "lambda": float(lam),
        "U": U_test,
        "q": q_test,
        "sigma": float(result["sigma"]),
        "frequency": float(result["frequency_Hz"]),
        "result": result,
    }


# ------------------------------------------------------------
# 3. Adaptive bracket + root refinement
# ------------------------------------------------------------

def solve_flutter_point_adaptive(
    Ma,
    U_exp,
    q_exp,
    f_exp,
    sigma_tol=2e-3,
    lambda_tol=2e-4,
    lambda_floor=0.55,
    lambda_ceiling=1.80,
    max_bracket_steps=7,
    max_root_iter=12,
):
    """
    Find the first Mode-1 neutral-stability crossing near the
    experimental condition without an expensive uniform scan.

    Starts at lambda=1.0. If stable, move upward.
    If unstable, move downward. Step size grows gradually.

    Returns a result dictionary even if no bracket is found.
    """

    print(
        f"\nMach {Ma:.3f}: starting adaptive bracket at lambda = 1.000"
    )

    center = _solve_validation_lambda(
        Ma=Ma,
        U_exp=U_exp,
        q_exp=q_exp,
        lam=1.0,
        previous_result=None,
    )

    print(
        f"  lambda=1.0000, sigma={center['sigma']:+.5f} 1/s, "
        f"f={center['frequency']:.4f} Hz"
    )

    # Direction: stable -> move upward; unstable -> move downward.
    direction = +1.0 if center["sigma"] < 0.0 else -1.0

    current = center
    bracket = None
    step = 0.12

    for j in range(max_bracket_steps):

        lam_next = current["lambda"] + direction * step
        lam_next = min(max(lam_next, lambda_floor), lambda_ceiling)

        if abs(lam_next - current["lambda"]) < 1e-12:
            break

        nxt = _solve_validation_lambda(
            Ma=Ma,
            U_exp=U_exp,
            q_exp=q_exp,
            lam=lam_next,
            previous_result=current["result"],
        )

        print(
            f"  lambda={lam_next:.4f}, sigma={nxt['sigma']:+.5f} 1/s, "
            f"f={nxt['frequency']:.4f} Hz"
        )

        # Sign change in either direction.
        if current["sigma"] * nxt["sigma"] <= 0.0:
            low = current if current["lambda"] < nxt["lambda"] else nxt
            high = nxt if current["lambda"] < nxt["lambda"] else current

            # Ensure low is stable and high is unstable.
            if low["sigma"] <= 0.0 and high["sigma"] >= 0.0:
                bracket = (low, high)
            elif high["sigma"] <= 0.0 and low["sigma"] >= 0.0:
                bracket = (high, low)
            else:
                # Unusual non-monotonic sign orientation: sort by sigma.
                stable = current if current["sigma"] <= 0.0 else nxt
                unstable = nxt if current["sigma"] <= 0.0 else current
                bracket = (stable, unstable)
            break

        current = nxt
        step *= 1.35

    if bracket is None:
        print(
            f"  WARNING: no Mode-1 sign-change bracket found for Mach {Ma:.3f} "
            f"within lambda [{lambda_floor:.2f}, {lambda_ceiling:.2f}]."
        )

        return {
            "Mach": float(Ma),
            "status": "NO_BRACKET",
            "U_exp_m_s": float(U_exp),
            "U_pred_m_s": np.nan,
            "U_error_pct": np.nan,
            "q_exp_Pa": float(q_exp),
            "q_pred_Pa": np.nan,
            "q_error_pct": np.nan,
            "f_exp_Hz": float(f_exp),
            "f_pred_Hz": np.nan,
            "f_error_pct": np.nan,
            "lambda_flutter": np.nan,
            "sigma_final": np.nan,
            "MAC": np.nan,
        }

    low, high = bracket

    print(
        f"  bracket: lambda {low['lambda']:.5f} "
        f"(sigma {low['sigma']:+.5f}) to "
        f"{high['lambda']:.5f} "
        f"(sigma {high['sigma']:+.5f})"
    )

    # --------------------------------------------------------
    # Safeguarded false-position refinement
    # --------------------------------------------------------

    root = None

    for it in range(1, max_root_iter + 1):

        lam_low = low["lambda"]
        lam_high = high["lambda"]
        sig_low = low["sigma"]
        sig_high = high["sigma"]

        lam_trial = (
            lam_low
            - sig_low * (lam_high - lam_low) / (sig_high - sig_low)
        )

        # Keep the trial away from an endpoint so false position
        # cannot stagnate.
        width = lam_high - lam_low
        margin = 0.08 * width

        if (
            lam_trial <= lam_low + margin
            or lam_trial >= lam_high - margin
        ):
            lam_trial = 0.5 * (lam_low + lam_high)

        ref = (
            low
            if abs(lam_trial - lam_low) <= abs(lam_high - lam_trial)
            else high
        )

        trial = _solve_validation_lambda(
            Ma=Ma,
            U_exp=U_exp,
            q_exp=q_exp,
            lam=lam_trial,
            previous_result=ref["result"],
        )

        print(
            f"    root {it:02d}: lambda={lam_trial:.7f}, "
            f"sigma={trial['sigma']:+.6f}, "
            f"f={trial['frequency']:.5f} Hz"
        )

        if (
            abs(trial["sigma"]) <= sigma_tol
            or (lam_high - lam_low) <= lambda_tol
        ):
            root = trial
            break

        if trial["sigma"] < 0.0:
            low = trial
        else:
            high = trial

    if root is None:
        root = min(
            [low, high],
            key=lambda item: abs(item["sigma"]),
        )

    velocity_error = (
        100.0 * (root["U"] - U_exp) / U_exp
    )

    q_error = (
        100.0 * (root["q"] - q_exp) / q_exp
    )

    frequency_error = (
        100.0 * (root["frequency"] - f_exp) / f_exp
    )

    return {
        "Mach": float(Ma),
        "status": "OK",
        "U_exp_m_s": float(U_exp),
        "U_pred_m_s": root["U"],
        "U_error_pct": velocity_error,
        "q_exp_Pa": float(q_exp),
        "q_pred_Pa": root["q"],
        "q_error_pct": q_error,
        "f_exp_Hz": float(f_exp),
        "f_pred_Hz": root["frequency"],
        "f_error_pct": frequency_error,
        "lambda_flutter": root["lambda"],
        "sigma_final": root["sigma"],
        "MAC": root["result"]["MAC"],
    }


# ------------------------------------------------------------
# 4. Run representative lower-Mach validation cases
# ------------------------------------------------------------

representative_results = []

for _, row in validation_points.iterrows():

    Ma = float(row["Mach"])
    U_exp = float(row["V_m_s"])
    q_exp = float(row["q_Pa"])
    f_exp = float(row["flutter_frequency_Hz"])

    print("\n" + "-" * 55)
    print(f"Solving representative Mach {Ma:.3f}")
    print("-" * 55)

    result = solve_flutter_point_adaptive(
        Ma=Ma,
        U_exp=U_exp,
        q_exp=q_exp,
        f_exp=f_exp,
    )

    representative_results.append(result)


# ------------------------------------------------------------
# 5. Add completed Mach 0.960 result from Cell 42
# ------------------------------------------------------------

representative_results.append({
    "Mach": 0.960,
    "status": "OK",
    "U_exp_m_s": U_exp_search,
    "U_pred_m_s": U_flutter,
    "U_error_pct": velocity_error_pct,
    "q_exp_Pa": q_exp_search,
    "q_pred_Pa": q_flutter,
    "q_error_pct": q_error_pct,
    "f_exp_Hz": f_exp_search,
    "f_pred_Hz": f_flutter,
    "f_error_pct": frequency_error_pct,
    "lambda_flutter": lambda_flutter,
    "sigma_final": flutter_result["sigma"],
    "MAC": flutter_result["MAC"],
})


# ------------------------------------------------------------
# 6. Final validation table
# ------------------------------------------------------------

agard_representative_validation = (
    pd.DataFrame(representative_results)
    .sort_values("Mach")
    .reset_index(drop=True)
)

print("\nRepresentative AGARD validation:")
display(agard_representative_validation)


# ------------------------------------------------------------
# 7. Aggregate statistics — valid solved rows only
# ------------------------------------------------------------

valid_rows = (
    agard_representative_validation["status"] == "OK"
)

if valid_rows.any():

    mean_abs_U_error = (
        agard_representative_validation.loc[
            valid_rows,
            "U_error_pct",
        ]
        .abs()
        .mean()
    )

    mean_abs_f_error = (
        agard_representative_validation.loc[
            valid_rows,
            "f_error_pct",
        ]
        .abs()
        .mean()
    )

    print("\nRepresentative validation statistics:")
    print(
        "Mean absolute flutter-velocity error = "
        f"{mean_abs_U_error:.3f}%"
    )
    print(
        "Mean absolute flutter-frequency error = "
        f"{mean_abs_f_error:.3f}%"
    )


# ------------------------------------------------------------
# 8. Comparison plots — solved points only
# ------------------------------------------------------------

plot_df = (
    agard_representative_validation[
        valid_rows
    ]
    .copy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    plot_df["Mach"],
    plot_df["U_exp_m_s"],
    "o-",
    label="Experiment",
)

plt.plot(
    plot_df["Mach"],
    plot_df["U_pred_m_s"],
    "s--",
    label="Wall + IPS DLM",
)

plt.xlabel("Mach number")
plt.ylabel("Flutter velocity (m/s)")
plt.title("AGARD 445.6 representative flutter validation")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    plot_df["Mach"],
    plot_df["f_exp_Hz"],
    "o-",
    label="Experiment",
)

plt.plot(
    plot_df["Mach"],
    plot_df["f_pred_Hz"],
    "s--",
    label="Wall + IPS DLM",
)

plt.xlabel("Mach number")
plt.ylabel("Flutter frequency (Hz)")
plt.title("AGARD 445.6 representative flutter-frequency validation")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


print("\nCell 43 runtime-safe representative validation completed.")


In [ ]:
# Cell 44 — Correct conventional p-k audit
# Mach 0.960, Mode 1 only
#
# IMPORTANT:
# Cells 40-43 used a frequency-consistent complex-stiffness
# eigenproblem. This cell replaces that eigenproblem with the
# conventional frozen-k p-k quadratic form:
#
#   M p^2
# + [C - q*QI/omega] p
# + [K - q*QR] = 0
#
# where:
#
#   Qhh(k) = QR + i QI
#   omega  = k * U
#
# At p = i*omega this exactly reproduces:
#
#   -omega^2 M + i omega C + K - q Qhh = 0
#
# We first audit ONLY the Mach 0.960 experimental condition.


# ------------------------------------------------------------
# 1. Recover cached 4-mode generalized aerodynamic matrices
#    at Mach 0.960
# ------------------------------------------------------------

def collect_cached_qhh(
    Ma_target,
    n_modes=4,
    mach_tolerance=1e-6,
):
    """
    Extract already-computed Qhh(k) matrices from Cell 40 cache.
    """

    k_values = []
    qhh_values = []

    for key, Q in _wall_qhh_cache.items():

        ma_key, k_key, nm_key = key

        if (
            nm_key == n_modes
            and abs(
                float(ma_key)
                - float(Ma_target)
            )
            < mach_tolerance
        ):

            k_values.append(
                float(k_key)
            )

            qhh_values.append(
                np.asarray(
                    Q,
                    dtype=complex,
                )
            )


    if len(k_values) == 0:

        return (
            np.array([]),
            np.empty(
                (
                    0,
                    n_modes,
                    n_modes,
                ),
                dtype=complex,
            ),
        )


    order = np.argsort(
        k_values
    )


    k_values = np.asarray(
        k_values
    )[order]


    qhh_values = np.asarray(
        qhh_values
    )[order]


    return (
        k_values,
        qhh_values,
    )


k_cache_096, Q_cache_096 = (
    collect_cached_qhh(
        Ma_target=0.960,
        n_modes=4,
    )
)


print(
    f"Cached Mach 0.960 Qhh matrices: "
    f"{len(k_cache_096)}"
)


# ------------------------------------------------------------
# 2. Make sure Mode-1 p-k region is covered
#
# Expected region is approximately k = 0.20...0.35 1/m.
#
# Usually Cell 42 already generated many points here.
# Only fill missing coverage if necessary.
# ------------------------------------------------------------

required_k_min = 0.20
required_k_max = 0.36


if (
    len(k_cache_096) < 5
    or np.min(k_cache_096) > required_k_min
    or np.max(k_cache_096) < required_k_max
):

    print(
        "\nCached coverage is insufficient."
    )

    print(
        "Computing a compact Mach 0.960 GAF table..."
    )


    fallback_k = np.linspace(
        required_k_min,
        required_k_max,
        7,
    )


    for k_test in fallback_k:

        calculate_Qhh_wall_ips(
            Ma=0.960,
            k_PA=float(
                k_test
            ),
            n_modes=4,
        )


    k_cache_096, Q_cache_096 = (
        collect_cached_qhh(
            Ma_target=0.960,
            n_modes=4,
        )
    )


print(
    f"k coverage = "
    f"{np.min(k_cache_096):.5f} "
    f"to "
    f"{np.max(k_cache_096):.5f} 1/m"
)


# ------------------------------------------------------------
# 3. Restrict interpolation to the Mode-1 neighbourhood
#
# This prevents high-frequency Mode-2/3/4 cached points from
# affecting the low-frequency interpolation.
# ------------------------------------------------------------

mode1_cache_mask = (
    (k_cache_096 >= 0.18)
    &
    (k_cache_096 <= 0.40)
)


k_mode1_cache = (
    k_cache_096[
        mode1_cache_mask
    ]
)


Q_mode1_cache = (
    Q_cache_096[
        mode1_cache_mask
    ]
)


print(
    f"Mode-1 interpolation points: "
    f"{len(k_mode1_cache)}"
)


assert len(
    k_mode1_cache
) >= 4


# ------------------------------------------------------------
# 4. Linear GAF interpolation
#
# Qhh is only 4x4, so interpolate each real and imaginary
# element independently.
# ------------------------------------------------------------

def interpolate_Qhh_mode1_096(
    k_query,
):
    """
    Interpolate Mach-0.960 wall+IPS Qhh within the
    already-computed Mode-1 k range.
    """

    k_query = float(
        k_query
    )


    if (
        k_query
        < np.min(
            k_mode1_cache
        )
        or
        k_query
        > np.max(
            k_mode1_cache
        )
    ):

        raise ValueError(
            f"k={k_query:.5f} lies outside cached "
            "Mode-1 interpolation range "
            f"[{np.min(k_mode1_cache):.5f}, "
            f"{np.max(k_mode1_cache):.5f}]."
        )


    Q_out = np.empty(
        (4, 4),
        dtype=complex,
    )


    for i in range(4):

        for j in range(4):

            real_part = np.interp(
                k_query,
                k_mode1_cache,
                Q_mode1_cache[
                    :,
                    i,
                    j,
                ].real,
            )


            imag_part = np.interp(
                k_query,
                k_mode1_cache,
                Q_mode1_cache[
                    :,
                    i,
                    j,
                ].imag,
            )


            Q_out[
                i,
                j,
            ] = (
                real_part
                + 1j
                * imag_part
            )


    return Q_out


# ------------------------------------------------------------
# 5. Correct frozen-k p-k eigenproblem
# ------------------------------------------------------------

def frozen_k_pk_correct(
    U,
    q_dyn,
    k_PA,
):
    """
    Conventional frozen-k p-k quadratic eigenproblem.

    Equation:

        M p^2
      + C_eff p
      + K_eff = 0

    with:

        K_eff = K - q*Re(Qhh)

        C_eff = C - q*Im(Qhh)/omega

        omega = k*U
    """

    M = (
        M_modal_SI.copy()
    )

    C = (
        C_modal_SI.copy()
    )

    K = (
        K_modal_SI.copy()
    )


    omega = (
        float(k_PA)
        * float(U)
    )


    if omega <= 0.0:

        raise ValueError(
            "omega must be positive."
        )


    Qhh = (
        interpolate_Qhh_mode1_096(
            k_PA
        )
    )


    QR = (
        np.real(
            Qhh
        )
    )

    QI = (
        np.imag(
            Qhh
        )
    )


    K_eff = (
        K
        - q_dyn
        * QR
    )


    C_eff = (
        C
        - (
            q_dyn
            / omega
        )
        * QI
    )


    M_inv = np.linalg.inv(
        M
    )


    Z = np.zeros(
        (4, 4)
    )

    I = np.eye(
        4
    )


    A = np.block([
        [
            Z,
            I,
        ],
        [
            -M_inv
            @ K_eff,

            -M_inv
            @ C_eff,
        ],
    ])


    poles, eigvectors = (
        np.linalg.eig(
            A
        )
    )


    # --------------------------------------------------------
    # Algebraic consistency audit
    #
    # Evaluate both formulations at p=i*omega.
    # They should agree to numerical precision.
    # --------------------------------------------------------

    p_harmonic = (
        1j
        * omega
    )


    D_pk = (
        M
        * p_harmonic**2

        + C_eff
        * p_harmonic

        + K_eff
    )


    D_harmonic = (
        -omega**2
        * M

        + 1j
        * omega
        * C

        + K

        - q_dyn
        * Qhh
    )


    algebra_error = (
        np.linalg.norm(
            D_pk
            - D_harmonic
        )
        /
        np.linalg.norm(
            D_harmonic
        )
    )


    return (
        poles,
        eigvectors,
        algebra_error,
    )


# ------------------------------------------------------------
# 6. MAC-tracked corrected Mode-1 p-k iteration
# ------------------------------------------------------------

def pk_iterate_mode1_correct_096(
    U,
    q_dyn,
    initial_k=None,
    tol=1e-7,
    max_iter=50,
    relaxation=0.65,
):
    """
    Correct p-k iteration for Mode 1 at Mach 0.960.
    """

    M = (
        M_modal_SI.copy()
    )


    if initial_k is None:

        k_current = (
            omega_analysis[0]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    reference_q = np.array([
        1.0,
        0.0,
        0.0,
        0.0,
    ], dtype=complex)


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    history = []

    selected_pole = None
    selected_q = None
    selected_mac = None
    final_algebra_error = None


    for iteration in range(
        1,
        max_iter + 1
    ):

        (
            poles,
            eigvectors,
            algebra_error,
        ) = frozen_k_pk_correct(
            U=U,
            q_dyn=q_dyn,
            k_PA=k_current,
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        candidate_macs = []
        candidate_vectors = []


        for idx in positive_indices:

            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M,
                )
            )


            candidate_macs.append(
                mac
            )

            candidate_vectors.append(
                q_candidate
            )


        best_local = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_index = (
            positive_indices[
                best_local
            ]
        )


        selected_pole = (
            poles[
                selected_index
            ]
        )


        selected_q = (
            candidate_vectors[
                best_local
            ]
        )


        selected_mac = float(
            candidate_macs[
                best_local
            ]
        )


        selected_q = (
            phase_align_modal_vector(
                selected_q,
                reference_q,
                M,
            )
        )


        selected_q = (
            mass_normalize_modal_vector(
                selected_q,
                M,
            )
        )


        omega_new = float(
            np.imag(
                selected_pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        history.append({
            "iteration":
                iteration,

            "k_current":
                k_current,

            "sigma_1_per_s":
                np.real(
                    selected_pole
                ),

            "frequency_Hz":
                omega_new
                / (
                    2.0
                    * np.pi
                ),

            "MAC":
                selected_mac,

            "relative_k_change":
                relative_change,

            "harmonic_algebra_error":
                algebra_error,
        })


        reference_q = (
            selected_q.copy()
        )


        k_current = (
            k_next
        )


        final_algebra_error = (
            algebra_error
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            "Correct Mode-1 p-k iteration "
            "did not converge."
        )


    sigma = float(
        np.real(
            selected_pole
        )
    )


    omega = float(
        np.imag(
            selected_pole
        )
    )


    damping_ratio = (
        -sigma
        / abs(
            selected_pole
        )
    )


    return {
        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            damping_ratio,

        "k_PA":
            omega
            / U,

        "MAC":
            selected_mac,

        "iterations":
            iteration,

        "algebra_error":
            final_algebra_error,

        "modal_vector":
            selected_q,

        "history":
            pd.DataFrame(
                history
            ),
    }


# ------------------------------------------------------------
# 7. Solve the published Mach 0.960 condition
# ------------------------------------------------------------

correct_pk_096 = (
    pk_iterate_mode1_correct_096(
        U=U_exp_search,
        q_dyn=q_exp_search,
        initial_k=
            mode1_cell40[
                "k_PA"
            ],
    )
)


# ------------------------------------------------------------
# 8. Compare provisional and corrected p-k
# ------------------------------------------------------------

comparison_cell44 = pd.DataFrame({
    "quantity": [
        "sigma (1/s)",
        "frequency (Hz)",
        "damping ratio",
        "k_PA (1/m)",
    ],

    "previous_complex_stiffness": [
        mode1_cell40[
            "sigma"
        ],

        mode1_cell40[
            "frequency_Hz"
        ],

        mode1_cell40[
            "damping_ratio"
        ],

        mode1_cell40[
            "k_PA"
        ],
    ],

    "correct_p_k": [
        correct_pk_096[
            "sigma"
        ],

        correct_pk_096[
            "frequency_Hz"
        ],

        correct_pk_096[
            "damping_ratio"
        ],

        correct_pk_096[
            "k_PA"
        ],
    ],
})


print(
    "\nMach 0.960 experimental-condition audit:"
)

display(
    comparison_cell44
)


print(
    "\nCorrect p-k Mode-1 result:"
)

print(
    f"sigma        = "
    f"{correct_pk_096['sigma']:.6f} 1/s"
)

print(
    f"frequency    = "
    f"{correct_pk_096['frequency_Hz']:.6f} Hz"
)

print(
    f"damping      = "
    f"{correct_pk_096['damping_ratio']:.6f}"
)

print(
    f"k_PA         = "
    f"{correct_pk_096['k_PA']:.6f} 1/m"
)

print(
    f"MAC          = "
    f"{correct_pk_096['MAC']:.6f}"
)

print(
    f"iterations   = "
    f"{correct_pk_096['iterations']}"
)

print(
    f"harmonic algebra error = "
    f"{correct_pk_096['algebra_error']:.3e}"
)


print(
    "\nExperimental flutter frequency:"
)

print(
    f"{f_exp_search:.6f} Hz"
)


print(
    "\nCell 44 corrected p-k audit completed."
)

In [ ]:
# ============================================================
# CELL 45 — FINAL CORRECTED p-k REPRESENTATIVE VALIDATION
# ============================================================
#
# Uses the aerodynamic GAF matrices already cached during
# Cells 42–44.
#
# No intentional new DLM sweep is performed here.
#
# Improvements over previous Cell 45:
#
#   1. Correct conventional p-k formulation
#   2. Cached Qhh interpolation
#   3. Guarded SMALL edge extrapolation
#   4. MAC-based Mode-1 tracking
#   5. Automatic flutter-root refinement
#
# Representative Mach numbers:
#
#       0.499
#       0.678
#       0.901
#       0.960
#
# ============================================================


# ------------------------------------------------------------
# 1. Collect cached aerodynamic matrices
# ------------------------------------------------------------

def get_cached_Qhh_data(
    Ma_target,
    n_modes=4,
    mach_tol=1e-6,
):

    k_vals = []
    q_vals = []


    for key, Q in _wall_qhh_cache.items():

        ma_key, k_key, nm_key = key


        if (
            int(nm_key) == n_modes
            and
            abs(
                float(ma_key)
                - float(Ma_target)
            )
            < mach_tol
        ):

            k_vals.append(
                float(k_key)
            )

            q_vals.append(
                np.asarray(
                    Q,
                    dtype=complex,
                )
            )


    if len(k_vals) < 2:

        raise RuntimeError(
            f"Insufficient cached aerodynamic data "
            f"for Mach {Ma_target:.3f}."
        )


    k_vals = np.asarray(
        k_vals,
        dtype=float,
    )

    q_vals = np.asarray(
        q_vals,
        dtype=complex,
    )


    # Sort by k
    order = np.argsort(
        k_vals
    )

    k_vals = (
        k_vals[
            order
        ]
    )

    q_vals = (
        q_vals[
            order
        ]
    )


    # Remove duplicate k values
    unique_k, unique_indices = (
        np.unique(
            k_vals,
            return_index=True,
        )
    )

    q_vals = (
        q_vals[
            unique_indices
        ]
    )


    return (
        unique_k,
        q_vals,
    )


# ------------------------------------------------------------
# 2. Build aerodynamic cache database
# ------------------------------------------------------------

cached_aero_database = {}


representative_machs = [
    0.499,
    0.678,
    0.901,
    0.960,
]


for Ma in representative_machs:

    (
        k_values,
        Q_values,
    ) = get_cached_Qhh_data(
        Ma_target=Ma,
        n_modes=4,
    )


    cached_aero_database[
        Ma
    ] = {
        "k":
            k_values,

        "Q":
            Q_values,
    }


    print(
        f"Mach {Ma:.3f}: "
        f"{len(k_values)} cached GAF matrices, "
        f"k = {k_values.min():.4f} "
        f"to {k_values.max():.4f} 1/m"
    )


# ------------------------------------------------------------
# 3. Small linear edge-extrapolation helper
# ------------------------------------------------------------

def interp_with_guarded_edges(
    x,
    xp,
    fp,
):
    """
    Normal interpolation inside the cached range.

    If x is slightly outside the range, use linear
    extrapolation from the closest two points.
    """

    xp = np.asarray(
        xp,
        dtype=float,
    )

    fp = np.asarray(
        fp,
    )


    # --------------------------------------------------------
    # Inside database
    # --------------------------------------------------------

    if (
        xp[0]
        <= x
        <= xp[-1]
    ):

        return np.interp(
            x,
            xp,
            fp,
        )


    # --------------------------------------------------------
    # Below database
    # --------------------------------------------------------

    if x < xp[0]:

        x0 = xp[0]
        x1 = xp[1]

        y0 = fp[0]
        y1 = fp[1]


        slope = (
            (y1 - y0)
            /
            (x1 - x0)
        )


        return (
            y0
            + slope
            * (
                x - x0
            )
        )


    # --------------------------------------------------------
    # Above database
    # --------------------------------------------------------

    x0 = xp[-2]
    x1 = xp[-1]

    y0 = fp[-2]
    y1 = fp[-1]


    slope = (
        (y1 - y0)
        /
        (x1 - x0)
    )


    return (
        y1
        + slope
        * (
            x - x1
        )
    )


# ------------------------------------------------------------
# 4. Generic cached Qhh interpolation
# ------------------------------------------------------------

def interpolate_cached_Qhh(
    Ma,
    k_query,
):

    Ma = float(
        Ma
    )

    k_query = float(
        k_query
    )


    # --------------------------------------------------------
    # Find nearest representative Mach
    # --------------------------------------------------------

    mach_key = min(
        cached_aero_database.keys(),

        key=lambda x:
            abs(
                x - Ma
            ),
    )


    if (
        abs(
            mach_key
            - Ma
        )
        > 1e-5
    ):

        raise ValueError(
            f"No aerodynamic cache available "
            f"for Mach {Ma:.5f}."
        )


    data = (
        cached_aero_database[
            mach_key
        ]
    )


    k_data = np.asarray(
        data[
            "k"
        ],
        dtype=float,
    )


    Q_data = np.asarray(
        data[
            "Q"
        ],
        dtype=complex,
    )


    k_min = float(
        k_data.min()
    )

    k_max = float(
        k_data.max()
    )

    k_span = (
        k_max
        - k_min
    )


    # --------------------------------------------------------
    # Guarded extrapolation limit
    #
    # Allow only 5% of the existing k-span outside the
    # aerodynamic database.
    #
    # This safely covers:
    #
    #     Mach 0.678 requested k ≈ 0.2584
    #
    # against cached minimum:
    #
    #     k_min ≈ 0.2607
    #
    # --------------------------------------------------------

    extrapolation_allowance = (
        0.05
        * k_span
    )


    lower_limit = (
        k_min
        - extrapolation_allowance
    )

    upper_limit = (
        k_max
        + extrapolation_allowance
    )


    if (
        k_query
        < lower_limit
        or
        k_query
        > upper_limit
    ):

        raise ValueError(
            f"\nRequested k = {k_query:.5f} 1/m "
            f"is too far outside the cached "
            f"Mach {Ma:.3f} aerodynamic range.\n"
            f"Cached range: "
            f"[{k_min:.5f}, {k_max:.5f}]\n"
            f"Allowed guarded range: "
            f"[{lower_limit:.5f}, {upper_limit:.5f}]\n"
            "A new DLM GAF calculation would be required."
        )


    # --------------------------------------------------------
    # Report extrapolation
    # --------------------------------------------------------

    if k_query < k_min:

        print(
            f"  Small lower-edge GAF extrapolation: "
            f"k={k_query:.5f} "
            f"< {k_min:.5f}"
        )


    elif k_query > k_max:

        print(
            f"  Small upper-edge GAF extrapolation: "
            f"k={k_query:.5f} "
            f"> {k_max:.5f}"
        )


    # --------------------------------------------------------
    # Interpolate every element of Qhh
    # --------------------------------------------------------

    Q_interp = np.empty(
        (4, 4),
        dtype=complex,
    )


    for i in range(4):

        for j in range(4):


            real_part = (
                interp_with_guarded_edges(
                    x=k_query,

                    xp=k_data,

                    fp=Q_data[
                        :,
                        i,
                        j,
                    ].real,
                )
            )


            imag_part = (
                interp_with_guarded_edges(
                    x=k_query,

                    xp=k_data,

                    fp=Q_data[
                        :,
                        i,
                        j,
                    ].imag,
                )
            )


            Q_interp[
                i,
                j,
            ] = (
                real_part
                + 1j
                * imag_part
            )


    return Q_interp


# ------------------------------------------------------------
# 5. Correct conventional frozen-k p-k eigenproblem
# ------------------------------------------------------------

def frozen_k_pk_cached(
    Ma,
    U,
    q_dyn,
    k_PA,
):

    M = (
        M_modal_SI.copy()
    )

    C = (
        C_modal_SI.copy()
    )

    K = (
        K_modal_SI.copy()
    )


    # --------------------------------------------------------
    # Harmonic circular frequency
    # --------------------------------------------------------

    omega = (
        float(k_PA)
        * float(U)
    )


    if omega <= 0.0:

        raise ValueError(
            "omega must be positive."
        )


    # --------------------------------------------------------
    # Interpolated generalized aerodynamics
    # --------------------------------------------------------

    Qhh = (
        interpolate_cached_Qhh(
            Ma=Ma,
            k_query=k_PA,
        )
    )


    QR = np.real(
        Qhh
    )

    QI = np.imag(
        Qhh
    )


    # --------------------------------------------------------
    # Correct p-k decomposition
    #
    # M p²
    # +
    # [C - q QI / omega] p
    # +
    # [K - q QR]
    # =
    # 0
    # --------------------------------------------------------

    K_eff = (
        K
        - q_dyn
        * QR
    )


    C_eff = (
        C
        - (
            q_dyn
            / omega
        )
        * QI
    )


    M_inv = np.linalg.inv(
        M
    )


    Z = np.zeros(
        (4, 4),
        dtype=float,
    )

    I = np.eye(
        4,
        dtype=float,
    )


    A = np.block([
        [
            Z,
            I,
        ],

        [
            -M_inv
            @ K_eff,

            -M_inv
            @ C_eff,
        ],
    ])


    poles, eigvectors = (
        np.linalg.eig(
            A
        )
    )


    return (
        poles,
        eigvectors,
    )


# ------------------------------------------------------------
# 6. MAC-tracked Mode-1 corrected p-k iteration
# ------------------------------------------------------------

def pk_mode1_cached(
    Ma,
    U,
    q_dyn,
    initial_k=None,
    initial_reference=None,
    tol=2e-7,
    max_iter=60,
    relaxation=0.65,
):

    M = (
        M_modal_SI.copy()
    )


    # --------------------------------------------------------
    # Initial k
    # --------------------------------------------------------

    if initial_k is None:

        k_current = (
            omega_analysis[0]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    # --------------------------------------------------------
    # Initial modal reference
    # --------------------------------------------------------

    if initial_reference is None:

        reference_q = np.array(
            [
                1.0,
                0.0,
                0.0,
                0.0,
            ],
            dtype=complex,
        )

    else:

        reference_q = np.asarray(
            initial_reference,
            dtype=complex,
        ).copy()


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    # --------------------------------------------------------
    # p-k iterations
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
        ) = frozen_k_pk_cached(
            Ma=Ma,
            U=U,
            q_dyn=q_dyn,
            k_PA=k_current,
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        if len(
            positive_indices
        ) == 0:

            raise RuntimeError(
                "No positive-frequency poles found."
            )


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_index = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_index
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        mac_selected = float(
            candidate_macs[
                best
            ]
        )


        # ----------------------------------------------------
        # Phase align
        # ----------------------------------------------------

        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M,
            )
        )


        # ----------------------------------------------------
        # Update k
        # ----------------------------------------------------

        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            f"Correct p-k did not converge "
            f"for Mach {Ma:.3f}."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {
        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            mac_selected,

        "iterations":
            iteration,

        "modal_vector":
            q_selected.copy(),
    }


# ------------------------------------------------------------
# 7. Correct flutter-root solver
# ------------------------------------------------------------

def solve_correct_flutter_root(
    Ma,
    U_exp,
    q_exp,
    lambda_guess,
):

    # --------------------------------------------------------
    # Initial search bracket around previous Cell-43 root
    # --------------------------------------------------------

    half_width = 0.04


    def evaluate_lambda(
        lam,
        reference=None,
    ):

        U = (
            lam
            * U_exp
        )


        q = (
            lam**2
            * q_exp
        )


        if reference is None:

            initial_k = None

            initial_reference = None


        else:

            initial_k = (
                reference[
                    "k_PA"
                ]
            )

            initial_reference = (
                reference[
                    "modal_vector"
                ]
            )


        result = (
            pk_mode1_cached(
                Ma=Ma,

                U=U,

                q_dyn=q,

                initial_k=
                    initial_k,

                initial_reference=
                    initial_reference,
            )
        )


        return {
            "lambda":
                float(
                    lam
                ),

            "U":
                float(
                    U
                ),

            "q":
                float(
                    q
                ),

            **result,
        }


    # --------------------------------------------------------
    # Initial bracket
    # --------------------------------------------------------

    lam_low = max(
        0.60,
        lambda_guess
        - half_width,
    )


    lam_high = min(
        1.60,
        lambda_guess
        + half_width,
    )


    low = (
        evaluate_lambda(
            lam_low
        )
    )


    high = (
        evaluate_lambda(
            lam_high,
            reference=low,
        )
    )


    # --------------------------------------------------------
    # Expand bracket if necessary
    # --------------------------------------------------------

    expansion = 0


    while (
        low[
            "sigma"
        ]
        * high[
            "sigma"
        ]
        > 0.0
    ):


        expansion += 1


        if expansion > 8:

            raise RuntimeError(
                f"Could not bracket corrected flutter "
                f"for Mach {Ma:.3f}."
            )


        half_width += 0.04


        lam_low = max(
            0.55,
            lambda_guess
            - half_width,
        )


        lam_high = min(
            1.80,
            lambda_guess
            + half_width,
        )


        low = (
            evaluate_lambda(
                lam_low
            )
        )


        high = (
            evaluate_lambda(
                lam_high,
                reference=low,
            )
        )


    # --------------------------------------------------------
    # Force low = stable, high = unstable
    # --------------------------------------------------------

    if (
        low[
            "sigma"
        ] > 0.0
        and
        high[
            "sigma"
        ] < 0.0
    ):

        low, high = (
            high,
            low,
        )


    print(
        f"  bracket: "
        f"lambda={low['lambda']:.5f}, "
        f"sigma={low['sigma']:+.5f} "
        f"to "
        f"lambda={high['lambda']:.5f}, "
        f"sigma={high['sigma']:+.5f}"
    )


    # --------------------------------------------------------
    # Root refinement
    # --------------------------------------------------------

    for root_iteration in range(
        1,
        21,
    ):


        sigma_low = (
            low[
                "sigma"
            ]
        )


        sigma_high = (
            high[
                "sigma"
            ]
        )


        # ----------------------------------------------------
        # False-position estimate
        # ----------------------------------------------------

        lam_trial = (
            low[
                "lambda"
            ]

            - sigma_low
            * (
                high[
                    "lambda"
                ]
                - low[
                    "lambda"
                ]
            )

            / (
                sigma_high
                - sigma_low
            )
        )


        # ----------------------------------------------------
        # Safeguard against endpoint stagnation
        # ----------------------------------------------------

        width = abs(
            high[
                "lambda"
            ]
            - low[
                "lambda"
            ]
        )


        lower_bound = min(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        upper_bound = max(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        margin = (
            0.05
            * width
        )


        if (
            lam_trial
            <= lower_bound
            + margin

            or

            lam_trial
            >= upper_bound
            - margin
        ):

            lam_trial = (
                0.5
                * (
                    low[
                        "lambda"
                    ]
                    + high[
                        "lambda"
                    ]
                )
            )


        # ----------------------------------------------------
        # Use closest endpoint for continuation
        # ----------------------------------------------------

        if (
            abs(
                lam_trial
                - low[
                    "lambda"
                ]
            )

            <=

            abs(
                high[
                    "lambda"
                ]
                - lam_trial
            )
        ):

            reference = (
                low
            )

        else:

            reference = (
                high
            )


        trial = (
            evaluate_lambda(
                lam_trial,
                reference=reference,
            )
        )


        print(
            f"    root {root_iteration:02d}: "
            f"lambda={trial['lambda']:.7f}, "
            f"sigma={trial['sigma']:+.6f}, "
            f"f={trial['frequency_Hz']:.5f} Hz"
        )


        # ----------------------------------------------------
        # Neutral stability reached
        # ----------------------------------------------------

        if (
            abs(
                trial[
                    "sigma"
                ]
            )
            < 1e-4
        ):

            return trial


        # ----------------------------------------------------
        # Update bracket
        # ----------------------------------------------------

        if (
            trial[
                "sigma"
            ] < 0.0
        ):

            low = (
                trial
            )

        else:

            high = (
                trial
            )


    # --------------------------------------------------------
    # Return closest point if iteration limit reached
    # --------------------------------------------------------

    return min(
        [
            low,
            high,
        ],

        key=lambda x:
            abs(
                x[
                    "sigma"
                ]
            ),
    )


# ------------------------------------------------------------
# 8. Recompute all four representative flutter points
# ------------------------------------------------------------

corrected_rows = []


for _, row in (
    agard_representative_validation.iterrows()
):


    Ma = float(
        row[
            "Mach"
        ]
    )


    U_exp = float(
        row[
            "U_exp_m_s"
        ]
    )


    q_exp = float(
        row[
            "q_exp_Pa"
        ]
    )


    f_exp = float(
        row[
            "f_exp_Hz"
        ]
    )


    lambda_old = float(
        row[
            "lambda_flutter"
        ]
    )


    print(
        "\n-------------------------------------------"
    )

    print(
        f"Corrected p-k flutter root: "
        f"Mach {Ma:.3f}"
    )

    print(
        "-------------------------------------------"
    )


    result = (
        solve_correct_flutter_root(
            Ma=Ma,

            U_exp=U_exp,

            q_exp=q_exp,

            lambda_guess=
                lambda_old,
        )
    )


    # --------------------------------------------------------
    # Errors
    # --------------------------------------------------------

    U_error = (
        100.0
        * (
            result[
                "U"
            ]
            - U_exp
        )
        / U_exp
    )


    q_error = (
        100.0
        * (
            result[
                "q"
            ]
            - q_exp
        )
        / q_exp
    )


    f_error = (
        100.0
        * (
            result[
                "frequency_Hz"
            ]
            - f_exp
        )
        / f_exp
    )


    corrected_rows.append({

        "Mach":
            Ma,

        "U_exp_m_s":
            U_exp,

        "U_previous_m_s":
            float(
                row[
                    "U_pred_m_s"
                ]
            ),

        "U_corrected_m_s":
            result[
                "U"
            ],

        "U_error_pct":
            U_error,

        "q_exp_Pa":
            q_exp,

        "q_corrected_Pa":
            result[
                "q"
            ],

        "q_error_pct":
            q_error,

        "f_exp_Hz":
            f_exp,

        "f_previous_Hz":
            float(
                row[
                    "f_pred_Hz"
                ]
            ),

        "f_corrected_Hz":
            result[
                "frequency_Hz"
            ],

        "f_error_pct":
            f_error,

        "lambda_previous":
            lambda_old,

        "lambda_corrected":
            result[
                "lambda"
            ],

        "sigma_final":
            result[
                "sigma"
            ],

        "MAC":
            result[
                "MAC"
            ],

        "pk_iterations":
            result[
                "iterations"
            ],
    })


# ------------------------------------------------------------
# 9. Final validation dataframe
# ------------------------------------------------------------

corrected_pk_validation = (
    pd.DataFrame(
        corrected_rows
    )
)


# ------------------------------------------------------------
# 10. Difference from provisional Cell-43 results
# ------------------------------------------------------------

corrected_pk_validation[
    "delta_U_vs_previous_pct"
] = (
    100.0
    * (
        corrected_pk_validation[
            "U_corrected_m_s"
        ]
        - corrected_pk_validation[
            "U_previous_m_s"
        ]
    )

    /
    corrected_pk_validation[
        "U_previous_m_s"
    ]
)


corrected_pk_validation[
    "delta_f_vs_previous_pct"
] = (
    100.0
    * (
        corrected_pk_validation[
            "f_corrected_Hz"
        ]
        - corrected_pk_validation[
            "f_previous_Hz"
        ]
    )

    /
    corrected_pk_validation[
        "f_previous_Hz"
    ]
)


# ------------------------------------------------------------
# 11. Final validation statistics
# ------------------------------------------------------------

corrected_U_MAE = (
    corrected_pk_validation[
        "U_error_pct"
    ]
    .abs()
    .mean()
)


corrected_f_MAE = (
    corrected_pk_validation[
        "f_error_pct"
    ]
    .abs()
    .mean()
)


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print(
    "\nFINAL corrected p-k validation:"
)

display(
    corrected_pk_validation
)


print(
    "\nCorrected validation statistics:"
)

print(
    f"Mean absolute flutter-velocity error = "
    f"{corrected_U_MAE:.3f}%"
)


print(
    f"Mean absolute flutter-frequency error = "
    f"{corrected_f_MAE:.3f}%"
)


print(
    "\nEffect of correcting the p-k formulation:"
)


display(
    corrected_pk_validation[
        [
            "Mach",
            "delta_U_vs_previous_pct",
            "delta_f_vs_previous_pct",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Corrected flutter-velocity plot
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    corrected_pk_validation[
        "Mach"
    ],

    corrected_pk_validation[
        "U_exp_m_s"
    ],

    "o-",

    label="Experiment",
)


plt.plot(
    corrected_pk_validation[
        "Mach"
    ],

    corrected_pk_validation[
        "U_corrected_m_s"
    ],

    "s--",

    label="Corrected p-k",
)


plt.xlabel(
    "Mach number"
)

plt.ylabel(
    "Flutter velocity (m/s)"
)

plt.title(
    "AGARD 445.6 corrected flutter validation"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 14. Corrected flutter-frequency plot
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    corrected_pk_validation[
        "Mach"
    ],

    corrected_pk_validation[
        "f_exp_Hz"
    ],

    "o-",

    label="Experiment",
)


plt.plot(
    corrected_pk_validation[
        "Mach"
    ],

    corrected_pk_validation[
        "f_corrected_Hz"
    ],

    "s--",

    label="Corrected p-k",
)


plt.xlabel(
    "Mach number"
)

plt.ylabel(
    "Flutter frequency (Hz)"
)

plt.title(
    "AGARD 445.6 corrected flutter-frequency validation"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nCell 45 corrected cached-GAF validation completed."
)

In [ ]:
# ============================================================
# CELL 46 — FINAL GAF INTERPOLATION SENSITIVITY
# ============================================================
#
# Purpose:
#
# Test whether the corrected flutter results depend materially
# on the very dense cached GAF database.
#
# Cases:
#
#   Full cache
#   Every 2nd cached GAF point
#   Every 4th cached GAF point
#
# No new DLM calculations.
#
# ============================================================

import copy
import io
import contextlib


# ------------------------------------------------------------
# 1. Save full aerodynamic database
# ------------------------------------------------------------

full_aero_database = copy.deepcopy(
    cached_aero_database
)


# ------------------------------------------------------------
# 2. Build thinned database
# ------------------------------------------------------------

def make_thinned_database(
    original_database,
    stride,
):

    thinned = {}

    for Ma, data in original_database.items():

        k = np.asarray(
            data["k"]
        )

        Q = np.asarray(
            data["Q"]
        )


        indices = list(
            range(
                0,
                len(k),
                stride,
            )
        )


        # Always retain final endpoint
        if indices[-1] != len(k) - 1:

            indices.append(
                len(k) - 1
            )


        indices = np.asarray(
            indices,
            dtype=int,
        )


        thinned[Ma] = {
            "k":
                k[
                    indices
                ].copy(),

            "Q":
                Q[
                    indices
                ].copy(),
        }


    return thinned


# ------------------------------------------------------------
# 3. Sensitivity cases
# ------------------------------------------------------------

sensitivity_cases = {
    "Full":
        full_aero_database,

    "Every_2nd":
        make_thinned_database(
            full_aero_database,
            stride=2,
        ),

    "Every_4th":
        make_thinned_database(
            full_aero_database,
            stride=4,
        ),
}


# ------------------------------------------------------------
# 4. Recompute roots silently
# ------------------------------------------------------------

sensitivity_rows = []


for case_name, database in sensitivity_cases.items():

    # Existing Cell-45 functions read this global database.
    cached_aero_database = database


    print(
        f"\nRunning interpolation case: "
        f"{case_name}"
    )


    for _, row in (
        corrected_pk_validation.iterrows()
    ):

        Ma = float(
            row["Mach"]
        )

        U_exp = float(
            row["U_exp_m_s"]
        )

        q_exp = float(
            row["q_exp_Pa"]
        )

        lambda_guess = float(
            row["lambda_corrected"]
        )


        # Suppress detailed root-iteration output
        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            result = (
                solve_correct_flutter_root(
                    Ma=Ma,
                    U_exp=U_exp,
                    q_exp=q_exp,
                    lambda_guess=
                        lambda_guess,
                )
            )


        sensitivity_rows.append({
            "case":
                case_name,

            "Mach":
                Ma,

            "n_GAF_points":
                len(
                    database[
                        Ma
                    ]["k"]
                ),

            "lambda_flutter":
                result[
                    "lambda"
                ],

            "U_flutter_m_s":
                result[
                    "U"
                ],

            "f_flutter_Hz":
                result[
                    "frequency_Hz"
                ],

            "sigma_final":
                result[
                    "sigma"
                ],
        })


# ------------------------------------------------------------
# 5. Restore full database
# ------------------------------------------------------------

cached_aero_database = (
    full_aero_database
)


# ------------------------------------------------------------
# 6. Assemble table
# ------------------------------------------------------------

gaf_sensitivity = pd.DataFrame(
    sensitivity_rows
)


# ------------------------------------------------------------
# 7. Add full-cache reference
# ------------------------------------------------------------

full_reference = (
    gaf_sensitivity[
        gaf_sensitivity[
            "case"
        ]
        == "Full"
    ][
        [
            "Mach",
            "U_flutter_m_s",
            "f_flutter_Hz",
        ]
    ]
    .rename(
        columns={
            "U_flutter_m_s":
                "U_reference",

            "f_flutter_Hz":
                "f_reference",
        }
    )
)


gaf_sensitivity = (
    gaf_sensitivity
    .merge(
        full_reference,
        on="Mach",
        how="left",
    )
)


# ------------------------------------------------------------
# 8. Difference from full cache
# ------------------------------------------------------------

gaf_sensitivity[
    "delta_U_pct"
] = (
    100.0
    * (
        gaf_sensitivity[
            "U_flutter_m_s"
        ]
        - gaf_sensitivity[
            "U_reference"
        ]
    )
    /
    gaf_sensitivity[
        "U_reference"
    ]
)


gaf_sensitivity[
    "delta_f_pct"
] = (
    100.0
    * (
        gaf_sensitivity[
            "f_flutter_Hz"
        ]
        - gaf_sensitivity[
            "f_reference"
        ]
    )
    /
    gaf_sensitivity[
        "f_reference"
    ]
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print(
    "\nGAF interpolation sensitivity:"
)

display(
    gaf_sensitivity[
        [
            "case",
            "Mach",
            "n_GAF_points",
            "U_flutter_m_s",
            "delta_U_pct",
            "f_flutter_Hz",
            "delta_f_pct",
            "sigma_final",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Maximum sensitivity
# ------------------------------------------------------------

non_full = (
    gaf_sensitivity[
        gaf_sensitivity[
            "case"
        ]
        != "Full"
    ]
)


max_U_sensitivity = (
    non_full[
        "delta_U_pct"
    ]
    .abs()
    .max()
)


max_f_sensitivity = (
    non_full[
        "delta_f_pct"
    ]
    .abs()
    .max()
)


print(
    "\nMaximum sensitivity to GAF thinning:"
)

print(
    f"Flutter velocity = "
    f"{max_U_sensitivity:.4f}%"
)

print(
    f"Flutter frequency = "
    f"{max_f_sensitivity:.4f}%"
)


print(
    "\nCell 46 GAF sensitivity completed."
)

In [ ]:
# ============================================================
# CELL 47 — PASSIVE REINFORCEMENT STRUCTURAL FRAMEWORK
# ============================================================
#
# INDUSTRY PROBLEM:
#
# Where should structural reinforcement be placed to increase
# flutter margin efficiently without excessive structural mass?
#
#
# APPROACH:
#
# Use the validated AGARD modal basis as a reduced-order
# structural test bed.
#
# A reinforcement region may increase:
#
#     bending stiffness
#     torsional stiffness
#
# and adds structural mass.
#
#
# IMPORTANT:
#
# We are NOT arbitrarily changing modal frequencies.
#
# The stiffness change is weighted using spatial bending and
# torsional deformation of the published mode shapes.
#
# The added mass matrix is calculated from the modal
# displacement field.
#
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline


# ------------------------------------------------------------
# 1. Wing geometry
# ------------------------------------------------------------

SPAN = 0.762000          # m
C_ROOT = 0.557784        # m
C_TIP = 0.3681984        # m

SWEEP_QC_DEG = 45.0

SWEEP_QC = np.deg2rad(
    SWEEP_QC_DEG
)


# ------------------------------------------------------------
# 2. Spanwise analysis stations
# ------------------------------------------------------------

n_passive_span = 121

y_passive = np.linspace(
    0.0,
    SPAN,
    n_passive_span,
)

eta_passive = (
    y_passive
    / SPAN
)


# ------------------------------------------------------------
# 3. Local chord geometry
# ------------------------------------------------------------

chord_passive = (
    C_ROOT
    + (
        C_TIP
        - C_ROOT
    )
    * eta_passive
)


# Quarter-chord line
x_qc_passive = (
    0.25
    * C_ROOT
    + y_passive
    * np.tan(
        SWEEP_QC
    )
)


# Leading-edge location
x_le_passive = (
    x_qc_passive
    - 0.25
    * chord_passive
)


# ------------------------------------------------------------
# 4. Helper for modal displacement at chord fraction xi
#
# xi = 0.0  -> LE
# xi = 0.5  -> mid-chord
# xi = 1.0  -> TE
# ------------------------------------------------------------

def modal_field_at_chord_fraction(
    xi,
):

    x_query = (
        x_le_passive
        + float(xi)
        * chord_passive
    )


    query_xy = np.column_stack([
        x_query,
        y_passive,
    ])


    H_query, _, _ = (
        build_ips_transfer(
            query_xy
        )
    )


    Phi_query = (
        H_query
        @ Phi_z_4
    )


    return Phi_query


# ------------------------------------------------------------
# 5. Modal fields at 25%, 50%, 75% chord
# ------------------------------------------------------------

Phi_25 = (
    modal_field_at_chord_fraction(
        0.25
    )
)

Phi_50 = (
    modal_field_at_chord_fraction(
        0.50
    )
)

Phi_75 = (
    modal_field_at_chord_fraction(
        0.75
    )
)


assert Phi_25.shape == (
    n_passive_span,
    4,
)

assert Phi_50.shape == (
    n_passive_span,
    4,
)

assert Phi_75.shape == (
    n_passive_span,
    4,
)


# ------------------------------------------------------------
# 6. Representative bending displacement
#
# Mid-chord displacement is used as the bending field.
# ------------------------------------------------------------

Phi_bending = (
    Phi_50.copy()
)


# ------------------------------------------------------------
# 7. Representative twist field
#
# Small-angle twist approximation:
#
# theta =
#
#     (w_75 - w_25)
#     ----------------
#          0.5 c
#
# Because modal coordinate q has units of length,
# theta coefficient has units 1/m.
# ------------------------------------------------------------

Phi_twist = (
    (
        Phi_75
        - Phi_25
    )
    /
    (
        0.50
        * chord_passive[:, None]
    )
)


# ------------------------------------------------------------
# 8. Smooth spanwise derivatives
#
# Bending strain indicator:
#
#       kappa_b = d²w/dy²
#
# Torsional strain indicator:
#
#       gamma_t = d(theta)/dy
# ------------------------------------------------------------

Phi_bending_curvature = np.zeros_like(
    Phi_bending
)

Phi_twist_gradient = np.zeros_like(
    Phi_twist
)


for mode_id in range(4):

    bending_spline = CubicSpline(
        y_passive,
        Phi_bending[
            :,
            mode_id
        ],
        bc_type="natural",
    )


    twist_spline = CubicSpline(
        y_passive,
        Phi_twist[
            :,
            mode_id
        ],
        bc_type="natural",
    )


    Phi_bending_curvature[
        :,
        mode_id
    ] = bending_spline(
        y_passive,
        2,
    )


    Phi_twist_gradient[
        :,
        mode_id
    ] = twist_spline(
        y_passive,
        1,
    )


# ------------------------------------------------------------
# 9. Modal families
#
# Based on the published mode descriptions already used in
# the benchmark:
#
# Mode 1 -> first bending
# Mode 2 -> first torsion
# Mode 3 -> second bending
# Mode 4 -> second torsion
# ------------------------------------------------------------

BENDING_MODES = [
    0,
    2,
]

TORSION_MODES = [
    1,
    3,
]


# ------------------------------------------------------------
# 10. Full-span modal strain-energy normalizers
# ------------------------------------------------------------

bending_energy_norm = np.zeros(
    4
)

torsion_energy_norm = np.zeros(
    4
)


for i in BENDING_MODES:

    bending_energy_norm[i] = (
        np.trapezoid(
            Phi_bending_curvature[
                :,
                i
            ]**2,
            y_passive,
        )
    )


for i in TORSION_MODES:

    torsion_energy_norm[i] = (
        np.trapezoid(
            Phi_twist_gradient[
                :,
                i
            ]**2,
            y_passive,
        )
    )


# ------------------------------------------------------------
# 11. Smooth reinforcement window
#
# eta_start, eta_end:
# span fractions
#
# A small cosine transition avoids artificial discontinuities.
# ------------------------------------------------------------

def reinforcement_window(
    eta_start,
    eta_end,
    transition=0.02,
):

    eta_start = float(
        eta_start
    )

    eta_end = float(
        eta_end
    )


    if not (
        0.0
        <= eta_start
        < eta_end
        <= 1.0
    ):

        raise ValueError(
            "Require 0 <= eta_start < eta_end <= 1."
        )


    w = np.zeros_like(
        eta_passive
    )


    # Core region
    core = (
        (eta_passive >= eta_start)
        &
        (eta_passive <= eta_end)
    )

    w[core] = 1.0


    # --------------------------------------------------------
    # Inboard cosine transition
    # --------------------------------------------------------

    if transition > 0.0:

        lower_0 = max(
            0.0,
            eta_start
            - transition,
        )


        lower_mask = (
            (eta_passive >= lower_0)
            &
            (eta_passive < eta_start)
        )


        if np.any(
            lower_mask
        ):

            s = (
                eta_passive[
                    lower_mask
                ]
                - lower_0
            ) / (
                eta_start
                - lower_0
            )


            w[
                lower_mask
            ] = (
                0.5
                * (
                    1.0
                    - np.cos(
                        np.pi
                        * s
                    )
                )
            )


        # ----------------------------------------------------
        # Outboard transition
        # ----------------------------------------------------

        upper_1 = min(
            1.0,
            eta_end
            + transition,
        )


        upper_mask = (
            (eta_passive > eta_end)
            &
            (eta_passive <= upper_1)
        )


        if np.any(
            upper_mask
        ):

            s = (
                eta_passive[
                    upper_mask
                ]
                - eta_end
            ) / (
                upper_1
                - eta_end
            )


            w[
                upper_mask
            ] = (
                0.5
                * (
                    1.0
                    + np.cos(
                        np.pi
                        * s
                    )
                )
            )


    return w


# ------------------------------------------------------------
# 12. Build passive modal matrices
# ------------------------------------------------------------

def build_passive_modal_matrices(
    eta_start,
    eta_end,
    bending_stiffness_gain=0.0,
    torsional_stiffness_gain=0.0,
    added_mass_kg=0.0,
):
    """
    Construct modified 4-mode structural matrices.

    Interpretation
    --------------
    bending_stiffness_gain = 0.10

        means a local +10% bending-stiffness reinforcement
        inside the selected region.

    torsional_stiffness_gain = 0.10

        means a local +10% torsional-stiffness reinforcement
        inside the selected region.

    added_mass_kg

        total physical reinforcement mass distributed across
        the selected span region.

    Notes
    -----
    Stiffness increments are based on normalized modal
    strain-energy participation.

    Added mass is projected through the mid-chord modal
    displacement field.
    """


    if bending_stiffness_gain < 0.0:

        raise ValueError(
            "bending_stiffness_gain must be >= 0."
        )


    if torsional_stiffness_gain < 0.0:

        raise ValueError(
            "torsional_stiffness_gain must be >= 0."
        )


    if added_mass_kg < 0.0:

        raise ValueError(
            "added_mass_kg must be >= 0."
        )


    window = reinforcement_window(
        eta_start,
        eta_end,
    )


    # ========================================================
    # BASELINE MATRICES
    # ========================================================

    M_new = (
        M_modal_SI.copy()
    )

    K_new = (
        K_modal_SI.copy()
    )

    C_new = (
        C_modal_SI.copy()
    )


    delta_K = np.zeros(
        (4, 4)
    )


    # ========================================================
    # BENDING STIFFNESS CONTRIBUTION
    # ========================================================

    for i in BENDING_MODES:

        for j in BENDING_MODES:


            numerator = (
                np.trapezoid(
                    window
                    * Phi_bending_curvature[
                        :,
                        i
                    ]
                    * Phi_bending_curvature[
                        :,
                        j
                    ],
                    y_passive,
                )
            )


            denominator = np.sqrt(
                bending_energy_norm[i]
                * bending_energy_norm[j]
            )


            participation = (
                numerator
                / denominator
            )


            delta_K[
                i,
                j
            ] += (
                bending_stiffness_gain
                * np.sqrt(
                    K_modal_SI[
                        i,
                        i
                    ]
                    * K_modal_SI[
                        j,
                        j
                    ]
                )
                * participation
            )


    # ========================================================
    # TORSIONAL STIFFNESS CONTRIBUTION
    # ========================================================

    for i in TORSION_MODES:

        for j in TORSION_MODES:


            numerator = (
                np.trapezoid(
                    window
                    * Phi_twist_gradient[
                        :,
                        i
                    ]
                    * Phi_twist_gradient[
                        :,
                        j
                    ],
                    y_passive,
                )
            )


            denominator = np.sqrt(
                torsion_energy_norm[i]
                * torsion_energy_norm[j]
            )


            participation = (
                numerator
                / denominator
            )


            delta_K[
                i,
                j
            ] += (
                torsional_stiffness_gain
                * np.sqrt(
                    K_modal_SI[
                        i,
                        i
                    ]
                    * K_modal_SI[
                        j,
                        j
                    ]
                )
                * participation
            )


    K_new += (
        delta_K
    )


    # ========================================================
    # ADDED MASS CONTRIBUTION
    # ========================================================

    delta_M = np.zeros(
        (4, 4)
    )


    window_integral = (
        np.trapezoid(
            window,
            y_passive,
        )
    )


    if (
        added_mass_kg > 0.0
        and
        window_integral > 0.0
    ):


        mass_per_length = (
            added_mass_kg
            / window_integral
        )


        for i in range(4):

            for j in range(4):


                delta_M[
                    i,
                    j
                ] = (
                    mass_per_length
                    * np.trapezoid(
                        window
                        * Phi_50[
                            :,
                            i
                        ]
                        * Phi_50[
                            :,
                            j
                        ],
                        y_passive,
                    )
                )


        M_new += (
            delta_M
        )


    # ========================================================
    # PHYSICAL CHECKS
    # ========================================================

    M_eigs = np.linalg.eigvalsh(
        M_new
    )

    K_eigs = np.linalg.eigvalsh(
        K_new
    )


    if np.min(
        M_eigs
    ) <= 0.0:

        raise RuntimeError(
            "Modified mass matrix is not positive definite."
        )


    if np.min(
        K_eigs
    ) <= 0.0:

        raise RuntimeError(
            "Modified stiffness matrix is not positive definite."
        )


    return {
        "M":
            M_new,

        "C":
            C_new,

        "K":
            K_new,

        "delta_M":
            delta_M,

        "delta_K":
            delta_K,

        "window":
            window,

        "added_mass_kg":
            added_mass_kg,
    }


# ------------------------------------------------------------
# 13. ZERO-MODIFICATION VERIFICATION
# ------------------------------------------------------------

zero_passive = (
    build_passive_modal_matrices(
        eta_start=0.40,
        eta_end=0.60,
        bending_stiffness_gain=0.0,
        torsional_stiffness_gain=0.0,
        added_mass_kg=0.0,
    )
)


zero_M_error = (
    np.linalg.norm(
        zero_passive["M"]
        - M_modal_SI
    )
)


zero_K_error = (
    np.linalg.norm(
        zero_passive["K"]
        - K_modal_SI
    )
)


assert np.allclose(
    zero_passive["M"],
    M_modal_SI,
)


assert np.allclose(
    zero_passive["K"],
    K_modal_SI,
)


# ------------------------------------------------------------
# 14. SMALL DEMONSTRATION MODIFICATION
#
# NOT an optimized design.
#
# This is only a numerical sanity check.
# ------------------------------------------------------------

demo_passive = (
    build_passive_modal_matrices(
        eta_start=0.50,
        eta_end=0.75,
        bending_stiffness_gain=0.10,
        torsional_stiffness_gain=0.10,
        added_mass_kg=0.20,
    )
)


# ------------------------------------------------------------
# 15. Calculate dry natural frequencies
# ------------------------------------------------------------

def generalized_frequencies(
    M,
    K,
):

    eigvals = np.linalg.eigvals(
        np.linalg.solve(
            M,
            K,
        )
    )


    eigvals = np.real(
        eigvals
    )


    eigvals = np.sort(
        eigvals
    )


    frequencies = (
        np.sqrt(
            eigvals
        )
        /
        (
            2.0
            * np.pi
        )
    )


    return frequencies


baseline_freq_check = (
    generalized_frequencies(
        M_modal_SI,
        K_modal_SI,
    )
)


demo_freq = (
    generalized_frequencies(
        demo_passive["M"],
        demo_passive["K"],
    )
)


# ------------------------------------------------------------
# 16. Report
# ------------------------------------------------------------

passive_framework_check = pd.DataFrame({
    "mode_id":
        [1, 2, 3, 4],

    "baseline_frequency_Hz":
        baseline_freq_check,

    "demo_frequency_Hz":
        demo_freq,

    "frequency_change_pct":
        100.0
        * (
            demo_freq
            - baseline_freq_check
        )
        / baseline_freq_check,
})


print(
    "PASSIVE REINFORCEMENT FRAMEWORK"
)

print(
    "\nZero-modification verification:"
)

print(
    f"Mass-matrix error     = "
    f"{zero_M_error:.3e}"
)

print(
    f"Stiffness-matrix error = "
    f"{zero_K_error:.3e}"
)


print(
    "\nDemonstration reinforcement:"
)

print(
    "Span region           = "
    "50% to 75%"
)

print(
    "Bending stiffness gain = 10%"
)

print(
    "Torsional stiffness gain = 10%"
)

print(
    "Added mass             = "
    "0.20 kg"
)


print(
    "\nDry-frequency response:"
)

display(
    passive_framework_check
)


print(
    "\nAdded modal mass matrix ΔM:"
)

display(
    pd.DataFrame(
        demo_passive[
            "delta_M"
        ]
    )
)


print(
    "\nAdded modal stiffness matrix ΔK:"
)

display(
    pd.DataFrame(
        demo_passive[
            "delta_K"
        ]
    )
)


# ------------------------------------------------------------
# 17. Plot deformation-energy indicators
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for mode_id in range(4):

    quantity = (
        Phi_bending_curvature[
            :,
            mode_id
        ]**2
        +
        Phi_twist_gradient[
            :,
            mode_id
        ]**2
    )


    quantity = (
        quantity
        / max(
            np.max(
                quantity
            ),
            1e-16,
        )
    )


    plt.plot(
        eta_passive,
        quantity,
        label=
            f"Mode {mode_id + 1}",
    )


plt.xlabel(
    "Normalized span η"
)

plt.ylabel(
    "Normalized deformation-energy indicator"
)

plt.title(
    "Where structural reinforcement can influence the modal dynamics"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nCell 47 passive structural framework completed."
)

In [ ]:
# ============================================================
# CELL 48 — CALIBRATED + REGULARIZED PASSIVE MODEL
# ============================================================
#
# Improvements relative to Cell 47:
#
#   1. Calibrate physical mass scale of AGARD Model 3
#   2. Regularize spanwise modal derivatives
#   3. Allow all modes to contain bending + torsion
#   4. Use physically consistent modal mass projection
#   5. Maintain ~1% structural modal damping after modification
#
# This remains a REDUCED-ORDER structural-tailoring model.
# It is not claimed to be a detailed FE reinforcement model.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.linalg import eigh


# ------------------------------------------------------------
# 1. AGARD weakened Model-3 physical panel mass
#
# Published experimental relation:
#
#       mu = m / (rho * V)
#
# therefore:
#
#       m = mu * rho * V
#
# V is the volume of a conical frustum whose:
#
#       height       = panel span
#       large dia.   = root chord
#       small dia.   = tip chord
#
# Use representative M = 0.960 Model-3 air point:
#
#       rho = 0.000123 slug/ft^3
#       mu  = 225.820
# ------------------------------------------------------------

M_TO_FT = 3.280839895
SLUG_TO_KG = 14.59390294

span_ft = (
    SPAN * M_TO_FT
)

root_chord_ft = (
    C_ROOT * M_TO_FT
)

tip_chord_ft = (
    C_TIP * M_TO_FT
)


reference_volume_ft3 = (
    np.pi
    * span_ft
    / 12.0
    * (
        root_chord_ft**2
        + root_chord_ft
        * tip_chord_ft
        + tip_chord_ft**2
    )
)


rho_model3_slug_ft3 = (
    0.000123
)

mass_ratio_model3 = (
    225.820
)


panel_mass_slug = (
    mass_ratio_model3
    * rho_model3_slug_ft3
    * reference_volume_ft3
)


BASELINE_PANEL_MASS_KG = (
    panel_mass_slug
    * SLUG_TO_KG
)


print(
    "AGARD Model-3 physical mass calibration:"
)

print(
    f"Reference volume = "
    f"{reference_volume_ft3:.6f} ft^3"
)

print(
    f"Baseline wing-panel mass = "
    f"{BASELINE_PANEL_MASS_KG:.6f} kg"
)


# ------------------------------------------------------------
# 2. Physical interpretation of Cell-47 demo
# ------------------------------------------------------------

cell47_demo_mass_fraction = (
    0.20
    / BASELINE_PANEL_MASS_KG
)


print(
    f"\nCell-47 0.20 kg demonstration mass = "
    f"{100.0 * cell47_demo_mass_fraction:.2f}% "
    "of the complete wing-panel mass."
)


# ------------------------------------------------------------
# 3. Regularize spanwise modal fields
#
# The raw IPS surfaces are accurate for displacement transfer,
# but second derivatives amplify small interpolation oscillations.
#
# Use a local polynomial Savitzky-Golay representation before
# structural-tailoring calculations.
# ------------------------------------------------------------

dy_passive = (
    y_passive[1]
    - y_passive[0]
)


SG_WINDOW = 17
SG_ORDER = 4


Phi_bending_smooth = np.zeros_like(
    Phi_bending
)

Phi_twist_smooth = np.zeros_like(
    Phi_twist
)

Phi_bending_curvature_reg = np.zeros_like(
    Phi_bending
)

Phi_twist_gradient_reg = np.zeros_like(
    Phi_twist
)


for mode_id in range(4):

    # --------------------------------------------------------
    # Smoothed modal displacement
    # --------------------------------------------------------

    Phi_bending_smooth[
        :,
        mode_id
    ] = savgol_filter(
        Phi_bending[
            :,
            mode_id
        ],

        window_length=
            SG_WINDOW,

        polyorder=
            SG_ORDER,

        deriv=
            0,

        mode=
            "interp",
    )


    # --------------------------------------------------------
    # Smoothed modal twist
    # --------------------------------------------------------

    Phi_twist_smooth[
        :,
        mode_id
    ] = savgol_filter(
        Phi_twist[
            :,
            mode_id
        ],

        window_length=
            SG_WINDOW,

        polyorder=
            SG_ORDER,

        deriv=
            0,

        mode=
            "interp",
    )


    # --------------------------------------------------------
    # Bending curvature:
    #
    #       d²Phi / dy²
    # --------------------------------------------------------

    Phi_bending_curvature_reg[
        :,
        mode_id
    ] = savgol_filter(
        Phi_bending[
            :,
            mode_id
        ],

        window_length=
            SG_WINDOW,

        polyorder=
            SG_ORDER,

        deriv=
            2,

        delta=
            dy_passive,

        mode=
            "interp",
    )


    # --------------------------------------------------------
    # Twist gradient:
    #
    #       d(theta) / dy
    # --------------------------------------------------------

    Phi_twist_gradient_reg[
        :,
        mode_id
    ] = savgol_filter(
        Phi_twist[
            :,
            mode_id
        ],

        window_length=
            SG_WINDOW,

        polyorder=
            SG_ORDER,

        deriv=
            1,

        delta=
            dy_passive,

        mode=
            "interp",
    )


# ------------------------------------------------------------
# 4. Full-span structural activity matrices
#
# These are deformation/strain PARTICIPATION measures.
#
# They are not claimed to be dimensional physical strain
# energies because local EI and GJ distributions are not being
# reconstructed here.
# ------------------------------------------------------------

B_full = np.zeros(
    (4, 4)
)

T_full = np.zeros(
    (4, 4)
)


for i in range(4):

    for j in range(4):

        B_full[
            i,
            j
        ] = np.trapezoid(
            Phi_bending_curvature_reg[
                :,
                i
            ]
            * Phi_bending_curvature_reg[
                :,
                j
            ],

            y_passive,
        )


        T_full[
            i,
            j
        ] = np.trapezoid(
            Phi_twist_gradient_reg[
                :,
                i
            ]
            * Phi_twist_gradient_reg[
                :,
                j
            ],

            y_passive,
        )


B_diag = np.diag(
    B_full
)

T_diag = np.diag(
    T_full
)


# ------------------------------------------------------------
# 5. Bending/torsional modal activity fractions
#
# These are reduced-order weighting factors, not claims of
# exact physical energy partition.
# ------------------------------------------------------------

activity_total = (
    B_diag
    + T_diag
)


bending_activity_fraction = (
    B_diag
    / activity_total
)


torsional_activity_fraction = (
    T_diag
    / activity_total
)


activity_table = pd.DataFrame({
    "mode_id":
        [1, 2, 3, 4],

    "bending_activity_fraction":
        bending_activity_fraction,

    "torsional_activity_fraction":
        torsional_activity_fraction,
})


print(
    "\nRegularized modal activity fractions:"
)

display(
    activity_table
)


# ------------------------------------------------------------
# 6. Reinforcement mass line
#
# Assume the passive reinforcement is approximately associated
# with a spar / structural line near 40% chord.
# ------------------------------------------------------------

REINFORCEMENT_CHORD_FRACTION = (
    0.40
)


Phi_mass_line = (
    modal_field_at_chord_fraction(
        REINFORCEMENT_CHORD_FRACTION
    )
)


# ------------------------------------------------------------
# 7. Constant modal damping builder
#
# Reconstruct C such that the MODIFIED dry modes retain
# approximately:
#
#       zeta = 1%
#
# This prevents structural redesign from accidentally changing
# damping simply because M and K changed.
# ------------------------------------------------------------

STRUCTURAL_ZETA = (
    0.01
)


def build_constant_modal_damping(
    M,
    K,
    zeta=STRUCTURAL_ZETA,
):

    eigvals, eigvecs = eigh(
        K,
        M,
    )


    if np.any(
        eigvals <= 0.0
    ):

        raise RuntimeError(
            "Non-positive structural eigenvalue."
        )


    omega = np.sqrt(
        eigvals
    )


    C_modal = np.diag(
        2.0
        * zeta
        * omega
    )


    # scipy.linalg.eigh(K,M) returns:
    #
    #       V.T @ M @ V = I
    #
    # therefore:
    #
    #       C = M V C_modal V.T M
    #

    C = (
        M
        @ eigvecs
        @ C_modal
        @ eigvecs.T
        @ M
    )


    # Numerical symmetry cleanup
    C = (
        0.5
        * (
            C
            + C.T
        )
    )


    return C


# ------------------------------------------------------------
# 8. Calibrated passive structural model
# ------------------------------------------------------------

def build_passive_modal_matrices_v2(
    eta_start,
    eta_end,
    bending_stiffness_gain=0.0,
    torsional_stiffness_gain=0.0,
    added_mass_fraction=0.0,
):
    """
    Reduced-order passive structural-tailoring model.

    Parameters
    ----------
    eta_start, eta_end
        Reinforcement region along normalized span.

    bending_stiffness_gain
        Local fractional bending-stiffness improvement.

        Example:
            0.10 = +10%

    torsional_stiffness_gain
        Local fractional torsional-stiffness improvement.

    added_mass_fraction
        TOTAL physical reinforcement mass as a fraction of the
        baseline AGARD wing-panel mass.

        Example:
            0.02 = added physical mass equal to 2% of panel mass.
    """


    if not (
        0.0
        <= eta_start
        < eta_end
        <= 1.0
    ):

        raise ValueError(
            "Require 0 <= eta_start < eta_end <= 1."
        )


    if (
        bending_stiffness_gain < 0.0
        or
        torsional_stiffness_gain < 0.0
        or
        added_mass_fraction < 0.0
    ):

        raise ValueError(
            "Passive design variables must be non-negative."
        )


    window = (
        reinforcement_window(
            eta_start,
            eta_end,
            transition=0.02,
        )
    )


    # ========================================================
    # 8A. WINDOWED BENDING + TORSIONAL PARTICIPATION
    # ========================================================

    B_window = np.zeros(
        (4, 4)
    )

    T_window = np.zeros(
        (4, 4)
    )


    for i in range(4):

        for j in range(4):

            B_window[
                i,
                j
            ] = np.trapezoid(
                window
                * Phi_bending_curvature_reg[
                    :,
                    i
                ]
                * Phi_bending_curvature_reg[
                    :,
                    j
                ],

                y_passive,
            )


            T_window[
                i,
                j
            ] = np.trapezoid(
                window
                * Phi_twist_gradient_reg[
                    :,
                    i
                ]
                * Phi_twist_gradient_reg[
                    :,
                    j
                ],

                y_passive,
            )


    # ========================================================
    # 8B. NORMALIZED PARTICIPATION MATRICES
    # ========================================================

    B_part = np.zeros(
        (4, 4)
    )

    T_part = np.zeros(
        (4, 4)
    )


    for i in range(4):

        for j in range(4):

            B_denominator = np.sqrt(
                max(
                    B_diag[i]
                    * B_diag[j],
                    1e-30,
                )
            )


            T_denominator = np.sqrt(
                max(
                    T_diag[i]
                    * T_diag[j],
                    1e-30,
                )
            )


            B_part[
                i,
                j
            ] = (
                B_window[
                    i,
                    j
                ]
                / B_denominator
            )


            T_part[
                i,
                j
            ] = (
                T_window[
                    i,
                    j
                ]
                / T_denominator
            )


    # ========================================================
    # 8C. STIFFNESS INCREMENT
    #
    # Important calibration property:
    #
    # For full-span uniform reinforcement with:
    #
    #       bending_gain = torsional_gain = g
    #
    # diagonal modal stiffness tends toward approximately:
    #
    #       Kii_new = (1 + g) Kii
    #
    # rather than double-counting bending and torsion.
    # ========================================================

    sqrt_K = np.sqrt(
        np.diag(
            K_modal_SI
        )
    )


    K_scale = np.outer(
        sqrt_K,
        sqrt_K,
    )


    bending_weight = np.sqrt(
        np.outer(
            bending_activity_fraction,
            bending_activity_fraction,
        )
    )


    torsion_weight = np.sqrt(
        np.outer(
            torsional_activity_fraction,
            torsional_activity_fraction,
        )
    )


    delta_K = (
        K_scale
        * (
            bending_stiffness_gain
            * bending_weight
            * B_part

            +

            torsional_stiffness_gain
            * torsion_weight
            * T_part
        )
    )


    # Force exact symmetry
    delta_K = (
        0.5
        * (
            delta_K
            + delta_K.T
        )
    )


    # ========================================================
    # 8D. PHYSICAL ADDED MASS
    # ========================================================

    added_mass_kg = (
        added_mass_fraction
        * BASELINE_PANEL_MASS_KG
    )


    delta_M = np.zeros(
        (4, 4)
    )


    window_length = np.trapezoid(
        window,
        y_passive,
    )


    if (
        added_mass_kg > 0.0
        and
        window_length > 0.0
    ):

        mass_per_length = (
            added_mass_kg
            / window_length
        )


        for i in range(4):

            for j in range(4):

                delta_M[
                    i,
                    j
                ] = (
                    mass_per_length
                    * np.trapezoid(
                        window
                        * Phi_mass_line[
                            :,
                            i
                        ]
                        * Phi_mass_line[
                            :,
                            j
                        ],

                        y_passive,
                    )
                )


    delta_M = (
        0.5
        * (
            delta_M
            + delta_M.T
        )
    )


    # ========================================================
    # 8E. MODIFIED STRUCTURAL MATRICES
    # ========================================================

    M_new = (
        M_modal_SI
        + delta_M
    )


    K_new = (
        K_modal_SI
        + delta_K
    )


    # Positive-definite checks
    if np.min(
        np.linalg.eigvalsh(
            M_new
        )
    ) <= 0.0:

        raise RuntimeError(
            "Modified M is not positive definite."
        )


    if np.min(
        np.linalg.eigvalsh(
            K_new
        )
    ) <= 0.0:

        raise RuntimeError(
            "Modified K is not positive definite."
        )


    C_new = (
        build_constant_modal_damping(
            M_new,
            K_new,
        )
    )


    return {
        "M":
            M_new,

        "C":
            C_new,

        "K":
            K_new,

        "delta_M":
            delta_M,

        "delta_K":
            delta_K,

        "window":
            window,

        "added_mass_kg":
            added_mass_kg,

        "added_mass_fraction":
            added_mass_fraction,

        "eta_start":
            eta_start,

        "eta_end":
            eta_end,
    }


# ------------------------------------------------------------
# 9. Frequency helper
# ------------------------------------------------------------

def generalized_frequencies_v2(
    M,
    K,
):

    eigvals, _ = eigh(
        K,
        M,
    )


    return (
        np.sqrt(
            eigvals
        )
        / (
            2.0
            * np.pi
        )
    )


# ------------------------------------------------------------
# 10. ZERO-MODIFICATION CHECK
# ------------------------------------------------------------

passive_zero_v2 = (
    build_passive_modal_matrices_v2(
        eta_start=0.50,
        eta_end=0.75,
        bending_stiffness_gain=0.0,
        torsional_stiffness_gain=0.0,
        added_mass_fraction=0.0,
    )
)


assert np.allclose(
    passive_zero_v2["M"],
    M_modal_SI,
)


assert np.allclose(
    passive_zero_v2["K"],
    K_modal_SI,
)


assert np.allclose(
    passive_zero_v2["C"],
    C_modal_SI,
    rtol=1e-8,
    atol=1e-8,
)


# ------------------------------------------------------------
# 11. MORE REALISTIC DEMONSTRATION
#
# Region:
#
#       50–75% span
#
# Structural modification:
#
#       +10% local bending proxy
#       +10% local torsion proxy
#
# Mass:
#
#       +2% wing-panel mass
#
# ------------------------------------------------------------

demo_v2 = (
    build_passive_modal_matrices_v2(
        eta_start=0.50,
        eta_end=0.75,
        bending_stiffness_gain=0.10,
        torsional_stiffness_gain=0.10,
        added_mass_fraction=0.02,
    )
)


baseline_freq_v2 = (
    generalized_frequencies_v2(
        M_modal_SI,
        K_modal_SI,
    )
)


demo_freq_v2 = (
    generalized_frequencies_v2(
        demo_v2["M"],
        demo_v2["K"],
    )
)


passive_v2_frequency_table = pd.DataFrame({
    "mode_id":
        [1, 2, 3, 4],

    "baseline_Hz":
        baseline_freq_v2,

    "demo_v2_Hz":
        demo_freq_v2,

    "change_pct":
        100.0
        * (
            demo_freq_v2
            - baseline_freq_v2
        )
        / baseline_freq_v2,
})


# ------------------------------------------------------------
# 12. REPORT
# ------------------------------------------------------------

print(
    "\nCALIBRATED PASSIVE FRAMEWORK V2"
)


print(
    f"\nBaseline panel mass = "
    f"{BASELINE_PANEL_MASS_KG:.4f} kg"
)


print(
    f"Demo added physical mass = "
    f"{demo_v2['added_mass_kg']:.4f} kg"
)


print(
    f"Demo added mass fraction = "
    f"{100.0 * demo_v2['added_mass_fraction']:.2f}%"
)


print(
    "\nDry-frequency response:"
)

display(
    passive_v2_frequency_table
)


# ------------------------------------------------------------
# 13. REGULARIZED STRUCTURAL ACTIVITY PLOT
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for mode_id in range(4):

    indicator = (
        Phi_bending_curvature_reg[
            :,
            mode_id
        ]**2

        +

        Phi_twist_gradient_reg[
            :,
            mode_id
        ]**2
    )


    indicator /= max(
        np.max(
            indicator
        ),
        1e-16,
    )


    plt.plot(
        eta_passive,
        indicator,
        label=
            f"Mode {mode_id + 1}",
    )


# Show design region that we will permit later
plt.axvline(
    0.05,
    linestyle="--",
    linewidth=1.0,
)

plt.axvline(
    0.95,
    linestyle="--",
    linewidth=1.0,
)


plt.xlabel(
    "Normalized span η"
)

plt.ylabel(
    "Normalized structural-activity indicator"
)

plt.title(
    "Regularized modal structural-activity distribution"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nCell 48 calibrated passive framework completed."
)

In [ ]:
# ============================================================
# CELL 49 — PASSIVE REINFORCEMENT LOCATION SCREEN
# ============================================================
#
# ENGINEERING QUESTION:
#
# For a FIXED reinforcement budget, where along the span
# should the reinforcement be placed to obtain the greatest
# increase in flutter velocity?
#
#
# Every candidate has exactly:
#
#       reinforcement length = 20% span
#       local bending gain    = +10%
#       local torsion gain    = +10%
#       added physical mass   = 2% wing-panel mass
#
# Only spanwise LOCATION changes.
#
#
# Screening condition:
#
#       Mach = 0.901
#
# Reason:
#
# The validated DLM/p-k model had good agreement with the
# experiment here while remaining away from the near-transonic
# Mach 0.960 limitation.
#
# ============================================================

import io
import contextlib


# ------------------------------------------------------------
# 1. Design condition
# ------------------------------------------------------------

PASSIVE_DESIGN_MACH = 0.901


baseline_row_0901 = (
    corrected_pk_validation[
        np.isclose(
            corrected_pk_validation["Mach"],
            PASSIVE_DESIGN_MACH,
        )
    ]
    .iloc[0]
)


U_exp_passive = float(
    baseline_row_0901[
        "U_exp_m_s"
    ]
)

q_exp_passive = float(
    baseline_row_0901[
        "q_exp_Pa"
    ]
)

U_flutter_baseline = float(
    baseline_row_0901[
        "U_corrected_m_s"
    ]
)

lambda_flutter_baseline = float(
    baseline_row_0901[
        "lambda_corrected"
    ]
)

f_flutter_baseline = float(
    baseline_row_0901[
        "f_corrected_Hz"
    ]
)


print(
    "PASSIVE LOCATION SCREEN"
)

print(
    f"\nDesign Mach = "
    f"{PASSIVE_DESIGN_MACH:.3f}"
)

print(
    f"Baseline predicted flutter velocity = "
    f"{U_flutter_baseline:.3f} m/s"
)

print(
    f"Baseline predicted flutter frequency = "
    f"{f_flutter_baseline:.3f} Hz"
)


# ------------------------------------------------------------
# 2. Fixed reinforcement budget
# ------------------------------------------------------------

PASSIVE_WINDOW_LENGTH = (
    0.20
)

PASSIVE_BENDING_GAIN = (
    0.10
)

PASSIVE_TORSION_GAIN = (
    0.10
)

PASSIVE_MASS_FRACTION = (
    0.02
)


PASSIVE_ADDED_MASS_KG = (
    PASSIVE_MASS_FRACTION
    * BASELINE_PANEL_MASS_KG
)


print(
    "\nFixed reinforcement budget:"
)

print(
    f"Span length       = "
    f"{100*PASSIVE_WINDOW_LENGTH:.1f}%"
)

print(
    f"Bending gain      = "
    f"{100*PASSIVE_BENDING_GAIN:.1f}%"
)

print(
    f"Torsional gain    = "
    f"{100*PASSIVE_TORSION_GAIN:.1f}%"
)

print(
    f"Added mass        = "
    f"{PASSIVE_ADDED_MASS_KG:.4f} kg "
    f"({100*PASSIVE_MASS_FRACTION:.1f}% panel mass)"
)


# ------------------------------------------------------------
# 3. p-k solver supporting modified structural matrices
# ------------------------------------------------------------

def frozen_k_pk_passive(
    Ma,
    U,
    q_dyn,
    k_PA,
    M_struct,
    C_struct,
    K_struct,
):
    """
    Conventional frozen-k p-k eigenproblem using modified
    structural matrices while retaining the validated
    aerodynamic generalized-force database.
    """


    omega = (
        float(k_PA)
        * float(U)
    )


    if omega <= 0.0:

        raise ValueError(
            "omega must be positive."
        )


    # Generalized aerodynamic matrix remains expressed in the
    # original four-mode coordinate basis.
    Qhh = (
        interpolate_cached_Qhh(
            Ma=Ma,
            k_query=k_PA,
        )
    )


    QR = np.real(
        Qhh
    )

    QI = np.imag(
        Qhh
    )


    K_eff = (
        K_struct
        - q_dyn
        * QR
    )


    C_eff = (
        C_struct
        - (
            q_dyn
            / omega
        )
        * QI
    )


    M_inv = np.linalg.inv(
        M_struct
    )


    Z = np.zeros(
        (4, 4),
        dtype=float,
    )

    I = np.eye(
        4,
        dtype=float,
    )


    A = np.block([
        [
            Z,
            I,
        ],

        [
            -M_inv @ K_eff,
            -M_inv @ C_eff,
        ],
    ])


    poles, eigvectors = (
        np.linalg.eig(
            A
        )
    )


    return (
        poles,
        eigvectors,
    )


# ------------------------------------------------------------
# 4. MAC-tracked Mode-1 p-k with modified structure
# ------------------------------------------------------------

def pk_mode1_passive(
    Ma,
    U,
    q_dyn,
    M_struct,
    C_struct,
    K_struct,
    initial_k=None,
    initial_reference=None,
    tol=2e-7,
    max_iter=60,
    relaxation=0.65,
):


    if initial_k is None:

        k_current = (
            omega_analysis[0]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    # --------------------------------------------------------
    # Initial branch reference
    # --------------------------------------------------------

    if initial_reference is None:

        reference_q = np.array(
            [
                1.0,
                0.0,
                0.0,
                0.0,
            ],
            dtype=complex,
        )

    else:

        reference_q = np.asarray(
            initial_reference,
            dtype=complex,
        ).copy()


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M_struct,
        )
    )


    # --------------------------------------------------------
    # p-k iteration
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
        ) = frozen_k_pk_passive(
            Ma=Ma,

            U=U,

            q_dyn=q_dyn,

            k_PA=k_current,

            M_struct=M_struct,

            C_struct=C_struct,

            K_struct=K_struct,
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        if len(
            positive_indices
        ) == 0:

            raise RuntimeError(
                "No positive-frequency poles found."
            )


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M_struct,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M_struct,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_index = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_index
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        mac_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M_struct,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M_struct,
            )
        )


        # ----------------------------------------------------
        # Update k
        # ----------------------------------------------------

        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            "Passive p-k iteration did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {
        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            mac_selected,

        "iterations":
            iteration,

        "modal_vector":
            q_selected.copy(),
    }


# ------------------------------------------------------------
# 5. Flutter-root search for one passive configuration
# ------------------------------------------------------------

def solve_passive_flutter_root(
    M_struct,
    C_struct,
    K_struct,
    lambda_guess,
):
    """
    Find Mode-1 flutter root at Mach 0.901 using only the
    already-cached aerodynamic GAF database.
    """


    def evaluate_lambda(
        lam,
        reference=None,
    ):


        U = (
            lam
            * U_exp_passive
        )


        q = (
            lam**2
            * q_exp_passive
        )


        if reference is None:

            initial_k = None

            initial_reference = None

        else:

            initial_k = (
                reference[
                    "k_PA"
                ]
            )

            initial_reference = (
                reference[
                    "modal_vector"
                ]
            )


        result = (
            pk_mode1_passive(
                Ma=PASSIVE_DESIGN_MACH,

                U=U,

                q_dyn=q,

                M_struct=M_struct,

                C_struct=C_struct,

                K_struct=K_struct,

                initial_k=
                    initial_k,

                initial_reference=
                    initial_reference,
            )
        )


        return {
            "lambda":
                float(
                    lam
                ),

            "U":
                float(
                    U
                ),

            "q":
                float(
                    q
                ),

            **result,
        }


    # --------------------------------------------------------
    # Initial bracket around validated baseline root
    # --------------------------------------------------------

    half_width = (
        0.06
    )


    for expansion in range(
        10
    ):


        lam_low = max(
            0.60,
            lambda_guess
            - half_width,
        )


        lam_high = min(
            1.80,
            lambda_guess
            + half_width,
        )


        low = (
            evaluate_lambda(
                lam_low
            )
        )


        high = (
            evaluate_lambda(
                lam_high,
                reference=low,
            )
        )


        if (
            low[
                "sigma"
            ]
            * high[
                "sigma"
            ]
            <= 0.0
        ):

            break


        half_width += (
            0.05
        )


    else:

        raise RuntimeError(
            "Could not bracket passive flutter root."
        )


    # Ensure low = stable
    if (
        low[
            "sigma"
        ] > 0.0
    ):

        low, high = (
            high,
            low,
        )


    # --------------------------------------------------------
    # Root refinement
    # --------------------------------------------------------

    for root_iteration in range(
        1,
        25
    ):


        sigma_low = (
            low[
                "sigma"
            ]
        )


        sigma_high = (
            high[
                "sigma"
            ]
        )


        lam_trial = (
            low[
                "lambda"
            ]

            - sigma_low
            * (
                high[
                    "lambda"
                ]
                - low[
                    "lambda"
                ]
            )

            / (
                sigma_high
                - sigma_low
            )
        )


        # Safeguard
        lower = min(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        upper = max(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        width = (
            upper
            - lower
        )


        if (
            lam_trial
            <= lower
            + 0.05
            * width

            or

            lam_trial
            >= upper
            - 0.05
            * width
        ):

            lam_trial = (
                0.5
                * (
                    low[
                        "lambda"
                    ]
                    + high[
                        "lambda"
                    ]
                )
            )


        reference = (
            low

            if abs(
                lam_trial
                - low[
                    "lambda"
                ]
            )

            <=

            abs(
                high[
                    "lambda"
                ]
                - lam_trial
            )

            else high
        )


        trial = (
            evaluate_lambda(
                lam_trial,
                reference=reference,
            )
        )


        if (
            abs(
                trial[
                    "sigma"
                ]
            )
            < 1e-4
        ):

            return trial


        if (
            trial[
                "sigma"
            ]
            < 0.0
        ):

            low = (
                trial
            )

        else:

            high = (
                trial
            )


    return min(
        [
            low,
            high,
        ],

        key=lambda x:
            abs(
                x[
                    "sigma"
                ]
            ),
    )


# ------------------------------------------------------------
# 6. Candidate span locations
#
# Fixed 20%-span reinforcement:
#
#     05-25%
#     10-30%
#     ...
#     75-95%
#
# This excludes the exact root and extreme tip.
# ------------------------------------------------------------

eta_start_candidates = np.arange(
    0.05,
    0.751,
    0.05,
)


location_scan_rows = []


# ------------------------------------------------------------
# 7. Evaluate every location
# ------------------------------------------------------------

for eta_start in eta_start_candidates:


    eta_end = (
        eta_start
        + PASSIVE_WINDOW_LENGTH
    )


    passive_design = (
        build_passive_modal_matrices_v2(
            eta_start=
                eta_start,

            eta_end=
                eta_end,

            bending_stiffness_gain=
                PASSIVE_BENDING_GAIN,

            torsional_stiffness_gain=
                PASSIVE_TORSION_GAIN,

            added_mass_fraction=
                PASSIVE_MASS_FRACTION,
        )
    )


    # Dry frequencies for interpretation
    dry_freq = (
        generalized_frequencies_v2(
            passive_design[
                "M"
            ],

            passive_design[
                "K"
            ],
        )
    )


    # Suppress interpolation edge messages during scan
    with contextlib.redirect_stdout(
        io.StringIO()
    ):


        flutter_result = (
            solve_passive_flutter_root(
                M_struct=
                    passive_design[
                        "M"
                    ],

                C_struct=
                    passive_design[
                        "C"
                    ],

                K_struct=
                    passive_design[
                        "K"
                    ],

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    flutter_improvement_pct = (
        100.0
        * (
            flutter_result[
                "U"
            ]
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    # --------------------------------------------------------
    # Improvement per added physical mass
    # --------------------------------------------------------

    improvement_per_100g = (
        flutter_improvement_pct

        / (
            PASSIVE_ADDED_MASS_KG
            / 0.1
        )
    )


    location_scan_rows.append({

        "eta_start":
            eta_start,

        "eta_end":
            eta_end,

        "eta_mid":
            0.5
            * (
                eta_start
                + eta_end
            ),

        "added_mass_kg":
            PASSIVE_ADDED_MASS_KG,

        "mode1_dry_Hz":
            dry_freq[0],

        "mode2_dry_Hz":
            dry_freq[1],

        "mode3_dry_Hz":
            dry_freq[2],

        "mode4_dry_Hz":
            dry_freq[3],

        "flutter_velocity_m_s":
            flutter_result[
                "U"
            ],

        "flutter_frequency_Hz":
            flutter_result[
                "frequency_Hz"
            ],

        "flutter_improvement_pct":
            flutter_improvement_pct,

        "improvement_pct_per_100g":
            improvement_per_100g,

        "sigma_final":
            flutter_result[
                "sigma"
            ],

        "MAC":
            flutter_result[
                "MAC"
            ],
    })


# ------------------------------------------------------------
# 8. Results table
# ------------------------------------------------------------

passive_location_scan = (
    pd.DataFrame(
        location_scan_rows
    )
)


# ------------------------------------------------------------
# 9. Best location
# ------------------------------------------------------------

best_location_index = (
    passive_location_scan[
        "flutter_improvement_pct"
    ]
    .idxmax()
)


best_passive_location = (
    passive_location_scan
    .loc[
        best_location_index
    ]
)


# ------------------------------------------------------------
# 10. Rank candidates
# ------------------------------------------------------------

passive_location_ranking = (
    passive_location_scan
    .sort_values(
        "flutter_improvement_pct",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nPASSIVE REINFORCEMENT LOCATION RESULTS"
)


display(
    passive_location_scan[
        [
            "eta_start",
            "eta_end",
            "flutter_velocity_m_s",
            "flutter_improvement_pct",
            "flutter_frequency_Hz",
            "mode1_dry_Hz",
            "mode2_dry_Hz",
        ]
    ]
)


print(
    "\nTop five locations:"
)


display(
    passive_location_ranking[
        [
            "eta_start",
            "eta_end",
            "flutter_velocity_m_s",
            "flutter_improvement_pct",
            "improvement_pct_per_100g",
        ]
    ]
    .head(
        5
    )
)


print(
    "\nBest fixed-budget location:"
)


print(
    f"Span region = "
    f"{100*best_passive_location['eta_start']:.0f}% "
    f"to "
    f"{100*best_passive_location['eta_end']:.0f}%"
)


print(
    f"Flutter velocity = "
    f"{best_passive_location['flutter_velocity_m_s']:.3f} m/s"
)


print(
    f"Baseline flutter velocity = "
    f"{U_flutter_baseline:.3f} m/s"
)


print(
    f"Flutter improvement = "
    f"{best_passive_location['flutter_improvement_pct']:+.3f}%"
)


print(
    f"Added physical mass = "
    f"{PASSIVE_ADDED_MASS_KG:.4f} kg"
)


# ------------------------------------------------------------
# 11. Location-response plot
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    passive_location_scan[
        "eta_mid"
    ],

    passive_location_scan[
        "flutter_improvement_pct"
    ],

    "o-",
)


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.axvline(
    best_passive_location[
        "eta_mid"
    ],

    linestyle="--",

    linewidth=1.0,

    label="Best location",
)


plt.xlabel(
    "Reinforcement region midpoint, η"
)

plt.ylabel(
    "Flutter-velocity change (%)"
)

plt.title(
    "Where should a fixed reinforcement budget be placed?"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 12. Flutter velocity plot
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    passive_location_scan[
        "eta_mid"
    ],

    passive_location_scan[
        "flutter_velocity_m_s"
    ],

    "o-",

    label="Reinforced",
)


plt.axhline(
    U_flutter_baseline,

    linestyle="--",

    linewidth=1.0,

    label="Baseline",
)


plt.xlabel(
    "Reinforcement region midpoint, η"
)

plt.ylabel(
    "Predicted flutter velocity (m/s)"
)

plt.title(
    "Passive flutter margin versus reinforcement location"
)

plt.grid(
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nCell 49 passive location screen completed."
)

In [ ]:
# ============================================================
# CELL 50 — PASSIVE DESIGN MECHANISM DECOMPOSITION
# ============================================================
#
# ENGINEERING QUESTION:
#
# WHY does the 35–55% span reinforcement improve flutter?
#
#
# Decompose the winning fixed-budget design into:
#
#   1. Baseline
#   2. Added mass only
#   3. Bending stiffness only
#   4. Torsional stiffness only
#   5. Bending + torsional stiffness, zero mass
#   6. Complete design: stiffness + physical mass
#
#
# This tells us whether the improvement comes mainly from:
#
#       bending stiffness
#       torsional stiffness
#       modal mass redistribution
#       or interaction between them.
#
# No new DLM calculations are required.
#
# ============================================================

import io
import contextlib


# ------------------------------------------------------------
# 1. Best region from Cell 49
# ------------------------------------------------------------

BEST_ETA_START = float(
    best_passive_location[
        "eta_start"
    ]
)

BEST_ETA_END = float(
    best_passive_location[
        "eta_end"
    ]
)


print(
    "PASSIVE MECHANISM DECOMPOSITION"
)

print(
    f"\nSelected region = "
    f"{100*BEST_ETA_START:.0f}% "
    f"to "
    f"{100*BEST_ETA_END:.0f}% span"
)


# ------------------------------------------------------------
# 2. Mechanism cases
# ------------------------------------------------------------

mechanism_cases = [

    {
        "case":
            "Baseline",

        "bending_gain":
            0.00,

        "torsion_gain":
            0.00,

        "mass_fraction":
            0.00,
    },

    {
        "case":
            "Mass only",

        "bending_gain":
            0.00,

        "torsion_gain":
            0.00,

        "mass_fraction":
            PASSIVE_MASS_FRACTION,
    },

    {
        "case":
            "Bending stiffness only",

        "bending_gain":
            PASSIVE_BENDING_GAIN,

        "torsion_gain":
            0.00,

        "mass_fraction":
            0.00,
    },

    {
        "case":
            "Torsional stiffness only",

        "bending_gain":
            0.00,

        "torsion_gain":
            PASSIVE_TORSION_GAIN,

        "mass_fraction":
            0.00,
    },

    {
        "case":
            "Both stiffness, no mass",

        "bending_gain":
            PASSIVE_BENDING_GAIN,

        "torsion_gain":
            PASSIVE_TORSION_GAIN,

        "mass_fraction":
            0.00,
    },

    {
        "case":
            "Complete design",

        "bending_gain":
            PASSIVE_BENDING_GAIN,

        "torsion_gain":
            PASSIVE_TORSION_GAIN,

        "mass_fraction":
            PASSIVE_MASS_FRACTION,
    },
]


# ------------------------------------------------------------
# 3. Solve each mechanism case
# ------------------------------------------------------------

mechanism_rows = []

mechanism_flutter_results = {}


for case in mechanism_cases:


    design = (
        build_passive_modal_matrices_v2(
            eta_start=
                BEST_ETA_START,

            eta_end=
                BEST_ETA_END,

            bending_stiffness_gain=
                case[
                    "bending_gain"
                ],

            torsional_stiffness_gain=
                case[
                    "torsion_gain"
                ],

            added_mass_fraction=
                case[
                    "mass_fraction"
                ],
        )
    )


    # --------------------------------------------------------
    # Dry structural frequencies
    # --------------------------------------------------------

    dry_frequency = (
        generalized_frequencies_v2(
            design[
                "M"
            ],

            design[
                "K"
            ],
        )
    )


    # --------------------------------------------------------
    # Flutter solution
    # --------------------------------------------------------

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        flutter = (
            solve_passive_flutter_root(
                M_struct=
                    design[
                        "M"
                    ],

                C_struct=
                    design[
                        "C"
                    ],

                K_struct=
                    design[
                        "K"
                    ],

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    mechanism_flutter_results[
        case[
            "case"
        ]
    ] = (
        flutter
    )


    # --------------------------------------------------------
    # Modal-coordinate composition at flutter
    #
    # This is NOT called an energy fraction.
    #
    # It simply shows relative amplitudes of the four original
    # structural modal coordinates in the aeroelastic mode.
    # --------------------------------------------------------

    q_flutter = np.asarray(
        flutter[
            "modal_vector"
        ],
        dtype=complex,
    )


    modal_amplitudes = np.abs(
        q_flutter
    )


    modal_amplitudes = (
        modal_amplitudes
        /
        max(
            np.max(
                modal_amplitudes
            ),
            1e-16,
        )
    )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    mechanism_rows.append({

        "case":
            case[
                "case"
            ],

        "bending_gain_pct":
            100.0
            * case[
                "bending_gain"
            ],

        "torsion_gain_pct":
            100.0
            * case[
                "torsion_gain"
            ],

        "added_mass_kg":
            design[
                "added_mass_kg"
            ],

        "mode1_dry_Hz":
            dry_frequency[0],

        "mode2_dry_Hz":
            dry_frequency[1],

        "mode3_dry_Hz":
            dry_frequency[2],

        "mode4_dry_Hz":
            dry_frequency[3],

        "flutter_velocity_m_s":
            flutter[
                "U"
            ],

        "flutter_frequency_Hz":
            flutter[
                "frequency_Hz"
            ],

        "flutter_mode1_amp":
            modal_amplitudes[0],

        "flutter_mode2_amp":
            modal_amplitudes[1],

        "flutter_mode3_amp":
            modal_amplitudes[2],

        "flutter_mode4_amp":
            modal_amplitudes[3],

        "sigma_final":
            flutter[
                "sigma"
            ],

        "MAC":
            flutter[
                "MAC"
            ],

        "delta_M_norm":
            np.linalg.norm(
                design[
                    "delta_M"
                ]
            ),

        "delta_K_norm":
            np.linalg.norm(
                design[
                    "delta_K"
                ]
            ),
    })


# ------------------------------------------------------------
# 4. Build results dataframe
# ------------------------------------------------------------

passive_mechanism_table = (
    pd.DataFrame(
        mechanism_rows
    )
)


# ------------------------------------------------------------
# 5. Baseline flutter reference
# ------------------------------------------------------------

baseline_mechanism_row = (
    passive_mechanism_table[
        passive_mechanism_table[
            "case"
        ]
        == "Baseline"
    ]
    .iloc[0]
)


U_baseline_mechanism = float(
    baseline_mechanism_row[
        "flutter_velocity_m_s"
    ]
)


# ------------------------------------------------------------
# 6. Flutter change
# ------------------------------------------------------------

passive_mechanism_table[
    "flutter_change_pct"
] = (
    100.0
    * (
        passive_mechanism_table[
            "flutter_velocity_m_s"
        ]
        - U_baseline_mechanism
    )
    /
    U_baseline_mechanism
)


# ------------------------------------------------------------
# 7. Extract principal mechanism cases
# ------------------------------------------------------------

def mechanism_row(
    name,
):

    return (
        passive_mechanism_table[
            passive_mechanism_table[
                "case"
            ]
            == name
        ]
        .iloc[0]
    )


row_mass = mechanism_row(
    "Mass only"
)

row_bending = mechanism_row(
    "Bending stiffness only"
)

row_torsion = mechanism_row(
    "Torsional stiffness only"
)

row_stiffness = mechanism_row(
    "Both stiffness, no mass"
)

row_complete = mechanism_row(
    "Complete design"
)


# ------------------------------------------------------------
# 8. Interaction / non-additivity
#
# If mass and stiffness effects were perfectly independent:
#
# U_complete_expected =
#
#       U_baseline
#     + (U_stiffness - U_baseline)
#     + (U_mass - U_baseline)
#
# Difference from this value quantifies interaction.
# ------------------------------------------------------------

U_expected_additive = (
    float(
        row_stiffness[
            "flutter_velocity_m_s"
        ]
    )

    +

    float(
        row_mass[
            "flutter_velocity_m_s"
        ]
    )

    -

    U_baseline_mechanism
)


interaction_velocity = (
    float(
        row_complete[
            "flutter_velocity_m_s"
        ]
    )

    - U_expected_additive
)


interaction_pct_baseline = (
    100.0
    * interaction_velocity
    / U_baseline_mechanism
)


# ------------------------------------------------------------
# 9. Report primary mechanism table
# ------------------------------------------------------------

print(
    "\nFlutter mechanism comparison:"
)


display(
    passive_mechanism_table[
        [
            "case",
            "added_mass_kg",
            "mode1_dry_Hz",
            "mode2_dry_Hz",
            "flutter_velocity_m_s",
            "flutter_change_pct",
            "flutter_frequency_Hz",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Relative modal-coordinate composition
# ------------------------------------------------------------

print(
    "\nRelative aeroelastic modal-coordinate amplitudes "
    "at each flutter boundary:"
)


display(
    passive_mechanism_table[
        [
            "case",
            "flutter_mode1_amp",
            "flutter_mode2_amp",
            "flutter_mode3_amp",
            "flutter_mode4_amp",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Numerical mechanism summary
# ------------------------------------------------------------

print(
    "\nMECHANISM SUMMARY"
)


print(
    f"\nMass-only flutter effect = "
    f"{row_mass['flutter_change_pct']:+.3f}%"
)


print(
    f"Bending-stiffness-only effect = "
    f"{row_bending['flutter_change_pct']:+.3f}%"
)


print(
    f"Torsional-stiffness-only effect = "
    f"{row_torsion['flutter_change_pct']:+.3f}%"
)


print(
    f"Combined stiffness, zero-mass effect = "
    f"{row_stiffness['flutter_change_pct']:+.3f}%"
)


print(
    f"Complete physical design effect = "
    f"{row_complete['flutter_change_pct']:+.3f}%"
)


print(
    f"\nStiffness/mass interaction contribution = "
    f"{interaction_pct_baseline:+.4f}% "
    "of baseline flutter velocity"
)


# ------------------------------------------------------------
# 12. Flutter-benefit bar chart
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 5)
)


plt.bar(
    passive_mechanism_table[
        "case"
    ],

    passive_mechanism_table[
        "flutter_change_pct"
    ],
)


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.ylabel(
    "Flutter-velocity change (%)"
)


plt.title(
    "Why does the 35–55% span reinforcement work?"
)


plt.xticks(
    rotation=25,
    ha="right",
)


plt.grid(
    axis="y",
    alpha=0.25,
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 13. Flutter mode composition
# ------------------------------------------------------------

modal_plot_data = (
    passive_mechanism_table[
        [
            "flutter_mode1_amp",
            "flutter_mode2_amp",
            "flutter_mode3_amp",
            "flutter_mode4_amp",
        ]
    ]
)


x = np.arange(
    len(
        passive_mechanism_table
    )
)


width = 0.18


plt.figure(
    figsize=(11, 5)
)


for mode_id in range(4):

    plt.bar(
        x
        + (
            mode_id
            - 1.5
        )
        * width,

        modal_plot_data.iloc[
            :,
            mode_id
        ],

        width=

            width,

        label=
            f"Structural Mode {mode_id + 1}",
    )


plt.xticks(
    x,
    passive_mechanism_table[
        "case"
    ],
    rotation=25,
    ha="right",
)


plt.ylabel(
    "Relative modal-coordinate amplitude"
)


plt.title(
    "Composition of the critical aeroelastic mode"
)


plt.grid(
    axis="y",
    alpha=0.25,
)


plt.legend()


plt.tight_layout()

plt.show()


print(
    "\nCell 50 passive mechanism decomposition completed."
)

In [ ]:
# ============================================================
# CELL 51 — PASSIVE TORSIONAL DESIGN TRADE SPACE
# ============================================================
#
# ENGINEERING QUESTION:
#
# After establishing that torsional stiffness dominates the
# passive flutter benefit:
#
#     How much local torsional-stiffness increase is required?
#
# and:
#
#     How sensitive is that benefit to reinforcement mass?
#
#
# IMPORTANT:
#
# Torsional-stiffness gain and mass are intentionally treated
# as INDEPENDENT requirements here.
#
# We are NOT claiming that a particular mass automatically
# produces a particular GJ increase.
#
# Siemens NX will later provide a physical geometry from which
# mass and torsional stiffness can be reconciled.
#
# ============================================================

import io
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Fixed span region from Cell 49
# ------------------------------------------------------------

TRADE_ETA_START = (
    BEST_ETA_START
)

TRADE_ETA_END = (
    BEST_ETA_END
)


print(
    "PASSIVE TORSIONAL DESIGN TRADE STUDY"
)

print(
    f"\nSelected reinforcement region = "
    f"{100*TRADE_ETA_START:.0f}% "
    f"to "
    f"{100*TRADE_ETA_END:.0f}% span"
)


# ------------------------------------------------------------
# 2. Design variables
# ------------------------------------------------------------

torsion_gain_values = np.array([
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
])


mass_fraction_values = np.array([
    0.00,
    0.01,
    0.02,
    0.03,
    0.04,
])


# ------------------------------------------------------------
# 3. Storage
# ------------------------------------------------------------

trade_rows = []


# ------------------------------------------------------------
# 4. Evaluate trade space
#
# Bending gain = 0
#
# because Cell 50 demonstrated that torsional stiffness is the
# dominant passive mechanism.
# ------------------------------------------------------------

for torsion_gain in torsion_gain_values:

    for mass_fraction in mass_fraction_values:


        design = (
            build_passive_modal_matrices_v2(
                eta_start=
                    TRADE_ETA_START,

                eta_end=
                    TRADE_ETA_END,

                bending_stiffness_gain=
                    0.0,

                torsional_stiffness_gain=
                    torsion_gain,

                added_mass_fraction=
                    mass_fraction,
            )
        )


        # ----------------------------------------------------
        # Dry structural frequencies
        # ----------------------------------------------------

        dry_freq = (
            generalized_frequencies_v2(
                design[
                    "M"
                ],

                design[
                    "K"
                ],
            )
        )


        # ----------------------------------------------------
        # Flutter root
        # ----------------------------------------------------

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            flutter = (
                solve_passive_flutter_root(
                    M_struct=
                        design[
                            "M"
                        ],

                    C_struct=
                        design[
                            "C"
                        ],

                    K_struct=
                        design[
                            "K"
                        ],

                    lambda_guess=
                        lambda_flutter_baseline,
                )
            )


        flutter_improvement = (
            100.0
            * (
                flutter[
                    "U"
                ]
                - U_flutter_baseline
            )
            / U_flutter_baseline
        )


        trade_rows.append({

            "torsion_gain_pct":
                100.0
                * torsion_gain,

            "mass_fraction_pct":
                100.0
                * mass_fraction,

            "added_mass_kg":
                design[
                    "added_mass_kg"
                ],

            "mode1_dry_Hz":
                dry_freq[0],

            "mode2_dry_Hz":
                dry_freq[1],

            "flutter_velocity_m_s":
                flutter[
                    "U"
                ],

            "flutter_frequency_Hz":
                flutter[
                    "frequency_Hz"
                ],

            "flutter_improvement_pct":
                flutter_improvement,

            "sigma_final":
                flutter[
                    "sigma"
                ],

            "MAC":
                flutter[
                    "MAC"
                ],
        })


# ------------------------------------------------------------
# 5. Dataframe
# ------------------------------------------------------------

passive_torsion_trade = (
    pd.DataFrame(
        trade_rows
    )
)


print(
    "\nPassive torsional trade-space results:"
)

display(
    passive_torsion_trade
)


# ------------------------------------------------------------
# 6. Pivot table:
#
# rows    = torsional stiffness gain
# columns = mass fraction
# values  = flutter improvement
# ------------------------------------------------------------

trade_pivot = (
    passive_torsion_trade
    .pivot(
        index=
            "torsion_gain_pct",

        columns=
            "mass_fraction_pct",

        values=
            "flutter_improvement_pct",
    )
)


print(
    "\nFlutter-velocity improvement (%)"
)

print(
    "Rows = local torsional-stiffness increase"
)

print(
    "Columns = added physical mass"
)


display(
    trade_pivot
)


# ------------------------------------------------------------
# 7. Zero-mass stiffness requirement
#
# Determine approximate torsional gain required for:
#
#       +2%
#       +3%
#       +5%
#
# flutter-margin targets.
#
# This is a REQUIREMENT estimate, not a hardware claim.
# ------------------------------------------------------------

zero_mass_trade = (
    passive_torsion_trade[
        np.isclose(
            passive_torsion_trade[
                "mass_fraction_pct"
            ],
            0.0,
        )
    ]
    .sort_values(
        "torsion_gain_pct"
    )
)


target_improvements = [
    2.0,
    3.0,
    5.0,
]


requirement_rows = []


for target in target_improvements:


    x = (
        zero_mass_trade[
            "flutter_improvement_pct"
        ]
        .to_numpy()
    )


    y = (
        zero_mass_trade[
            "torsion_gain_pct"
        ]
        .to_numpy()
    )


    if (
        target >= np.min(x)
        and
        target <= np.max(x)
    ):

        required_gain = (
            np.interp(
                target,
                x,
                y,
            )
        )

    else:

        required_gain = np.nan


    requirement_rows.append({

        "target_flutter_improvement_pct":
            target,

        "estimated_required_torsion_gain_pct":
            required_gain,
    })


torsional_requirement_table = (
    pd.DataFrame(
        requirement_rows
    )
)


print(
    "\nApproximate torsional-stiffness requirements "
    "(zero added-mass structural comparison):"
)


display(
    torsional_requirement_table
)


# ------------------------------------------------------------
# 8. Plot torsional gain versus flutter improvement
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for mass_fraction in mass_fraction_values:


    subset = (
        passive_torsion_trade[
            np.isclose(
                passive_torsion_trade[
                    "mass_fraction_pct"
                ],

                100.0
                * mass_fraction,
            )
        ]
        .sort_values(
            "torsion_gain_pct"
        )
    )


    plt.plot(
        subset[
            "torsion_gain_pct"
        ],

        subset[
            "flutter_improvement_pct"
        ],

        "o-",

        label=
            f"{100*mass_fraction:.0f}% added mass",
    )


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.xlabel(
    "Local torsional-stiffness increase (%)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Passive flutter benefit versus torsional-stiffness requirement"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 9. Plot mass sensitivity at each torsion level
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for torsion_gain in torsion_gain_values:


    subset = (
        passive_torsion_trade[
            np.isclose(
                passive_torsion_trade[
                    "torsion_gain_pct"
                ],

                100.0
                * torsion_gain,
            )
        ]
        .sort_values(
            "mass_fraction_pct"
        )
    )


    plt.plot(
        subset[
            "mass_fraction_pct"
        ],

        subset[
            "flutter_improvement_pct"
        ],

        "o-",

        label=
            f"+{100*torsion_gain:.0f}% GJ proxy",
    )


plt.xlabel(
    "Added physical mass (% of panel mass)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Mass sensitivity of the passive torsional concept"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 10. Simple summary metrics
# ------------------------------------------------------------

best_trade_index = (
    passive_torsion_trade[
        "flutter_improvement_pct"
    ]
    .idxmax()
)


best_trade_case = (
    passive_torsion_trade
    .loc[
        best_trade_index
    ]
)


print(
    "\nHighest flutter improvement in the investigated "
    "passive trade space:"
)


print(
    f"Torsional-stiffness gain = "
    f"{best_trade_case['torsion_gain_pct']:.1f}%"
)


print(
    f"Added mass = "
    f"{best_trade_case['mass_fraction_pct']:.1f}% "
    "of panel mass"
)


print(
    f"Added physical mass = "
    f"{best_trade_case['added_mass_kg']:.4f} kg"
)


print(
    f"Flutter improvement = "
    f"{best_trade_case['flutter_improvement_pct']:+.3f}%"
)


print(
    "\nCell 51 passive torsional trade study completed."
)

In [ ]:
# ============================================================
# CELL 51A — EXTEND MACH 0.901 GAF DATABASE FOR ACTIVE CONTROL
# ============================================================
#
# Cell 52 moved the p-k iteration slightly beyond the current
# Mach-0.901 aerodynamic-frequency cache:
#
#       old maximum k ≈ 0.3451 1/m
#
# Active control may shift the aeroelastic frequency, so we
# calculate a FEW additional DLM generalized aerodynamic
# matrices rather than relying on larger extrapolation.
#
# This is a one-time cache extension.
# ============================================================


ACTIVE_GAF_MACH = 0.901


# ------------------------------------------------------------
# 1. Additional k points
#
# Existing cache already reaches approximately:
#
#       0.345 1/m
#
# Extend smoothly to:
#
#       0.400 1/m
# ------------------------------------------------------------

additional_active_k = np.array([
    0.350,
    0.360,
    0.380,
    0.400,
])


print(
    "Extending Mach 0.901 aerodynamic GAF database..."
)


# ------------------------------------------------------------
# 2. Calculate only genuinely missing GAF points
# ------------------------------------------------------------

existing_keys_0901 = [
    key
    for key in _wall_qhh_cache.keys()
    if (
        abs(
            float(key[0])
            - ACTIVE_GAF_MACH
        )
        < 1e-6
        and int(key[2]) == 4
    )
]


existing_k_0901 = np.array(
    [
        float(key[1])
        for key in existing_keys_0901
    ]
)


for k_test in additional_active_k:


    already_available = (
        existing_k_0901.size > 0
        and
        np.min(
            np.abs(
                existing_k_0901
                - k_test
            )
        )
        < 1e-8
    )


    if already_available:

        print(
            f"k = {k_test:.3f} 1/m already cached."
        )

        continue


    print(
        f"Calculating new DLM GAF at "
        f"k = {k_test:.3f} 1/m ..."
    )


    _ = calculate_Qhh_wall_ips(
        Ma=ACTIVE_GAF_MACH,
        k_PA=float(
            k_test
        ),
        n_modes=4,
    )


# ------------------------------------------------------------
# 3. Rebuild Mach-0.901 aerodynamic database
# ------------------------------------------------------------

k_extended_0901, Q_extended_0901 = (
    get_cached_Qhh_data(
        Ma_target=ACTIVE_GAF_MACH,
        n_modes=4,
    )
)


cached_aero_database[
    ACTIVE_GAF_MACH
] = {
    "k":
        k_extended_0901,

    "Q":
        Q_extended_0901,
}


# ------------------------------------------------------------
# 4. Report
# ------------------------------------------------------------

print(
    "\nUpdated Mach 0.901 GAF database:"
)

print(
    f"Number of matrices = "
    f"{len(k_extended_0901)}"
)

print(
    f"k minimum = "
    f"{k_extended_0901.min():.5f} 1/m"
)

print(
    f"k maximum = "
    f"{k_extended_0901.max():.5f} 1/m"
)


assert (
    k_extended_0901.max()
    >= 0.400
)


print(
    "\nMach 0.901 active-control GAF extension completed."
)

In [ ]:
# ============================================================
# CELL 52 — ACTIVE TORSIONAL FLUTTER-SUPPRESSION BASELINE
# ============================================================
#
# ENGINEERING QUESTION:
#
# Can active torsional damping raise the flutter boundary
# without passive structural reinforcement?
#
#
# ACTIVE CONCEPT:
#
# Distributed torsional actuator:
#
#       35% to 55% span
#
# Collocated regional twist-rate measurement:
#
#       theta_dot_sensor
#
# Feedback law:
#
#       T_command = -C_theta * theta_dot_sensor
#
#
# This first active model is IDEAL:
#
#       no actuator lag yet
#       no time delay yet
#       no saturation yet
#       no sensor noise yet
#
# Those will be added after establishing active authority.
#
# No new DLM calculations are required.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import contextlib


# ------------------------------------------------------------
# 1. Freeze passive reference concept for later comparison
# ------------------------------------------------------------

PASSIVE_REFERENCE_ETA_START = 0.35
PASSIVE_REFERENCE_ETA_END = 0.55

PASSIVE_REFERENCE_GJ_GAIN = 0.20
PASSIVE_REFERENCE_MASS_FRACTION = 0.02


passive_reference_design = (
    build_passive_modal_matrices_v2(
        eta_start=
            PASSIVE_REFERENCE_ETA_START,

        eta_end=
            PASSIVE_REFERENCE_ETA_END,

        bending_stiffness_gain=
            0.0,

        torsional_stiffness_gain=
            PASSIVE_REFERENCE_GJ_GAIN,

        added_mass_fraction=
            PASSIVE_REFERENCE_MASS_FRACTION,
    )
)


with contextlib.redirect_stdout(
    io.StringIO()
):

    passive_reference_flutter = (
        solve_passive_flutter_root(
            M_struct=
                passive_reference_design["M"],

            C_struct=
                passive_reference_design["C"],

            K_struct=
                passive_reference_design["K"],

            lambda_guess=
                lambda_flutter_baseline,
        )
    )


passive_reference_improvement = (
    100.0
    * (
        passive_reference_flutter["U"]
        - U_flutter_baseline
    )
    / U_flutter_baseline
)


print(
    "REFERENCE PASSIVE CONCEPT"
)

print(
    f"Region              = "
    f"{100*PASSIVE_REFERENCE_ETA_START:.0f}% "
    f"to "
    f"{100*PASSIVE_REFERENCE_ETA_END:.0f}% span"
)

print(
    f"Local GJ proxy gain = "
    f"{100*PASSIVE_REFERENCE_GJ_GAIN:.1f}%"
)

print(
    f"Added mass          = "
    f"{100*PASSIVE_REFERENCE_MASS_FRACTION:.1f}% "
    "of panel mass"
)

print(
    f"Flutter improvement = "
    f"{passive_reference_improvement:+.3f}%"
)


# ------------------------------------------------------------
# 2. Active actuator region
#
# Start with the same dynamically sensitive region identified
# by the passive study.
#
# Later we can optimize actuator location separately.
# ------------------------------------------------------------

ACTIVE_ETA_START = 0.35
ACTIVE_ETA_END = 0.55


active_window = (
    reinforcement_window(
        ACTIVE_ETA_START,
        ACTIVE_ETA_END,
        transition=0.02,
    )
)


active_window_integral = (
    np.trapezoid(
        active_window,
        y_passive,
    )
)


if active_window_integral <= 0.0:

    raise RuntimeError(
        "Invalid active actuator region."
    )


# ------------------------------------------------------------
# 3. Normalize actuator torque distribution
#
# torque_shape has units:
#
#       1/m
#
# so that:
#
#       integral torque_shape dy = 1
#
# A commanded total torque T then corresponds to:
#
#       distributed torque per span
#
#       m_t(y) = T * torque_shape(y)
# ------------------------------------------------------------

torque_shape = (
    active_window
    / active_window_integral
)


torque_shape_check = (
    np.trapezoid(
        torque_shape,
        y_passive,
    )
)


assert np.isclose(
    torque_shape_check,
    1.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 4. Generalized actuator vector
#
# Virtual work:
#
#       delta W
#
#       = integral m_t(y) delta theta(y) dy
#
#       = T * B^T delta q
#
#
# Therefore:
#
#       B_i = integral torque_shape * Phi_twist_i dy
#
#
# B has units approximately 1/m because q has units of length.
# ------------------------------------------------------------

B_active = np.zeros(
    4
)


for mode_id in range(4):

    B_active[
        mode_id
    ] = (
        np.trapezoid(
            torque_shape
            * Phi_twist_smooth[
                :,
                mode_id
            ],

            y_passive,
        )
    )


# ------------------------------------------------------------
# 5. Collocated twist sensor
#
# Regional measured twist:
#
#       theta_sensor = B^T q
#
# Therefore the same vector is used for sensing.
#
# This gives ideal collocated damping:
#
#       Delta C = C_theta * B B^T
#
# which is positive semidefinite for:
#
#       C_theta >= 0
# ------------------------------------------------------------

S_active = (
    B_active.copy()
)


print(
    "\nACTIVE ACTUATOR / SENSOR MODEL"
)

print(
    f"Actuator region = "
    f"{100*ACTIVE_ETA_START:.0f}% "
    f"to "
    f"{100*ACTIVE_ETA_END:.0f}% span"
)


active_vector_table = pd.DataFrame({
    "mode_id":
        [1, 2, 3, 4],

    "actuator_coefficient_1_per_m":
        B_active,

    "sensor_coefficient_1_per_m":
        S_active,
})


display(
    active_vector_table
)


# ------------------------------------------------------------
# 6. Define a physically scaled controller-gain reference
#
# Choose C_theta_ref such that its contribution to the
# Mode-1 modal damping diagonal approximately equals the
# baseline structural Mode-1 damping:
#
#       C_theta_ref * B1^2 = C11_baseline
#
#
# Units:
#
#       N m s / rad
#
# ------------------------------------------------------------

B1_squared = (
    B_active[0]**2
)


if B1_squared < 1e-14:

    raise RuntimeError(
        "Mode-1 actuator authority is too small."
    )


C_THETA_REFERENCE = (
    C_modal_SI[
        0,
        0
    ]
    / B1_squared
)


print(
    f"\nReference active damping gain = "
    f"{C_THETA_REFERENCE:.6f} N m s/rad"
)


# ------------------------------------------------------------
# 7. Gain sweep
#
# Multipliers relative to the physically scaled reference.
# ------------------------------------------------------------

active_gain_multipliers = np.array([
    0.00,
    0.25,
    0.50,
    1.00,
    1.50,
    2.00,
    3.00,
    4.00,
])


active_rows = []


# ------------------------------------------------------------
# 8. Evaluate active-only configurations
#
# Structure remains BASELINE:
#
#       M = baseline
#       K = baseline
#
# Only damping changes.
# ------------------------------------------------------------

for gain_multiplier in active_gain_multipliers:


    C_theta = (
        gain_multiplier
        * C_THETA_REFERENCE
    )


    # --------------------------------------------------------
    # Active damping matrix
    # --------------------------------------------------------

    delta_C_active = (
        C_theta
        * np.outer(
            B_active,
            S_active,
        )
    )


    # Symmetry check for collocated configuration
    assert np.allclose(
        delta_C_active,
        delta_C_active.T,
        atol=1e-12,
    )


    active_damping_eigs = (
        np.linalg.eigvalsh(
            delta_C_active
        )
    )


    if (
        np.min(
            active_damping_eigs
        )
        < -1e-10
    ):

        raise RuntimeError(
            "Collocated active damping matrix "
            "is not positive semidefinite."
        )


    C_active_case = (
        C_modal_SI
        + delta_C_active
    )


    # --------------------------------------------------------
    # Flutter calculation
    # --------------------------------------------------------

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        active_flutter = (
            solve_passive_flutter_root(
                M_struct=
                    M_modal_SI,

                C_struct=
                    C_active_case,

                K_struct=
                    K_modal_SI,

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    active_improvement = (
        100.0
        * (
            active_flutter["U"]
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    # --------------------------------------------------------
    # Effective added Mode-1 damping
    #
    # Useful for understanding controller strength.
    # --------------------------------------------------------

    delta_C11 = (
        delta_C_active[
            0,
            0
        ]
    )


    C11_ratio = (
        delta_C11
        / C_modal_SI[
            0,
            0
        ]
    )


    active_rows.append({

        "gain_multiplier":
            gain_multiplier,

        "C_theta_Nms_per_rad":
            C_theta,

        "delta_C11_Ns_per_m":
            delta_C11,

        "delta_C11_vs_baseline":
            C11_ratio,

        "flutter_velocity_m_s":
            active_flutter["U"],

        "flutter_frequency_Hz":
            active_flutter[
                "frequency_Hz"
            ],

        "flutter_improvement_pct":
            active_improvement,

        "sigma_final":
            active_flutter["sigma"],

        "MAC":
            active_flutter["MAC"],
    })


# ------------------------------------------------------------
# 9. Results dataframe
# ------------------------------------------------------------

active_gain_sweep = (
    pd.DataFrame(
        active_rows
    )
)


print(
    "\nACTIVE TORSIONAL DAMPING SWEEP"
)


display(
    active_gain_sweep
)


# ------------------------------------------------------------
# 10. Find gain required to match passive reference
#
# We do not extrapolate outside calculated results.
# ------------------------------------------------------------

active_sorted = (
    active_gain_sweep
    .sort_values(
        "flutter_improvement_pct"
    )
)


active_improvements = (
    active_sorted[
        "flutter_improvement_pct"
    ]
    .to_numpy()
)


active_gains = (
    active_sorted[
        "C_theta_Nms_per_rad"
    ]
    .to_numpy()
)


if (
    passive_reference_improvement
    >= np.min(
        active_improvements
    )
    and
    passive_reference_improvement
    <= np.max(
        active_improvements
    )
):

    C_theta_match_passive = (
        np.interp(
            passive_reference_improvement,
            active_improvements,
            active_gains,
        )
    )

else:

    C_theta_match_passive = (
        np.nan
    )


print(
    "\nPASSIVE vs ACTIVE REQUIREMENT"
)


print(
    f"Reference passive improvement = "
    f"{passive_reference_improvement:+.3f}%"
)


if np.isfinite(
    C_theta_match_passive
):

    print(
        "Estimated ideal active damping gain required "
        "to match passive improvement = "
        f"{C_theta_match_passive:.6f} N m s/rad"
    )

else:

    print(
        "The active sweep did not yet span the "
        "reference passive improvement."
    )


# ------------------------------------------------------------
# 11. Plot active authority
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    active_gain_sweep[
        "C_theta_Nms_per_rad"
    ],

    active_gain_sweep[
        "flutter_improvement_pct"
    ],

    "o-",
)


plt.axhline(
    passive_reference_improvement,

    linestyle="--",

    linewidth=1.0,

    label="Reference passive concept",
)


plt.xlabel(
    "Ideal torsional damping gain "
    r"$C_\theta$ (N m s/rad)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Active torsional damping authority"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 12. Plot flutter velocity
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    active_gain_sweep[
        "gain_multiplier"
    ],

    active_gain_sweep[
        "flutter_velocity_m_s"
    ],

    "o-",

    label="Active-only",
)


plt.axhline(
    U_flutter_baseline,

    linestyle="--",

    linewidth=1.0,

    label="Baseline",
)


plt.axhline(
    passive_reference_flutter["U"],

    linestyle=":",

    linewidth=1.0,

    label="Reference passive",
)


plt.xlabel(
    "Active damping gain / reference gain"
)


plt.ylabel(
    "Predicted flutter velocity (m/s)"
)


plt.title(
    "Active flutter suppression — ideal actuator"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


print(
    "\nCell 52 ideal active flutter-suppression baseline completed."
)

In [ ]:
# ============================================================
# CELL 52A — ACTIVE DAMPING SIGN / PHYSICS AUDIT
# ============================================================
#
# Question:
#
# Is Cell 52 destabilizing flutter because:
#
#   A) the control/damping sign is coded incorrectly?
#
# or
#
#   B) positive structural damping is genuinely shifting the
#      coupled aeroelastic system unfavorably?
#
#
# We verify this using DRY structural poles first.
#
# No DLM calculations are required.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Dry structural state-space poles
# ------------------------------------------------------------

def dry_structural_poles(
    M,
    C,
    K,
):

    n = M.shape[0]

    M_inv = np.linalg.inv(
        M
    )

    Z = np.zeros(
        (n, n)
    )

    I = np.eye(
        n
    )

    A = np.block([
        [
            Z,
            I,
        ],
        [
            -M_inv @ K,
            -M_inv @ C,
        ],
    ])

    poles, eigvectors = np.linalg.eig(
        A
    )

    return poles, eigvectors


# ------------------------------------------------------------
# 2. Gains to audit
# ------------------------------------------------------------

audit_gain_multipliers = np.array([
    0.00,
    0.25,
    0.50,
    1.00,
    2.00,
    4.00,
])


audit_rows = []


# ------------------------------------------------------------
# 3. Structural damping audit
# ------------------------------------------------------------

for multiplier in audit_gain_multipliers:

    C_theta = (
        multiplier
        * C_THETA_REFERENCE
    )


    delta_C = (
        C_theta
        * np.outer(
            B_active,
            B_active,
        )
    )


    C_total = (
        C_modal_SI
        + delta_C
    )


    # --------------------------------------------------------
    # Positive-semidefinite check
    # --------------------------------------------------------

    delta_C_eigs = (
        np.linalg.eigvalsh(
            delta_C
        )
    )


    # --------------------------------------------------------
    # Dry poles
    # --------------------------------------------------------

    poles, eigvectors = (
        dry_structural_poles(
            M_modal_SI,
            C_total,
            K_modal_SI,
        )
    )


    positive_frequency_indices = np.where(
        np.imag(
            poles
        )
        > 0.0
    )[0]


    positive_poles = (
        poles[
            positive_frequency_indices
        ]
    )


    # Sort by frequency
    order = np.argsort(
        np.imag(
            positive_poles
        )
    )


    positive_poles = (
        positive_poles[
            order
        ]
    )


    # --------------------------------------------------------
    # Store each dry mode
    # --------------------------------------------------------

    for mode_id, pole in enumerate(
        positive_poles,
        start=1,
    ):

        sigma = float(
            np.real(
                pole
            )
        )

        omega = float(
            np.imag(
                pole
            )
        )

        frequency = (
            omega
            / (
                2.0
                * np.pi
            )
        )

        damping_ratio = (
            -sigma
            / abs(
                pole
            )
        )


        audit_rows.append({

            "gain_multiplier":
                multiplier,

            "C_theta_Nms_per_rad":
                C_theta,

            "mode_id":
                mode_id,

            "sigma_dry_1_per_s":
                sigma,

            "frequency_dry_Hz":
                frequency,

            "damping_ratio_dry":
                damping_ratio,

            "delta_C_min_eigenvalue":
                np.min(
                    delta_C_eigs
                ),

            "delta_C_max_eigenvalue":
                np.max(
                    delta_C_eigs
                ),
        })


# ------------------------------------------------------------
# 4. Assemble results
# ------------------------------------------------------------

active_dry_audit = pd.DataFrame(
    audit_rows
)


print(
    "ACTIVE CONTROL DRY-POLE AUDIT"
)


display(
    active_dry_audit
)


# ------------------------------------------------------------
# 5. Compact Mode-1 table
# ------------------------------------------------------------

mode1_dry_audit = (
    active_dry_audit[
        active_dry_audit[
            "mode_id"
        ]
        == 1
    ]
    .copy()
)


print(
    "\nMode-1 dry structural response:"
)


display(
    mode1_dry_audit[
        [
            "gain_multiplier",
            "C_theta_Nms_per_rad",
            "sigma_dry_1_per_s",
            "frequency_dry_Hz",
            "damping_ratio_dry",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Compare dry damping with flutter result from Cell 52
# ------------------------------------------------------------

active_damping_diagnosis = (
    active_gain_sweep[
        [
            "gain_multiplier",
            "C_theta_Nms_per_rad",
            "flutter_velocity_m_s",
            "flutter_improvement_pct",
        ]
    ]
    .merge(
        mode1_dry_audit[
            [
                "gain_multiplier",
                "sigma_dry_1_per_s",
                "damping_ratio_dry",
            ]
        ],
        on="gain_multiplier",
        how="left",
    )
)


print(
    "\nDry structural damping versus aeroelastic flutter:"
)


display(
    active_damping_diagnosis
)


# ------------------------------------------------------------
# 7. Check that positive feedback gain really adds damping
# ------------------------------------------------------------

baseline_mode1_sigma = float(
    mode1_dry_audit[
        np.isclose(
            mode1_dry_audit[
                "gain_multiplier"
            ],
            0.0,
        )
    ][
        "sigma_dry_1_per_s"
    ]
    .iloc[0]
)


max_gain_mode1_sigma = float(
    mode1_dry_audit[
        np.isclose(
            mode1_dry_audit[
                "gain_multiplier"
            ],
            4.0,
        )
    ][
        "sigma_dry_1_per_s"
    ]
    .iloc[0]
)


print(
    "\nSIGN CHECK"
)


print(
    f"Baseline Mode-1 dry sigma = "
    f"{baseline_mode1_sigma:.6f} 1/s"
)


print(
    f"4x-gain Mode-1 dry sigma = "
    f"{max_gain_mode1_sigma:.6f} 1/s"
)


if (
    max_gain_mode1_sigma
    < baseline_mode1_sigma
):

    print(
        "\nPASS: Positive controller gain moves the "
        "dry structural pole farther into the stable half-plane."
    )

    print(
        "Therefore the Cell-52 loss of flutter margin is NOT "
        "a simple controller-sign error."
    )

else:

    print(
        "\nWARNING: Positive controller gain does not increase "
        "dry structural stability."
    )

    print(
        "The feedback sign/implementation must be corrected "
        "before continuing."
    )


# ------------------------------------------------------------
# 8. Plot dry damping
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.plot(
    mode1_dry_audit[
        "gain_multiplier"
    ],

    mode1_dry_audit[
        "damping_ratio_dry"
    ],

    "o-",
)


plt.xlabel(
    "Active gain / reference gain"
)


plt.ylabel(
    "Dry Mode-1 damping ratio"
)


plt.title(
    "Does the active controller add structural damping?"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 9. Contrast with aeroelastic flutter trend
# ------------------------------------------------------------

fig, ax1 = plt.subplots(
    figsize=(9, 5)
)


ax1.plot(
    active_damping_diagnosis[
        "gain_multiplier"
    ],

    active_damping_diagnosis[
        "damping_ratio_dry"
    ],

    "o-",

    label="Dry Mode-1 damping ratio",
)


ax1.set_xlabel(
    "Active gain / reference gain"
)


ax1.set_ylabel(
    "Dry Mode-1 damping ratio"
)


ax2 = ax1.twinx()


ax2.plot(
    active_damping_diagnosis[
        "gain_multiplier"
    ],

    active_damping_diagnosis[
        "flutter_velocity_m_s"
    ],

    "s--",

    label="Flutter velocity",
)


ax2.set_ylabel(
    "Flutter velocity (m/s)"
)


ax1.grid(
    alpha=0.25
)


plt.title(
    "Structural damping versus aeroelastic stability"
)


fig.tight_layout()

plt.show()


print(
    "\nCell 52A active damping audit completed."
)

In [ ]:
# ============================================================
# CELL 53 — IDEAL ACTIVE PD TORSIONAL CONTROLLER MAP
# ============================================================
#
# PURPOSE:
#
# Cell 52 showed:
#
#     pure twist-rate feedback (active damping)
#     REDUCES flutter margin.
#
# Now investigate:
#
#     T_act = -K_theta * theta
#             -C_theta * theta_dot
#
# using the same collocated actuator/sensor vector.
#
#
# This is an IDEAL controller:
#
#     no actuator lag
#     no delay
#     no saturation
#
# Those effects will be introduced AFTER identifying a
# stabilizing controller region.
#
#
# IMPORTANT:
#
# This remains a Mode-1 flutter-branch SCREENING study.
# Final selected controllers will later receive an
# all-branch verification.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import contextlib


# ------------------------------------------------------------
# 1. Controller structural influence matrix
# ------------------------------------------------------------

BBT_active = (
    np.outer(
        B_active,
        B_active,
    )
)


# ------------------------------------------------------------
# 2. Active-stiffness reference gain
#
# Define K_theta_ref so that:
#
#       Delta K_11 = 10% of baseline K_11
#
# Therefore:
#
#       Ktheta_ref * B1^2
#       -----------------
#            K11
#
#           = 0.10
#
# Units:
#
#       N m / rad
#
# This is only a convenient controller scaling.
# It is NOT equivalent to claiming +10% physical GJ.
# ------------------------------------------------------------

ACTIVE_K11_REFERENCE_FRACTION = (
    0.10
)


K_THETA_REFERENCE = (
    ACTIVE_K11_REFERENCE_FRACTION
    * K_modal_SI[
        0,
        0
    ]
    / (
        B_active[0]**2
    )
)


print(
    "IDEAL ACTIVE PD CONTROLLER MAP"
)


print(
    f"\nK_theta reference = "
    f"{K_THETA_REFERENCE:.6f} N m/rad"
)


print(
    "Reference definition:"
)


print(
    f"At K multiplier = 1.0, "
    f"Delta K11 = "
    f"{100*ACTIVE_K11_REFERENCE_FRACTION:.1f}% "
    "of baseline K11."
)


print(
    f"\nC_theta reference = "
    f"{C_THETA_REFERENCE:.6f} N m s/rad"
)


print(
    "At C multiplier = 1.0, "
    "Delta C11 approximately equals baseline C11."
)


# ------------------------------------------------------------
# 3. Controller gain grid
#
# Because pure damping was harmful, use finer resolution near
# small C gains.
# ------------------------------------------------------------

K_gain_multipliers = np.array([
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
    1.50,
    2.00,
])


C_gain_multipliers = np.array([
    0.00,
    0.10,
    0.25,
    0.50,
    1.00,
])


# ------------------------------------------------------------
# 4. Storage
# ------------------------------------------------------------

pd_rows = []


# ------------------------------------------------------------
# 5. Sweep active stiffness + damping
# ------------------------------------------------------------

for K_mult in K_gain_multipliers:

    for C_mult in C_gain_multipliers:


        K_theta = (
            K_mult
            * K_THETA_REFERENCE
        )


        C_theta = (
            C_mult
            * C_THETA_REFERENCE
        )


        # ----------------------------------------------------
        # Controller-induced structural matrices
        # ----------------------------------------------------

        delta_K_control = (
            K_theta
            * BBT_active
        )


        delta_C_control = (
            C_theta
            * BBT_active
        )


        K_controlled = (
            K_modal_SI
            + delta_K_control
        )


        C_controlled = (
            C_modal_SI
            + delta_C_control
        )


        M_controlled = (
            M_modal_SI.copy()
        )


        # ----------------------------------------------------
        # Dry closed-loop structural check
        # ----------------------------------------------------

        dry_poles, _ = (
            dry_structural_poles(
                M_controlled,
                C_controlled,
                K_controlled,
            )
        )


        positive_dry = (
            dry_poles[
                np.imag(
                    dry_poles
                )
                > 0.0
            ]
        )


        dry_damping_ratios = (
            -np.real(
                positive_dry
            )
            /
            np.abs(
                positive_dry
            )
        )


        min_dry_damping = float(
            np.min(
                dry_damping_ratios
            )
        )


        dry_stable = bool(
            np.all(
                np.real(
                    dry_poles
                )
                < 0.0
            )
        )


        # ----------------------------------------------------
        # Aeroelastic flutter calculation
        # ----------------------------------------------------

        status = (
            "OK"
        )


        try:

            with contextlib.redirect_stdout(
                io.StringIO()
            ):

                flutter = (
                    solve_passive_flutter_root(
                        M_struct=
                            M_controlled,

                        C_struct=
                            C_controlled,

                        K_struct=
                            K_controlled,

                        lambda_guess=
                            lambda_flutter_baseline,
                    )
                )


            U_flutter_case = float(
                flutter[
                    "U"
                ]
            )


            f_flutter_case = float(
                flutter[
                    "frequency_Hz"
                ]
            )


            sigma_final = float(
                flutter[
                    "sigma"
                ]
            )


            MAC_final = float(
                flutter[
                    "MAC"
                ]
            )


            improvement = (
                100.0
                * (
                    U_flutter_case
                    - U_flutter_baseline
                )
                / U_flutter_baseline
            )


        except ValueError:

            # Usually means controller moved k outside the
            # currently available GAF database.

            status = (
                "GAF_RANGE"
            )


            U_flutter_case = np.nan
            f_flutter_case = np.nan
            sigma_final = np.nan
            MAC_final = np.nan
            improvement = np.nan


        except RuntimeError:

            status = (
                "NO_ROOT"
            )


            U_flutter_case = np.nan
            f_flutter_case = np.nan
            sigma_final = np.nan
            MAC_final = np.nan
            improvement = np.nan


        # ----------------------------------------------------
        # Generalized Mode-1 equivalent changes
        # ----------------------------------------------------

        delta_K11_fraction = (
            delta_K_control[
                0,
                0
            ]
            /
            K_modal_SI[
                0,
                0
            ]
        )


        delta_C11_fraction = (
            delta_C_control[
                0,
                0
            ]
            /
            C_modal_SI[
                0,
                0
            ]
        )


        pd_rows.append({

            "K_multiplier":
                K_mult,

            "C_multiplier":
                C_mult,

            "K_theta_Nm_per_rad":
                K_theta,

            "C_theta_Nms_per_rad":
                C_theta,

            "delta_K11_pct":
                100.0
                * delta_K11_fraction,

            "delta_C11_pct":
                100.0
                * delta_C11_fraction,

            "dry_stable":
                dry_stable,

            "min_dry_damping_ratio":
                min_dry_damping,

            "flutter_velocity_m_s":
                U_flutter_case,

            "flutter_frequency_Hz":
                f_flutter_case,

            "flutter_improvement_pct":
                improvement,

            "sigma_final":
                sigma_final,

            "MAC":
                MAC_final,

            "status":
                status,
        })


# ------------------------------------------------------------
# 6. Results dataframe
# ------------------------------------------------------------

active_PD_map = (
    pd.DataFrame(
        pd_rows
    )
)


print(
    "\nACTIVE PD CONTROLLER RESULTS"
)


display(
    active_PD_map
)


# ------------------------------------------------------------
# 7. Valid controller cases
# ------------------------------------------------------------

valid_PD = (
    active_PD_map[
        (
            active_PD_map[
                "status"
            ]
            == "OK"
        )
        &
        (
            active_PD_map[
                "dry_stable"
            ]
            == True
        )
    ]
    .copy()
)


if len(
    valid_PD
) == 0:

    raise RuntimeError(
        "No valid active PD controller cases were obtained."
    )


# ------------------------------------------------------------
# 8. Best ideal active controller in investigated grid
# ------------------------------------------------------------

best_PD_index = (
    valid_PD[
        "flutter_improvement_pct"
    ]
    .idxmax()
)


best_active_PD = (
    valid_PD
    .loc[
        best_PD_index
    ]
)


print(
    "\nBEST IDEAL ACTIVE PD CASE"
)


print(
    f"K multiplier = "
    f"{best_active_PD['K_multiplier']:.2f}"
)


print(
    f"C multiplier = "
    f"{best_active_PD['C_multiplier']:.2f}"
)


print(
    f"K_theta = "
    f"{best_active_PD['K_theta_Nm_per_rad']:.3f} N m/rad"
)


print(
    f"C_theta = "
    f"{best_active_PD['C_theta_Nms_per_rad']:.3f} "
    "N m s/rad"
)


print(
    f"Equivalent Delta K11 = "
    f"{best_active_PD['delta_K11_pct']:.2f}%"
)


print(
    f"Equivalent Delta C11 = "
    f"{best_active_PD['delta_C11_pct']:.2f}%"
)


print(
    f"Flutter velocity = "
    f"{best_active_PD['flutter_velocity_m_s']:.3f} m/s"
)


print(
    f"Flutter improvement = "
    f"{best_active_PD['flutter_improvement_pct']:+.3f}%"
)


print(
    f"Minimum dry damping ratio = "
    f"{best_active_PD['min_dry_damping_ratio']:.5f}"
)


# ------------------------------------------------------------
# 9. Does ideal active control match passive reference?
# ------------------------------------------------------------

print(
    "\nPASSIVE / ACTIVE COMPARISON"
)


print(
    f"Passive reference improvement = "
    f"{passive_reference_improvement:+.3f}%"
)


print(
    f"Best ideal active improvement = "
    f"{best_active_PD['flutter_improvement_pct']:+.3f}%"
)


if (
    best_active_PD[
        "flutter_improvement_pct"
    ]
    >= passive_reference_improvement
):

    print(
        "The investigated ideal active controller CAN "
        "match or exceed the passive reference."
    )

else:

    print(
        "The investigated ideal active controller does NOT "
        "yet match the passive reference."
    )


# ------------------------------------------------------------
# 10. Heatmap table
# ------------------------------------------------------------

PD_pivot = (
    active_PD_map
    .pivot(
        index=
            "C_multiplier",

        columns=
            "K_multiplier",

        values=
            "flutter_improvement_pct",
    )
)


print(
    "\nFlutter improvement (%)"
)

print(
    "Rows = active damping multiplier"
)

print(
    "Columns = active stiffness multiplier"
)


display(
    PD_pivot
)


# ------------------------------------------------------------
# 11. Heatmap
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.imshow(
    PD_pivot.to_numpy(),
    aspect="auto",
    origin="lower",
)


plt.colorbar(
    label="Flutter-velocity improvement (%)"
)


plt.xticks(
    np.arange(
        len(
            PD_pivot.columns
        )
    ),
    [
        f"{x:.2f}"
        for x in PD_pivot.columns
    ],
)


plt.yticks(
    np.arange(
        len(
            PD_pivot.index
        )
    ),
    [
        f"{x:.2f}"
        for x in PD_pivot.index
    ],
)


plt.xlabel(
    "Active stiffness multiplier"
)


plt.ylabel(
    "Active damping multiplier"
)


plt.title(
    "Ideal active PD controller stability map"
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 12. Controller curves at each damping level
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for C_mult in C_gain_multipliers:


    subset = (
        valid_PD[
            np.isclose(
                valid_PD[
                    "C_multiplier"
                ],
                C_mult,
            )
        ]
        .sort_values(
            "K_multiplier"
        )
    )


    if len(
        subset
    ) == 0:

        continue


    plt.plot(
        subset[
            "K_multiplier"
        ],

        subset[
            "flutter_improvement_pct"
        ],

        "o-",

        label=
            f"C multiplier = {C_mult:.2f}",
    )


plt.axhline(
    passive_reference_improvement,
    linestyle="--",
    linewidth=1.0,
    label="Passive reference",
)


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.xlabel(
    "Active stiffness multiplier"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Stiffness feedback versus rate feedback"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 13. Store selected ideal controller
# ------------------------------------------------------------

IDEAL_ACTIVE_K_THETA = float(
    best_active_PD[
        "K_theta_Nm_per_rad"
    ]
)


IDEAL_ACTIVE_C_THETA = float(
    best_active_PD[
        "C_theta_Nms_per_rad"
    ]
)


print(
    "\nSelected ideal-control candidate stored as:"
)


print(
    f"IDEAL_ACTIVE_K_THETA = "
    f"{IDEAL_ACTIVE_K_THETA:.6f}"
)


print(
    f"IDEAL_ACTIVE_C_THETA = "
    f"{IDEAL_ACTIVE_C_THETA:.6f}"
)


print(
    "\nCell 53 ideal active PD map completed."
)

In [ ]:
# ============================================================
# CELL 54 — MINIMUM-AUTHORITY ACTIVE CONTROLLER
#           + ALL-BRANCH AEROELASTIC VERIFICATION
# ============================================================
#
# ENGINEERING OBJECTIVE:
#
# Find the MINIMUM ideal active torsional stiffness feedback
# required to match the reference passive flutter benefit.
#
# Then verify ALL FOUR aeroelastic branches at that condition.
#
#
# Controller:
#
#       T_act = -K_theta * theta_sensor
#
#       C_theta = 0
#
#
# Why C_theta = 0?
#
# Cell 53 showed that twist-rate feedback reduced the flutter
# boundary throughout the investigated controller map.
#
#
# IMPORTANT:
#
# This remains an ideal zero-lag controller.
#
# After this verification:
#
#       actuator dynamics
#       phase lag
#       time delay
#       torque authority
#
# will be introduced.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import contextlib


# ------------------------------------------------------------
# 1. Target active-control performance
# ------------------------------------------------------------

ACTIVE_TARGET_IMPROVEMENT = float(
    passive_reference_improvement
)


print(
    "MINIMUM-AUTHORITY ACTIVE CONTROLLER DESIGN"
)


print(
    f"\nTarget flutter improvement = "
    f"{ACTIVE_TARGET_IMPROVEMENT:.4f}%"
)


print(
    "Target is the reference passive concept."
)


# ------------------------------------------------------------
# 2. Pure-stiffness controller flutter solver
# ------------------------------------------------------------

def solve_active_stiffness_case(
    K_multiplier,
):
    """
    Solve the Mode-1 flutter boundary for a pure active
    torsional-stiffness controller.
    """

    K_theta = (
        float(K_multiplier)
        * K_THETA_REFERENCE
    )


    delta_K_control = (
        K_theta
        * BBT_active
    )


    K_controlled = (
        K_modal_SI
        + delta_K_control
    )


    C_controlled = (
        C_modal_SI.copy()
    )


    M_controlled = (
        M_modal_SI.copy()
    )


    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        flutter = (
            solve_passive_flutter_root(
                M_struct=
                    M_controlled,

                C_struct=
                    C_controlled,

                K_struct=
                    K_controlled,

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    improvement = (
        100.0
        * (
            flutter[
                "U"
            ]
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    return {
        "K_multiplier":
            float(
                K_multiplier
            ),

        "K_theta":
            K_theta,

        "M":
            M_controlled,

        "C":
            C_controlled,

        "K":
            K_controlled,

        "flutter":
            flutter,

        "improvement_pct":
            improvement,
    }


# ------------------------------------------------------------
# 3. Verify target bracket
#
# Cell 53 already showed:
#
# K_mult = 0.00 -> ~0%
# K_mult = 0.25 -> ~4.52%
#
# The passive target (~3.47%) therefore lies inside.
# ------------------------------------------------------------

active_low = (
    solve_active_stiffness_case(
        0.0
    )
)


active_high = (
    solve_active_stiffness_case(
        0.25
    )
)


assert (
    active_low[
        "improvement_pct"
    ]
    <= ACTIVE_TARGET_IMPROVEMENT
)


assert (
    active_high[
        "improvement_pct"
    ]
    >= ACTIVE_TARGET_IMPROVEMENT
)


print(
    "\nInitial controller bracket:"
)


print(
    f"K multiplier = 0.0000 -> "
    f"{active_low['improvement_pct']:+.4f}%"
)


print(
    f"K multiplier = 0.2500 -> "
    f"{active_high['improvement_pct']:+.4f}%"
)


# ------------------------------------------------------------
# 4. Bisection search for minimum gain matching passive
# ------------------------------------------------------------

K_low = 0.0
K_high = 0.25

target_tolerance = (
    0.01
)  # percentage-point flutter improvement


active_gain_search_rows = []


for iteration in range(
    1,
    21,
):


    K_trial = (
        0.5
        * (
            K_low
            + K_high
        )
    )


    trial = (
        solve_active_stiffness_case(
            K_trial
        )
    )


    active_gain_search_rows.append({

        "iteration":
            iteration,

        "K_multiplier":
            K_trial,

        "K_theta_Nm_per_rad":
            trial[
                "K_theta"
            ],

        "flutter_velocity_m_s":
            trial[
                "flutter"
            ][
                "U"
            ],

        "flutter_improvement_pct":
            trial[
                "improvement_pct"
            ],

        "target_error_pct_point":
            trial[
                "improvement_pct"
            ]
            - ACTIVE_TARGET_IMPROVEMENT,
    })


    error = (
        trial[
            "improvement_pct"
        ]
        - ACTIVE_TARGET_IMPROVEMENT
    )


    if (
        abs(
            error
        )
        <= target_tolerance
    ):

        active_reference_case = (
            trial
        )

        break


    if error < 0.0:

        K_low = (
            K_trial
        )

    else:

        K_high = (
            K_trial
        )


else:

    active_reference_case = (
        trial
    )


active_gain_search = (
    pd.DataFrame(
        active_gain_search_rows
    )
)


# ------------------------------------------------------------
# 5. Freeze engineering reference controller
# ------------------------------------------------------------

ACTIVE_REFERENCE_K_MULTIPLIER = float(
    active_reference_case[
        "K_multiplier"
    ]
)


ACTIVE_REFERENCE_K_THETA = float(
    active_reference_case[
        "K_theta"
    ]
)


ACTIVE_REFERENCE_C_THETA = (
    0.0
)


ACTIVE_REFERENCE_M = (
    active_reference_case[
        "M"
    ]
)


ACTIVE_REFERENCE_C = (
    active_reference_case[
        "C"
    ]
)


ACTIVE_REFERENCE_K = (
    active_reference_case[
        "K"
    ]
)


ACTIVE_REFERENCE_FLUTTER = (
    active_reference_case[
        "flutter"
    ]
)


ACTIVE_REFERENCE_IMPROVEMENT = float(
    active_reference_case[
        "improvement_pct"
    ]
)


print(
    "\nController-gain search:"
)

display(
    active_gain_search
)


print(
    "\nMINIMUM-AUTHORITY IDEAL ACTIVE REFERENCE"
)


print(
    f"K multiplier = "
    f"{ACTIVE_REFERENCE_K_MULTIPLIER:.6f}"
)


print(
    f"K_theta = "
    f"{ACTIVE_REFERENCE_K_THETA:.6f} N m/rad"
)


print(
    f"C_theta = "
    f"{ACTIVE_REFERENCE_C_THETA:.6f} N m s/rad"
)


print(
    f"Flutter velocity = "
    f"{ACTIVE_REFERENCE_FLUTTER['U']:.3f} m/s"
)


print(
    f"Flutter improvement = "
    f"{ACTIVE_REFERENCE_IMPROVEMENT:+.3f}%"
)


# ------------------------------------------------------------
# 6. Actuator authority per measured twist
#
# Linear eigenanalysis cannot define absolute oscillation
# amplitude, so do NOT quote a unique peak actuator torque.
#
# Instead calculate required torque per unit measured twist.
# ------------------------------------------------------------

TORQUE_PER_DEGREE = (
    ACTIVE_REFERENCE_K_THETA
    * np.deg2rad(
        1.0
    )
)


TORQUE_PER_MRAD = (
    ACTIVE_REFERENCE_K_THETA
    * 1.0e-3
)


print(
    "\nIDEAL ACTUATOR AUTHORITY"
)


print(
    f"Torque per 1 degree measured regional twist = "
    f"{TORQUE_PER_DEGREE:.4f} N m"
)


print(
    f"Torque per 1 mrad measured regional twist = "
    f"{TORQUE_PER_MRAD:.4f} N m"
)


# ============================================================
# PART B — EXTEND GAF DATABASE FOR ALL FOUR MODES
# ============================================================


# ------------------------------------------------------------
# 7. Additional Mach-0.901 frequency parameters
#
# Existing database primarily covered Mode 1.
#
# Extend to the Mode-2/3/4 frequency range.
# ------------------------------------------------------------

all_branch_k_extension = np.array([
    0.45,
    0.55,
    0.70,
    0.85,
    1.00,
    1.20,
    1.45,
    1.70,
    1.95,
    2.20,
    2.40,
])


print(
    "\nExtending Mach 0.901 GAF database "
    "for all-branch verification..."
)


existing_k_0901 = np.array(
    [
        float(
            key[1]
        )

        for key
        in _wall_qhh_cache.keys()

        if (
            abs(
                float(
                    key[0]
                )
                - PASSIVE_DESIGN_MACH
            )
            < 1e-6

            and

            int(
                key[2]
            )
            == 4
        )
    ]
)


for k_test in all_branch_k_extension:


    already_available = (
        existing_k_0901.size > 0

        and

        np.min(
            np.abs(
                existing_k_0901
                - k_test
            )
        )
        < 1e-8
    )


    if already_available:

        print(
            f"k={k_test:.2f} already available."
        )

        continue


    print(
        f"Calculating DLM GAF at "
        f"k={k_test:.2f} 1/m ..."
    )


    _ = (
        calculate_Qhh_wall_ips(
            Ma=
                PASSIVE_DESIGN_MACH,

            k_PA=
                float(
                    k_test
                ),

            n_modes=
                4,
        )
    )


# ------------------------------------------------------------
# 8. Refresh Mach-0.901 cached aerodynamic database
# ------------------------------------------------------------

k_all_0901, Q_all_0901 = (
    get_cached_Qhh_data(
        Ma_target=
            PASSIVE_DESIGN_MACH,

        n_modes=
            4,
    )
)


cached_aero_database[
    PASSIVE_DESIGN_MACH
] = {

    "k":
        k_all_0901,

    "Q":
        Q_all_0901,
}


print(
    "\nUpdated Mach-0.901 GAF coverage:"
)


print(
    f"Number of GAF matrices = "
    f"{len(k_all_0901)}"
)


print(
    f"k range = "
    f"{k_all_0901.min():.4f} "
    f"to "
    f"{k_all_0901.max():.4f} 1/m"
)


# ============================================================
# PART C — GENERIC ALL-BRANCH p-k SOLVER
# ============================================================


# ------------------------------------------------------------
# 9. Generic branch tracker
# ------------------------------------------------------------

def pk_controlled_branch(
    mode_index,
    U,
    q_dyn,
    M_struct,
    C_struct,
    K_struct,
    initial_k=None,
    tol=2e-7,
    max_iter=80,
    relaxation=0.65,
):


    if not (
        0
        <= mode_index
        < 4
    ):

        raise ValueError(
            "mode_index must be 0,1,2,3."
        )


    # --------------------------------------------------------
    # Initial reduced-frequency guess
    # --------------------------------------------------------

    if initial_k is None:

        k_current = (
            omega_analysis[
                mode_index
            ]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    # --------------------------------------------------------
    # Original structural modal-coordinate reference
    # --------------------------------------------------------

    reference_q = np.zeros(
        4,
        dtype=complex,
    )


    reference_q[
        mode_index
    ] = 1.0


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M_struct,
        )
    )


    # --------------------------------------------------------
    # p-k iteration
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iter + 1
    ):


        poles, eigvectors = (
            frozen_k_pk_passive(
                Ma=
                    PASSIVE_DESIGN_MACH,

                U=
                    U,

                q_dyn=
                    q_dyn,

                k_PA=
                    k_current,

                M_struct=
                    M_struct,

                C_struct=
                    C_struct,

                K_struct=
                    K_struct,
            )
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M_struct,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M_struct,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_idx = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_idx
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        MAC_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M_struct,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M_struct,
            )
        )


        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            f"Controlled branch "
            f"{mode_index+1} did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    damping_ratio = (
        -sigma
        / abs(
            pole
        )
    )


    return {

        "mode_id":
            mode_index + 1,

        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            damping_ratio,

        "k_PA":
            omega
            / U,

        "MAC":
            MAC_selected,

        "iterations":
            iteration,
    }


# ------------------------------------------------------------
# 10. Active reference flutter condition
# ------------------------------------------------------------

U_active_reference = float(
    ACTIVE_REFERENCE_FLUTTER[
        "U"
    ]
)


lambda_active_reference = (
    U_active_reference
    / U_exp_passive
)


q_active_reference = (
    lambda_active_reference**2
    * q_exp_passive
)


print(
    "\nALL-BRANCH ACTIVE VERIFICATION CONDITION"
)


print(
    f"Mach = "
    f"{PASSIVE_DESIGN_MACH:.3f}"
)


print(
    f"U = "
    f"{U_active_reference:.3f} m/s"
)


print(
    f"q = "
    f"{q_active_reference:.3f} Pa"
)


# ------------------------------------------------------------
# 11. Solve all four branches
# ------------------------------------------------------------

all_branch_results = []


for mode_index in range(
    4
):


    print(
        f"Solving controlled branch "
        f"{mode_index+1}..."
    )


    result = (
        pk_controlled_branch(
            mode_index=
                mode_index,

            U=
                U_active_reference,

            q_dyn=
                q_active_reference,

            M_struct=
                ACTIVE_REFERENCE_M,

            C_struct=
                ACTIVE_REFERENCE_C,

            K_struct=
                ACTIVE_REFERENCE_K,
        )
    )


    all_branch_results.append(
        result
    )


active_all_branch_table = (
    pd.DataFrame(
        all_branch_results
    )
)


# ------------------------------------------------------------
# 12. Identify least-stable branch
# ------------------------------------------------------------

least_stable_index = (
    active_all_branch_table[
        "sigma"
    ]
    .idxmax()
)


least_stable_branch = (
    active_all_branch_table
    .loc[
        least_stable_index
    ]
)


print(
    "\nACTIVE ALL-BRANCH VERIFICATION"
)


display(
    active_all_branch_table
)


print(
    "\nLeast-stable branch:"
)


print(
    f"Mode = "
    f"{int(least_stable_branch['mode_id'])}"
)


print(
    f"sigma = "
    f"{least_stable_branch['sigma']:+.6e} 1/s"
)


print(
    f"frequency = "
    f"{least_stable_branch['frequency_Hz']:.4f} Hz"
)


# ------------------------------------------------------------
# 13. Verification decision
# ------------------------------------------------------------

non_mode1 = (
    active_all_branch_table[
        active_all_branch_table[
            "mode_id"
        ]
        != 1
    ]
)


other_modes_stable = bool(
    np.all(
        non_mode1[
            "sigma"
        ]
        < 0.0
    )
)


mode1_neutral = bool(
    abs(
        active_all_branch_table
        .loc[
            active_all_branch_table[
                "mode_id"
            ]
            == 1,
            "sigma",
        ]
        .iloc[0]
    )
    < 1e-3
)


print(
    "\nVERIFICATION RESULT"
)


if (
    mode1_neutral
    and
    other_modes_stable
):

    print(
        "PASS: Mode 1 is neutral while all other "
        "tracked aeroelastic branches remain stable."
    )


    print(
        "The minimum-authority ideal active controller "
        "is suitable for the next actuator-dynamics study."
    )


else:

    print(
        "WARNING: another branch or the Mode-1 root "
        "requires further investigation before accepting "
        "the active controller."
    )


# ------------------------------------------------------------
# 14. Branch damping plot
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.bar(
    active_all_branch_table[
        "mode_id"
    ],

    active_all_branch_table[
        "sigma"
    ],
)


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.xlabel(
    "Tracked aeroelastic branch"
)


plt.ylabel(
    r"Pole real part $\sigma$ (1/s)"
)


plt.title(
    "All-branch stability at the active flutter boundary"
)


plt.xticks(
    [
        1,
        2,
        3,
        4,
    ]
)


plt.grid(
    axis="y",
    alpha=0.25,
)


plt.tight_layout()

plt.show()


print(
    "\nCell 54 minimum-authority active controller "
    "and all-branch verification completed."
)

In [ ]:
# ============================================================
# CELL 55 — ACTUATOR BANDWIDTH + DELAY SENSITIVITY
# ============================================================
#
# ENGINEERING QUESTION:
#
# How much of the ideal active flutter benefit survives:
#
#       finite actuator bandwidth
#       +
#       sensing/computation/actuation delay?
#
#
# Controller:
#
#       T_cmd = -K_theta * theta_sensor
#
# with:
#
#       K_theta = minimum-authority value from Cell 54
#
#
# Actuator dynamics:
#
#                omega_a
#       H_a = -------------
#             omega_a + iω
#
#
# Delay:
#
#       H_d = exp(-i ω tau)
#
#
# Effective frequency-domain controller:
#
#       K_ctrl(ω)
#
#       = K_theta
#         H_a(iω)
#         exp(-iω tau)
#         B B^T
#
#
# This remains a linear frequency-domain study.
#
# Saturation / absolute torque limits are NOT included yet.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import contextlib


# ============================================================
# PART A — COMPLETE LOW-k AERODYNAMIC COVERAGE
# ============================================================

LOW_K_EXTENSION = np.array([
    0.14,
    0.16,
    0.18,
    0.20,
])


print(
    "Checking low-k GAF coverage for active-control dynamics..."
)


existing_k_0901 = np.asarray(
    cached_aero_database[
        PASSIVE_DESIGN_MACH
    ][
        "k"
    ]
)


for k_test in LOW_K_EXTENSION:

    already_available = (
        np.min(
            np.abs(
                existing_k_0901
                - k_test
            )
        )
        < 1e-8
    )


    if already_available:

        print(
            f"k = {k_test:.2f} 1/m already available."
        )

        continue


    print(
        f"Calculating Mach 0.901 GAF at "
        f"k = {k_test:.2f} 1/m ..."
    )


    _ = calculate_Qhh_wall_ips(
        Ma=
            PASSIVE_DESIGN_MACH,

        k_PA=
            float(
                k_test
            ),

        n_modes=
            4,
    )


# ------------------------------------------------------------
# Refresh aerodynamic database
# ------------------------------------------------------------

k_active_db, Q_active_db = (
    get_cached_Qhh_data(
        Ma_target=
            PASSIVE_DESIGN_MACH,

        n_modes=
            4,
    )
)


cached_aero_database[
    PASSIVE_DESIGN_MACH
] = {
    "k":
        k_active_db,

    "Q":
        Q_active_db,
}


print(
    "\nActive-control aerodynamic database:"
)

print(
    f"k range = "
    f"{k_active_db.min():.4f} "
    f"to "
    f"{k_active_db.max():.4f} 1/m"
)


# ============================================================
# PART B — ACTUATOR TRANSFER FUNCTION
# ============================================================

def actuator_transfer_function(
    omega,
    bandwidth_Hz,
    delay_s,
):
    """
    First-order actuator + pure delay.

    Returns:
        H(i omega)

    bandwidth_Hz:
        actuator -3 dB bandwidth

    delay_s:
        total sensing/computation/actuation delay
    """

    omega = float(
        omega
    )


    bandwidth_Hz = float(
        bandwidth_Hz
    )


    delay_s = float(
        delay_s
    )


    if bandwidth_Hz <= 0.0:

        raise ValueError(
            "Actuator bandwidth must be positive."
        )


    if delay_s < 0.0:

        raise ValueError(
            "Delay must be non-negative."
        )


    omega_a = (
        2.0
        * np.pi
        * bandwidth_Hz
    )


    H_actuator = (
        omega_a
        /
        (
            omega_a
            + 1j
            * omega
        )
    )


    H_delay = np.exp(
        -1j
        * omega
        * delay_s
    )


    return (
        H_actuator
        * H_delay
    )


# ============================================================
# PART C — FROZEN-k p-k WITH DYNAMIC CONTROLLER
# ============================================================

def frozen_k_pk_dynamic_active(
    Ma,
    U,
    q_dyn,
    k_PA,
    K_theta,
    actuator_bandwidth_Hz,
    delay_s,
):
    """
    Frozen-k p-k eigenproblem including a frequency-dependent
    active stiffness controller.

    Harmonic dynamic stiffness:

        -ω²M
        + iωC
        + K
        - q Qhh
        + Kctrl(ω)

    where:

        Kctrl = K_theta H(iω) B B^T

    Split into real stiffness + equivalent damping:

        K_eff =
            K
            + Re(Kctrl)
            - q Re(Qhh)

        C_eff =
            C
            + Im(Kctrl)/ω
            - q Im(Qhh)/ω
    """

    M = (
        M_modal_SI.copy()
    )

    C = (
        C_modal_SI.copy()
    )

    K = (
        K_modal_SI.copy()
    )


    omega = (
        float(k_PA)
        * float(U)
    )


    if omega <= 0.0:

        raise ValueError(
            "omega must be positive."
        )


    # --------------------------------------------------------
    # Aerodynamic GAF
    # --------------------------------------------------------

    Qhh = (
        interpolate_cached_Qhh(
            Ma=Ma,
            k_query=k_PA,
        )
    )


    QR = np.real(
        Qhh
    )

    QI = np.imag(
        Qhh
    )


    # --------------------------------------------------------
    # Actuator/controller transfer
    # --------------------------------------------------------

    H_controller = (
        actuator_transfer_function(
            omega=
                omega,

            bandwidth_Hz=
                actuator_bandwidth_Hz,

            delay_s=
                delay_s,
        )
    )


    K_controller_complex = (
        K_theta
        * H_controller
        * BBT_active
    )


    Kc_real = np.real(
        K_controller_complex
    )


    Kc_imag = np.imag(
        K_controller_complex
    )


    # --------------------------------------------------------
    # Correct frozen-k quadratic matrices
    # --------------------------------------------------------

    K_eff = (
        K
        + Kc_real
        - q_dyn
        * QR
    )


    C_eff = (
        C
        + (
            Kc_imag
            / omega
        )
        - (
            q_dyn
            / omega
        )
        * QI
    )


    M_inv = np.linalg.inv(
        M
    )


    Z = np.zeros(
        (4, 4)
    )

    I = np.eye(
        4
    )


    A = np.block([
        [
            Z,
            I,
        ],

        [
            -M_inv @ K_eff,
            -M_inv @ C_eff,
        ],
    ])


    poles, eigvectors = (
        np.linalg.eig(
            A
        )
    )


    return (
        poles,
        eigvectors,
        H_controller,
    )


# ============================================================
# PART D — MODE-1 DYNAMIC-CONTROLLER p-k ITERATION
# ============================================================

def pk_mode1_dynamic_active(
    U,
    q_dyn,
    K_theta,
    actuator_bandwidth_Hz,
    delay_s,
    initial_k=None,
    initial_reference=None,
    tol=2e-7,
    max_iter=80,
    relaxation=0.65,
):

    M = (
        M_modal_SI.copy()
    )


    if initial_k is None:

        k_current = (
            omega_analysis[0]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    if initial_reference is None:

        reference_q = np.array(
            [
                1.0,
                0.0,
                0.0,
                0.0,
            ],
            dtype=complex,
        )

    else:

        reference_q = np.asarray(
            initial_reference,
            dtype=complex,
        ).copy()


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    selected_H = None


    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
            H_controller,
        ) = (
            frozen_k_pk_dynamic_active(
                Ma=
                    PASSIVE_DESIGN_MACH,

                U=
                    U,

                q_dyn=
                    q_dyn,

                k_PA=
                    k_current,

                K_theta=
                    K_theta,

                actuator_bandwidth_Hz=
                    actuator_bandwidth_Hz,

                delay_s=
                    delay_s,
            )
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_idx = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_idx
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        MAC_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M,
            )
        )


        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        selected_H = (
            H_controller
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            "Dynamic active p-k iteration "
            "did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {
        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            MAC_selected,

        "iterations":
            iteration,

        "modal_vector":
            q_selected.copy(),

        "H_controller":
            selected_H,

        "H_magnitude":
            abs(
                selected_H
            ),

        "H_phase_deg":
            np.angle(
                selected_H,
                deg=True,
            ),
    }


# ============================================================
# PART E — FLUTTER ROOT FOR DYNAMIC ACTUATOR
# ============================================================

def solve_dynamic_active_flutter(
    K_theta,
    actuator_bandwidth_Hz,
    delay_s,
    lambda_guess=None,
):


    if lambda_guess is None:

        lambda_guess = (
            ACTIVE_REFERENCE_FLUTTER[
                "U"
            ]
            / U_exp_passive
        )


    def evaluate_lambda(
        lam,
        reference=None,
    ):

        U = (
            lam
            * U_exp_passive
        )


        q = (
            lam**2
            * q_exp_passive
        )


        if reference is None:

            initial_k = None
            initial_reference = None

        else:

            initial_k = (
                reference[
                    "k_PA"
                ]
            )

            initial_reference = (
                reference[
                    "modal_vector"
                ]
            )


        result = (
            pk_mode1_dynamic_active(
                U=
                    U,

                q_dyn=
                    q,

                K_theta=
                    K_theta,

                actuator_bandwidth_Hz=
                    actuator_bandwidth_Hz,

                delay_s=
                    delay_s,

                initial_k=
                    initial_k,

                initial_reference=
                    initial_reference,
            )
        )


        return {
            "lambda":
                float(
                    lam
                ),

            "U":
                float(
                    U
                ),

            "q":
                float(
                    q
                ),

            **result,
        }


    # --------------------------------------------------------
    # Bracket
    # --------------------------------------------------------

    half_width = (
        0.08
    )


    for expansion in range(
        12
    ):


        lam_low = max(
            0.55,
            lambda_guess
            - half_width,
        )


        lam_high = min(
            1.80,
            lambda_guess
            + half_width,
        )


        low = (
            evaluate_lambda(
                lam_low
            )
        )


        high = (
            evaluate_lambda(
                lam_high,
                reference=low,
            )
        )


        if (
            low[
                "sigma"
            ]
            * high[
                "sigma"
            ]
            <= 0.0
        ):

            break


        half_width += (
            0.06
        )


    else:

        raise RuntimeError(
            "Could not bracket dynamic-controller flutter root."
        )


    if (
        low[
            "sigma"
        ]
        > 0.0
    ):

        low, high = (
            high,
            low,
        )


    # --------------------------------------------------------
    # Root refinement
    # --------------------------------------------------------

    for root_iteration in range(
        1,
        26
    ):


        sig_low = (
            low[
                "sigma"
            ]
        )


        sig_high = (
            high[
                "sigma"
            ]
        )


        lam_trial = (
            low[
                "lambda"
            ]

            - sig_low
            * (
                high[
                    "lambda"
                ]
                - low[
                    "lambda"
                ]
            )

            / (
                sig_high
                - sig_low
            )
        )


        lower = min(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        upper = max(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        width = (
            upper
            - lower
        )


        if (
            lam_trial
            <= lower
            + 0.05
            * width

            or

            lam_trial
            >= upper
            - 0.05
            * width
        ):

            lam_trial = (
                0.5
                * (
                    low[
                        "lambda"
                    ]
                    + high[
                        "lambda"
                    ]
                )
            )


        reference = (
            low

            if abs(
                lam_trial
                - low[
                    "lambda"
                ]
            )

            <=

            abs(
                high[
                    "lambda"
                ]
                - lam_trial
            )

            else high
        )


        trial = (
            evaluate_lambda(
                lam_trial,
                reference=reference,
            )
        )


        if (
            abs(
                trial[
                    "sigma"
                ]
            )
            < 1e-4
        ):

            return trial


        if (
            trial[
                "sigma"
            ]
            < 0.0
        ):

            low = (
                trial
            )

        else:

            high = (
                trial
            )


    return min(
        [
            low,
            high,
        ],

        key=lambda x:
            abs(
                x[
                    "sigma"
                ]
            ),
    )


# ============================================================
# PART F — BANDWIDTH / DELAY TRADE SPACE
# ============================================================

actuator_bandwidth_values_Hz = np.array([
    25.0,
    50.0,
    100.0,
    200.0,
    500.0,
])


delay_values_ms = np.array([
    0.0,
    1.0,
    2.0,
    5.0,
    10.0,
])


dynamic_active_rows = []


print(
    "\nRunning actuator bandwidth / delay sweep..."
)


for bandwidth_Hz in actuator_bandwidth_values_Hz:

    for delay_ms in delay_values_ms:


        delay_s = (
            delay_ms
            / 1000.0
        )


        print(
            f"  bandwidth = "
            f"{bandwidth_Hz:6.1f} Hz, "
            f"delay = "
            f"{delay_ms:4.1f} ms"
        )


        try:

            with contextlib.redirect_stdout(
                io.StringIO()
            ):

                result = (
                    solve_dynamic_active_flutter(
                        K_theta=
                            ACTIVE_REFERENCE_K_THETA,

                        actuator_bandwidth_Hz=
                            bandwidth_Hz,

                        delay_s=
                            delay_s,
                    )
                )


            improvement = (
                100.0
                * (
                    result[
                        "U"
                    ]
                    - U_flutter_baseline
                )
                / U_flutter_baseline
            )


            retained_ideal_pct = (
                100.0
                * improvement
                / ACTIVE_REFERENCE_IMPROVEMENT
            )


            status = (
                "OK"
            )


            U_flutter = (
                result[
                    "U"
                ]
            )


            f_flutter = (
                result[
                    "frequency_Hz"
                ]
            )


            H_mag = (
                result[
                    "H_magnitude"
                ]
            )


            H_phase = (
                result[
                    "H_phase_deg"
                ]
            )


            sigma = (
                result[
                    "sigma"
                ]
            )


        except (
            ValueError,
            RuntimeError,
        ):

            status = (
                "FAILED"
            )


            U_flutter = np.nan
            f_flutter = np.nan
            improvement = np.nan
            retained_ideal_pct = np.nan
            H_mag = np.nan
            H_phase = np.nan
            sigma = np.nan


        dynamic_active_rows.append({

            "bandwidth_Hz":
                bandwidth_Hz,

            "delay_ms":
                delay_ms,

            "flutter_velocity_m_s":
                U_flutter,

            "flutter_frequency_Hz":
                f_flutter,

            "flutter_improvement_pct":
                improvement,

            "ideal_benefit_retained_pct":
                retained_ideal_pct,

            "controller_magnitude":
                H_mag,

            "controller_phase_deg":
                H_phase,

            "sigma_final":
                sigma,

            "status":
                status,
        })


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

active_dynamic_trade = (
    pd.DataFrame(
        dynamic_active_rows
    )
)


print(
    "\nACTUATOR DYNAMICS RESULTS"
)


display(
    active_dynamic_trade
)


# ============================================================
# PART G — PERFORMANCE MAP
# ============================================================

dynamic_pivot = (
    active_dynamic_trade
    .pivot(
        index=
            "delay_ms",

        columns=
            "bandwidth_Hz",

        values=
            "flutter_improvement_pct",
    )
)


print(
    "\nFlutter-velocity improvement (%)"
)

print(
    "Rows = total delay (ms)"
)

print(
    "Columns = actuator bandwidth (Hz)"
)


display(
    dynamic_pivot
)


# ------------------------------------------------------------
# Heatmap
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.imshow(
    dynamic_pivot.to_numpy(),
    aspect="auto",
    origin="lower",
)


plt.colorbar(
    label="Flutter-velocity improvement (%)"
)


plt.xticks(
    np.arange(
        len(
            dynamic_pivot.columns
        )
    ),

    [
        f"{x:.0f}"
        for x in dynamic_pivot.columns
    ],
)


plt.yticks(
    np.arange(
        len(
            dynamic_pivot.index
        )
    ),

    [
        f"{x:.0f}"
        for x in dynamic_pivot.index
    ],
)


plt.xlabel(
    "Actuator bandwidth (Hz)"
)


plt.ylabel(
    "Total delay (ms)"
)


plt.title(
    "How actuator dynamics erode ideal flutter suppression"
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Curves
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


for delay_ms in delay_values_ms:


    subset = (
        active_dynamic_trade[
            np.isclose(
                active_dynamic_trade[
                    "delay_ms"
                ],
                delay_ms,
            )
        ]
        .sort_values(
            "bandwidth_Hz"
        )
    )


    plt.plot(
        subset[
            "bandwidth_Hz"
        ],

        subset[
            "flutter_improvement_pct"
        ],

        "o-",

        label=
            f"{delay_ms:.0f} ms delay",
    )


plt.axhline(
    ACTIVE_REFERENCE_IMPROVEMENT,

    linestyle="--",

    linewidth=1.0,

    label="Ideal active reference",
)


plt.axhline(
    passive_reference_improvement,

    linestyle=":",

    linewidth=1.0,

    label="Passive reference",
)


plt.axhline(
    0.0,
    linewidth=1.0,
)


plt.xlabel(
    "Actuator bandwidth (Hz)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Practical actuator bandwidth and delay requirements"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# PART H — BEST / WORST PRACTICAL CASES
# ============================================================

valid_dynamic = (
    active_dynamic_trade[
        active_dynamic_trade[
            "status"
        ]
        == "OK"
    ]
    .copy()
)


best_dynamic = (
    valid_dynamic.loc[
        valid_dynamic[
            "flutter_improvement_pct"
        ]
        .idxmax()
    ]
)


worst_dynamic = (
    valid_dynamic.loc[
        valid_dynamic[
            "flutter_improvement_pct"
        ]
        .idxmin()
    ]
)


print(
    "\nACTUATOR-DYNAMICS SUMMARY"
)


print(
    f"\nIdeal active improvement = "
    f"{ACTIVE_REFERENCE_IMPROVEMENT:+.3f}%"
)


print(
    f"Passive reference improvement = "
    f"{passive_reference_improvement:+.3f}%"
)


print(
    "\nBest investigated practical actuator case:"
)


print(
    f"Bandwidth = "
    f"{best_dynamic['bandwidth_Hz']:.0f} Hz"
)


print(
    f"Delay = "
    f"{best_dynamic['delay_ms']:.1f} ms"
)


print(
    f"Flutter improvement = "
    f"{best_dynamic['flutter_improvement_pct']:+.3f}%"
)


print(
    f"Ideal benefit retained = "
    f"{best_dynamic['ideal_benefit_retained_pct']:.1f}%"
)


print(
    f"Controller phase at flutter = "
    f"{best_dynamic['controller_phase_deg']:.2f} deg"
)


print(
    "\nWorst investigated actuator case:"
)


print(
    f"Bandwidth = "
    f"{worst_dynamic['bandwidth_Hz']:.0f} Hz"
)


print(
    f"Delay = "
    f"{worst_dynamic['delay_ms']:.1f} ms"
)


print(
    f"Flutter improvement = "
    f"{worst_dynamic['flutter_improvement_pct']:+.3f}%"
)


print(
    f"Controller phase at flutter = "
    f"{worst_dynamic['controller_phase_deg']:.2f} deg"
)


print(
    "\nCell 55 actuator bandwidth/delay study completed."
)

In [ ]:
# ============================================================
# CELL 56 — DYNAMIC CONTROLLER PHASE / ROBUSTNESS AUDIT
# ============================================================
#
# PURPOSE
# -------
# Cell 55 showed that actuator phase lag can INCREASE flutter
# margin beyond the zero-lag active reference.
#
# Before accepting this:
#
#   1. quantify the controller's effective stiffness/damping
#      contribution at flutter;
#
#   2. verify dry closed-loop stability using explicit actuator
#      dynamics and Padé delay approximations;
#
#   3. verify ALL FOUR aeroelastic branches for representative
#      phase-shaped controller cases.
#
#
# NO NEW DLM GAF CALCULATIONS ARE EXPECTED.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import tf2ss


# ============================================================
# PART A — INTERPRET CELL-55 CONTROLLER PHASE
# ============================================================

phase_audit_rows = []


for _, row in active_dynamic_trade.iterrows():

    if row["status"] != "OK":
        continue

    omega_flutter = (
        2.0
        * np.pi
        * float(
            row["flutter_frequency_Hz"]
        )
    )

    H = (
        float(
            row["controller_magnitude"]
        )
        * np.exp(
            1j
            * np.deg2rad(
                float(
                    row["controller_phase_deg"]
                )
            )
        )
    )

    # --------------------------------------------------------
    # Controller contribution:
    #
    # Kctrl = Ktheta H B B^T
    #
    # Re(Kctrl) -> equivalent active stiffness
    #
    # Im(Kctrl)/omega -> equivalent damping contribution
    # --------------------------------------------------------

    delta_K_ctrl = (
        ACTIVE_REFERENCE_K_THETA
        * np.real(H)
        * BBT_active
    )

    delta_C_ctrl = (
        ACTIVE_REFERENCE_K_THETA
        * np.imag(H)
        / omega_flutter
        * BBT_active
    )

    delta_K11_pct = (
        100.0
        * delta_K_ctrl[0, 0]
        / K_modal_SI[0, 0]
    )

    delta_C11_pct = (
        100.0
        * delta_C_ctrl[0, 0]
        / C_modal_SI[0, 0]
    )

    phase_audit_rows.append({

        "bandwidth_Hz":
            float(
                row["bandwidth_Hz"]
            ),

        "delay_ms":
            float(
                row["delay_ms"]
            ),

        "flutter_improvement_pct":
            float(
                row["flutter_improvement_pct"]
            ),

        "flutter_frequency_Hz":
            float(
                row["flutter_frequency_Hz"]
            ),

        "H_magnitude":
            abs(H),

        "H_phase_deg":
            np.angle(
                H,
                deg=True,
            ),

        "effective_delta_K11_pct":
            delta_K11_pct,

        "effective_delta_C11_pct":
            delta_C11_pct,
    })


controller_phase_audit = (
    pd.DataFrame(
        phase_audit_rows
    )
)


print(
    "CONTROLLER PHASE INTERPRETATION"
)


display(
    controller_phase_audit
)


# ============================================================
# PART B — DRY CLOSED-LOOP STATE-SPACE MODEL
# ============================================================
#
# Unlike the frozen-frequency flutter calculation, here we
# explicitly include actuator internal states.
#
#
# Actuator:
#
#                omega_a
# H_a(s) = --------------------
#            s + omega_a
#
#
# Delay approximated using Padé [1/1] or [2/2].
#
#
# Controller:
#
#       T = -K_theta H(s) theta
#
# ============================================================


def pade_delay_coefficients(
    delay_s,
    order=1,
):
    """
    Polynomial coefficients in descending powers of s
    approximating exp(-s*tau).
    """

    tau = float(
        delay_s
    )


    if tau < 0.0:
        raise ValueError(
            "Delay must be non-negative."
        )


    if tau == 0.0:

        return (
            np.array([1.0]),
            np.array([1.0]),
        )


    if order == 1:

        # (1 - s tau/2) / (1 + s tau/2)

        numerator = np.array([
            -tau / 2.0,
            1.0,
        ])

        denominator = np.array([
            tau / 2.0,
            1.0,
        ])


    elif order == 2:

        # 1 - s tau/2 + s² tau²/12
        # ---------------------------
        # 1 + s tau/2 + s² tau²/12

        numerator = np.array([
            tau**2 / 12.0,
            -tau / 2.0,
            1.0,
        ])

        denominator = np.array([
            tau**2 / 12.0,
            tau / 2.0,
            1.0,
        ])


    else:

        raise ValueError(
            "Supported Padé orders are 1 and 2."
        )


    return (
        numerator,
        denominator,
    )


def actuator_controller_state_space(
    bandwidth_Hz,
    delay_s,
    pade_order=1,
):
    """
    State-space realization of:

        H(s)
        =
        omega_a/(s+omega_a)
        *
        exp(-s*tau)

    with Padé approximation for delay.
    """

    bandwidth_Hz = float(
        bandwidth_Hz
    )


    if bandwidth_Hz <= 0.0:

        raise ValueError(
            "Bandwidth must be positive."
        )


    omega_a = (
        2.0
        * np.pi
        * bandwidth_Hz
    )


    # First-order actuator
    num_act = np.array([
        omega_a
    ])

    den_act = np.array([
        1.0,
        omega_a,
    ])


    # Delay approximation
    num_delay, den_delay = (
        pade_delay_coefficients(
            delay_s=
                delay_s,

            order=
                pade_order,
        )
    )


    num_total = np.polymul(
        num_act,
        num_delay,
    )


    den_total = np.polymul(
        den_act,
        den_delay,
    )


    A_h, B_h, C_h, D_h = (
        tf2ss(
            num_total,
            den_total,
        )
    )


    return (
        A_h,
        B_h,
        C_h,
        D_h,
    )


def dry_closed_loop_dynamic_controller(
    bandwidth_Hz,
    delay_s,
    pade_order=1,
):
    """
    Exact finite-dimensional dry closed-loop model after
    Padé approximation of the delay.

    Structural states:
        q
        qdot

    Controller states:
        x_h
    """

    M = (
        M_modal_SI.copy()
    )

    C = (
        C_modal_SI.copy()
    )

    K = (
        K_modal_SI.copy()
    )


    A_h, B_h, C_h, D_h = (
        actuator_controller_state_space(
            bandwidth_Hz=
                bandwidth_Hz,

            delay_s=
                delay_s,

            pade_order=
                pade_order,
        )
    )


    n_struct = 4

    n_ctrl = (
        A_h.shape[0]
    )


    M_inv = np.linalg.inv(
        M
    )


    B_col = (
        B_active.reshape(
            -1,
            1,
        )
    )


    B_row = (
        B_active.reshape(
            1,
            -1,
        )
    )


    D_scalar = float(
        D_h[
            0,
            0
        ]
    )


    # --------------------------------------------------------
    # Structural acceleration:
    #
    # M qdd
    # + C qdot
    # + K q
    # =
    # -B Ktheta y_controller
    #
    # y_controller =
    #
    #     C_h x_h
    #     +
    #     D_h B^T q
    # --------------------------------------------------------

    A_vq = (
        -M_inv
        @ (
            K
            + ACTIVE_REFERENCE_K_THETA
            * D_scalar
            * (
                B_col
                @ B_row
            )
        )
    )


    A_vv = (
        -M_inv
        @ C
    )


    A_vx = (
        -M_inv
        @ (
            ACTIVE_REFERENCE_K_THETA
            * B_col
            @ C_h
        )
    )


    # Controller state:
    #
    # xdot =
    #
    #     A_h x
    #     +
    #     B_h theta
    #
    # theta = B^T q

    A_xq = (
        B_h
        @ B_row
    )


    Z44 = np.zeros(
        (4, 4)
    )


    I44 = np.eye(
        4
    )


    Z4c = np.zeros(
        (
            4,
            n_ctrl,
        )
    )


    Zc4 = np.zeros(
        (
            n_ctrl,
            4,
        )
    )


    A_closed = np.block([

        [
            Z44,
            I44,
            Z4c,
        ],

        [
            A_vq,
            A_vv,
            A_vx,
        ],

        [
            A_xq,
            Zc4,
            A_h,
        ],
    ])


    poles = np.linalg.eigvals(
        A_closed
    )


    return {
        "poles":
            poles,

        "max_real_pole":
            float(
                np.max(
                    np.real(
                        poles
                    )
                )
            ),

        "stable":
            bool(
                np.all(
                    np.real(
                        poles
                    )
                < 0.0
            )
        ),
    }


# ============================================================
# PART C — SELECT REPRESENTATIVE CELL-55 CASES
# ============================================================
#
# Do not call any of these "optimal" yet.
#
# Case A:
#     nearly zero-phase / high bandwidth
#
# Case B:
#     moderate phase shaping
#
# Case C:
#     strongest Cell-55 result
#
# ============================================================

dynamic_candidate_definitions = [

    {
        "candidate":
            "High-BW low-delay",

        "bandwidth_Hz":
            500.0,

        "delay_ms":
            0.0,
    },

    {
        "candidate":
            "Moderate phase-shaped",

        "bandwidth_Hz":
            100.0,

        "delay_ms":
            5.0,
    },

    {
        "candidate":
            "High phase-shaped",

        "bandwidth_Hz":
            500.0,

        "delay_ms":
            10.0,
    },
]


candidate_rows = []


for definition in dynamic_candidate_definitions:

    bandwidth = float(
        definition[
            "bandwidth_Hz"
        ]
    )

    delay_ms = float(
        definition[
            "delay_ms"
        ]
    )


    source_row = (
        active_dynamic_trade[
            np.isclose(
                active_dynamic_trade[
                    "bandwidth_Hz"
                ],
                bandwidth,
            )
            &
            np.isclose(
                active_dynamic_trade[
                    "delay_ms"
                ],
                delay_ms,
            )
        ]
        .iloc[0]
    )


    # --------------------------------------------------------
    # Dry closed-loop stability:
    # compare Padé [1/1] and [2/2]
    # --------------------------------------------------------

    dry_pade1 = (
        dry_closed_loop_dynamic_controller(
            bandwidth_Hz=
                bandwidth,

            delay_s=
                delay_ms
                / 1000.0,

            pade_order=
                1,
        )
    )


    dry_pade2 = (
        dry_closed_loop_dynamic_controller(
            bandwidth_Hz=
                bandwidth,

            delay_s=
                delay_ms
                / 1000.0,

            pade_order=
                2,
        )
    )


    candidate_rows.append({

        "candidate":
            definition[
                "candidate"
            ],

        "bandwidth_Hz":
            bandwidth,

        "delay_ms":
            delay_ms,

        "flutter_velocity_m_s":
            float(
                source_row[
                    "flutter_velocity_m_s"
                ]
            ),

        "flutter_improvement_pct":
            float(
                source_row[
                    "flutter_improvement_pct"
                ]
            ),

        "flutter_frequency_Hz":
            float(
                source_row[
                    "flutter_frequency_Hz"
                ]
            ),

        "controller_phase_deg":
            float(
                source_row[
                    "controller_phase_deg"
                ]
            ),

        "controller_magnitude":
            float(
                source_row[
                    "controller_magnitude"
                ]
            ),

        "dry_stable_pade1":
            dry_pade1[
                "stable"
            ],

        "dry_max_real_pade1":
            dry_pade1[
                "max_real_pole"
            ],

        "dry_stable_pade2":
            dry_pade2[
                "stable"
            ],

        "dry_max_real_pade2":
            dry_pade2[
                "max_real_pole"
            ],
    })


dynamic_candidate_summary = (
    pd.DataFrame(
        candidate_rows
    )
)


print(
    "\nREPRESENTATIVE DYNAMIC-CONTROLLER CANDIDATES"
)


display(
    dynamic_candidate_summary
)


# ============================================================
# PART D — GENERIC ALL-BRANCH DYNAMIC p-k SOLVER
# ============================================================

def pk_dynamic_active_branch(
    mode_index,
    U,
    q_dyn,
    K_theta,
    bandwidth_Hz,
    delay_s,
    initial_k=None,
    tol=2e-7,
    max_iter=80,
    relaxation=0.65,
):
    """
    Track one aeroelastic branch with frequency-dependent
    actuator dynamics.
    """

    if not (
        0
        <= mode_index
        < 4
    ):

        raise ValueError(
            "mode_index must be 0,1,2,3."
        )


    M = (
        M_modal_SI.copy()
    )


    if initial_k is None:

        k_current = (
            omega_analysis[
                mode_index
            ]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    reference_q = np.zeros(
        4,
        dtype=complex,
    )


    reference_q[
        mode_index
    ] = 1.0


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M,
        )
    )


    selected_H = None


    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
            H_controller,
        ) = (
            frozen_k_pk_dynamic_active(
                Ma=
                    PASSIVE_DESIGN_MACH,

                U=
                    U,

                q_dyn=
                    q_dyn,

                k_PA=
                    k_current,

                K_theta=
                    K_theta,

                actuator_bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,
            )
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:

            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_index = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_index
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        MAC_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M,
            )
        )


        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        selected_H = (
            H_controller
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            f"Dynamic active branch "
            f"{mode_index+1} did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {

        "mode_id":
            mode_index + 1,

        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            -sigma
            / abs(
                pole
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            MAC_selected,

        "iterations":
            iteration,

        "controller_magnitude":
            abs(
                selected_H
            ),

        "controller_phase_deg":
            np.angle(
                selected_H,
                deg=True,
            ),
    }


# ============================================================
# PART E — ALL-BRANCH VERIFICATION OF EACH CANDIDATE
# ============================================================

dynamic_all_branch_rows = []


for _, candidate in dynamic_candidate_summary.iterrows():

    name = (
        candidate[
            "candidate"
        ]
    )


    bandwidth = float(
        candidate[
            "bandwidth_Hz"
        ]
    )


    delay_ms = float(
        candidate[
            "delay_ms"
        ]
    )


    delay_s = (
        delay_ms
        / 1000.0
    )


    U_case = float(
        candidate[
            "flutter_velocity_m_s"
        ]
    )


    lambda_case = (
        U_case
        / U_exp_passive
    )


    q_case = (
        lambda_case**2
        * q_exp_passive
    )


    print(
        f"\nAll-branch verification: "
        f"{name}"
    )


    print(
        f"U = {U_case:.3f} m/s, "
        f"bandwidth = {bandwidth:.0f} Hz, "
        f"delay = {delay_ms:.1f} ms"
    )


    for mode_index in range(
        4
    ):


        result = (
            pk_dynamic_active_branch(
                mode_index=
                    mode_index,

                U=
                    U_case,

                q_dyn=
                    q_case,

                K_theta=
                    ACTIVE_REFERENCE_K_THETA,

                bandwidth_Hz=
                    bandwidth,

                delay_s=
                    delay_s,
            )
        )


        dynamic_all_branch_rows.append({

            "candidate":
                name,

            "bandwidth_Hz":
                bandwidth,

            "delay_ms":
                delay_ms,

            **result,
        })


dynamic_all_branch_table = (
    pd.DataFrame(
        dynamic_all_branch_rows
    )
)


print(
    "\nDYNAMIC-CONTROLLER ALL-BRANCH RESULTS"
)


display(
    dynamic_all_branch_table
)


# ============================================================
# PART F — VERIFICATION SUMMARY
# ============================================================

verification_rows = []


for name in (
    dynamic_candidate_summary[
        "candidate"
    ]
):


    branches = (
        dynamic_all_branch_table[
            dynamic_all_branch_table[
                "candidate"
            ]
            == name
        ]
    )


    mode1_sigma = float(
        branches[
            branches[
                "mode_id"
            ]
            == 1
        ][
            "sigma"
        ]
        .iloc[0]
    )


    other_modes_max_sigma = float(
        branches[
            branches[
                "mode_id"
            ]
            != 1
        ][
            "sigma"
        ]
        .max()
    )


    candidate_info = (
        dynamic_candidate_summary[
            dynamic_candidate_summary[
                "candidate"
            ]
            == name
        ]
        .iloc[0]
    )


    verification_rows.append({

        "candidate":
            name,

        "flutter_improvement_pct":
            candidate_info[
                "flutter_improvement_pct"
            ],

        "controller_phase_deg":
            candidate_info[
                "controller_phase_deg"
            ],

        "dry_stable_pade1":
            candidate_info[
                "dry_stable_pade1"
            ],

        "dry_stable_pade2":
            candidate_info[
                "dry_stable_pade2"
            ],

        "mode1_sigma":
            mode1_sigma,

        "max_other_branch_sigma":
            other_modes_max_sigma,

        "all_other_branches_stable":
            (
                other_modes_max_sigma
                < 0.0
            ),
    })


dynamic_verification_summary = (
    pd.DataFrame(
        verification_rows
    )
)


print(
    "\nFINAL DYNAMIC-CONTROLLER VERIFICATION SUMMARY"
)


display(
    dynamic_verification_summary
)


# ============================================================
# PART G — PHASE VERSUS PERFORMANCE
# ============================================================

plt.figure(
    figsize=(9, 5)
)


for bandwidth in sorted(
    controller_phase_audit[
        "bandwidth_Hz"
    ]
    .unique()
):


    subset = (
        controller_phase_audit[
            np.isclose(
                controller_phase_audit[
                    "bandwidth_Hz"
                ],
                bandwidth,
            )
        ]
        .sort_values(
            "H_phase_deg"
        )
    )


    plt.plot(
        subset[
            "H_phase_deg"
        ],

        subset[
            "flutter_improvement_pct"
        ],

        "o-",

        label=
            f"{bandwidth:.0f} Hz",
    )


plt.axhline(
    passive_reference_improvement,
    linestyle="--",
    linewidth=1.0,
    label="Passive reference",
)


plt.xlabel(
    "Controller phase at flutter (deg)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Flutter benefit is strongly phase-dependent"
)


plt.grid(
    alpha=0.25
)


plt.legend(
    title="Bandwidth"
)


plt.tight_layout()

plt.show()


print(
    "\nCell 56 dynamic-controller phase and "
    "all-branch verification completed."
)

In [ ]:
# ============================================================
# CELL 57 — ROBUST PRACTICAL ACTIVE-CONTROLLER SCREEN
# ============================================================
#
# PURPOSE
# -------
#
# Cell 55 showed large phase-shaped flutter benefits.
#
# Cell 56 showed that some of those controllers are actually
# DRY CLOSED-LOOP UNSTABLE or sensitive to the delay model.
#
#
# This cell therefore:
#
#   1. checks ALL 25 bandwidth/delay cases;
#   2. uses Padé [1/1], [2/2], and [3/3];
#   3. requires dry stability for ALL THREE approximations;
#   4. excludes zero-delay cases from the practical shortlist;
#   5. ranks surviving finite-delay cases;
#   6. all-branch verifies the top three.
#
#
# No new DLM calculations.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PART A — EXTEND PADÉ APPROXIMATION TO ORDER 3
# ============================================================

def pade_delay_coefficients(
    delay_s,
    order=1,
):
    """
    Padé approximation of exp(-s*tau).

    Supported:
        [1/1]
        [2/2]
        [3/3]
    """

    tau = float(
        delay_s
    )


    if tau < 0.0:

        raise ValueError(
            "Delay must be non-negative."
        )


    if tau == 0.0:

        return (
            np.array([1.0]),
            np.array([1.0]),
        )


    # --------------------------------------------------------
    # [1/1]
    #
    #       1 - s*tau/2
    #       -----------
    #       1 + s*tau/2
    # --------------------------------------------------------

    if order == 1:

        numerator = np.array([
            -tau / 2.0,
            1.0,
        ])


        denominator = np.array([
            tau / 2.0,
            1.0,
        ])


    # --------------------------------------------------------
    # [2/2]
    #
    # 1 - x/2 + x²/12
    # ----------------
    # 1 + x/2 + x²/12
    # --------------------------------------------------------

    elif order == 2:

        numerator = np.array([
            tau**2 / 12.0,
            -tau / 2.0,
            1.0,
        ])


        denominator = np.array([
            tau**2 / 12.0,
            tau / 2.0,
            1.0,
        ])


    # --------------------------------------------------------
    # [3/3]
    #
    # 120 - 60x + 12x² - x³
    # -----------------------
    # 120 + 60x + 12x² + x³
    # --------------------------------------------------------

    elif order == 3:

        numerator = np.array([
            -tau**3 / 120.0,
            tau**2 / 10.0,
            -tau / 2.0,
            1.0,
        ])


        denominator = np.array([
            tau**3 / 120.0,
            tau**2 / 10.0,
            tau / 2.0,
            1.0,
        ])


    else:

        raise ValueError(
            "Supported Padé orders are 1, 2, and 3."
        )


    return (
        numerator,
        denominator,
    )


print(
    "Padé delay approximation extended to order [3/3]."
)


# ============================================================
# PART B — DRY STABILITY FOR ALL CELL-55 CASES
# ============================================================

robust_active_rows = []


for _, row in active_dynamic_trade.iterrows():


    if row["status"] != "OK":

        continue


    bandwidth = float(
        row[
            "bandwidth_Hz"
        ]
    )


    delay_ms = float(
        row[
            "delay_ms"
        ]
    )


    delay_s = (
        delay_ms
        / 1000.0
    )


    # --------------------------------------------------------
    # Three different delay approximations
    # --------------------------------------------------------

    dry_1 = (
        dry_closed_loop_dynamic_controller(
            bandwidth_Hz=
                bandwidth,

            delay_s=
                delay_s,

            pade_order=
                1,
        )
    )


    dry_2 = (
        dry_closed_loop_dynamic_controller(
            bandwidth_Hz=
                bandwidth,

            delay_s=
                delay_s,

            pade_order=
                2,
        )
    )


    dry_3 = (
        dry_closed_loop_dynamic_controller(
            bandwidth_Hz=
                bandwidth,

            delay_s=
                delay_s,

            pade_order=
                3,
        )
    )


    # --------------------------------------------------------
    # Dry stability margin
    #
    # Positive number = stable margin.
    #
    #       margin = - max Re(p)
    #
    # --------------------------------------------------------

    margin_1 = (
        -dry_1[
            "max_real_pole"
        ]
    )


    margin_2 = (
        -dry_2[
            "max_real_pole"
        ]
    )


    margin_3 = (
        -dry_3[
            "max_real_pole"
        ]
    )


    minimum_margin = float(
        min(
            margin_1,
            margin_2,
            margin_3,
        )
    )


    stable_all_three = bool(
        dry_1[
            "stable"
        ]
        and
        dry_2[
            "stable"
        ]
        and
        dry_3[
            "stable"
        ]
    )


    robust_active_rows.append({

        "bandwidth_Hz":
            bandwidth,

        "delay_ms":
            delay_ms,

        "flutter_velocity_m_s":
            float(
                row[
                    "flutter_velocity_m_s"
                ]
            ),

        "flutter_frequency_Hz":
            float(
                row[
                    "flutter_frequency_Hz"
                ]
            ),

        "flutter_improvement_pct":
            float(
                row[
                    "flutter_improvement_pct"
                ]
            ),

        "controller_magnitude":
            float(
                row[
                    "controller_magnitude"
                ]
            ),

        "controller_phase_deg":
            float(
                row[
                    "controller_phase_deg"
                ]
            ),

        "pade1_stable":
            dry_1[
                "stable"
            ],

        "pade1_max_real":
            dry_1[
                "max_real_pole"
            ],

        "pade2_stable":
            dry_2[
                "stable"
            ],

        "pade2_max_real":
            dry_2[
                "max_real_pole"
            ],

        "pade3_stable":
            dry_3[
                "stable"
            ],

        "pade3_max_real":
            dry_3[
                "max_real_pole"
            ],

        "stable_all_three":
            stable_all_three,

        "minimum_dry_stability_margin":
            minimum_margin,
    })


active_robustness_screen = (
    pd.DataFrame(
        robust_active_rows
    )
)


print(
    "\nALL ACTUATOR CASES — DRY ROBUSTNESS SCREEN"
)


display(
    active_robustness_screen
)


# ============================================================
# PART C — PRACTICAL FINITE-DELAY SUBSET
# ============================================================

practical_robust_active = (
    active_robustness_screen[
        (
            active_robustness_screen[
                "delay_ms"
            ]
            > 0.0
        )
        &
        (
            active_robustness_screen[
                "stable_all_three"
            ]
            == True
        )
    ]
    .copy()
)


practical_robust_active = (
    practical_robust_active
    .sort_values(
        [
            "flutter_improvement_pct",
            "minimum_dry_stability_margin",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nFINITE-DELAY CASES STABLE FOR PADÉ "
    "[1/1], [2/2], AND [3/3]"
)


if len(
    practical_robust_active
) == 0:

    print(
        "No finite-delay controller survived all three "
        "dry-stability checks."
    )


else:

    display(
        practical_robust_active[
            [
                "bandwidth_Hz",
                "delay_ms",
                "flutter_improvement_pct",
                "flutter_frequency_Hz",
                "controller_phase_deg",
                "minimum_dry_stability_margin",
            ]
        ]
    )


# ============================================================
# PART D — COMPARE AGAINST PASSIVE REQUIREMENT
# ============================================================

if len(
    practical_robust_active
) > 0:


    practical_robust_active[
        "beats_passive_reference"
    ] = (
        practical_robust_active[
            "flutter_improvement_pct"
        ]
        >
        passive_reference_improvement
    )


    n_beating_passive = int(
        practical_robust_active[
            "beats_passive_reference"
        ]
        .sum()
    )


    print(
        f"\nNumber of robust finite-delay cases "
        f"exceeding passive reference = "
        f"{n_beating_passive}"
    )


# ============================================================
# PART E — SHORTLIST TOP THREE ROBUST FINITE-DELAY CASES
# ============================================================

N_SHORTLIST = min(
    3,
    len(
        practical_robust_active
    ),
)


active_practical_shortlist = (
    practical_robust_active
    .head(
        N_SHORTLIST
    )
    .copy()
)


print(
    "\nROBUST ACTIVE SHORTLIST"
)


display(
    active_practical_shortlist[
        [
            "bandwidth_Hz",
            "delay_ms",
            "flutter_velocity_m_s",
            "flutter_improvement_pct",
            "controller_phase_deg",
            "minimum_dry_stability_margin",
        ]
    ]
)


# ============================================================
# PART F — ALL-BRANCH VERIFICATION OF SHORTLIST
# ============================================================

shortlist_branch_rows = []


for shortlist_id, candidate in (
    active_practical_shortlist
    .iterrows()
):


    bandwidth = float(
        candidate[
            "bandwidth_Hz"
        ]
    )


    delay_ms = float(
        candidate[
            "delay_ms"
        ]
    )


    delay_s = (
        delay_ms
        / 1000.0
    )


    U_case = float(
        candidate[
            "flutter_velocity_m_s"
        ]
    )


    lambda_case = (
        U_case
        / U_exp_passive
    )


    q_case = (
        lambda_case**2
        * q_exp_passive
    )


    print(
        f"\nAll-branch verification:"
    )


    print(
        f"  shortlist {shortlist_id + 1}"
    )


    print(
        f"  bandwidth = "
        f"{bandwidth:.1f} Hz"
    )


    print(
        f"  delay = "
        f"{delay_ms:.1f} ms"
    )


    print(
        f"  U = "
        f"{U_case:.3f} m/s"
    )


    branch_failed = False


    for mode_index in range(
        4
    ):


        try:

            result = (
                pk_dynamic_active_branch(
                    mode_index=
                        mode_index,

                    U=
                        U_case,

                    q_dyn=
                        q_case,

                    K_theta=
                        ACTIVE_REFERENCE_K_THETA,

                    bandwidth_Hz=
                        bandwidth,

                    delay_s=
                        delay_s,
                )
            )


            shortlist_branch_rows.append({

                "shortlist_id":
                    shortlist_id + 1,

                "bandwidth_Hz":
                    bandwidth,

                "delay_ms":
                    delay_ms,

                **result,
            })


        except (
            ValueError,
            RuntimeError,
        ):


            branch_failed = True


            shortlist_branch_rows.append({

                "shortlist_id":
                    shortlist_id + 1,

                "bandwidth_Hz":
                    bandwidth,

                "delay_ms":
                    delay_ms,

                "mode_id":
                    mode_index + 1,

                "sigma":
                    np.nan,

                "frequency_Hz":
                    np.nan,

                "damping_ratio":
                    np.nan,

                "k_PA":
                    np.nan,

                "MAC":
                    np.nan,

                "iterations":
                    np.nan,

                "controller_magnitude":
                    np.nan,

                "controller_phase_deg":
                    np.nan,
            })


shortlist_all_branch = (
    pd.DataFrame(
        shortlist_branch_rows
    )
)


print(
    "\nSHORTLIST ALL-BRANCH RESULTS"
)


display(
    shortlist_all_branch
)


# ============================================================
# PART G — FINAL SHORTLIST VERIFICATION TABLE
# ============================================================

shortlist_summary_rows = []


for shortlist_id in range(
    1,
    N_SHORTLIST + 1
):


    candidate = (
        active_practical_shortlist
        .iloc[
            shortlist_id - 1
        ]
    )


    branches = (
        shortlist_all_branch[
            shortlist_all_branch[
                "shortlist_id"
            ]
            == shortlist_id
        ]
    )


    branch_data_complete = bool(
        branches[
            "sigma"
        ]
        .notna()
        .all()
    )


    if branch_data_complete:


        mode1_sigma = float(
            branches[
                branches[
                    "mode_id"
                ]
                == 1
            ][
                "sigma"
            ]
            .iloc[0]
        )


        other_max_sigma = float(
            branches[
                branches[
                    "mode_id"
                ]
                != 1
            ][
                "sigma"
            ]
            .max()
        )


        other_branches_stable = bool(
            other_max_sigma
            < 0.0
        )


        mode1_neutral = bool(
            abs(
                mode1_sigma
            )
            < 1e-3
        )


    else:


        mode1_sigma = np.nan
        other_max_sigma = np.nan

        other_branches_stable = False
        mode1_neutral = False


    fully_verified = bool(
        candidate[
            "stable_all_three"
        ]
        and
        branch_data_complete
        and
        mode1_neutral
        and
        other_branches_stable
    )


    shortlist_summary_rows.append({

        "shortlist_id":
            shortlist_id,

        "bandwidth_Hz":
            candidate[
                "bandwidth_Hz"
            ],

        "delay_ms":
            candidate[
                "delay_ms"
            ],

        "flutter_improvement_pct":
            candidate[
                "flutter_improvement_pct"
            ],

        "controller_phase_deg":
            candidate[
                "controller_phase_deg"
            ],

        "minimum_dry_stability_margin":
            candidate[
                "minimum_dry_stability_margin"
            ],

        "mode1_sigma":
            mode1_sigma,

        "max_other_branch_sigma":
            other_max_sigma,

        "fully_verified":
            fully_verified,
    })


active_practical_verification = (
    pd.DataFrame(
        shortlist_summary_rows
    )
)


print(
    "\nROBUST PRACTICAL ACTIVE VERIFICATION"
)


display(
    active_practical_verification
)


# ============================================================
# PART H — DO NOT AUTOMATICALLY CALL HIGHEST BENEFIT 'BEST'
# ============================================================

verified_cases = (
    active_practical_verification[
        active_practical_verification[
            "fully_verified"
        ]
        == True
    ]
    .copy()
)


if len(
    verified_cases
) > 0:


    highest_verified = (
        verified_cases
        .loc[
            verified_cases[
                "flutter_improvement_pct"
            ]
            .idxmax()
        ]
    )


    print(
        "\nHighest-benefit fully verified finite-delay case:"
    )


    print(
        f"Bandwidth = "
        f"{highest_verified['bandwidth_Hz']:.1f} Hz"
    )


    print(
        f"Delay = "
        f"{highest_verified['delay_ms']:.1f} ms"
    )


    print(
        f"Flutter improvement = "
        f"{highest_verified['flutter_improvement_pct']:+.3f}%"
    )


    print(
        f"Controller phase = "
        f"{highest_verified['controller_phase_deg']:.2f} deg"
    )


    print(
        f"Minimum dry stability margin = "
        f"{highest_verified['minimum_dry_stability_margin']:.4f} 1/s"
    )


    print(
        "\nThis is a candidate, NOT yet declared the final "
        "controller. Stability margin and implementation "
        "requirements still matter."
    )


else:


    print(
        "\nNo finite-delay controller has yet passed every "
        "robustness requirement."
    )


# ============================================================
# PART I — ROBUSTNESS MAP
# ============================================================

robust_map = (
    active_robustness_screen
    .pivot(
        index=
            "delay_ms",

        columns=
            "bandwidth_Hz",

        values=
            "minimum_dry_stability_margin",
    )
)


plt.figure(
    figsize=(9, 5)
)


plt.imshow(
    robust_map.to_numpy(),
    aspect="auto",
    origin="lower",
)


plt.colorbar(
    label=
        "Minimum dry stability margin "
        "across Padé orders (1/s)"
)


plt.xticks(
    np.arange(
        len(
            robust_map.columns
        )
    ),

    [
        f"{x:.0f}"
        for x in robust_map.columns
    ],
)


plt.yticks(
    np.arange(
        len(
            robust_map.index
        )
    ),

    [
        f"{x:.0f}"
        for x in robust_map.index
    ],
)


plt.xlabel(
    "Actuator bandwidth (Hz)"
)


plt.ylabel(
    "Total delay (ms)"
)


plt.title(
    "Dry closed-loop robustness of phase-shaped controllers"
)


plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Performance versus robustness
# ------------------------------------------------------------

finite_delay_screen = (
    active_robustness_screen[
        active_robustness_screen[
            "delay_ms"
        ]
        > 0.0
    ]
)


plt.figure(
    figsize=(9, 5)
)


stable_points = (
    finite_delay_screen[
        finite_delay_screen[
            "stable_all_three"
        ]
        == True
    ]
)


unstable_points = (
    finite_delay_screen[
        finite_delay_screen[
            "stable_all_three"
        ]
        == False
    ]
)


plt.scatter(
    stable_points[
        "minimum_dry_stability_margin"
    ],

    stable_points[
        "flutter_improvement_pct"
    ],

    label=
        "Stable for Padé 1/2/3",
)


plt.scatter(
    unstable_points[
        "minimum_dry_stability_margin"
    ],

    unstable_points[
        "flutter_improvement_pct"
    ],

    marker="x",

    label=
        "Delay-model instability",
)


plt.axhline(
    passive_reference_improvement,

    linestyle="--",

    linewidth=1.0,

    label=
        "Passive reference",
)


plt.axvline(
    0.0,

    linewidth=1.0,
)


plt.xlabel(
    "Minimum dry closed-loop stability margin (1/s)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Active-control performance versus robustness"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


print(
    "\nCell 57 robust practical active-controller "
    "screen completed."
)

In [ ]:
# ============================================================
# CELL 58 — HYBRID PASSIVE + ACTIVE FLUTTER TRADE SPACE
# ============================================================
#
# ENGINEERING QUESTION
# --------------------
#
# Can modest passive torsional reinforcement:
#
#       provide fail-safe flutter margin
#
# while a REDUCED-authority active controller:
#
#       provides the remaining performance?
#
#
# TARGET:
#
#       approximately +6% flutter velocity
#
#
# Reference robust active architecture:
#
#       actuator bandwidth = 25 Hz
#       total delay        = 5 ms
#
#
# IMPORTANT:
#
# Passive GJ gain and mass are linked here only through a
# NOTIONAL trade-study scaling:
#
#       +20% GJ proxy  <->  +2% panel mass
#
# NX will later replace this assumption using real geometry.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import contextlib


# ============================================================
# 1. FREEZE ROBUST ACTIVE REFERENCE
# ============================================================

ROBUST_ACTIVE_BANDWIDTH_HZ = 25.0

ROBUST_ACTIVE_DELAY_MS = 5.0

ROBUST_ACTIVE_DELAY_S = (
    ROBUST_ACTIVE_DELAY_MS
    / 1000.0
)


ROBUST_ACTIVE_K_THETA = (
    ACTIVE_REFERENCE_K_THETA
)


ROBUST_ACTIVE_IMPROVEMENT = (
    float(
        active_robustness_screen[
            np.isclose(
                active_robustness_screen[
                    "bandwidth_Hz"
                ],
                ROBUST_ACTIVE_BANDWIDTH_HZ,
            )
            &
            np.isclose(
                active_robustness_screen[
                    "delay_ms"
                ],
                ROBUST_ACTIVE_DELAY_MS,
            )
        ][
            "flutter_improvement_pct"
        ]
        .iloc[0]
    )
)


print(
    "HYBRID PASSIVE + ACTIVE DESIGN STUDY"
)


print(
    "\nRobust active reference:"
)


print(
    f"K_theta = "
    f"{ROBUST_ACTIVE_K_THETA:.6f} N m/rad"
)


print(
    f"Bandwidth = "
    f"{ROBUST_ACTIVE_BANDWIDTH_HZ:.1f} Hz"
)


print(
    f"Delay = "
    f"{ROBUST_ACTIVE_DELAY_MS:.1f} ms"
)


print(
    f"Active-only flutter improvement = "
    f"{ROBUST_ACTIVE_IMPROVEMENT:+.3f}%"
)


# ============================================================
# 2. HYBRID DESIGN VARIABLES
# ============================================================

hybrid_GJ_gain_values = np.array([
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
])


hybrid_active_fraction_values = np.array([
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
])


HYBRID_TARGET_IMPROVEMENT = (
    6.0
)


# ------------------------------------------------------------
# Notional mass/GJ scaling:
#
#       20% GJ -> 2% panel mass
#
# therefore:
#
#       mass_fraction = 0.1 * GJ_gain
# ------------------------------------------------------------

def notional_mass_fraction_from_GJ(
    GJ_gain,
):

    return (
        0.10
        * float(
            GJ_gain
        )
    )


# ============================================================
# 3. GENERIC DYNAMIC-ACTIVE FROZEN-k SOLVER
#    SUPPORTING MODIFIED PASSIVE STRUCTURE
# ============================================================

def frozen_k_pk_dynamic_hybrid(
    Ma,
    U,
    q_dyn,
    k_PA,
    M_struct,
    C_struct,
    K_struct,
    K_theta,
    actuator_bandwidth_Hz,
    delay_s,
):


    omega = (
        float(k_PA)
        * float(U)
    )


    if omega <= 0.0:

        raise ValueError(
            "omega must be positive."
        )


    # --------------------------------------------------------
    # Aerodynamics
    # --------------------------------------------------------

    Qhh = (
        interpolate_cached_Qhh(
            Ma=Ma,
            k_query=k_PA,
        )
    )


    QR = np.real(
        Qhh
    )


    QI = np.imag(
        Qhh
    )


    # --------------------------------------------------------
    # Dynamic controller
    # --------------------------------------------------------

    H_controller = (
        actuator_transfer_function(
            omega=
                omega,

            bandwidth_Hz=
                actuator_bandwidth_Hz,

            delay_s=
                delay_s,
        )
    )


    K_controller_complex = (
        K_theta
        * H_controller
        * BBT_active
    )


    Kc_real = np.real(
        K_controller_complex
    )


    Kc_imag = np.imag(
        K_controller_complex
    )


    # --------------------------------------------------------
    # Frozen-k p-k matrices
    # --------------------------------------------------------

    K_eff = (
        K_struct
        + Kc_real
        - q_dyn
        * QR
    )


    C_eff = (
        C_struct

        + Kc_imag
        / omega

        - q_dyn
        * QI
        / omega
    )


    M_inv = np.linalg.inv(
        M_struct
    )


    Z = np.zeros(
        (4, 4)
    )


    I = np.eye(
        4
    )


    A = np.block([
        [
            Z,
            I,
        ],

        [
            -M_inv @ K_eff,
            -M_inv @ C_eff,
        ],
    ])


    poles, eigvectors = (
        np.linalg.eig(
            A
        )
    )


    return (
        poles,
        eigvectors,
        H_controller,
    )


# ============================================================
# 4. MODE-1 HYBRID p-k ITERATION
# ============================================================

def pk_mode1_dynamic_hybrid(
    U,
    q_dyn,
    M_struct,
    C_struct,
    K_struct,
    K_theta,
    bandwidth_Hz,
    delay_s,
    initial_k=None,
    initial_reference=None,
    tol=2e-7,
    max_iter=80,
    relaxation=0.65,
):


    if initial_k is None:

        k_current = (
            omega_analysis[0]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    if initial_reference is None:

        reference_q = np.array(
            [
                1.0,
                0.0,
                0.0,
                0.0,
            ],
            dtype=complex,
        )

    else:

        reference_q = np.asarray(
            initial_reference,
            dtype=complex,
        ).copy()


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M_struct,
        )
    )


    selected_H = (
        1.0
        + 0.0j
    )


    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
            H_controller,
        ) = (
            frozen_k_pk_dynamic_hybrid(
                Ma=
                    PASSIVE_DESIGN_MACH,

                U=
                    U,

                q_dyn=
                    q_dyn,

                k_PA=
                    k_current,

                M_struct=
                    M_struct,

                C_struct=
                    C_struct,

                K_struct=
                    K_struct,

                K_theta=
                    K_theta,

                actuator_bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,
            )
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M_struct,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M_struct,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_idx = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_idx
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        MAC_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M_struct,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M_struct,
            )
        )


        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        selected_H = (
            H_controller
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            "Hybrid p-k iteration did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {

        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            MAC_selected,

        "iterations":
            iteration,

        "modal_vector":
            q_selected.copy(),

        "controller_magnitude":
            abs(
                selected_H
            ),

        "controller_phase_deg":
            np.angle(
                selected_H,
                deg=True,
            ),
    }


# ============================================================
# 5. HYBRID FLUTTER ROOT
# ============================================================

def solve_hybrid_flutter_root(
    M_struct,
    C_struct,
    K_struct,
    K_theta,
    bandwidth_Hz=
        ROBUST_ACTIVE_BANDWIDTH_HZ,
    delay_s=
        ROBUST_ACTIVE_DELAY_S,
    lambda_guess=None,
):


    if lambda_guess is None:

        lambda_guess = (
            lambda_flutter_baseline
        )


    def evaluate_lambda(
        lam,
        reference=None,
    ):


        U = (
            lam
            * U_exp_passive
        )


        q = (
            lam**2
            * q_exp_passive
        )


        if reference is None:

            initial_k = None
            initial_reference = None

        else:

            initial_k = (
                reference[
                    "k_PA"
                ]
            )

            initial_reference = (
                reference[
                    "modal_vector"
                ]
            )


        result = (
            pk_mode1_dynamic_hybrid(
                U=
                    U,

                q_dyn=
                    q,

                M_struct=
                    M_struct,

                C_struct=
                    C_struct,

                K_struct=
                    K_struct,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,

                initial_k=
                    initial_k,

                initial_reference=
                    initial_reference,
            )
        )


        return {

            "lambda":
                float(
                    lam
                ),

            "U":
                float(
                    U
                ),

            "q":
                float(
                    q
                ),

            **result,
        }


    # --------------------------------------------------------
    # Bracket
    # --------------------------------------------------------

    half_width = (
        0.10
    )


    for expansion in range(
        12
    ):


        lam_low = max(
            0.55,
            lambda_guess
            - half_width,
        )


        lam_high = min(
            1.90,
            lambda_guess
            + half_width,
        )


        low = (
            evaluate_lambda(
                lam_low
            )
        )


        high = (
            evaluate_lambda(
                lam_high,
                reference=low,
            )
        )


        if (
            low[
                "sigma"
            ]
            * high[
                "sigma"
            ]
            <= 0.0
        ):

            break


        half_width += (
            0.06
        )


    else:

        raise RuntimeError(
            "Could not bracket hybrid flutter root."
        )


    if (
        low[
            "sigma"
        ]
        > 0.0
    ):

        low, high = (
            high,
            low,
        )


    # --------------------------------------------------------
    # Root refinement
    # --------------------------------------------------------

    for root_iteration in range(
        1,
        26
    ):


        sig_low = (
            low[
                "sigma"
            ]
        )


        sig_high = (
            high[
                "sigma"
            ]
        )


        lam_trial = (
            low[
                "lambda"
            ]

            - sig_low
            * (
                high[
                    "lambda"
                ]
                - low[
                    "lambda"
                ]
            )

            / (
                sig_high
                - sig_low
            )
        )


        lower = min(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        upper = max(
            low[
                "lambda"
            ],
            high[
                "lambda"
            ],
        )


        width = (
            upper
            - lower
        )


        if (
            lam_trial
            <= lower
            + 0.05
            * width

            or

            lam_trial
            >= upper
            - 0.05
            * width
        ):

            lam_trial = (
                0.5
                * (
                    low[
                        "lambda"
                    ]
                    + high[
                        "lambda"
                    ]
                )
            )


        reference = (
            low

            if abs(
                lam_trial
                - low[
                    "lambda"
                ]
            )

            <=

            abs(
                high[
                    "lambda"
                ]
                - lam_trial
            )

            else high
        )


        trial = (
            evaluate_lambda(
                lam_trial,
                reference=reference,
            )
        )


        if (
            abs(
                trial[
                    "sigma"
                ]
            )
            < 1e-4
        ):

            return trial


        if (
            trial[
                "sigma"
            ]
            < 0.0
        ):

            low = (
                trial
            )

        else:

            high = (
                trial
            )


    return min(
        [
            low,
            high,
        ],

        key=lambda x:
            abs(
                x[
                    "sigma"
                ]
            ),
    )


# ============================================================
# 6. GENERIC DRY CLOSED-LOOP HYBRID CHECK
# ============================================================

def dry_closed_loop_hybrid(
    M_struct,
    C_struct,
    K_struct,
    K_theta,
    bandwidth_Hz,
    delay_s,
    pade_order,
):


    (
        A_h,
        B_h,
        C_h,
        D_h,
    ) = (
        actuator_controller_state_space(
            bandwidth_Hz=
                bandwidth_Hz,

            delay_s=
                delay_s,

            pade_order=
                pade_order,
        )
    )


    n_ctrl = (
        A_h.shape[0]
    )


    M_inv = np.linalg.inv(
        M_struct
    )


    B_col = (
        B_active.reshape(
            -1,
            1,
        )
    )


    B_row = (
        B_active.reshape(
            1,
            -1,
        )
    )


    D_scalar = float(
        D_h[
            0,
            0
        ]
    )


    A_vq = (
        -M_inv
        @ (
            K_struct

            + K_theta
            * D_scalar
            * (
                B_col
                @ B_row
            )
        )
    )


    A_vv = (
        -M_inv
        @ C_struct
    )


    A_vx = (
        -M_inv
        @ (
            K_theta
            * B_col
            @ C_h
        )
    )


    A_xq = (
        B_h
        @ B_row
    )


    Z44 = np.zeros(
        (4, 4)
    )


    I44 = np.eye(
        4
    )


    Z4c = np.zeros(
        (
            4,
            n_ctrl,
        )
    )


    Zc4 = np.zeros(
        (
            n_ctrl,
            4,
        )
    )


    A_closed = np.block([

        [
            Z44,
            I44,
            Z4c,
        ],

        [
            A_vq,
            A_vv,
            A_vx,
        ],

        [
            A_xq,
            Zc4,
            A_h,
        ],
    ])


    poles = np.linalg.eigvals(
        A_closed
    )


    max_real = float(
        np.max(
            np.real(
                poles
            )
        )
    )


    return {

        "stable":
            bool(
                max_real
                < 0.0
            ),

        "max_real_pole":
            max_real,

        "stability_margin":
            -max_real,
    }


# ============================================================
# 7. PASSIVE-ONLY REFERENCE FOR EACH GJ LEVEL
# ============================================================

passive_level_results = {}


for GJ_gain in hybrid_GJ_gain_values:


    mass_fraction = (
        notional_mass_fraction_from_GJ(
            GJ_gain
        )
    )


    design = (
        build_passive_modal_matrices_v2(
            eta_start=
                0.35,

            eta_end=
                0.55,

            bending_stiffness_gain=
                0.0,

            torsional_stiffness_gain=
                GJ_gain,

            added_mass_fraction=
                mass_fraction,
        )
    )


    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        passive_flutter = (
            solve_passive_flutter_root(
                M_struct=
                    design[
                        "M"
                    ],

                C_struct=
                    design[
                        "C"
                    ],

                K_struct=
                    design[
                        "K"
                    ],

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    passive_improvement = (
        100.0
        * (
            passive_flutter[
                "U"
            ]
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    passive_level_results[
        float(
            GJ_gain
        )
    ] = {

        "design":
            design,

        "flutter":
            passive_flutter,

        "improvement_pct":
            passive_improvement,

        "mass_fraction":
            mass_fraction,
    }


# ============================================================
# 8. HYBRID GRID
# ============================================================

hybrid_rows = []


for GJ_gain in hybrid_GJ_gain_values:


    passive_data = (
        passive_level_results[
            float(
                GJ_gain
            )
        ]
    )


    design = (
        passive_data[
            "design"
        ]
    )


    passive_improvement = float(
        passive_data[
            "improvement_pct"
        ]
    )


    mass_fraction = float(
        passive_data[
            "mass_fraction"
        ]
    )


    added_mass_kg = (
        mass_fraction
        * BASELINE_PANEL_MASS_KG
    )


    for active_fraction in (
        hybrid_active_fraction_values
    ):


        K_theta_hybrid = (
            active_fraction
            * ROBUST_ACTIVE_K_THETA
        )


        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            result = (
                solve_hybrid_flutter_root(
                    M_struct=
                        design[
                            "M"
                        ],

                    C_struct=
                        design[
                            "C"
                        ],

                    K_struct=
                        design[
                            "K"
                        ],

                    K_theta=
                        K_theta_hybrid,
                )
            )


        hybrid_improvement = (
            100.0
            * (
                result[
                    "U"
                ]
                - U_flutter_baseline
            )
            / U_flutter_baseline
        )


        # ----------------------------------------------------
        # Dry robustness under Padé 1/2/3
        # ----------------------------------------------------

        dry_results = []


        for pade_order in [
            1,
            2,
            3,
        ]:


            dry_result = (
                dry_closed_loop_hybrid(
                    M_struct=
                        design[
                            "M"
                        ],

                    C_struct=
                        design[
                            "C"
                        ],

                    K_struct=
                        design[
                            "K"
                        ],

                    K_theta=
                        K_theta_hybrid,

                    bandwidth_Hz=
                        ROBUST_ACTIVE_BANDWIDTH_HZ,

                    delay_s=
                        ROBUST_ACTIVE_DELAY_S,

                    pade_order=
                        pade_order,
                )
            )


            dry_results.append(
                dry_result
            )


        stable_all_pade = bool(
            all(
                x[
                    "stable"
                ]
                for x
                in dry_results
            )
        )


        minimum_dry_margin = float(
            min(
                x[
                    "stability_margin"
                ]
                for x
                in dry_results
            )
        )


        # ----------------------------------------------------
        # Active authority reduction
        # ----------------------------------------------------

        active_authority_reduction = (
            100.0
            * (
                1.0
                - active_fraction
            )
        )


        commanded_torque_per_deg = (
            K_theta_hybrid
            * np.deg2rad(
                1.0
            )
        )


        hybrid_rows.append({

            "GJ_gain_pct":
                100.0
                * GJ_gain,

            "mass_fraction_pct":
                100.0
                * mass_fraction,

            "added_mass_kg":
                added_mass_kg,

            "passive_fail_safe_improvement_pct":
                passive_improvement,

            "active_fraction":
                active_fraction,

            "active_authority_reduction_pct":
                active_authority_reduction,

            "K_theta_Nm_per_rad":
                K_theta_hybrid,

            "commanded_torque_per_deg_Nm":
                commanded_torque_per_deg,

            "flutter_velocity_m_s":
                result[
                    "U"
                ],

            "flutter_frequency_Hz":
                result[
                    "frequency_Hz"
                ],

            "hybrid_improvement_pct":
                hybrid_improvement,

            "controller_magnitude":
                result[
                    "controller_magnitude"
                ],

            "controller_phase_deg":
                result[
                    "controller_phase_deg"
                ],

            "stable_all_pade":
                stable_all_pade,

            "minimum_dry_stability_margin":
                minimum_dry_margin,

            "meets_6pct_target":
                (
                    hybrid_improvement
                    >= HYBRID_TARGET_IMPROVEMENT
                ),
        })


# ============================================================
# 9. RESULTS
# ============================================================

hybrid_trade = (
    pd.DataFrame(
        hybrid_rows
    )
)


print(
    "\nHYBRID TRADE SPACE"
)


display(
    hybrid_trade
)


# ============================================================
# 10. CONSISTENCY CHECK:
#     zero active authority must reproduce passive-only result
# ============================================================

zero_active = (
    hybrid_trade[
        np.isclose(
            hybrid_trade[
                "active_fraction"
            ],
            0.0,
        )
    ]
)


hybrid_passive_consistency_error = (
    (
        zero_active[
            "hybrid_improvement_pct"
        ]
        -
        zero_active[
            "passive_fail_safe_improvement_pct"
        ]
    )
    .abs()
    .max()
)


print(
    "\nZero-active consistency error:"
)


print(
    f"{hybrid_passive_consistency_error:.6e} percentage points"
)


# ============================================================
# 11. DESIGNS MEETING TARGET + DRY ROBUSTNESS
# ============================================================

hybrid_target_candidates = (
    hybrid_trade[
        (
            hybrid_trade[
                "meets_6pct_target"
            ]
            == True
        )
        &
        (
            hybrid_trade[
                "stable_all_pade"
            ]
            == True
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Sort FIRST by lowest active authority,
# then lowest added mass,
# then larger dry stability margin.
#
# This is NOT a universal optimum.
# It is a requirement-oriented shortlist.
# ------------------------------------------------------------

hybrid_target_candidates = (
    hybrid_target_candidates
    .sort_values(
        [
            "active_fraction",
            "added_mass_kg",
            "minimum_dry_stability_margin",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nROBUST HYBRID DESIGNS MEETING +6% TARGET"
)


display(
    hybrid_target_candidates[
        [
            "GJ_gain_pct",
            "mass_fraction_pct",
            "passive_fail_safe_improvement_pct",
            "active_fraction",
            "active_authority_reduction_pct",
            "K_theta_Nm_per_rad",
            "commanded_torque_per_deg_Nm",
            "hybrid_improvement_pct",
            "minimum_dry_stability_margin",
        ]
    ]
)


# ============================================================
# 12. HYBRID PERFORMANCE MAP
# ============================================================

hybrid_pivot = (
    hybrid_trade
    .pivot(
        index=
            "active_fraction",

        columns=
            "GJ_gain_pct",

        values=
            "hybrid_improvement_pct",
    )
)


plt.figure(
    figsize=(9, 5)
)


plt.imshow(
    hybrid_pivot.to_numpy(),
    aspect="auto",
    origin="lower",
)


plt.colorbar(
    label=
        "Flutter-velocity improvement (%)"
)


plt.xticks(
    np.arange(
        len(
            hybrid_pivot.columns
        )
    ),

    [
        f"{x:.0f}"
        for x
        in hybrid_pivot.columns
    ],
)


plt.yticks(
    np.arange(
        len(
            hybrid_pivot.index
        )
    ),

    [
        f"{100*x:.0f}%"
        for x
        in hybrid_pivot.index
    ],
)


plt.xlabel(
    "Local passive torsional-stiffness gain (%)"
)


plt.ylabel(
    "Active authority fraction"
)


plt.title(
    "Hybrid passive-active flutter-margin trade space"
)


plt.tight_layout()

plt.show()


# ============================================================
# 13. FAIL-SAFE MARGIN VERSUS ACTIVE AUTHORITY
# ============================================================

plt.figure(
    figsize=(9, 5)
)


for GJ_gain in hybrid_GJ_gain_values:


    subset = (
        hybrid_trade[
            np.isclose(
                hybrid_trade[
                    "GJ_gain_pct"
                ],
                100.0
                * GJ_gain,
            )
        ]
        .sort_values(
            "active_fraction"
        )
    )


    plt.plot(
        100.0
        * subset[
            "active_fraction"
        ],

        subset[
            "hybrid_improvement_pct"
        ],

        "o-",

        label=
            f"+{100*GJ_gain:.0f}% GJ proxy",
    )


plt.axhline(
    HYBRID_TARGET_IMPROVEMENT,

    linestyle="--",

    linewidth=1.0,

    label="+6% target",
)


plt.axhline(
    passive_reference_improvement,

    linestyle=":",

    linewidth=1.0,

    label="Passive reference",
)


plt.xlabel(
    "Active controller authority (% of robust active reference)"
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "How passive tailoring reduces active-control demand"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# 14. SAVE SHORTLIST FOR NEXT CELL
# ============================================================

if len(
    hybrid_target_candidates
) > 0:


    HYBRID_SHORTLIST = (
        hybrid_target_candidates
        .head(
            min(
                3,
                len(
                    hybrid_target_candidates
                ),
            )
        )
        .copy()
    )


    print(
        "\nHYBRID SHORTLIST FOR ALL-BRANCH VERIFICATION"
    )


    display(
        HYBRID_SHORTLIST
    )


else:


    HYBRID_SHORTLIST = pd.DataFrame()


    print(
        "\nNo investigated hybrid configuration "
        "met the +6% robust target."
    )


print(
    "\nCell 58 hybrid passive-active trade study completed."
)

In [ ]:
# ============================================================
# CELL 59 — HYBRID ALL-BRANCH + ACTIVE-FAILURE VERIFICATION
# ============================================================
#
# PURPOSE
# -------
#
# Verify the shortlisted hybrid architectures from Cell 58.
#
# For each candidate:
#
#   1. verify all four aeroelastic branches at the HYBRID
#      flutter boundary;
#
#   2. confirm dry closed-loop stability under Padé
#      [1/1], [2/2], [3/3];
#
#   3. calculate the controller-OFF flutter boundary;
#
#   4. quantify retained fail-safe flutter margin;
#
#   5. compare active torque / authority requirement.
#
#
# IMPORTANT:
#
# "Controller failure" here means:
#
#       K_theta -> 0
#
# followed by calculation of the new STEADY flutter boundary.
#
# This is NOT a transient failure-response simulation.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import contextlib


# ============================================================
# PART A — GENERIC HYBRID ALL-BRANCH p-k TRACKER
# ============================================================

def pk_dynamic_hybrid_branch(
    mode_index,
    U,
    q_dyn,
    M_struct,
    C_struct,
    K_struct,
    K_theta,
    bandwidth_Hz,
    delay_s,
    initial_k=None,
    tol=2e-7,
    max_iter=80,
    relaxation=0.65,
):
    """
    Track one aeroelastic branch for a modified passive
    structure plus frequency-dependent active controller.
    """

    if not (
        0
        <= mode_index
        < 4
    ):

        raise ValueError(
            "mode_index must be 0,1,2,3."
        )


    if initial_k is None:

        k_current = (
            omega_analysis[
                mode_index
            ]
            / U
        )

    else:

        k_current = float(
            initial_k
        )


    reference_q = np.zeros(
        4,
        dtype=complex,
    )


    reference_q[
        mode_index
    ] = 1.0


    reference_q = (
        mass_normalize_modal_vector(
            reference_q,
            M_struct,
        )
    )


    selected_H = (
        1.0
        + 0.0j
    )


    for iteration in range(
        1,
        max_iter + 1
    ):


        (
            poles,
            eigvectors,
            H_controller,
        ) = (
            frozen_k_pk_dynamic_hybrid(
                Ma=
                    PASSIVE_DESIGN_MACH,

                U=
                    U,

                q_dyn=
                    q_dyn,

                k_PA=
                    k_current,

                M_struct=
                    M_struct,

                C_struct=
                    C_struct,

                K_struct=
                    K_struct,

                K_theta=
                    K_theta,

                actuator_bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,
            )
        )


        positive_indices = np.where(
            np.imag(
                poles
            )
            > 0.0
        )[0]


        if len(
            positive_indices
        ) == 0:

            raise RuntimeError(
                "No positive-frequency poles found."
            )


        candidate_macs = []

        candidate_vectors = []


        for idx in positive_indices:


            q_candidate = (
                eigvectors[
                    :4,
                    idx,
                ]
            )


            q_candidate = (
                mass_normalize_modal_vector(
                    q_candidate,
                    M_struct,
                )
            )


            mac = (
                complex_modal_mac(
                    reference_q,
                    q_candidate,
                    M_struct,
                )
            )


            candidate_macs.append(
                mac
            )


            candidate_vectors.append(
                q_candidate
            )


        best = int(
            np.argmax(
                candidate_macs
            )
        )


        selected_idx = (
            positive_indices[
                best
            ]
        )


        pole = (
            poles[
                selected_idx
            ]
        )


        q_selected = (
            candidate_vectors[
                best
            ]
        )


        MAC_selected = float(
            candidate_macs[
                best
            ]
        )


        q_selected = (
            phase_align_modal_vector(
                q_selected,
                reference_q,
                M_struct,
            )
        )


        q_selected = (
            mass_normalize_modal_vector(
                q_selected,
                M_struct,
            )
        )


        omega_new = float(
            np.imag(
                pole
            )
        )


        k_raw = (
            omega_new
            / U
        )


        k_next = (
            relaxation
            * k_raw

            + (
                1.0
                - relaxation
            )
            * k_current
        )


        relative_change = (
            abs(
                k_next
                - k_current
            )
            /
            max(
                abs(
                    k_current
                ),
                1e-14,
            )
        )


        reference_q = (
            q_selected.copy()
        )


        k_current = (
            k_next
        )


        selected_H = (
            H_controller
        )


        if (
            relative_change
            < tol
        ):

            break


    else:

        raise RuntimeError(
            f"Hybrid branch {mode_index+1} "
            "did not converge."
        )


    sigma = float(
        np.real(
            pole
        )
    )


    omega = float(
        np.imag(
            pole
        )
    )


    return {

        "mode_id":
            mode_index + 1,

        "sigma":
            sigma,

        "frequency_Hz":
            omega
            / (
                2.0
                * np.pi
            ),

        "damping_ratio":
            -sigma
            / abs(
                pole
            ),

        "k_PA":
            omega
            / U,

        "MAC":
            MAC_selected,

        "iterations":
            iteration,

        "controller_magnitude":
            abs(
                selected_H
            ),

        "controller_phase_deg":
            np.angle(
                selected_H,
                deg=True,
            ),
    }


# ============================================================
# PART B — PREPARE SHORTLIST STRUCTURAL MODELS
# ============================================================

if len(
    HYBRID_SHORTLIST
) == 0:

    raise RuntimeError(
        "HYBRID_SHORTLIST from Cell 58 is empty."
    )


hybrid_verification_rows = []

hybrid_branch_rows = []


# ============================================================
# PART C — VERIFY EVERY SHORTLIST CANDIDATE
# ============================================================

for shortlist_index, candidate in (
    HYBRID_SHORTLIST
    .reset_index(
        drop=True
    )
    .iterrows()
):


    candidate_id = (
        shortlist_index
        + 1
    )


    GJ_gain_pct = float(
        candidate[
            "GJ_gain_pct"
        ]
    )


    GJ_gain = (
        GJ_gain_pct
        / 100.0
    )


    mass_fraction = (
        float(
            candidate[
                "mass_fraction_pct"
            ]
        )
        / 100.0
    )


    active_fraction = float(
        candidate[
            "active_fraction"
        ]
    )


    K_theta = float(
        candidate[
            "K_theta_Nm_per_rad"
        ]
    )


    U_hybrid = float(
        candidate[
            "flutter_velocity_m_s"
        ]
    )


    q_hybrid = (
        (
            U_hybrid
            / U_exp_passive
        )**2
        * q_exp_passive
    )


    # --------------------------------------------------------
    # Rebuild passive structural component explicitly
    # --------------------------------------------------------

    design = (
        build_passive_modal_matrices_v2(
            eta_start=
                0.35,

            eta_end=
                0.55,

            bending_stiffness_gain=
                0.0,

            torsional_stiffness_gain=
                GJ_gain,

            added_mass_fraction=
                mass_fraction,
        )
    )


    M_hybrid = (
        design[
            "M"
        ]
    )


    C_hybrid = (
        design[
            "C"
        ]
    )


    K_hybrid = (
        design[
            "K"
        ]
    )


    print(
        "\n--------------------------------------------"
    )


    print(
        f"HYBRID SHORTLIST {candidate_id}"
    )


    print(
        f"GJ gain          = {GJ_gain_pct:.1f}%"
    )


    print(
        f"Added mass       = "
        f"{100*mass_fraction:.1f}% panel mass"
    )


    print(
        f"Active authority = "
        f"{100*active_fraction:.1f}%"
    )


    print(
        f"Hybrid U_flutter = "
        f"{U_hybrid:.3f} m/s"
    )


    # ========================================================
    # 1. ALL-BRANCH AEROELASTIC VERIFICATION
    # ========================================================

    candidate_branch_results = []


    for mode_index in range(
        4
    ):


        print(
            f"  Solving hybrid branch "
            f"{mode_index+1}..."
        )


        result = (
            pk_dynamic_hybrid_branch(
                mode_index=
                    mode_index,

                U=
                    U_hybrid,

                q_dyn=
                    q_hybrid,

                M_struct=
                    M_hybrid,

                C_struct=
                    C_hybrid,

                K_struct=
                    K_hybrid,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    ROBUST_ACTIVE_BANDWIDTH_HZ,

                delay_s=
                    ROBUST_ACTIVE_DELAY_S,
            )
        )


        candidate_branch_results.append(
            result
        )


        hybrid_branch_rows.append({

            "candidate_id":
                candidate_id,

            "GJ_gain_pct":
                GJ_gain_pct,

            "active_fraction":
                active_fraction,

            **result,
        })


    branch_df = (
        pd.DataFrame(
            candidate_branch_results
        )
    )


    mode1_sigma = float(
        branch_df[
            branch_df[
                "mode_id"
            ]
            == 1
        ][
            "sigma"
        ]
        .iloc[0]
    )


    max_other_sigma = float(
        branch_df[
            branch_df[
                "mode_id"
            ]
            != 1
        ][
            "sigma"
        ]
        .max()
    )


    mode1_neutral = bool(
        abs(
            mode1_sigma
        )
        < 1e-3
    )


    other_branches_stable = bool(
        max_other_sigma
        < 0.0
    )


    # ========================================================
    # 2. DRY CLOSED-LOOP ROBUSTNESS
    # ========================================================

    dry_results = []


    for pade_order in [
        1,
        2,
        3,
    ]:


        dry_result = (
            dry_closed_loop_hybrid(
                M_struct=
                    M_hybrid,

                C_struct=
                    C_hybrid,

                K_struct=
                    K_hybrid,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    ROBUST_ACTIVE_BANDWIDTH_HZ,

                delay_s=
                    ROBUST_ACTIVE_DELAY_S,

                pade_order=
                    pade_order,
            )
        )


        dry_results.append(
            dry_result
        )


    dry_stable_all = bool(
        all(
            x[
                "stable"
            ]
            for x
            in dry_results
        )
    )


    min_dry_margin = float(
        min(
            x[
                "stability_margin"
            ]
            for x
            in dry_results
        )
    )


    # ========================================================
    # 3. CONTROLLER-OFF / FAIL-SAFE FLUTTER BOUNDARY
    # ========================================================

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        fail_safe_flutter = (
            solve_passive_flutter_root(
                M_struct=
                    M_hybrid,

                C_struct=
                    C_hybrid,

                K_struct=
                    K_hybrid,

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    U_fail_safe = float(
        fail_safe_flutter[
            "U"
        ]
    )


    fail_safe_improvement = (
        100.0
        * (
            U_fail_safe
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    # --------------------------------------------------------
    # Boundary lost when active control disappears
    # --------------------------------------------------------

    active_loss_velocity_drop = (
        U_hybrid
        - U_fail_safe
    )


    active_loss_velocity_drop_pct = (
        100.0
        * active_loss_velocity_drop
        / U_hybrid
    )


    # ========================================================
    # 4. ACTUATOR AUTHORITY
    # ========================================================

    torque_per_deg = (
        K_theta
        * np.deg2rad(
            1.0
        )
    )


    authority_reduction_pct = (
        100.0
        * (
            1.0
            - active_fraction
        )
    )


    # ========================================================
    # 5. OVERALL VERIFICATION
    # ========================================================

    fully_verified = bool(
        mode1_neutral
        and
        other_branches_stable
        and
        dry_stable_all
        and
        fail_safe_improvement
        > 0.0
    )


    hybrid_verification_rows.append({

        "candidate_id":
            candidate_id,

        "GJ_gain_pct":
            GJ_gain_pct,

        "added_mass_kg":
            float(
                candidate[
                    "added_mass_kg"
                ]
            ),

        "active_fraction":
            active_fraction,

        "active_authority_reduction_pct":
            authority_reduction_pct,

        "K_theta_Nm_per_rad":
            K_theta,

        "torque_per_deg_Nm":
            torque_per_deg,

        "hybrid_flutter_velocity_m_s":
            U_hybrid,

        "hybrid_improvement_pct":
            float(
                candidate[
                    "hybrid_improvement_pct"
                ]
            ),

        "controller_off_flutter_velocity_m_s":
            U_fail_safe,

        "controller_off_improvement_pct":
            fail_safe_improvement,

        "flutter_velocity_lost_on_control_failure_m_s":
            active_loss_velocity_drop,

        "flutter_boundary_drop_on_failure_pct":
            active_loss_velocity_drop_pct,

        "mode1_sigma":
            mode1_sigma,

        "max_other_branch_sigma":
            max_other_sigma,

        "minimum_dry_stability_margin":
            min_dry_margin,

        "mode1_neutral":
            mode1_neutral,

        "other_branches_stable":
            other_branches_stable,

        "dry_stable_pade123":
            dry_stable_all,

        "fully_verified":
            fully_verified,
    })


# ============================================================
# PART D — RESULTS TABLES
# ============================================================

hybrid_all_branch_table = (
    pd.DataFrame(
        hybrid_branch_rows
    )
)


hybrid_verification_table = (
    pd.DataFrame(
        hybrid_verification_rows
    )
)


print(
    "\nHYBRID ALL-BRANCH RESULTS"
)


display(
    hybrid_all_branch_table
)


print(
    "\nHYBRID ARCHITECTURE VERIFICATION"
)


display(
    hybrid_verification_table
)


# ============================================================
# PART E — VERIFIED CANDIDATES
# ============================================================

verified_hybrids = (
    hybrid_verification_table[
        hybrid_verification_table[
            "fully_verified"
        ]
        == True
    ]
    .copy()
)


if len(
    verified_hybrids
) == 0:

    raise RuntimeError(
        "No hybrid shortlist candidate passed "
        "all verification checks."
    )


# ============================================================
# PART F — REQUIREMENT-ORIENTED SELECTION
# ============================================================
#
# We are NOT maximizing flutter improvement.
#
# Priority:
#
#   1. at least +6% hybrid benefit;
#   2. largest reduction in active authority;
#   3. largest fail-safe passive margin;
#   4. lower physical added mass.
#
# ============================================================

verified_hybrids = (
    verified_hybrids[
        verified_hybrids[
            "hybrid_improvement_pct"
        ]
        >= 6.0
    ]
    .copy()
)


verified_hybrids = (
    verified_hybrids
    .sort_values(
        [
            "active_authority_reduction_pct",
            "controller_off_improvement_pct",
            "added_mass_kg",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nVERIFIED HYBRIDS MEETING +6% REQUIREMENT"
)


display(
    verified_hybrids
)


# ============================================================
# PART G — SELECT PROVISIONAL HYBRID REFERENCE
# ============================================================

HYBRID_REFERENCE = (
    verified_hybrids
    .iloc[0]
)


HYBRID_REFERENCE_GJ_GAIN_PCT = float(
    HYBRID_REFERENCE[
        "GJ_gain_pct"
    ]
)


HYBRID_REFERENCE_ACTIVE_FRACTION = float(
    HYBRID_REFERENCE[
        "active_fraction"
    ]
)


HYBRID_REFERENCE_K_THETA = float(
    HYBRID_REFERENCE[
        "K_theta_Nm_per_rad"
    ]
)


print(
    "\nPROVISIONAL HYBRID REFERENCE"
)


print(
    f"Passive GJ gain = "
    f"{HYBRID_REFERENCE_GJ_GAIN_PCT:.1f}%"
)


print(
    f"Active authority = "
    f"{100*HYBRID_REFERENCE_ACTIVE_FRACTION:.1f}% "
    "of robust active reference"
)


print(
    f"Active gain K_theta = "
    f"{HYBRID_REFERENCE_K_THETA:.3f} N m/rad"
)


print(
    f"Hybrid flutter improvement = "
    f"{HYBRID_REFERENCE['hybrid_improvement_pct']:+.3f}%"
)


print(
    f"Controller-off flutter improvement = "
    f"{HYBRID_REFERENCE['controller_off_improvement_pct']:+.3f}%"
)


print(
    f"Torque per degree = "
    f"{HYBRID_REFERENCE['torque_per_deg_Nm']:.3f} N m/deg"
)


print(
    f"Minimum dry stability margin = "
    f"{HYBRID_REFERENCE['minimum_dry_stability_margin']:.4f} 1/s"
)


# ============================================================
# PART H — ARCHITECTURE COMPARISON
# ============================================================

architecture_comparison = pd.DataFrame({

    "architecture": [
        "Baseline",
        "Passive reference",
        "Robust active-only",
        "Hybrid reference",
        "Hybrid after active failure",
    ],

    "flutter_improvement_pct": [
        0.0,

        passive_reference_improvement,

        ROBUST_ACTIVE_IMPROVEMENT,

        HYBRID_REFERENCE[
            "hybrid_improvement_pct"
        ],

        HYBRID_REFERENCE[
            "controller_off_improvement_pct"
        ],
    ],

    "active_authority_pct": [
        0.0,
        0.0,
        100.0,

        100.0
        * HYBRID_REFERENCE_ACTIVE_FRACTION,

        0.0,
    ],
})


print(
    "\nARCHITECTURE COMPARISON"
)


display(
    architecture_comparison
)


plt.figure(
    figsize=(10, 5)
)


plt.bar(
    architecture_comparison[
        "architecture"
    ],

    architecture_comparison[
        "flutter_improvement_pct"
    ],
)


plt.axhline(
    6.0,
    linestyle="--",
    linewidth=1.0,
    label="+6% design target",
)


plt.ylabel(
    "Flutter-velocity improvement (%)"
)


plt.title(
    "Passive, active and hybrid architecture comparison"
)


plt.xticks(
    rotation=20,
    ha="right",
)


plt.grid(
    axis="y",
    alpha=0.25,
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# PART I — FAIL-SAFE VERSUS ACTIVE DEMAND
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.scatter(
    verified_hybrids[
        "active_authority_reduction_pct"
    ],

    verified_hybrids[
        "controller_off_improvement_pct"
    ],
    s=80,
)


for _, row in verified_hybrids.iterrows():


    plt.annotate(
        f"+{row['GJ_gain_pct']:.0f}% GJ",

        (
            row[
                "active_authority_reduction_pct"
            ],

            row[
                "controller_off_improvement_pct"
            ],
        ),

        xytext=(
            5,
            5,
        ),

        textcoords=
            "offset points",
    )


plt.xlabel(
    "Reduction in active-control authority (%)"
)


plt.ylabel(
    "Flutter improvement remaining "
    "after active-control loss (%)"
)


plt.title(
    "Hybrid architecture: active demand versus fail-safe margin"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


print(
    "\nCell 59 hybrid all-branch and "
    "active-failure verification completed."
)

In [ ]:
# ============================================================
# CELL 60 — PHYSICS-BASED DOE FOR ML/AI SURROGATE
# ============================================================
#
# OBJECTIVE
# ---------
#
# Generate an independent physics dataset spanning the hybrid
# passive-active design space.
#
# ML WILL NOT generate the aeroelastic truth.
#
# Each training label comes from the reduced-order physics
# model developed and verified in Cells 1–59.
#
#
# DESIGN VARIABLES
#
#   1. reinforcement midpoint eta_mid
#   2. local torsional-stiffness gain
#   3. active-control authority fraction
#   4. actuator bandwidth
#   5. total control delay
#
#
# OUTPUTS / ML TARGETS
#
#   hybrid flutter improvement
#   controller-off / fail-safe improvement
#   minimum dry closed-loop stability margin
#
#
# A selected ML design will later be re-evaluated with the
# physics solver and all four aeroelastic branches.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import contextlib
import time

from scipy.stats import qmc


# ============================================================
# 1. REPRODUCIBLE DOE SETTINGS
# ============================================================

DOE_RANDOM_SEED = 4456

N_PHYSICS_DOE = 80


# Reinforcement width remains fixed at 20% span.
DOE_REINFORCEMENT_WIDTH = 0.20


print(
    "PHYSICS-BASED HYBRID DOE FOR ML"
)

print(
    f"\nNumber of requested physics cases = "
    f"{N_PHYSICS_DOE}"
)

print(
    f"Fixed reinforcement span width = "
    f"{100*DOE_REINFORCEMENT_WIDTH:.1f}%"
)


# ============================================================
# 2. DESIGN-SPACE BOUNDS
# ============================================================
#
# Bounds are intentionally centred around regions already
# shown to be useful by the deterministic study.
#
# eta_mid:
#     0.35 to 0.55
#
# GJ gain:
#     8% to 25%
#
# active fraction:
#     25% to 80% of robust active-only authority
#
# bandwidth:
#     20 to 60 Hz
#
# delay:
#     1 to 8 ms
#
# ============================================================

DOE_BOUNDS = {

    "eta_mid": (
        0.35,
        0.55,
    ),

    "GJ_gain": (
        0.08,
        0.25,
    ),

    "active_fraction": (
        0.25,
        0.80,
    ),

    "bandwidth_Hz": (
        20.0,
        60.0,
    ),

    "delay_ms": (
        1.0,
        8.0,
    ),
}


doe_variable_names = list(
    DOE_BOUNDS.keys()
)


lower_bounds = np.array(
    [
        DOE_BOUNDS[name][0]
        for name
        in doe_variable_names
    ]
)


upper_bounds = np.array(
    [
        DOE_BOUNDS[name][1]
        for name
        in doe_variable_names
    ]
)


print(
    "\nDOE design-variable bounds:"
)


for name in doe_variable_names:

    print(
        f"{name:18s}: "
        f"{DOE_BOUNDS[name][0]} "
        f"to "
        f"{DOE_BOUNDS[name][1]}"
    )


# ============================================================
# 3. LATIN-HYPERCUBE SAMPLE
# ============================================================
#
# Latin hypercube gives better coverage than naïve random
# sampling for the same number of expensive physics cases.
#
# ============================================================

lhs_sampler = (
    qmc.LatinHypercube(
        d=len(
            doe_variable_names
        ),
        seed=DOE_RANDOM_SEED,
    )
)


lhs_unit = (
    lhs_sampler.random(
        n=N_PHYSICS_DOE
    )
)


lhs_physical = (
    qmc.scale(
        lhs_unit,
        lower_bounds,
        upper_bounds,
    )
)


doe_inputs = pd.DataFrame(
    lhs_physical,
    columns=doe_variable_names,
)


# ------------------------------------------------------------
# Convert midpoint -> start/end coordinates
# ------------------------------------------------------------

doe_inputs[
    "eta_start"
] = (
    doe_inputs[
        "eta_mid"
    ]
    - 0.5
    * DOE_REINFORCEMENT_WIDTH
)


doe_inputs[
    "eta_end"
] = (
    doe_inputs[
        "eta_mid"
    ]
    + 0.5
    * DOE_REINFORCEMENT_WIDTH
)


# ------------------------------------------------------------
# Notional mass model
#
# Still NOT a physical NX result.
#
# 20% GJ gain -> 2% panel mass
# ------------------------------------------------------------

doe_inputs[
    "mass_fraction"
] = (
    0.10
    * doe_inputs[
        "GJ_gain"
    ]
)


# ------------------------------------------------------------
# Active gain
# ------------------------------------------------------------

doe_inputs[
    "K_theta_Nm_per_rad"
] = (
    doe_inputs[
        "active_fraction"
    ]
    * ROBUST_ACTIVE_K_THETA
)


# ------------------------------------------------------------
# Added physical mass
# ------------------------------------------------------------

doe_inputs[
    "added_mass_kg"
] = (
    doe_inputs[
        "mass_fraction"
    ]
    * BASELINE_PANEL_MASS_KG
)


# ------------------------------------------------------------
# Torque requirement per degree of sensed regional twist
# ------------------------------------------------------------

doe_inputs[
    "torque_per_deg_Nm"
] = (
    doe_inputs[
        "K_theta_Nm_per_rad"
    ]
    * np.deg2rad(
        1.0
    )
)


print(
    "\nFirst five DOE inputs:"
)


display(
    doe_inputs.head()
)


# ============================================================
# 4. PHYSICS DOE
# ============================================================

physics_doe_rows = []


doe_start_time = time.time()


for case_index, row in doe_inputs.iterrows():


    case_number = (
        case_index
        + 1
    )


    if (
        case_number == 1
        or
        case_number % 10 == 0
        or
        case_number == N_PHYSICS_DOE
    ):

        elapsed = (
            time.time()
            - doe_start_time
        )


        print(
            f"Physics DOE case "
            f"{case_number:3d}/"
            f"{N_PHYSICS_DOE} "
            f"| elapsed "
            f"{elapsed:.1f} s"
        )


    eta_start = float(
        row[
            "eta_start"
        ]
    )


    eta_end = float(
        row[
            "eta_end"
        ]
    )


    GJ_gain = float(
        row[
            "GJ_gain"
        ]
    )


    mass_fraction = float(
        row[
            "mass_fraction"
        ]
    )


    active_fraction = float(
        row[
            "active_fraction"
        ]
    )


    K_theta = float(
        row[
            "K_theta_Nm_per_rad"
        ]
    )


    bandwidth_Hz = float(
        row[
            "bandwidth_Hz"
        ]
    )


    delay_ms = float(
        row[
            "delay_ms"
        ]
    )


    delay_s = (
        delay_ms
        / 1000.0
    )


    # --------------------------------------------------------
    # Build passive structural design
    # --------------------------------------------------------

    design = (
        build_passive_modal_matrices_v2(
            eta_start=
                eta_start,

            eta_end=
                eta_end,

            bending_stiffness_gain=
                0.0,

            torsional_stiffness_gain=
                GJ_gain,

            added_mass_fraction=
                mass_fraction,
        )
    )


    M_design = (
        design[
            "M"
        ]
    )


    C_design = (
        design[
            "C"
        ]
    )


    K_design = (
        design[
            "K"
        ]
    )


    status = (
        "OK"
    )


    try:

        # ====================================================
        # HYBRID FLUTTER BOUNDARY
        # ====================================================

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            hybrid_result = (
                solve_hybrid_flutter_root(
                    M_struct=
                        M_design,

                    C_struct=
                        C_design,

                    K_struct=
                        K_design,

                    K_theta=
                        K_theta,

                    bandwidth_Hz=
                        bandwidth_Hz,

                    delay_s=
                        delay_s,

                    lambda_guess=
                        lambda_flutter_baseline,
                )
            )


        U_hybrid = float(
            hybrid_result[
                "U"
            ]
        )


        f_hybrid = float(
            hybrid_result[
                "frequency_Hz"
            ]
        )


        hybrid_improvement = (
            100.0
            * (
                U_hybrid
                - U_flutter_baseline
            )
            / U_flutter_baseline
        )


        controller_phase = float(
            hybrid_result[
                "controller_phase_deg"
            ]
        )


        controller_magnitude = float(
            hybrid_result[
                "controller_magnitude"
            ]
        )


        # ====================================================
        # CONTROLLER-OFF / FAIL-SAFE FLUTTER
        # ====================================================

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            fail_safe_result = (
                solve_passive_flutter_root(
                    M_struct=
                        M_design,

                    C_struct=
                        C_design,

                    K_struct=
                        K_design,

                    lambda_guess=
                        lambda_flutter_baseline,
                )
            )


        U_fail_safe = float(
            fail_safe_result[
                "U"
            ]
        )


        fail_safe_improvement = (
            100.0
            * (
                U_fail_safe
                - U_flutter_baseline
            )
            / U_flutter_baseline
        )


        # ====================================================
        # DRY CLOSED-LOOP ROBUSTNESS
        #
        # Require three independent Padé delay approximations.
        # ====================================================

        dry_checks = []


        for pade_order in [
            1,
            2,
            3,
        ]:


            dry_result = (
                dry_closed_loop_hybrid(
                    M_struct=
                        M_design,

                    C_struct=
                        C_design,

                    K_struct=
                        K_design,

                    K_theta=
                        K_theta,

                    bandwidth_Hz=
                        bandwidth_Hz,

                    delay_s=
                        delay_s,

                    pade_order=
                        pade_order,
                )
            )


            dry_checks.append(
                dry_result
            )


        stable_pade123 = bool(
            all(
                result[
                    "stable"
                ]
                for result
                in dry_checks
            )
        )


        minimum_dry_margin = float(
            min(
                result[
                    "stability_margin"
                ]
                for result
                in dry_checks
            )
        )


        # ====================================================
        # CONTROL-LOSS PENALTY
        # ====================================================

        velocity_loss_if_control_off = (
            U_hybrid
            - U_fail_safe
        )


        velocity_loss_if_control_off_pct = (
            100.0
            * velocity_loss_if_control_off
            / U_hybrid
        )


    except Exception as exc:


        status = (
            type(
                exc
            ).__name__
        )


        U_hybrid = np.nan
        f_hybrid = np.nan

        hybrid_improvement = np.nan

        controller_phase = np.nan
        controller_magnitude = np.nan

        U_fail_safe = np.nan
        fail_safe_improvement = np.nan

        stable_pade123 = False
        minimum_dry_margin = np.nan

        velocity_loss_if_control_off = np.nan
        velocity_loss_if_control_off_pct = np.nan


    # ========================================================
    # STORE CASE
    # ========================================================

    physics_doe_rows.append({

        "case_id":
            case_number,

        # -----------------------------
        # ML FEATURES
        # -----------------------------

        "eta_mid":
            float(
                row[
                    "eta_mid"
                ]
            ),

        "eta_start":
            eta_start,

        "eta_end":
            eta_end,

        "GJ_gain_pct":
            100.0
            * GJ_gain,

        "mass_fraction_pct":
            100.0
            * mass_fraction,

        "added_mass_kg":
            float(
                row[
                    "added_mass_kg"
                ]
            ),

        "active_fraction":
            active_fraction,

        "active_authority_pct":
            100.0
            * active_fraction,

        "K_theta_Nm_per_rad":
            K_theta,

        "torque_per_deg_Nm":
            float(
                row[
                    "torque_per_deg_Nm"
                ]
            ),

        "bandwidth_Hz":
            bandwidth_Hz,

        "delay_ms":
            delay_ms,

        # -----------------------------
        # PHYSICS LABELS
        # -----------------------------

        "hybrid_flutter_velocity_m_s":
            U_hybrid,

        "hybrid_flutter_frequency_Hz":
            f_hybrid,

        "hybrid_improvement_pct":
            hybrid_improvement,

        "fail_safe_flutter_velocity_m_s":
            U_fail_safe,

        "fail_safe_improvement_pct":
            fail_safe_improvement,

        "velocity_loss_if_control_off_m_s":
            velocity_loss_if_control_off,

        "velocity_loss_if_control_off_pct":
            velocity_loss_if_control_off_pct,

        "minimum_dry_stability_margin":
            minimum_dry_margin,

        "stable_pade123":
            stable_pade123,

        "controller_phase_deg":
            controller_phase,

        "controller_magnitude":
            controller_magnitude,

        "status":
            status,
    })


# ============================================================
# 5. BUILD DOE DATAFRAME
# ============================================================

hybrid_ml_doe = pd.DataFrame(
    physics_doe_rows
)


doe_runtime = (
    time.time()
    - doe_start_time
)


print(
    "\nPHYSICS DOE COMPLETE"
)


print(
    f"Runtime = "
    f"{doe_runtime:.1f} s"
)


print(
    f"Successful cases = "
    f"{(hybrid_ml_doe['status'] == 'OK').sum()} "
    f"/ {len(hybrid_ml_doe)}"
)


# ============================================================
# 6. VALID ML DATASET
# ============================================================

hybrid_ml_valid = (
    hybrid_ml_doe[
        (
            hybrid_ml_doe[
                "status"
            ]
            == "OK"
        )
        &
        (
            hybrid_ml_doe[
                "hybrid_improvement_pct"
            ]
            .notna()
        )
        &
        (
            hybrid_ml_doe[
                "fail_safe_improvement_pct"
            ]
            .notna()
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    f"Valid ML rows = "
    f"{len(hybrid_ml_valid)}"
)


# ============================================================
# 7. DATA QUALITY CHECKS
# ============================================================

if len(
    hybrid_ml_valid
) < 50:

    print(
        "\nWARNING:"
    )

    print(
        "Fewer than 50 valid physics cases are available."
    )

    print(
        "We should augment the DOE before trusting an ML model."
    )


duplicate_feature_rows = (
    hybrid_ml_valid[
        [
            "eta_mid",
            "GJ_gain_pct",
            "active_fraction",
            "bandwidth_Hz",
            "delay_ms",
        ]
    ]
    .duplicated()
    .sum()
)


print(
    f"\nDuplicate design points = "
    f"{duplicate_feature_rows}"
)


print(
    "\nPhysics-output summary:"
)


display(
    hybrid_ml_valid[
        [
            "hybrid_improvement_pct",
            "fail_safe_improvement_pct",
            "minimum_dry_stability_margin",
            "added_mass_kg",
            "active_authority_pct",
            "torque_per_deg_Nm",
        ]
    ]
    .describe()
)


# ============================================================
# 8. TARGET / ROBUSTNESS COVERAGE
# ============================================================

n_target = int(
    (
        hybrid_ml_valid[
            "hybrid_improvement_pct"
        ]
        >= 6.0
    )
    .sum()
)


n_stable = int(
    (
        hybrid_ml_valid[
            "stable_pade123"
        ]
        == True
    )
    .sum()
)


n_target_and_stable = int(
    (
        (
            hybrid_ml_valid[
                "hybrid_improvement_pct"
            ]
            >= 6.0
        )
        &
        (
            hybrid_ml_valid[
                "stable_pade123"
            ]
            == True
        )
    )
    .sum()
)


print(
    "\nDOE coverage:"
)


print(
    f"Cases with >= +6% hybrid improvement = "
    f"{n_target}"
)


print(
    f"Cases stable for Padé 1/2/3 = "
    f"{n_stable}"
)


print(
    f"Cases satisfying BOTH = "
    f"{n_target_and_stable}"
)


# ============================================================
# 9. VISUAL CHECK OF DESIGN SPACE
# ============================================================

plt.figure(
    figsize=(9, 5)
)


scatter = plt.scatter(
    hybrid_ml_valid[
        "GJ_gain_pct"
    ],

    100.0
    * hybrid_ml_valid[
        "active_fraction"
    ],

    c=
        hybrid_ml_valid[
            "hybrid_improvement_pct"
        ],

    s=55,
)


plt.colorbar(
    scatter,
    label=
        "Physics flutter improvement (%)",
)


plt.scatter(
    [
        HYBRID_REFERENCE_GJ_GAIN_PCT
    ],

    [
        100.0
        * HYBRID_REFERENCE_ACTIVE_FRACTION
    ],

    marker="*",
    s=220,
    label="Deterministic reference",
)


plt.xlabel(
    "Local torsional-stiffness gain (%)"
)


plt.ylabel(
    "Active authority (% of robust active reference)"
)


plt.title(
    "Physics DOE coverage for ML surrogate"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# 10. PERFORMANCE / FAIL-SAFE TRADE SPACE
# ============================================================

plt.figure(
    figsize=(9, 5)
)


scatter = plt.scatter(
    hybrid_ml_valid[
        "fail_safe_improvement_pct"
    ],

    hybrid_ml_valid[
        "hybrid_improvement_pct"
    ],

    c=
        hybrid_ml_valid[
            "active_authority_pct"
        ],

    s=55,
)


plt.colorbar(
    scatter,
    label=
        "Active authority (%)",
)


plt.axhline(
    6.0,
    linestyle="--",
    linewidth=1.0,
)


plt.xlabel(
    "Controller-off flutter improvement (%)"
)


plt.ylabel(
    "Hybrid flutter improvement (%)"
)


plt.title(
    "Physics DOE: performance versus fail-safe capability"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


# ============================================================
# 11. SAVE PHYSICS DATASET
# ============================================================

DOE_CSV_PATH = (
    "/content/FLEX445_hybrid_physics_DOE.csv"
)


hybrid_ml_doe.to_csv(
    DOE_CSV_PATH,
    index=False,
)


print(
    "\nSaved physics DOE to:"
)


print(
    DOE_CSV_PATH
)


print(
    "\nCell 60 physics-based ML DOE completed."
)

In [ ]:
# ============================================================
# CELL 61 — FIXED
# ML SURROGATE TRAINING + VALIDATION
#
# FIX:
# Every model is now independently cloned before fitting.
# No estimator object is shared between targets.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
)

from sklearn.linear_model import Ridge

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)


# ============================================================
# 1. SETTINGS
# ============================================================

ML_RANDOM_SEED = 4456


ML_FEATURES = [
    "eta_mid",
    "GJ_gain_pct",
    "active_fraction",
    "bandwidth_Hz",
    "delay_ms",
]


ML_TARGETS = [
    "hybrid_improvement_pct",
    "fail_safe_improvement_pct",
    "minimum_dry_stability_margin",
]


ml_dataset = (
    hybrid_ml_valid[
        ML_FEATURES
        + ML_TARGETS
    ]
    .copy()
    .dropna()
    .reset_index(drop=True)
)


print(
    "ML SURROGATE TRAINING — CORRECTED"
)

print(
    f"\nDataset rows = {len(ml_dataset)}"
)

print(
    f"Input features = {len(ML_FEATURES)}"
)

print(
    f"Regression targets = {len(ML_TARGETS)}"
)


# ============================================================
# 2. COMMON TRAIN / TEST SPLIT
# ============================================================

X = (
    ml_dataset[
        ML_FEATURES
    ]
)


all_indices = np.arange(
    len(
        ml_dataset
    )
)


train_indices, test_indices = (
    train_test_split(
        all_indices,

        test_size=0.20,

        random_state=
            ML_RANDOM_SEED,
    )
)


X_train = (
    X.iloc[
        train_indices
    ]
)


X_test = (
    X.iloc[
        test_indices
    ]
)


print(
    f"\nTraining rows = {len(X_train)}"
)

print(
    f"Test rows = {len(X_test)}"
)


# ============================================================
# 3. CROSS-VALIDATION
# ============================================================

cv = (
    KFold(
        n_splits=5,

        shuffle=True,

        random_state=
            ML_RANDOM_SEED,
    )
)


# ============================================================
# 4. FRESH MODEL TEMPLATES
#
# IMPORTANT:
# These are templates only.
#
# Every actual fitted estimator will be created using:
#
#       clone(template)
#
# ============================================================

MODEL_TEMPLATES = {

    "Polynomial_Ridge":

        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False,
                )
            ),

            (
                "scale",
                StandardScaler(),
            ),

            (
                "model",
                Ridge(
                    alpha=1.0,
                )
            ),
        ]),


    "Random_Forest":

        RandomForestRegressor(
            n_estimators=500,

            min_samples_leaf=2,

            random_state=
                ML_RANDOM_SEED,

            n_jobs=-1,
        ),


    "Extra_Trees":

        ExtraTreesRegressor(
            n_estimators=500,

            min_samples_leaf=2,

            random_state=
                ML_RANDOM_SEED,

            n_jobs=-1,
        ),


    "Gradient_Boosting":

        GradientBoostingRegressor(
            n_estimators=250,

            learning_rate=0.03,

            max_depth=2,

            loss="squared_error",

            random_state=
                ML_RANDOM_SEED,
        ),
}


# ============================================================
# 5. TRAIN INDEPENDENT MODELS
# ============================================================

model_comparison_rows = []


# IMPORTANT:
#
# trained_models[target][model]
#
# now contains a genuinely independent fitted estimator.
# ============================================================

trained_models = {}


for target_name in ML_TARGETS:


    print(
        "\n=========================================="
    )

    print(
        f"TARGET: {target_name}"
    )

    print(
        "=========================================="
    )


    y = (
        ml_dataset[
            target_name
        ]
    )


    y_train = (
        y.iloc[
            train_indices
        ]
    )


    y_test = (
        y.iloc[
            test_indices
        ]
    )


    trained_models[
        target_name
    ] = {}


    for (
        model_name,
        template
    ) in MODEL_TEMPLATES.items():


        # ----------------------------------------------------
        # CRITICAL FIX:
        # Create a new independent estimator.
        # ----------------------------------------------------

        model = clone(
            template
        )


        model.fit(
            X_train,
            y_train,
        )


        y_pred = (
            model.predict(
                X_test
            )
        )


        test_r2 = (
            r2_score(
                y_test,
                y_pred,
            )
        )


        test_mae = (
            mean_absolute_error(
                y_test,
                y_pred,
            )
        )


        test_rmse = (
            np.sqrt(
                mean_squared_error(
                    y_test,
                    y_pred,
                )
            )
        )


        # ----------------------------------------------------
        # CV also gets its own estimator clone internally.
        # ----------------------------------------------------

        cv_r2_scores = (
            cross_val_score(
                clone(
                    template
                ),

                X,
                y,

                cv=cv,

                scoring="r2",

                n_jobs=-1,
            )
        )


        cv_mae_scores = (
            -cross_val_score(
                clone(
                    template
                ),

                X,
                y,

                cv=cv,

                scoring=
                    "neg_mean_absolute_error",

                n_jobs=-1,
            )
        )


        cv_r2_mean = float(
            np.mean(
                cv_r2_scores
            )
        )


        cv_r2_std = float(
            np.std(
                cv_r2_scores
            )
        )


        cv_mae_mean = float(
            np.mean(
                cv_mae_scores
            )
        )


        # ----------------------------------------------------
        # Store THIS fitted object.
        # It will never be refitted.
        # ----------------------------------------------------

        trained_models[
            target_name
        ][
            model_name
        ] = model


        model_comparison_rows.append({

            "target":
                target_name,

            "model":
                model_name,

            "test_R2":
                test_r2,

            "test_MAE":
                test_mae,

            "test_RMSE":
                test_rmse,

            "CV_R2_mean":
                cv_r2_mean,

            "CV_R2_std":
                cv_r2_std,

            "CV_MAE_mean":
                cv_mae_mean,
        })


        print(
            f"{model_name:20s}"
            f" | test R2 = {test_r2:7.4f}"
            f" | MAE = {test_mae:8.5f}"
            f" | CV R2 = {cv_r2_mean:7.4f}"
        )


# ============================================================
# 6. MODEL COMPARISON
# ============================================================

ml_model_comparison = (
    pd.DataFrame(
        model_comparison_rows
    )
)


print(
    "\nFULL MODEL COMPARISON"
)


display(
    ml_model_comparison
    .sort_values(
        [
            "target",
            "CV_R2_mean",
        ],

        ascending=[
            True,
            False,
        ],
    )
)


# ============================================================
# 7. SELECT BEST MODEL FOR EACH TARGET
#
# Criterion:
#
#       highest mean 5-fold CV R²
#
# ============================================================

ML_BEST_MODELS = {}

best_model_rows = []


for target_name in ML_TARGETS:


    target_results = (
        ml_model_comparison[
            ml_model_comparison[
                "target"
            ]
            == target_name
        ]
    )


    best_row_index = (
        target_results[
            "CV_R2_mean"
        ]
        .idxmax()
    )


    best_row = (
        target_results
        .loc[
            best_row_index
        ]
    )


    best_model_name = str(
        best_row[
            "model"
        ]
    )


    # --------------------------------------------------------
    # CRITICAL FIX:
    #
    # Fresh estimator for final full-data surrogate.
    # --------------------------------------------------------

    final_model = clone(
        MODEL_TEMPLATES[
            best_model_name
        ]
    )


    final_model.fit(
        X,

        ml_dataset[
            target_name
        ],
    )


    ML_BEST_MODELS[
        target_name
    ] = {

        "name":
            best_model_name,

        "model":
            final_model,

        "test_R2":
            float(
                best_row[
                    "test_R2"
                ]
            ),

        "test_MAE":
            float(
                best_row[
                    "test_MAE"
                ]
            ),

        "CV_R2_mean":
            float(
                best_row[
                    "CV_R2_mean"
                ]
            ),

        "CV_R2_std":
            float(
                best_row[
                    "CV_R2_std"
                ]
            ),

        "CV_MAE_mean":
            float(
                best_row[
                    "CV_MAE_mean"
                ]
            ),
    }


    best_model_rows.append({

        "target":
            target_name,

        "selected_model":
            best_model_name,

        "test_R2":
            best_row[
                "test_R2"
            ],

        "test_MAE":
            best_row[
                "test_MAE"
            ],

        "CV_R2_mean":
            best_row[
                "CV_R2_mean"
            ],

        "CV_R2_std":
            best_row[
                "CV_R2_std"
            ],

        "CV_MAE_mean":
            best_row[
                "CV_MAE_mean"
            ],
    })


ml_selected_models = (
    pd.DataFrame(
        best_model_rows
    )
)


print(
    "\nSELECTED SURROGATE MODELS"
)


display(
    ml_selected_models
)


# ============================================================
# 8. EXPLICIT OBJECT-INDEPENDENCE AUDIT
# ============================================================

selected_object_ids = [

    id(
        ML_BEST_MODELS[
            target_name
        ][
            "model"
        ]
    )

    for target_name in ML_TARGETS
]


all_independent = bool(
    len(
        set(
            selected_object_ids
        )
    )
    ==
    len(
        selected_object_ids
    )
)


print(
    "\nMODEL OBJECT INDEPENDENCE CHECK"
)


print(
    f"Unique model objects = "
    f"{len(set(selected_object_ids))}"
)


print(
    f"Expected            = "
    f"{len(ML_TARGETS)}"
)


assert all_independent, (
    "ERROR: selected surrogate models "
    "still share estimator objects."
)


print(
    "PASS: every target owns an independent fitted model."
)


# ============================================================
# 9. CORRECT HOLDOUT PARITY PLOTS
# ============================================================

for target_name in ML_TARGETS:


    model_name = (
        ML_BEST_MODELS[
            target_name
        ][
            "name"
        ]
    )


    # This is the independently fitted TRAIN-only version.
    evaluation_model = (
        trained_models[
            target_name
        ][
            model_name
        ]
    )


    y_true = (
        ml_dataset[
            target_name
        ]
        .iloc[
            test_indices
        ]
        .to_numpy()
    )


    y_pred = (
        evaluation_model.predict(
            X_test
        )
    )


    plot_min = min(
        np.min(
            y_true
        ),
        np.min(
            y_pred
        ),
    )


    plot_max = max(
        np.max(
            y_true
        ),
        np.max(
            y_pred
        ),
    )


    span = max(
        plot_max
        - plot_min,
        1e-12,
    )


    margin = (
        0.05
        * span
    )


    plt.figure(
        figsize=(6, 6)
    )


    plt.scatter(
        y_true,
        y_pred,
        s=60,
    )


    plt.plot(
        [
            plot_min - margin,
            plot_max + margin,
        ],

        [
            plot_min - margin,
            plot_max + margin,
        ],

        "--",
        linewidth=1.0,
    )


    plt.xlabel(
        "Physics-model value"
    )


    plt.ylabel(
        "ML-surrogate prediction"
    )


    plt.title(
        f"{target_name}\n"
        f"{model_name}"
    )


    plt.grid(
        alpha=0.25
    )


    plt.tight_layout()

    plt.show()


# ============================================================
# 10. CORRECT UNSEEN TEST PREDICTIONS
# ============================================================

test_prediction_table = (
    ml_dataset
    .iloc[
        test_indices
    ][
        ML_FEATURES
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


for target_name in ML_TARGETS:


    model_name = (
        ML_BEST_MODELS[
            target_name
        ][
            "name"
        ]
    )


    evaluation_model = (
        trained_models[
            target_name
        ][
            model_name
        ]
    )


    y_true = (
        ml_dataset[
            target_name
        ]
        .iloc[
            test_indices
        ]
        .to_numpy()
    )


    y_pred = (
        evaluation_model.predict(
            X_test
        )
    )


    test_prediction_table[
        f"{target_name}_physics"
    ] = y_true


    test_prediction_table[
        f"{target_name}_ML"
    ] = y_pred


    test_prediction_table[
        f"{target_name}_error"
    ] = (
        y_pred
        - y_true
    )


print(
    "\nUNSEEN TEST-SET PREDICTIONS — CORRECTED"
)


display(
    test_prediction_table
)


# ============================================================
# 11. DETERMINISTIC HYBRID REFERENCE CHECK
# ============================================================

reference_feature_row = pd.DataFrame({

    "eta_mid": [
        0.45
    ],

    "GJ_gain_pct": [
        HYBRID_REFERENCE_GJ_GAIN_PCT
    ],

    "active_fraction": [
        HYBRID_REFERENCE_ACTIVE_FRACTION
    ],

    "bandwidth_Hz": [
        ROBUST_ACTIVE_BANDWIDTH_HZ
    ],

    "delay_ms": [
        ROBUST_ACTIVE_DELAY_MS
    ],
})


reference_predictions = {}


for target_name in ML_TARGETS:


    model = (
        ML_BEST_MODELS[
            target_name
        ][
            "model"
        ]
    )


    reference_predictions[
        target_name
    ] = float(
        model.predict(
            reference_feature_row
        )[0]
    )


reference_check = pd.DataFrame({

    "quantity": [
        "Hybrid improvement (%)",
        "Fail-safe improvement (%)",
        "Minimum dry stability margin (1/s)",
    ],

    "physics_value": [
        float(
            HYBRID_REFERENCE[
                "hybrid_improvement_pct"
            ]
        ),

        float(
            HYBRID_REFERENCE[
                "controller_off_improvement_pct"
            ]
        ),

        float(
            HYBRID_REFERENCE[
                "minimum_dry_stability_margin"
            ]
        ),
    ],

    "ML_prediction": [
        reference_predictions[
            "hybrid_improvement_pct"
        ],

        reference_predictions[
            "fail_safe_improvement_pct"
        ],

        reference_predictions[
            "minimum_dry_stability_margin"
        ],
    ],
})


reference_check[
    "signed_error"
] = (
    reference_check[
        "ML_prediction"
    ]
    -
    reference_check[
        "physics_value"
    ]
)


reference_check[
    "absolute_error"
] = (
    reference_check[
        "signed_error"
    ]
    .abs()
)


print(
    "\nDETERMINISTIC REFERENCE — CORRECTED ML SANITY CHECK"
)


display(
    reference_check
)


# ============================================================
# 12. QUALITY GATES
# ============================================================

quality_requirements = {

    "hybrid_improvement_pct":
        0.90,

    "fail_safe_improvement_pct":
        0.90,

    "minimum_dry_stability_margin":
        0.75,
}


quality_rows = []


for target_name in ML_TARGETS:


    achieved = (
        ML_BEST_MODELS[
            target_name
        ][
            "CV_R2_mean"
        ]
    )


    required = (
        quality_requirements[
            target_name
        ]
    )


    passed = bool(
        achieved
        >= required
    )


    quality_rows.append({

        "target":
            target_name,

        "CV_R2_achieved":
            achieved,

        "CV_R2_required":
            required,

        "quality_gate_pass":
            passed,
    })


ml_quality_gate = (
    pd.DataFrame(
        quality_rows
    )
)


print(
    "\nML SURROGATE QUALITY GATES"
)


display(
    ml_quality_gate
)


ALL_ML_QUALITY_GATES_PASS = bool(
    ml_quality_gate[
        "quality_gate_pass"
    ]
    .all()
)


if ALL_ML_QUALITY_GATES_PASS:

    print(
        "\nPASS:"
    )

    print(
        "All corrected surrogate-quality gates are satisfied."
    )

    print(
        "The independent surrogate models may proceed "
        "to large design-space screening."
    )


else:

    print(
        "\nDOE AUGMENTATION REQUIRED:"
    )

    print(
        "Do not proceed to large ML optimization."
    )


print(
    "\nCell 61 FIXED ML surrogate training "
    "and validation completed."
)

In [ ]:
# ============================================================
# CELL 62 — LARGE ML SURROGATE DESIGN SEARCH
#            + ENGINEERING CONSTRAINTS + PARETO FRONT
# ============================================================
#
# PURPOSE
# -------
#
# Use the validated ML surrogates from Cell 61 to screen
# 100,000 hybrid passive-active architectures.
#
# IMPORTANT:
#
# 1. Search remains STRICTLY inside the original physics-DOE
#    bounds. No ML extrapolation.
#
# 2. ML is used only for fast screening.
#
# 3. Final candidate designs will be re-run through the
#    full physics solver in Cell 63.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import qmc


# ============================================================
# 1. SEARCH SETTINGS
# ============================================================

N_ML_SEARCH = 100_000

ML_SEARCH_SEED = 4456


# ------------------------------------------------------------
# Engineering screening requirements
#
# These are PROJECT requirements, not certification values.
# ------------------------------------------------------------

REQ_HYBRID_IMPROVEMENT = 6.0          # %
REQ_FAIL_SAFE_IMPROVEMENT = 3.0       # %
REQ_MIN_DRY_MARGIN = 0.40             # 1/s

MAX_ACTIVE_AUTHORITY = 0.60           # fraction
MAX_MASS_FRACTION = 0.025             # 2.5% panel mass


print(
    "LARGE SURROGATE-ASSISTED HYBRID DESIGN SEARCH"
)

print(
    f"\nCandidate architectures = {N_ML_SEARCH:,}"
)

print(
    "\nEngineering screening requirements:"
)

print(
    f"Hybrid flutter improvement >= "
    f"{REQ_HYBRID_IMPROVEMENT:.1f}%"
)

print(
    f"Fail-safe flutter improvement >= "
    f"{REQ_FAIL_SAFE_IMPROVEMENT:.1f}%"
)

print(
    f"Minimum dry stability margin >= "
    f"{REQ_MIN_DRY_MARGIN:.2f} 1/s"
)

print(
    f"Active authority <= "
    f"{100*MAX_ACTIVE_AUTHORITY:.0f}%"
)

print(
    f"Added mass <= "
    f"{100*MAX_MASS_FRACTION:.1f}% "
    f"of baseline panel mass"
)


# ============================================================
# 2. GENERATE 100,000 DESIGNS
#
# Search bounds are IDENTICAL to Cell 60 DOE bounds.
# ============================================================

search_sampler = (
    qmc.LatinHypercube(
        d=5,
        seed=ML_SEARCH_SEED,
    )
)


search_unit = (
    search_sampler.random(
        n=N_ML_SEARCH
    )
)


search_lower = np.array([
    DOE_BOUNDS["eta_mid"][0],
    100.0 * DOE_BOUNDS["GJ_gain"][0],
    DOE_BOUNDS["active_fraction"][0],
    DOE_BOUNDS["bandwidth_Hz"][0],
    DOE_BOUNDS["delay_ms"][0],
])


search_upper = np.array([
    DOE_BOUNDS["eta_mid"][1],
    100.0 * DOE_BOUNDS["GJ_gain"][1],
    DOE_BOUNDS["active_fraction"][1],
    DOE_BOUNDS["bandwidth_Hz"][1],
    DOE_BOUNDS["delay_ms"][1],
])


search_physical = (
    qmc.scale(
        search_unit,
        search_lower,
        search_upper,
    )
)


ml_search = pd.DataFrame(
    search_physical,
    columns=[
        "eta_mid",
        "GJ_gain_pct",
        "active_fraction",
        "bandwidth_Hz",
        "delay_ms",
    ],
)


# ============================================================
# 3. DERIVED ENGINEERING QUANTITIES
# ============================================================

ml_search[
    "eta_start"
] = (
    ml_search[
        "eta_mid"
    ]
    - 0.10
)


ml_search[
    "eta_end"
] = (
    ml_search[
        "eta_mid"
    ]
    + 0.10
)


# ------------------------------------------------------------
# Same NOTIONAL mass model used in Cells 58–60:
#
#   +20% GJ <-> +2% panel mass
#
# therefore
#
#   mass fraction = GJ gain fraction * 0.10
# ------------------------------------------------------------

ml_search[
    "mass_fraction"
] = (
    0.001
    * ml_search[
        "GJ_gain_pct"
    ]
)


ml_search[
    "mass_fraction_pct"
] = (
    100.0
    * ml_search[
        "mass_fraction"
    ]
)


ml_search[
    "added_mass_kg"
] = (
    ml_search[
        "mass_fraction"
    ]
    * BASELINE_PANEL_MASS_KG
)


ml_search[
    "active_authority_pct"
] = (
    100.0
    * ml_search[
        "active_fraction"
    ]
)


ml_search[
    "K_theta_Nm_per_rad"
] = (
    ml_search[
        "active_fraction"
    ]
    * ROBUST_ACTIVE_K_THETA
)


ml_search[
    "torque_per_deg_Nm"
] = (
    ml_search[
        "K_theta_Nm_per_rad"
    ]
    * np.deg2rad(
        1.0
    )
)


# ============================================================
# 4. ML PREDICTIONS
# ============================================================

X_search = (
    ml_search[
        ML_FEATURES
    ]
)


for target_name in ML_TARGETS:

    model = (
        ML_BEST_MODELS[
            target_name
        ][
            "model"
        ]
    )

    ml_search[
        f"pred_{target_name}"
    ] = (
        model.predict(
            X_search
        )
    )


print(
    "\nML evaluation complete."
)


# ============================================================
# 5. CONSTRAINT FILTERING
# ============================================================

ml_search[
    "constraint_hybrid"
] = (
    ml_search[
        "pred_hybrid_improvement_pct"
    ]
    >= REQ_HYBRID_IMPROVEMENT
)


ml_search[
    "constraint_fail_safe"
] = (
    ml_search[
        "pred_fail_safe_improvement_pct"
    ]
    >= REQ_FAIL_SAFE_IMPROVEMENT
)


ml_search[
    "constraint_stability"
] = (
    ml_search[
        "pred_minimum_dry_stability_margin"
    ]
    >= REQ_MIN_DRY_MARGIN
)


ml_search[
    "constraint_active"
] = (
    ml_search[
        "active_fraction"
    ]
    <= MAX_ACTIVE_AUTHORITY
)


ml_search[
    "constraint_mass"
] = (
    ml_search[
        "mass_fraction"
    ]
    <= MAX_MASS_FRACTION
)


ml_search[
    "feasible"
] = (
    ml_search[
        [
            "constraint_hybrid",
            "constraint_fail_safe",
            "constraint_stability",
            "constraint_active",
            "constraint_mass",
        ]
    ]
    .all(
        axis=1
    )
)


ml_feasible = (
    ml_search[
        ml_search[
            "feasible"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\nSURROGATE SEARCH RESULTS"
)

print(
    f"Total designs = "
    f"{len(ml_search):,}"
)

print(
    f"Feasible designs = "
    f"{len(ml_feasible):,}"
)

print(
    f"Feasible fraction = "
    f"{100*len(ml_feasible)/len(ml_search):.2f}%"
)


if len(
    ml_feasible
) == 0:

    raise RuntimeError(
        "No surrogate designs satisfy the engineering "
        "requirements."
    )


# ============================================================
# 6. REQUIREMENT MARGINS
# ============================================================

ml_feasible[
    "hybrid_margin_above_requirement"
] = (
    ml_feasible[
        "pred_hybrid_improvement_pct"
    ]
    - REQ_HYBRID_IMPROVEMENT
)


ml_feasible[
    "fail_safe_margin_above_requirement"
] = (
    ml_feasible[
        "pred_fail_safe_improvement_pct"
    ]
    - REQ_FAIL_SAFE_IMPROVEMENT
)


ml_feasible[
    "dry_margin_above_requirement"
] = (
    ml_feasible[
        "pred_minimum_dry_stability_margin"
    ]
    - REQ_MIN_DRY_MARGIN
)


# ============================================================
# 7. 2-OBJECTIVE PARETO FRONT
#
# Once performance / fail-safe / stability requirements have
# been enforced as HARD constraints, the practical trade is:
#
#       minimize added structural mass
#
#       minimize active-control authority
#
#
# A point is Pareto-efficient if no other feasible point has:
#
#       <= mass
#       <= active authority
#
# with at least one strict improvement.
#
# ============================================================

pareto_source = (
    ml_feasible
    .sort_values(
        [
            "added_mass_kg",
            "active_fraction",
        ]
    )
    .reset_index(
        drop=True
    )
)


pareto_indices = []

best_active_so_far = np.inf


for idx, row in (
    pareto_source.iterrows()
):

    active_fraction = float(
        row[
            "active_fraction"
        ]
    )

    if (
        active_fraction
        < best_active_so_far
    ):

        pareto_indices.append(
            idx
        )

        best_active_so_far = (
            active_fraction
        )


ml_pareto = (
    pareto_source
    .iloc[
        pareto_indices
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    f"\nPareto-efficient feasible designs = "
    f"{len(ml_pareto)}"
)


# ============================================================
# 8. ENGINEERING SHORTLIST STRATEGIES
#
# Instead of one arbitrary scalar score, choose candidates
# representing different design philosophies.
# ============================================================

shortlist_rows = []


# ------------------------------------------------------------
# A. Minimum active authority
# ------------------------------------------------------------

idx = (
    ml_feasible[
        "active_fraction"
    ]
    .idxmin()
)


candidate = (
    ml_feasible
    .loc[
        idx
    ]
    .copy()
)


candidate[
    "selection_reason"
] = (
    "Minimum active authority"
)


shortlist_rows.append(
    candidate
)


# ------------------------------------------------------------
# B. Minimum structural mass
# ------------------------------------------------------------

idx = (
    ml_feasible[
        "added_mass_kg"
    ]
    .idxmin()
)


candidate = (
    ml_feasible
    .loc[
        idx
    ]
    .copy()
)


candidate[
    "selection_reason"
] = (
    "Minimum added mass"
)


shortlist_rows.append(
    candidate
)


# ------------------------------------------------------------
# C. Highest fail-safe margin
# ------------------------------------------------------------

idx = (
    ml_feasible[
        "pred_fail_safe_improvement_pct"
    ]
    .idxmax()
)


candidate = (
    ml_feasible
    .loc[
        idx
    ]
    .copy()
)


candidate[
    "selection_reason"
] = (
    "Highest fail-safe margin"
)


shortlist_rows.append(
    candidate
)


# ------------------------------------------------------------
# D. Highest hybrid performance
# ------------------------------------------------------------

idx = (
    ml_feasible[
        "pred_hybrid_improvement_pct"
    ]
    .idxmax()
)


candidate = (
    ml_feasible
    .loc[
        idx
    ]
    .copy()
)


candidate[
    "selection_reason"
] = (
    "Highest hybrid improvement"
)


shortlist_rows.append(
    candidate
)


# ------------------------------------------------------------
# E. Balanced Pareto candidate
#
# Normalize mass and active authority over Pareto front and
# choose the point closest to the ideal lower-left corner.
# ------------------------------------------------------------

pareto_balanced = (
    ml_pareto.copy()
)


mass_min = (
    pareto_balanced[
        "added_mass_kg"
    ]
    .min()
)


mass_max = (
    pareto_balanced[
        "added_mass_kg"
    ]
    .max()
)


active_min = (
    pareto_balanced[
        "active_fraction"
    ]
    .min()
)


active_max = (
    pareto_balanced[
        "active_fraction"
    ]
    .max()
)


pareto_balanced[
    "mass_normalized"
] = (
    (
        pareto_balanced[
            "added_mass_kg"
        ]
        - mass_min
    )
    /
    max(
        mass_max
        - mass_min,
        1e-12,
    )
)


pareto_balanced[
    "active_normalized"
] = (
    (
        pareto_balanced[
            "active_fraction"
        ]
        - active_min
    )
    /
    max(
        active_max
        - active_min,
        1e-12,
    )
)


pareto_balanced[
    "distance_from_ideal"
] = np.sqrt(

    pareto_balanced[
        "mass_normalized"
    ]**2

    +

    pareto_balanced[
        "active_normalized"
    ]**2
)


idx = (
    pareto_balanced[
        "distance_from_ideal"
    ]
    .idxmin()
)


candidate = (
    pareto_balanced
    .loc[
        idx
    ]
    .copy()
)


candidate[
    "selection_reason"
] = (
    "Balanced Pareto trade"
)


shortlist_rows.append(
    candidate
)


# ============================================================
# 9. BUILD UNIQUE SHORTLIST
# ============================================================

ML_PHYSICS_SHORTLIST = (
    pd.DataFrame(
        shortlist_rows
    )
    .drop_duplicates(
        subset=[
            "eta_mid",
            "GJ_gain_pct",
            "active_fraction",
            "bandwidth_Hz",
            "delay_ms",
        ]
    )
    .reset_index(
        drop=True
    )
)


ML_PHYSICS_SHORTLIST[
    "candidate_id"
] = (
    np.arange(
        len(
            ML_PHYSICS_SHORTLIST
        )
    )
    + 1
)


print(
    "\nML-DERIVED CANDIDATES FOR PHYSICS RE-VERIFICATION"
)


display(
    ML_PHYSICS_SHORTLIST[
        [
            "candidate_id",
            "selection_reason",

            "eta_mid",
            "eta_start",
            "eta_end",

            "GJ_gain_pct",
            "mass_fraction_pct",
            "added_mass_kg",

            "active_authority_pct",
            "K_theta_Nm_per_rad",
            "torque_per_deg_Nm",

            "bandwidth_Hz",
            "delay_ms",

            "pred_hybrid_improvement_pct",
            "pred_fail_safe_improvement_pct",
            "pred_minimum_dry_stability_margin",
        ]
    ]
)


# ============================================================
# 10. COMPARE WITH DETERMINISTIC REFERENCE
# ============================================================

reference_row = pd.DataFrame({

    "design": [
        "Deterministic reference"
    ],

    "GJ_gain_pct": [
        HYBRID_REFERENCE_GJ_GAIN_PCT
    ],

    "active_authority_pct": [
        100.0
        * HYBRID_REFERENCE_ACTIVE_FRACTION
    ],

    "hybrid_improvement_pct": [
        float(
            HYBRID_REFERENCE[
                "hybrid_improvement_pct"
            ]
        )
    ],

    "fail_safe_improvement_pct": [
        float(
            HYBRID_REFERENCE[
                "controller_off_improvement_pct"
            ]
        )
    ],

    "minimum_dry_stability_margin": [
        float(
            HYBRID_REFERENCE[
                "minimum_dry_stability_margin"
            ]
        )
    ],
})


ml_shortlist_comparison = pd.DataFrame({

    "design": [
        f"ML candidate {i}"
        for i
        in ML_PHYSICS_SHORTLIST[
            "candidate_id"
        ]
    ],

    "GJ_gain_pct":
        ML_PHYSICS_SHORTLIST[
            "GJ_gain_pct"
        ]
        .to_numpy(),

    "active_authority_pct":
        ML_PHYSICS_SHORTLIST[
            "active_authority_pct"
        ]
        .to_numpy(),

    "hybrid_improvement_pct":
        ML_PHYSICS_SHORTLIST[
            "pred_hybrid_improvement_pct"
        ]
        .to_numpy(),

    "fail_safe_improvement_pct":
        ML_PHYSICS_SHORTLIST[
            "pred_fail_safe_improvement_pct"
        ]
        .to_numpy(),

    "minimum_dry_stability_margin":
        ML_PHYSICS_SHORTLIST[
            "pred_minimum_dry_stability_margin"
        ]
        .to_numpy(),
})


design_comparison = pd.concat(
    [
        reference_row,
        ml_shortlist_comparison,
    ],
    ignore_index=True,
)


print(
    "\nDETERMINISTIC REFERENCE VS ML SHORTLIST"
)


display(
    design_comparison
)


# ============================================================
# 11. PARETO PLOT
# ============================================================

plt.figure(
    figsize=(9, 6)
)


plt.scatter(
    ml_feasible[
        "mass_fraction_pct"
    ],

    ml_feasible[
        "active_authority_pct"
    ],

    s=10,
    alpha=0.25,

    label="Feasible ML designs",
)


plt.plot(
    ml_pareto[
        "mass_fraction_pct"
    ],

    ml_pareto[
        "active_authority_pct"
    ],

    "o-",

    markersize=4,

    label="Pareto front",
)


plt.scatter(
    ML_PHYSICS_SHORTLIST[
        "mass_fraction_pct"
    ],

    ML_PHYSICS_SHORTLIST[
        "active_authority_pct"
    ],

    marker="*",
    s=180,

    label="Physics re-verification shortlist",
)


plt.scatter(
    [
        0.10
        * HYBRID_REFERENCE_GJ_GAIN_PCT
    ],

    [
        100.0
        * HYBRID_REFERENCE_ACTIVE_FRACTION
    ],

    marker="X",
    s=140,

    label="Deterministic reference",
)


plt.xlabel(
    "Notional added mass (% of panel mass)"
)


plt.ylabel(
    "Active-control authority (%)"
)


plt.title(
    "Feasible hybrid design space and Pareto frontier"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# 12. PERFORMANCE / FAIL-SAFE MAP
# ============================================================

plt.figure(
    figsize=(9, 6)
)


scatter = plt.scatter(
    ml_feasible[
        "pred_fail_safe_improvement_pct"
    ],

    ml_feasible[
        "pred_hybrid_improvement_pct"
    ],

    c=
        ml_feasible[
            "active_authority_pct"
        ],

    s=12,
    alpha=0.45,
)


plt.colorbar(
    scatter,
    label=
        "Active authority (%)",
)


plt.scatter(
    ML_PHYSICS_SHORTLIST[
        "pred_fail_safe_improvement_pct"
    ],

    ML_PHYSICS_SHORTLIST[
        "pred_hybrid_improvement_pct"
    ],

    marker="*",
    s=180,

    label="Selected ML candidates",
)


plt.axhline(
    REQ_HYBRID_IMPROVEMENT,
    linestyle="--",
    linewidth=1.0,
)


plt.axvline(
    REQ_FAIL_SAFE_IMPROVEMENT,
    linestyle="--",
    linewidth=1.0,
)


plt.xlabel(
    "Predicted controller-off flutter improvement (%)"
)


plt.ylabel(
    "Predicted hybrid flutter improvement (%)"
)


plt.title(
    "Surrogate-assisted hybrid architecture search"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# 13. FEATURE VALUES OF PARETO FRONT
# ============================================================

print(
    "\nPARETO FRONT SUMMARY"
)


display(
    ml_pareto[
        [
            "eta_mid",
            "GJ_gain_pct",
            "mass_fraction_pct",
            "active_authority_pct",
            "bandwidth_Hz",
            "delay_ms",

            "pred_hybrid_improvement_pct",
            "pred_fail_safe_improvement_pct",
            "pred_minimum_dry_stability_margin",
        ]
    ]
    .describe()
)


# ============================================================
# 14. SAVE SEARCH RESULTS
# ============================================================

ML_SEARCH_CSV = (
    "/content/FLEX445_ML_design_search.csv"
)


ML_PARETO_CSV = (
    "/content/FLEX445_ML_pareto_front.csv"
)


ML_SHORTLIST_CSV = (
    "/content/FLEX445_ML_physics_shortlist.csv"
)


ml_search.to_csv(
    ML_SEARCH_CSV,
    index=False,
)


ml_pareto.to_csv(
    ML_PARETO_CSV,
    index=False,
)


ML_PHYSICS_SHORTLIST.to_csv(
    ML_SHORTLIST_CSV,
    index=False,
)


print(
    "\nSaved:"
)

print(
    ML_SEARCH_CSV
)

print(
    ML_PARETO_CSV
)

print(
    ML_SHORTLIST_CSV
)


print(
    "\nCell 62 surrogate-assisted "
    "100,000-design search completed."
)

In [ ]:
# ============================================================
# CELL 63 — PHYSICS RE-VERIFICATION OF ML-DERIVED DESIGNS
# ============================================================
#
# PURPOSE
# -------
#
# The ML surrogate is NOT the final authority.
#
# Re-run every shortlisted ML architecture through the
# full reduced-order aeroelastic physics model.
#
#
# VERIFY:
#
#   1. hybrid flutter boundary
#   2. controller-off / fail-safe flutter boundary
#   3. Padé 1/2/3 dry closed-loop stability
#   4. all four aeroelastic branches
#   5. ML prediction errors
#   6. engineering requirements
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import contextlib


# ============================================================
# 1. REQUIREMENTS
# ============================================================

PHYSICS_REQ_HYBRID = 6.0
PHYSICS_REQ_FAIL_SAFE = 3.0
PHYSICS_REQ_DRY_MARGIN = 0.40


print(
    "ML CANDIDATE PHYSICS RE-VERIFICATION"
)

print(
    f"\nCandidates = {len(ML_PHYSICS_SHORTLIST)}"
)


# ============================================================
# 2. STORAGE
# ============================================================

verification_rows = []

verification_branch_rows = []


# ============================================================
# 3. LOOP THROUGH ML SHORTLIST
# ============================================================

for _, candidate in (
    ML_PHYSICS_SHORTLIST
    .sort_values("candidate_id")
    .iterrows()
):


    candidate_id = int(
        candidate[
            "candidate_id"
        ]
    )


    reason = str(
        candidate[
            "selection_reason"
        ]
    )


    eta_mid = float(
        candidate[
            "eta_mid"
        ]
    )


    eta_start = float(
        candidate[
            "eta_start"
        ]
    )


    eta_end = float(
        candidate[
            "eta_end"
        ]
    )


    GJ_gain_pct = float(
        candidate[
            "GJ_gain_pct"
        ]
    )


    GJ_gain = (
        GJ_gain_pct
        / 100.0
    )


    mass_fraction_pct = float(
        candidate[
            "mass_fraction_pct"
        ]
    )


    mass_fraction = (
        mass_fraction_pct
        / 100.0
    )


    active_fraction = float(
        candidate[
            "active_fraction"
        ]
    )


    K_theta = float(
        candidate[
            "K_theta_Nm_per_rad"
        ]
    )


    bandwidth_Hz = float(
        candidate[
            "bandwidth_Hz"
        ]
    )


    delay_ms = float(
        candidate[
            "delay_ms"
        ]
    )


    delay_s = (
        delay_ms
        / 1000.0
    )


    print(
        "\n================================================"
    )

    print(
        f"ML CANDIDATE {candidate_id}"
    )

    print(
        f"Selection reason  = {reason}"
    )

    print(
        f"Span region       = "
        f"{100*eta_start:.1f}% to "
        f"{100*eta_end:.1f}%"
    )

    print(
        f"GJ gain           = "
        f"{GJ_gain_pct:.3f}%"
    )

    print(
        f"Notional mass     = "
        f"{mass_fraction_pct:.3f}%"
    )

    print(
        f"Active authority  = "
        f"{100*active_fraction:.3f}%"
    )

    print(
        f"Bandwidth         = "
        f"{bandwidth_Hz:.2f} Hz"
    )

    print(
        f"Delay             = "
        f"{delay_ms:.2f} ms"
    )


    # ========================================================
    # 4. REBUILD PASSIVE STRUCTURAL MODEL
    # ========================================================

    design = (
        build_passive_modal_matrices_v2(
            eta_start=
                eta_start,

            eta_end=
                eta_end,

            bending_stiffness_gain=
                0.0,

            torsional_stiffness_gain=
                GJ_gain,

            added_mass_fraction=
                mass_fraction,
        )
    )


    M_design = (
        design[
            "M"
        ]
    )


    C_design = (
        design[
            "C"
        ]
    )


    K_design = (
        design[
            "K"
        ]
    )


    # ========================================================
    # 5. FULL-PHYSICS HYBRID FLUTTER ROOT
    # ========================================================

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        hybrid_physics = (
            solve_hybrid_flutter_root(
                M_struct=
                    M_design,

                C_struct=
                    C_design,

                K_struct=
                    K_design,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    U_hybrid = float(
        hybrid_physics[
            "U"
        ]
    )


    f_hybrid = float(
        hybrid_physics[
            "frequency_Hz"
        ]
    )


    hybrid_improvement = (
        100.0
        * (
            U_hybrid
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    print(
        f"\nPhysics hybrid result:"
    )

    print(
        f"U_flutter = "
        f"{U_hybrid:.3f} m/s"
    )

    print(
        f"Improvement = "
        f"{hybrid_improvement:+.4f}%"
    )

    print(
        f"Frequency = "
        f"{f_hybrid:.4f} Hz"
    )


    # ========================================================
    # 6. FULL-PHYSICS CONTROLLER-OFF FLUTTER
    # ========================================================

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        fail_safe_physics = (
            solve_passive_flutter_root(
                M_struct=
                    M_design,

                C_struct=
                    C_design,

                K_struct=
                    K_design,

                lambda_guess=
                    lambda_flutter_baseline,
            )
        )


    U_fail_safe = float(
        fail_safe_physics[
            "U"
        ]
    )


    fail_safe_improvement = (
        100.0
        * (
            U_fail_safe
            - U_flutter_baseline
        )
        / U_flutter_baseline
    )


    print(
        f"Controller-off improvement = "
        f"{fail_safe_improvement:+.4f}%"
    )


    # ========================================================
    # 7. DRY ROBUSTNESS — PADÉ 1/2/3
    # ========================================================

    dry_results = []


    for pade_order in [
        1,
        2,
        3,
    ]:


        dry_result = (
            dry_closed_loop_hybrid(
                M_struct=
                    M_design,

                C_struct=
                    C_design,

                K_struct=
                    K_design,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,

                pade_order=
                    pade_order,
            )
        )


        dry_results.append(
            dry_result
        )


    dry_stable_all = bool(
        all(
            result[
                "stable"
            ]
            for result
            in dry_results
        )
    )


    minimum_dry_margin = float(
        min(
            result[
                "stability_margin"
            ]
            for result
            in dry_results
        )
    )


    print(
        f"Minimum dry stability margin = "
        f"{minimum_dry_margin:.5f} 1/s"
    )


    print(
        f"Padé 1/2/3 stable = "
        f"{dry_stable_all}"
    )


    # ========================================================
    # 8. ALL-BRANCH AEROELASTIC VERIFICATION
    # ========================================================

    q_hybrid = (
        (
            U_hybrid
            / U_exp_passive
        )**2
        * q_exp_passive
    )


    branch_results = []


    for mode_index in range(
        4
    ):


        print(
            f"  Checking aeroelastic branch "
            f"{mode_index + 1}..."
        )


        branch_result = (
            pk_dynamic_hybrid_branch(
                mode_index=
                    mode_index,

                U=
                    U_hybrid,

                q_dyn=
                    q_hybrid,

                M_struct=
                    M_design,

                C_struct=
                    C_design,

                K_struct=
                    K_design,

                K_theta=
                    K_theta,

                bandwidth_Hz=
                    bandwidth_Hz,

                delay_s=
                    delay_s,
            )
        )


        branch_results.append(
            branch_result
        )


        verification_branch_rows.append({

            "candidate_id":
                candidate_id,

            "selection_reason":
                reason,

            **branch_result,
        })


    branch_df = pd.DataFrame(
        branch_results
    )


    mode1_sigma = float(
        branch_df[
            branch_df[
                "mode_id"
            ]
            == 1
        ][
            "sigma"
        ]
        .iloc[0]
    )


    max_other_sigma = float(
        branch_df[
            branch_df[
                "mode_id"
            ]
            != 1
        ][
            "sigma"
        ]
        .max()
    )


    mode1_neutral = bool(
        abs(
            mode1_sigma
        )
        < 1e-3
    )


    other_branches_stable = bool(
        max_other_sigma
        < 0.0
    )


    # ========================================================
    # 9. ML PREDICTION ERRORS
    # ========================================================

    ML_hybrid = float(
        candidate[
            "pred_hybrid_improvement_pct"
        ]
    )


    ML_fail_safe = float(
        candidate[
            "pred_fail_safe_improvement_pct"
        ]
    )


    ML_dry = float(
        candidate[
            "pred_minimum_dry_stability_margin"
        ]
    )


    hybrid_prediction_error = (
        ML_hybrid
        - hybrid_improvement
    )


    fail_safe_prediction_error = (
        ML_fail_safe
        - fail_safe_improvement
    )


    dry_prediction_error = (
        ML_dry
        - minimum_dry_margin
    )


    # ========================================================
    # 10. REQUIREMENT CHECKS
    # ========================================================

    pass_hybrid = bool(
        hybrid_improvement
        >= PHYSICS_REQ_HYBRID
    )


    pass_fail_safe = bool(
        fail_safe_improvement
        >= PHYSICS_REQ_FAIL_SAFE
    )


    pass_dry_margin = bool(
        minimum_dry_margin
        >= PHYSICS_REQ_DRY_MARGIN
    )


    fully_verified = bool(

        pass_hybrid

        and pass_fail_safe

        and pass_dry_margin

        and dry_stable_all

        and mode1_neutral

        and other_branches_stable
    )


    verification_rows.append({

        "candidate_id":
            candidate_id,

        "selection_reason":
            reason,

        "eta_mid":
            eta_mid,

        "eta_start":
            eta_start,

        "eta_end":
            eta_end,

        "GJ_gain_pct":
            GJ_gain_pct,

        "mass_fraction_pct":
            mass_fraction_pct,

        "added_mass_kg":
            float(
                candidate[
                    "added_mass_kg"
                ]
            ),

        "active_authority_pct":
            100.0
            * active_fraction,

        "K_theta_Nm_per_rad":
            K_theta,

        "torque_per_deg_Nm":
            float(
                candidate[
                    "torque_per_deg_Nm"
                ]
            ),

        "bandwidth_Hz":
            bandwidth_Hz,

        "delay_ms":
            delay_ms,

        # ----------------------------------------
        # ML predictions
        # ----------------------------------------

        "ML_hybrid_improvement_pct":
            ML_hybrid,

        "ML_fail_safe_improvement_pct":
            ML_fail_safe,

        "ML_dry_margin":
            ML_dry,

        # ----------------------------------------
        # Physics results
        # ----------------------------------------

        "physics_flutter_velocity_m_s":
            U_hybrid,

        "physics_flutter_frequency_Hz":
            f_hybrid,

        "physics_hybrid_improvement_pct":
            hybrid_improvement,

        "physics_fail_safe_velocity_m_s":
            U_fail_safe,

        "physics_fail_safe_improvement_pct":
            fail_safe_improvement,

        "physics_minimum_dry_margin":
            minimum_dry_margin,

        # ----------------------------------------
        # Prediction errors
        # ----------------------------------------

        "hybrid_prediction_error_pct_point":
            hybrid_prediction_error,

        "fail_safe_prediction_error_pct_point":
            fail_safe_prediction_error,

        "dry_margin_prediction_error":
            dry_prediction_error,

        # ----------------------------------------
        # Branch verification
        # ----------------------------------------

        "mode1_sigma":
            mode1_sigma,

        "max_other_branch_sigma":
            max_other_sigma,

        "mode1_neutral":
            mode1_neutral,

        "other_branches_stable":
            other_branches_stable,

        # ----------------------------------------
        # Requirements
        # ----------------------------------------

        "pass_hybrid_requirement":
            pass_hybrid,

        "pass_fail_safe_requirement":
            pass_fail_safe,

        "pass_dry_margin_requirement":
            pass_dry_margin,

        "dry_stable_pade123":
            dry_stable_all,

        "fully_verified":
            fully_verified,
    })


# ============================================================
# 11. RESULTS TABLES
# ============================================================

ml_physics_verification = (
    pd.DataFrame(
        verification_rows
    )
)


ml_physics_branch_verification = (
    pd.DataFrame(
        verification_branch_rows
    )
)


print(
    "\n=============================================="
)

print(
    "ML → PHYSICS VERIFICATION RESULTS"
)

print(
    "=============================================="
)


display(
    ml_physics_verification
)


print(
    "\nALL-BRANCH PHYSICS RESULTS"
)


display(
    ml_physics_branch_verification
)


# ============================================================
# 12. VERIFIED DESIGNS ONLY
# ============================================================

verified_ml_designs = (
    ml_physics_verification[
        ml_physics_verification[
            "fully_verified"
        ]
        == True
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\nFULLY VERIFIED ML-DERIVED DESIGNS"
)


display(
    verified_ml_designs
)


print(
    f"\nVerified candidates = "
    f"{len(verified_ml_designs)} / "
    f"{len(ml_physics_verification)}"
)


# ============================================================
# 13. SURROGATE ERROR SUMMARY
# ============================================================

error_summary = pd.DataFrame({

    "quantity": [
        "Hybrid improvement",
        "Fail-safe improvement",
        "Minimum dry stability margin",
    ],

    "mean_absolute_error": [

        ml_physics_verification[
            "hybrid_prediction_error_pct_point"
        ]
        .abs()
        .mean(),

        ml_physics_verification[
            "fail_safe_prediction_error_pct_point"
        ]
        .abs()
        .mean(),

        ml_physics_verification[
            "dry_margin_prediction_error"
        ]
        .abs()
        .mean(),
    ],

    "maximum_absolute_error": [

        ml_physics_verification[
            "hybrid_prediction_error_pct_point"
        ]
        .abs()
        .max(),

        ml_physics_verification[
            "fail_safe_prediction_error_pct_point"
        ]
        .abs()
        .max(),

        ml_physics_verification[
            "dry_margin_prediction_error"
        ]
        .abs()
        .max(),
    ],
})


print(
    "\nSHORTLIST SURROGATE ERROR"
)


display(
    error_summary
)


# ============================================================
# 14. PHYSICS VS ML PLOT
# ============================================================

plt.figure(
    figsize=(7, 6)
)


plt.scatter(
    ml_physics_verification[
        "ML_hybrid_improvement_pct"
    ],

    ml_physics_verification[
        "physics_hybrid_improvement_pct"
    ],

    s=90,
)


for _, row in (
    ml_physics_verification.iterrows()
):


    plt.annotate(
        f"C{int(row['candidate_id'])}",

        (
            row[
                "ML_hybrid_improvement_pct"
            ],

            row[
                "physics_hybrid_improvement_pct"
            ],
        ),

        xytext=(5, 5),

        textcoords=
            "offset points",
    )


plot_min = min(
    ml_physics_verification[
        "ML_hybrid_improvement_pct"
    ]
    .min(),

    ml_physics_verification[
        "physics_hybrid_improvement_pct"
    ]
    .min(),
)


plot_max = max(
    ml_physics_verification[
        "ML_hybrid_improvement_pct"
    ]
    .max(),

    ml_physics_verification[
        "physics_hybrid_improvement_pct"
    ]
    .max(),
)


plt.plot(
    [
        plot_min,
        plot_max,
    ],

    [
        plot_min,
        plot_max,
    ],

    "--",
    linewidth=1.0,
)


plt.axhline(
    PHYSICS_REQ_HYBRID,
    linestyle=":",
    linewidth=1.0,

    label="+6% requirement",
)


plt.xlabel(
    "ML-predicted flutter improvement (%)"
)


plt.ylabel(
    "Physics-verified flutter improvement (%)"
)


plt.title(
    "ML candidate predictions versus physics re-verification"
)


plt.grid(
    alpha=0.25
)


plt.legend()


plt.tight_layout()

plt.show()


# ============================================================
# 15. VERIFIED MASS / ACTIVE TRADE
# ============================================================

if len(
    verified_ml_designs
) > 0:


    plt.figure(
        figsize=(8, 6)
    )


    plt.scatter(
        verified_ml_designs[
            "mass_fraction_pct"
        ],

        verified_ml_designs[
            "active_authority_pct"
        ],

        s=100,
    )


    for _, row in (
        verified_ml_designs.iterrows()
    ):


        plt.annotate(
            f"C{int(row['candidate_id'])}",

            (
                row[
                    "mass_fraction_pct"
                ],

                row[
                    "active_authority_pct"
                ],
            ),

            xytext=(5, 5),

            textcoords=
                "offset points",
        )


    plt.scatter(
        [
            2.0
        ],

        [
            50.0
        ],

        marker="X",
        s=150,

        label=
            "Deterministic reference",
    )


    plt.xlabel(
        "Notional added mass (% of panel mass)"
    )


    plt.ylabel(
        "Active-control authority (%)"
    )


    plt.title(
        "Physics-verified ML design trade"
    )


    plt.grid(
        alpha=0.25
    )


    plt.legend()


    plt.tight_layout()

    plt.show()


# ============================================================
# 16. REQUIREMENT-ORIENTED FINAL ML CANDIDATE
#
# Do not simply maximize performance.
#
# Among verified candidates:
#
#   first minimize active authority,
#   then minimize mass,
#   then maximize fail-safe margin.
#
# ============================================================

if len(
    verified_ml_designs
) > 0:


    final_ranked = (
        verified_ml_designs
        .sort_values(
            [
                "active_authority_pct",
                "mass_fraction_pct",
                "physics_fail_safe_improvement_pct",
            ],

            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    ML_FINAL_REFERENCE = (
        final_ranked
        .iloc[0]
    )


    print(
        "\nPROVISIONAL ML-DERIVED PHYSICS-VERIFIED REFERENCE"
    )


    print(
        f"Candidate = "
        f"{int(ML_FINAL_REFERENCE['candidate_id'])}"
    )


    print(
        f"Selection origin = "
        f"{ML_FINAL_REFERENCE['selection_reason']}"
    )


    print(
        f"Span region = "
        f"{100*ML_FINAL_REFERENCE['eta_start']:.1f}% "
        f"to "
        f"{100*ML_FINAL_REFERENCE['eta_end']:.1f}%"
    )


    print(
        f"GJ gain = "
        f"{ML_FINAL_REFERENCE['GJ_gain_pct']:.3f}%"
    )


    print(
        f"Notional added mass = "
        f"{ML_FINAL_REFERENCE['mass_fraction_pct']:.3f}%"
    )


    print(
        f"Active authority = "
        f"{ML_FINAL_REFERENCE['active_authority_pct']:.3f}%"
    )


    print(
        f"Physics hybrid improvement = "
        f"{ML_FINAL_REFERENCE['physics_hybrid_improvement_pct']:+.4f}%"
    )


    print(
        f"Physics fail-safe improvement = "
        f"{ML_FINAL_REFERENCE['physics_fail_safe_improvement_pct']:+.4f}%"
    )


    print(
        f"Dry stability margin = "
        f"{ML_FINAL_REFERENCE['physics_minimum_dry_margin']:.5f} 1/s"
    )


else:


    ML_FINAL_REFERENCE = None


    print(
        "\nNo ML-derived candidate survived "
        "full physics verification."
    )


print(
    "\nCell 63 ML candidate physics "
    "re-verification completed."
)